In [ ]:
# ── Repository folders (RELATIVE paths: this notebook is in Code/) ──
from pathlib import Path
REPO_DIR    = Path.cwd().parent if Path.cwd().name == "Code" else Path.cwd()
DATA_DIR    = REPO_DIR / "Data"       # feature CSVs, tracking, per-individual Excel
RESULTS_DIR = REPO_DIR / "Results"    # saved sessions (e.g. reference_best) + outputs
VIDEO_ROOT  = REPO_DIR / "Videos"     # videos NOT included in the repository: put them here
_LOCAL_VIDEOS = Path("E:/01_Travail/CREC/First_Capture/VIDEO/capture")   # local copy (author's PC)
if not VIDEO_ROOT.is_dir() and _LOCAL_VIDEOS.is_dir():   # no Videos/ -> local videos
    VIDEO_ROOT = _LOCAL_VIDEOS

import json
import webbrowser
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches
from pathlib import Path
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import kneighbors_graph
import igraph as ig
import leidenalg
import umap
import pickle

%matplotlib inline
#%matplotlib qt

# ══ DATASET CHOICE ═══════════════════════════════════════════
DATASET = "cut_new"   # "cut_old" = old cut file  + old features (5)
                      # "cut_new" = new cut file  + new features (9)
                      #             (Frame_start -> strike, ~95 k frames)
                      # "full"    = full file     + new features (9)
                      #             (Start -> strike, ~1.2 M frames)

# ── CSV files
SMOOTH_FILES_CUT_OLD = {     # add_features_cut_2.py (old computation)
    1:  DATA_DIR / "cut_cuttlefish_with_features_smooth_1.csv",
   # 5:  DATA_DIR / "cut_cuttlefish_with_features_smooth_5.csv",
    #10: DATA_DIR / "cut_cuttlefish_with_features_smooth_10.csv",
    #20: DATA_DIR / "cut_cuttlefish_with_features_smooth_20.csv",
    #50: DATA_DIR / "cut_cuttlefish_with_features_smooth_50.csv",
}
SMOOTH_FILES_CUT_NEW = {     # add_features_full.py, SOURCE = "cut"
    1:  DATA_DIR / "cut_cuttlefish_with_new_features_smooth_1.csv",
   # 5:  DATA_DIR / "cut_cuttlefish_with_new_features_smooth_5.csv",
    #10: DATA_DIR / "cut_cuttlefish_with_new_features_smooth_10.csv",
    #20: DATA_DIR / "cut_cuttlefish_with_new_features_smooth_20.csv",
    #50: DATA_DIR / "cut_cuttlefish_with_new_features_smooth_50.csv",
}
SMOOTH_FILES_FULL = {        # add_features_full.py, SOURCE = "full"
    1:  DATA_DIR / "full_cuttlefish_with_features_smooth_1.csv",
}

# ── Features
FEATURES_OLD = ["distance_to_shrimp", "approach_angle", "speed_cuttle_smooth",
                "orientation_variation", "jerk_cuttle"]

# FEATURES_OLD = [
#     "distance_to_shrimp", "approach_angle", "orientation_variation",
#     "speed_cuttle_smooth", "accel_cuttle" ,"jerk_cuttle", "head_turn", "eye_convergence"
# ]

# New variables (add_features_full.py) — all except distance_change
FEATURES_NEW = [
    # 6 variables, each = one behavioural axis
    "distance_to_shrimp",        # near or far from the prey (log)
    "attack_side",               # where the cuttlefish is around the prey: 0° = in front of its head, 180° = behind
    "bearing_to_prey",           # is the cuttlefish aiming at the prey? 0° = straight ahead
    "cuttle_approach_speed",     # cuttlefish moves towards the prey (+) / backs off (-) / still (0)
    "orbit_speed",               # cuttlefish turns around the prey (°/s) -> circling
    "who_moves",                 # who moves: + = cuttlefish, - = shrimp, 0 = both / neither
    # Available in the file but removed from the clustering:
    # "speed_cuttle_log",        # redundant: approach + orbit already decompose the movement
    # "orientation_variation",   # correlated with speed (r=0.79), rotation covered by orbit_speed
    # "approach_angle",          # relative body orientation, replaced by attack_side
    # "approach_angle_variation",# half driven by the shrimp, replaced by orbit_speed
    # "speed_shrimp_log",        # prey behaviour (context, to describe afterwards)
    # "shrimp_retreat_speed",    # same
    # "distance_change",         # mixes cuttlefish + shrimp (r=0.87 with shrimp_retreat_speed)
]

# ── Feature weights (1.0 = no weighting) ────────────────────
#   Applied AFTER the StandardScaler: a weight W gives the feature W times
#   more importance in the distance between frames.
#   A feature missing from the dictionary has a weight of 1.0.
FEATURE_WEIGHTS_OLD = {      # for "cut_old" (current weights taken from cell 1)
    "orientation_variation":          8.0,
    "distance_to_shrimp":             1.0,
    "approach_angle":                 1.0,
    "speed_cuttle_smooth":            1.0,
    "jerk_cuttle":                    1.0
}

FEATURE_WEIGHTS_NEW = {      # for "cut_new" and "full"
    "distance_to_shrimp":        1.0,
    "attack_side":               3.0,
    "bearing_to_prey":           1.0,
    "cuttle_approach_speed":     2.0,   # movement: higher weight so that Leiden
    "orbit_speed":               5.0,   #   splits states by MOVEMENT rather than
    "who_moves":                 4.0,   #   by position alone
}

# ── Excluded individuals (dubious data) ──────────────────────
EXCLUDE_ANIMALS = [
    152,    # anomalous cut file (4314 frames instead of ~251) -> creates its own state
    133,    # missing from the Excel (no reliable strike frame) -> creates 2 states on its own
]

# ── Orbit threshold ───────────────────────────────────────────
#   orbit_speed is recomputed at loading (cell 2) from orbit_speed_deg_s:
#   below ORBIT_MIN_DEG_S (°/s) = slight curve, not circling -> 0, then log.
#   Without a threshold, log1p makes 3-6 °/s almost as strong as 15-50 °/s.
#   Can be changed here without rerunning add_features_full.py (0 = no threshold).
ORBIT_MIN_DEG_S = 0.0   # tested: 8 degraded the groups (single orbit state); 0 or 3 advised

# ── Leiden parameters
LEIDEN_K    = 300
LEIDEN_RES  = 1.0
LEIDEN_K2   = 3
LEIDEN_RES2 = 5.0

# ── Settings depending on DATASET ───────────────────────────
#   FRAME_STEP: Leiden + UMAP on 1 frame out of FRAME_STEP (the others: by neighbours)
FRAME_STEP = 1
VIDEO_DIR  = VIDEO_ROOT / "videos_catch"   # videoN_cut.mp4

if DATASET == "cut_old":
    SMOOTH_FILES = SMOOTH_FILES_CUT_OLD
    FEATURES     = list(FEATURES_OLD)
    OUT_DIR      = RESULTS_DIR / "analysis_results"
elif DATASET == "cut_new":
    SMOOTH_FILES = SMOOTH_FILES_CUT_NEW
    FEATURES     = list(FEATURES_NEW)
    OUT_DIR      = RESULTS_DIR / "analysis_results_cut_new"
    LEIDEN_K     = 300   # K=300 / res=0.5: more homogeneous clusters than with K=1000
    LEIDEN_RES   = 1.0   #   res 1.0: states that separate the orbit better (user choice)
elif DATASET == "full":
    SMOOTH_FILES = SMOOTH_FILES_FULL
    FEATURES     = list(FEATURES_NEW)
    OUT_DIR      = RESULTS_DIR / "analysis_results_full"
    VIDEO_DIR    = VIDEO_ROOT / "01_VIDEO_Cropped"   # videoN.mp4
    FRAME_STEP   = 3     # ~400 k frames for Leiden instead of 1.2 M
    LEIDEN_K     = 200   # 1000 neighbours × 400 k frames does not fit in memory
    LEIDEN_RES   = 0.3   # K=200 / res=0.3 -> ~17 clusters (comparable to the cut)
                         # if MemoryError: LEIDEN_K = 100 and LEIDEN_RES = 0.1
else:
    raise ValueError(f"DATASET = {DATASET!r} unknown — 'cut_old', 'cut_new' or 'full'")

USE_FULL = DATASET == "full"          # full file?
USE_NEW  = DATASET != "cut_old"       # new features?
FEATURE_WEIGHTS = dict(FEATURE_WEIGHTS_NEW if USE_NEW else FEATURE_WEIGHTS_OLD)
_unknown = [f for f in FEATURE_WEIGHTS if f not in FEATURES]
if _unknown:
    print(f"[WARN] weights defined for unused features (ignored): {_unknown}")

OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Palettes
PALETTE = [
    "#FF4FB8","#C97E48","#8C6BB1","#9C9C9C","#F5E21E",
    "#3A8891","#E45756","#4CAF50","#1F77B4","#FF9F1C",
    "#A05195","#2CB1BC","#7F7F7F","#BCBD22","#17BECF",
    "#D62728","#9467BD","#8C564B","#E377C2","#AEC7E8",
]
SUPER_PAL = [
    "#FF4444","#44AAFF","#44FF88","#FFD700","#FF44FF",
    "#FF8844","#44FFFF","#FF44AA","#88FF44","#AA44FF",
]
# ── Semantic aliases ─────────────────────────────────
CLUSTER_PALETTE  = PALETTE    # behavioural states C0, C1, ...
STRATEGY_PALETTE = SUPER_PAL  # strategies A, B, C, ... (and Ward groups)

# Fixed colormaps — single source of truth
CMAP_CLUSTER = mpl.colors.ListedColormap(PALETTE)
NORM_CLUSTER = mpl.colors.BoundaryNorm(range(len(PALETTE) + 1), ncolors=len(PALETTE))

print(f"[OK] Config — DATASET={DATASET} | {len(FEATURES)} features | "
      f"weights ≠ 1: { {f: w for f, w in FEATURE_WEIGHTS.items() if w != 1.0} or 'none'} | {len(SMOOTH_FILES)} file(s) | "
      f"1 frame / {FRAME_STEP} for Leiden | results → {OUT_DIR}")

In [ ]:
def run_leiden(df, features, k=100, res=1.0, k2=3, res2=5.0, seed=42,
               feature_weights=None):
    """
    Leiden level 1 (clusters) + level 2 (superclusters).

    feature_weights : optional dict {feature_name: weight}
        The weight is applied AFTER StandardScaler so that it is actually
        effective — multiplying by W gives W times more importance in
        the Euclidean distance between frames.
        E.g. : {"orientation_variation": 3.0} → this feature counts
             3× more than the others in the clustering.
    """

    # ── Level 1 — clusters on the frames ───────────────────
    X_raw = df[features].values.copy().astype(float)

    # StandardScaler FIRST — all features at std=1
    X = StandardScaler().fit_transform(X_raw)

    # Weights AFTER StandardScaler
    # Before scaling: different stds → multiplying has no uniform effect
    # After scaling: std=1 for all → multiplying by W gives exactly
    #                W times more importance in the Euclidean distance
    if feature_weights:
        for i, feat in enumerate(features):
            w = feature_weights.get(feat, 1.0)
            if w != 1.0:
                X[:, i] *= w
                print(f"  [WEIGHT] {feat} × {w}  (std : 1.0 → {w:.1f})")

    A         = kneighbors_graph(X, n_neighbors=k, mode='distance')
    A         = A.tocoo()
    keep      = A.data != 0                         # comme A.nonzero()
    s, t      = A.row[keep], A.col[keep]
    dist_vals = A.data[keep]
    w_vals    = 1 - dist_vals / (dist_vals.max() + 1e-9)

    # tableaux numpy (pas de liste Python de tuples) : même graphe, bien moins de mémoire
    g = ig.Graph(n=len(X), edges=np.column_stack([s, t]), directed=False)
    g.es["weight"] = w_vals

    p = leidenalg.find_partition(
        g, leidenalg.RBConfigurationVertexPartition,
        weights="weight",
        resolution_parameter=res, seed=seed
    )
    df            = df.copy()
    df["cluster"] = p.membership
    n_clusters    = df["cluster"].nunique()

    # ── Level 2 — superclusters on the centroids ───────────
    mean_c  = df.groupby("cluster")[features].mean()
    mean_cs = StandardScaler().fit_transform(mean_c)

    A2      = kneighbors_graph(mean_cs,
                                n_neighbors=min(k2, len(mean_c) - 1),
                                mode='distance')
    s2, t2  = A2.nonzero()
    dist2   = np.array(A2[s2, t2]).flatten()
    w2      = 1 - dist2 / (dist2.max() + 1e-9)

    g2 = ig.Graph(n=len(mean_c), edges=list(zip(s2, t2)), directed=False)
    g2.es["weight"] = w2.tolist()

    p2   = leidenalg.find_partition(
        g2, leidenalg.RBConfigurationVertexPartition,
        weights="weight",
        resolution_parameter=res2, seed=seed
    )
    c2sc               = {cid: sl for cid, sl in zip(mean_c.index, p2.membership)}
    df["supercluster"] = df["cluster"].map(c2sc)
    n_super            = df["supercluster"].nunique()

    print(f"  → {n_clusters} clusters | {n_super} superclusters")
    return df, X, n_clusters, n_super


def run_umap(X, seed=42):
    return umap.UMAP(n_components=2, random_state=seed).fit_transform(X)


# ── Feature weights ────────────────────────────────────────
# FEATURE_WEIGHTS is defined in CELL 0 depending on DATASET
# (FEATURE_WEIGHTS_OLD for "cut_old", FEATURE_WEIGHTS_NEW for "cut_new" / "full")

print("[OK] Functions defined")

In [ ]:
results = {}

USE_CACHE = True   # False = force a full recomputation

def _cache_params(fpath):
    """Everything that, if it changes, invalidates the cache
    (features, weights, Leiden parameters, modified source CSV)."""
    st = Path(fpath).stat()
    return {"features": list(FEATURES), "weights": dict(FEATURE_WEIGHTS),
            "leiden": [LEIDEN_K, LEIDEN_RES, LEIDEN_K2, LEIDEN_RES2]
                      + ([FRAME_STEP] if FRAME_STEP > 1 else []),
            "exclude": sorted(EXCLUDE_ANIMALS),
            "orbit_min": ORBIT_MIN_DEG_S,
            "csv": [str(fpath), st.st_size, st.st_mtime]}

def _leiden_subsampled(df_w, step=FRAME_STEP, k_prop=15):
    """
    Leiden + UMAP on 1 frame out of `step` (per individual). Every other frame
    gets the MAJORITY cluster and the MEAN UMAP position of its k_prop
    nearest neighbours in feature space -> every frame gets
    a cluster, the sequences stay complete.
    """
    from sklearn.neighbors import NearestNeighbors
    df_w = df_w.sort_values(["animal_id", "frame"]).reset_index(drop=True)
    samp = (df_w.groupby("animal_id").cumcount() % step == 0).to_numpy()
    df_s, X_s, n_cl, n_su = run_leiden(
        df_w[samp].reset_index(drop=True), FEATURES,
        k=LEIDEN_K, res=LEIDEN_RES, k2=LEIDEN_K2, res2=LEIDEN_RES2,
        feature_weights=FEATURE_WEIGHTS)
    Z_s = run_umap(X_s)

    # same transform as run_leiden (StandardScaler of the subsample + weights)
    raw_s = df_w.loc[samp, FEATURES].to_numpy(float)
    mu, sd = raw_s.mean(0), raw_s.std(0)
    sd[sd == 0] = 1.0
    wts   = np.array([FEATURE_WEIGHTS.get(f, 1.0) for f in FEATURES])
    X_all = (df_w[FEATURES].to_numpy(float) - mu) / sd * wts

    # non-sampled frames: vote of the k_prop sampled neighbours
    lab_s = df_s["cluster"].to_numpy()
    nn    = NearestNeighbors(n_neighbors=k_prop).fit(X_s)
    other = ~samp
    _, idx = nn.kneighbors(X_all[other])
    counts = np.zeros((other.sum(), n_cl))
    for j in range(k_prop):
        counts[np.arange(other.sum()), lab_s[idx[:, j]]] += 1
    cluster = np.empty(len(df_w), dtype=int)
    cluster[samp], cluster[other] = lab_s, counts.argmax(1)
    Z = np.empty((len(df_w), 2))
    Z[samp], Z[other] = Z_s, Z_s[idx].mean(1)

    c2sc = df_s.groupby("cluster")["supercluster"].first()
    df_w["cluster"]        = cluster
    df_w["supercluster"]   = df_w["cluster"].map(c2sc)
    df_w["leiden_sampled"] = samp

    # check: on sampled frames, does the neighbours' vote (without
    # themselves) give back their true Leiden cluster?
    rng = np.random.default_rng(0)
    h   = rng.choice(len(X_s), min(5000, len(X_s)), replace=False)
    _, ih = nn.kneighbors(X_s[h], n_neighbors=k_prop + 1)
    vote = np.array([np.bincount(lab_s[r[1:]], minlength=n_cl).argmax() for r in ih])
    print(f"  [SUBSAMPLE] Leiden/UMAP on {samp.sum()} frames (1/{step}) | "
          f"{other.sum()} frames assigned by neighbours | vote consistency {np.mean(vote == lab_s[h]):.0%}")
    return df_w, X_all, Z, n_cl, n_su


for window, fpath in SMOOTH_FILES.items():
    print(f"\n── Smooth window={window} ──")
    cache  = OUT_DIR / f"smooth_{window}_results.pkl"
    params = _cache_params(fpath)

    # ── Try the cache ──
    if USE_CACHE and cache.exists():
        with open(cache, "rb") as f:
            res_w = pickle.load(f)
        if res_w.get("params") == params:
            results[window] = res_w
            print(f"  [CACHE] {cache.name} reloaded — "
                  f"{res_w['n_clusters']} clusters | {res_w['n_superclusters']} superclusters")
            continue
        print(f"  [CACHE] different parameters or old cache -> recomputing")

    df_w = pd.read_csv(fpath)
    df_w = df_w[~df_w["animal_id"].isin(EXCLUDE_ANIMALS)].reset_index(drop=True)
    # orbit: ORBIT_MIN_DEG_S threshold (cell 0) then log
    if "orbit_speed_deg_s" in df_w.columns and "orbit_speed" in FEATURES:
        df_w["orbit_speed"] = np.log1p(np.clip(df_w["orbit_speed_deg_s"] - ORBIT_MIN_DEG_S, 0, None))
    print(f"  {len(df_w)} frames | {df_w['animal_id'].nunique()} individuals")

    if FRAME_STEP > 1:
        df_w, X_w, Z_w, n_cl, n_su = _leiden_subsampled(df_w)
    else:
        df_w, X_w, n_cl, n_su = run_leiden(
            df_w, FEATURES,
            k=LEIDEN_K, res=LEIDEN_RES,
            k2=LEIDEN_K2, res2=LEIDEN_RES2,
            feature_weights=FEATURE_WEIGHTS    # ← added
        )
        Z_w = run_umap(X_w)

    res_w = {"df": df_w, "X": X_w, "Z": Z_w,
             "n_clusters": n_cl, "n_superclusters": n_su, "window": window,
             "features": list(FEATURES), "params": params}
    results[window] = res_w

    df_w.to_csv(OUT_DIR / f"smooth_{window}_clustered.csv", index=False)
    with open(cache, "wb") as f:
        pickle.dump(res_w, f)
    print(f"  [OK] → smooth_{window}_clustered.csv")

print("\n[OK] All files processed")

In [ ]:
# ===============================================================
#  CELL — Manual clusters by GRAPH PROPAGATION (Leiden-style)
#  You place seeds, and the points attach to them by following the
#  5D kNN links (geodesic distance), not in a straight line.
#
#  Prerequisite: results[window] with df, Z, X (scaled+weighted), features
# ===============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.widgets import Slider, Button, RadioButtons
from sklearn.neighbors import kneighbors_graph
from scipy.sparse.csgraph import dijkstra

MANUAL_PALETTE = ["#e6194B", "#3cb44b", "#4363d8", "#f58231", "#911eb4",
                  "#42d4f4", "#f032e6", "#bfef45", "#469990", "#9A6324"]


def open_graph_clustering(results, window, k_graph=15):
    """
    k_graph : number of neighbours of the propagation graph (like LEIDEN_K,
              but smaller for fine control).

    Workflow :
      1. left click on the UMAP -> places a seed of the current cluster
      2. "Propagate"            -> each point joins the closest seed
                                   BY FOLLOWING THE LINKS
      3. "New cluster"          -> next colour, new seeds
      4. right click on a point -> forces it into the current cluster
      5. "Export"               -> graph_cluster column
    """
    R = results[window]
    df    = R["df"].reset_index(drop=True)
    Z     = np.asarray(R["Z"])
    Xproc = np.asarray(R["X"])              # 5D scaled + weighted
    feats = R["features"]

    # ── 5D kNN graph, built ONCE (geodesic distances afterwards) ──
    print(f"[GRAPH] building the {k_graph}-NN graph on {len(df)} points…")
    G = kneighbors_graph(Xproc, k_graph, mode="distance")
    G = G.maximum(G.T)                      # symmetrise: reciprocal link
    print("[GRAPH] ready.")

    state = {"seeds": {0: []}, "labels": np.full(len(df), -1, int),
             "cur": 0, "widgets": {}}

    fig = plt.figure(figsize=(13, 8.5))
    ax  = fig.add_axes([0.05, 0.08, 0.60, 0.86])
    ax.set_title(f"UMAP smooth={window} — L click = seed · R click = force",
                 fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

    ax.scatter(Z[:, 0], Z[:, 1], s=5, c="#dddddd", zorder=1)
    lab_sc  = ax.scatter(Z[:, 0], Z[:, 1], s=7, zorder=2,
                         facecolors=[(0, 0, 0, 0)] * len(df))
    seed_sc = ax.scatter([], [], s=180, marker="*", zorder=6,
                         facecolors="none", edgecolors="black", linewidths=1.8)
    info = fig.text(0.68, 0.92, "", fontsize=8, va="top", family="monospace")

    ax_k = fig.add_axes([0.70, 0.30, 0.25, 0.03])
    s_k  = Slider(ax_k, "max distance", 0.0, 1.0, valinit=1.0)   # 1.0 = unlimited

    def _mk(name, y, label, color=None):
        axb = fig.add_axes([0.70, y, 0.25, 0.05])
        bt = Button(axb, label, color=color) if color else Button(axb, label)
        state["widgets"][name] = bt
        return bt

    b_prop = _mk("prop", 0.60, "Propagate",        "#dfe")
    b_new  = _mk("new",  0.53, "New cluster")
    b_undo = _mk("undo", 0.46, "Clear current cluster")
    b_save = _mk("save", 0.39, "Export CSV",       "#fed")
    state["widgets"]["s_k"] = s_k

    def _col(cid):
        return MANUAL_PALETTE[cid % len(MANUAL_PALETTE)]

    def _redraw():
        rgba = [(0, 0, 0, 0)] * len(df)
        for i, l in enumerate(state["labels"]):
            if l >= 0:
                rgba[i] = mcolors.to_rgba(_col(l), 0.85)
        lab_sc.set_facecolors(rgba)
        alls = [s for lst in state["seeds"].values() for s in lst]
        seed_sc.set_offsets(Z[alls] if alls else np.empty((0, 2)))
        if alls:
            seed_sc.set_edgecolors([_col(c) for c, lst in state["seeds"].items()
                                    for _ in lst])
        fig.canvas.draw_idle()

    def _nearest(xy):
        return int(np.argmin(np.linalg.norm(Z - xy, axis=1)))

    def on_click(event):
        if event.inaxes is not ax or event.xdata is None:
            return
        j = _nearest([event.xdata, event.ydata])
        if event.button == 1:                      # seed
            state["seeds"].setdefault(state["cur"], []).append(j)
            info.set_text(f"Seed added (cluster {state['cur']}) : #{j}\n"
                          f"total seeds in cluster : "
                          f"{len(state['seeds'][state['cur']])}")
        elif event.button == 3:                    # force a point
            state["labels"][j] = state["cur"]
            info.set_text(f"Point #{j} forced -> cluster {state['cur']}")
        _redraw()

    def on_propagate(event):
        seeds_by_c = {c: s for c, s in state["seeds"].items() if s}
        if not seeds_by_c:
            info.set_text("Place at least one seed (left click) first.")
            return
        # geodesic distance from ALL the seeds, per cluster
        cids = sorted(seeds_by_c)
        dmat = np.full((len(cids), len(df)), np.inf)
        for r, c in enumerate(cids):
            d = dijkstra(G, indices=seeds_by_c[c], min_only=True)
            dmat[r] = d
        best = dmat.argmin(0)
        dmin = dmat.min(0)

        # threshold: do not attach beyond a given geodesic distance
        if s_k.val < 1.0:
            finite = dmin[np.isfinite(dmin)]
            dist_threshold = np.quantile(finite, s_k.val) if len(finite) else np.inf
        else:
            dist_threshold = np.inf

        lab = np.full(len(df), -1, int)
        ok = np.isfinite(dmin) & (dmin <= dist_threshold)
        lab[ok] = np.array(cids)[best[ok]]
        # keep the manually forced points
        forced = state["labels"] >= 0
        lab[forced] = state["labels"][forced]
        state["labels"] = lab
        _redraw()

        n = [(c, int((lab == c).sum())) for c in cids]
        txt = "Propagation (geodesic) :\n"
        txt += "\n".join(f"  cluster {c} : {v} points" for c, v in n)
        txt += f"\n  unassigned : {int((lab < 0).sum())}"
        info.set_text(txt)

    def on_new(event):
        if state["seeds"].get(state["cur"]):
            state["cur"] += 1
            state["seeds"][state["cur"]] = []
            info.set_text(f"New cluster {state['cur']}  ({_col(state['cur'])})\n"
                          f"place its seeds with a left click.")

    def on_undo(event):
        state["labels"][state["labels"] == state["cur"]] = -1
        state["seeds"][state["cur"]] = []
        _redraw()
        info.set_text(f"Cluster {state['cur']} cleared (points + seeds).")

    def on_save(event):
        out = df.copy()
        out["graph_cluster"] = state["labels"]
        path = f"umap_graph_clusters_smooth{window}.csv"
        out.to_csv(path, index=False)
        info.set_text(f"Exported : {path}\n"
                      f"{int((state['labels'] >= 0).sum())} points classified, "
                      f"{state['cur'] + 1} clusters.")

    fig.canvas.mpl_connect("button_press_event", on_click)
    b_prop.on_clicked(on_propagate)
    b_new.on_clicked(on_new)
    b_undo.on_clicked(on_undo)
    b_save.on_clicked(on_save)

    _redraw()
    info.set_text("1. left click = seed\n2. Propagate\n"
                  "3. New cluster, start again\n"
                  "right click = force a point")
    plt.show()
    return state


# vw = open_graph_clustering(results, window=10, k_graph=15)  # keep the variable!

In [ ]:
# smoothing comparison plots

n_smooth = len(results)
fig, axes = plt.subplots(2, n_smooth, figsize=(5 * n_smooth, 10))
if n_smooth == 1:
    axes = axes.reshape(2, 1)

for col, (window, res_w) in enumerate(sorted(results.items())):
    Z_w  = res_w["Z"]
    df_w = res_w["df"]
    n_cl = res_w["n_clusters"]
    n_su = res_w["n_superclusters"]

    norm_su = mpl.colors.BoundaryNorm(range(n_su + 1), ncolors=n_su)

    axes[0, col].scatter(Z_w[:, 0], Z_w[:, 1], c=df_w["cluster"],
                         cmap=CMAP_CLUSTER, norm=NORM_CLUSTER, s=2, alpha=0.5)
    axes[0, col].set_title(f"Smooth={window}\n{n_cl} clusters", fontsize=10)
    axes[0, col].axis("off")

    axes[1, col].scatter(Z_w[:, 0], Z_w[:, 1], c=df_w["supercluster"],
                         cmap="tab10", norm=norm_su, s=2, alpha=0.5)
    axes[1, col].set_title(f"{n_su} superclusters", fontsize=10)
    axes[1, col].axis("off")

fig.suptitle("Smoothing comparison — Clusters (top) / Superclusters (bottom)", fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / "comparison_smoothings.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
SELECTED_WINDOW = 1

In [ ]:
# ── Exploratory UMAP — change FEATURES_EXPLORE to test ──
# Uses the same pipeline as run_leiden (StandardScaler + weights)
# so that the UMAPs are comparable

SMOOTH_EXPLORE   = 1
FEATURES_EXPLORE = [          # ← change here to explore
    "distance_to_shrimp",
    "approach_angle",
    "orientation_variation",
    "speed_cuttle_smooth",
    #"accel_cuttle",
    "jerk_cuttle",           # ← comment/uncomment to test
    # "head_turn",
    # "eye_convergence",
]
WEIGHTS_EXPLORE = {            # ← same logic as FEATURE_WEIGHTS
    "orientation_variation": 8.0,
}
if USE_NEW:      # the old features do not exist in the new files
    FEATURES_EXPLORE = list(FEATURES)

# ── Computation ────────────────────────────────────────────────────
df_exp = results[SMOOTH_EXPLORE]["df"].copy()

X_raw = df_exp[FEATURES_EXPLORE].values.copy().astype(float)
X_exp = StandardScaler().fit_transform(X_raw)

# Weights after StandardScaler — same logic as run_leiden
for i, feat in enumerate(FEATURES_EXPLORE):
    w = WEIGHTS_EXPLORE.get(feat, 1.0)
    if w != 1.0:
        X_exp[:, i] *= w
        print(f"  [WEIGHT] {feat} × {w}")

print(f"[OK] X computed : {X_exp.shape} | computing UMAP...")
Z_exp = run_umap(X_exp)
print(f"[OK] UMAP computed")

# ── Plot ──────────────────────────────────────────────────────
n_feats = len(FEATURES_EXPLORE)
fig, axes = plt.subplots(1, n_feats, figsize=(4 * n_feats, 4))
if n_feats == 1: axes = [axes]

for ax, feat in zip(axes, FEATURES_EXPLORE):
    vals = df_exp[feat].values
    sc   = ax.scatter(Z_exp[:, 0], Z_exp[:, 1],
                      c=vals, cmap="viridis",
                      s=1, alpha=0.4,
                      vmin=np.percentile(vals, 2),
                      vmax=np.percentile(vals, 98))
    plt.colorbar(sc, ax=ax, fraction=0.04, pad=0.02)
    ax.set_title(feat.replace("_", "\n"), fontsize=8)
    ax.axis("off")

fig.suptitle(
    f"Exploratory UMAP — smooth={SMOOTH_EXPLORE} | "
    f"{len(FEATURES_EXPLORE)} features | weights={WEIGHTS_EXPLORE}",
    fontsize=9)
plt.tight_layout()
plt.show()

# ── Check: if FEATURES_EXPLORE == FEATURES
#    and WEIGHTS_EXPLORE == FEATURE_WEIGHTS
#    → UMAP identical to the pipeline one
print(f"\nTo reproduce the pipeline UMAP :")
print(f"  FEATURES_EXPLORE = FEATURES = {FEATURES}")
print(f"  WEIGHTS_EXPLORE  = FEATURE_WEIGHTS = {FEATURE_WEIGHTS}")

In [ ]:
# ===============================================================
#  CELL 0 — COLOUR REGISTRY (single source of truth)
#  Run BEFORE all plots.
# ===============================================================
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# ===============================================================
#  CELL A — Extraction of the results for the selected window
#  ⚠ MUST run BEFORE CELL 0 (which needs n_clusters)
# ===============================================================
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

df              = results[SELECTED_WINDOW]["df"].copy()
X_scaled        = results[SELECTED_WINDOW]["X"]
Z               = results[SELECTED_WINDOW]["Z"]
n_clusters      = results[SELECTED_WINDOW]["n_clusters"]
n_superclusters = results[SELECTED_WINDOW]["n_superclusters"]

print(f"[DATA] smooth={SELECTED_WINDOW} | {len(df)} rows | "
      f"{n_clusters} clusters | {n_superclusters} superclusters | "
      f"Z {Z.shape}")

# Check: labels must cover 0..n-1 without gaps, otherwise the
# colours (indexed by label) and the counters get shifted.
for col, n in [("cluster", n_clusters), ("supercluster", n_superclusters)]:
    if col in df.columns:
        seen = sorted(pd.to_numeric(df[col], errors="coerce").dropna().unique().astype(int))
        missing_labels = sorted(set(range(n)) - set(seen))
        extra  = sorted(set(seen) - set(range(n)))
        status_str = "OK" if not (missing_labels or extra) else f"missing={missing_labels} out_of_range={extra}"
        print(f"  {col:<13} : {len(seen)} labels present — {status_str}")

def build_color_registry(n_clusters, n_superclusters):
    """
    Builds ONE colour table, then derives everything else from it
    (cmap, norm, hex list). The two cannot diverge:
    the cmap is literally built from the list.
    """
    # 20 well-separated colours: the 10 of tab10 first
    # (the most distinct), then the 10 light variants of tab20.
    tab10 = [mcolors.to_hex(plt.get_cmap("tab10")(i)) for i in range(10)]
    tab20 = plt.get_cmap("tab20")
    light_colors = [mcolors.to_hex(tab20(i)) for i in range(1, 20, 2)]
    BASE = tab10 + light_colors                      # 20 colours, optimal order

    def _color_list(n):
        return [BASE[i % len(BASE)] for i in range(max(1, int(n)))]

    def _cmap_norm(colors):
        cmap = mcolors.ListedColormap(colors)
        # bounds at -0.5, 0.5, 1.5… -> each integer falls at the CENTRE of its bin
        norm = mcolors.BoundaryNorm(np.arange(len(colors) + 1) - 0.5,
                                    ncolors=len(colors))
        return cmap, norm

    reg = {}
    for key, n, pfx in [("cluster", n_clusters, "C"),
                        ("supercluster", n_superclusters, "S")]:
        cols = _color_list(n)
        cmap, norm = _cmap_norm(cols)
        reg[key] = {"colors": cols, "cmap": cmap, "norm": norm, "prefix": pfx}
    return reg


COLOR_REG = build_color_registry(n_clusters, n_superclusters)

# ── Backward-compatible aliases: your old code keeps working ────────
CMAP_CLUSTER = COLOR_REG["cluster"]["cmap"]
NORM_CLUSTER = COLOR_REG["cluster"]["norm"]
CMAP_SUPER   = COLOR_REG["supercluster"]["cmap"]
NORM_SUPER   = COLOR_REG["supercluster"]["norm"]
PALETTE      = COLOR_REG["cluster"]["colors"]        # used by the viewer
SUPER_PAL    = COLOR_REG["supercluster"]["colors"]   # used by the viewer


# ── THE ONLY 4 FUNCTIONS TO USE IN PLOTS ────────────────────
def group_color(label, group_col="cluster"):
    """Hex colour of a label. Use EVERYWHERE, no exceptions."""
    d = COLOR_REG[group_col]
    return d["colors"][int(label) % len(d["colors"])]


def group_prefix(group_col="cluster"):
    return COLOR_REG[group_col]["prefix"]


def group_cmap_norm(group_col="cluster"):
    d = COLOR_REG[group_col]
    return d["cmap"], d["norm"]


def group_handles(labels, group_col="cluster", lw=3, style="line"):
    """Handles ready to pass to ax.legend(handles=...)."""
    uniq = sorted({int(x) for x in np.asarray(labels).ravel()})
    pfx  = group_prefix(group_col)
    if style == "patch":
        return [Patch(color=group_color(l, group_col), label=f"{pfx}{l}") for l in uniq]
    return [Line2D([0], [0], color=group_color(l, group_col), lw=lw, label=f"{pfx}{l}")
            for l in uniq]


def preview_colors():
    """Control swatch: look at it once to validate visually."""
    fig, axes = plt.subplots(2, 1, figsize=(12, 2.4))
    for ax, key in zip(axes, ["cluster", "supercluster"]):
        cols = COLOR_REG[key]["colors"]
        for i, c in enumerate(cols):
            ax.add_patch(mpl.patches.Rectangle((i, 0), 1, 1, color=c))
            ax.text(i + 0.5, 0.5, f"{COLOR_REG[key]['prefix']}{i}",
                    ha="center", va="center", fontsize=8,
                    color="white" if i % 20 < 10 else "black", fontweight="bold")
        ax.set_xlim(0, len(cols)); ax.set_ylim(0, 1)
        ax.set_yticks([]); ax.set_xticks([])
        ax.set_title(f"{key}  ({len(cols)} groups)", fontsize=9, loc="left")
    plt.tight_layout(); plt.show()

def build_animal_colors(animal_ids, cmap_name="hsv"):
    """
    Stable colour per individual. Sorting guarantees that an animal's
    colour does NOT depend on the order in which the IDs arrive:
    same result from one run to the next.
    """
    ids = sorted({str(a).strip() for a in animal_ids}, key=natural_key)
    cm  = plt.get_cmap(cmap_name)
    COLOR_REG["animal"] = {
        "colors": {a: mcolors.to_hex(cm(i / max(1, len(ids))))
                   for i, a in enumerate(ids)},
        "prefix": "",
    }
    return COLOR_REG["animal"]["colors"]


def animal_color(aid):
    """Colour of an individual — tolerates str/int/leading zeros."""
    return COLOR_REG.get("animal", {}).get("colors", {}) \
                    .get(str(aid).strip(), "#888888")


def text_on(bg_color):
    """
    Black or white depending on the perceived luminance of the background (ITU-R BT.601).
    Avoids unreadable white text on the light tab20 shades.
    """
    r, g, b = mcolors.to_rgb(bg_color)
    return "black" if (0.299 * r + 0.587 * g + 0.114 * b) > 0.55 else "white"


def resolve_animal_id(df, raw, id_col="animal_id"):
    """
    Input -> EXACT value present in df[id_col].
    Centralised here because used by the viewer, the trajectory AND the UMAP.
    """
    vals = df[id_col].dropna().unique()
    s    = str(raw).strip()
    for v in vals:
        if str(v).strip() == s:
            return v
    try:
        num = float(s)
        for v in vals:
            try:
                if float(str(v).strip()) == num:
                    return v
            except ValueError:
                pass
    except ValueError:
        pass
    return None


def natural_key(a):
    """
    Sorts 2 before 10 (and not the reverse), tolerates text IDs.
    The tuple avoids a TypeError on mixed types: numeric values
    come first (rank 0), the others next in alphabetical order.
    """
    s = str(a).strip()
    try:
        return (0, float(s), "")
    except ValueError:
        return (1, 0.0, s)


build_animal_colors(df["animal_id"].dropna().unique())
animal_ids = sorted(df["animal_id"].dropna().unique(), key=natural_key)
n_animals  = len(animal_ids)
animal_colors_per_point = df["animal_id"].map(animal_color).to_numpy()
print(f"[COLOURS] + {n_animals} individuals registered in the registry.")

preview_colors()
print(f"[COLOURS] {n_clusters} clusters, {n_superclusters} superclusters — "
      f"single registry installed.")

In [ ]:
SELECTED_WINDOW = 1

In [ ]:
# ===============================================================
#  CELL B — Group analysis: radars, heatmap, dendrogram
#  PREREQUISITES: CELL A then CELL 0 (colour registry).
#
#  Fixes vs previous version:
#   [A] hard-coded tab10 -> group_cmap_norm() (S1 was not the
#       same colour here and in the UMAP)
#   [B] BoundaryNorm built on len(unique_g) -> colour shift
#       if a group is empty. The registry indexes by LABEL.
#   [C] darkened radar titles: the light shades were
#       unreadable on a white background
#   [D] dendrogram and heatmap: labels coloured by group
#   [E] overlaid radar (direct comparison of all groups)
#   [F] safeguards: <2 groups, missing features, constant feature
# ===============================================================

import seaborn as sns
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import dendrogram, linkage


def _darken(color_in, f=0.62):
    """
    Darkens a colour so it can be written as text on a white background.
    The light tab20 variants (#aec7e8, #ffbb78…) are unreadable
    as is; the RGB channels are multiplied by f < 1.
    """
    r, g, b = mcolors.to_rgb(color_in)
    return mcolors.to_hex((r * f, g * f, b * f))


def _color_ticks(ax, ordered_labels, group_col, axis="x"):
    """
    Colours the axis labels according to the group they refer to.
    ordered_labels : list of texts already set on the axis, e.g. ["C3","C0",…]
    """
    pfx    = group_prefix(group_col)
    getter = ax.get_xticklabels if axis == "x" else ax.get_yticklabels
    for tick in getter():
        txt = tick.get_text().strip()
        if not txt.startswith(pfx):
            continue
        try:
            g = int(txt[len(pfx):])
        except ValueError:
            continue
        tick.set_color(_darken(group_color(g, group_col), 0.75))
        tick.set_fontweight("bold")


def plot_cluster_analysis(df, features, group_col="cluster", n_cols_radar=4,
                          save=True):
    """
    For a given grouping (cluster or supercluster):
      1. Individual radars + overlaid radar
      2. Heatmap of inter-group distances (min-max normalised profiles)
      3. Hierarchical dendrogram

    All colours come from group_color(): identical to
    plot_umap, plot_animal_trajectory and the interactive viewer.
    """
    if "group_color" not in globals():
        raise RuntimeError("Run CELL 0 first (colour registry).")
    if group_col not in df.columns:
        raise KeyError(f"Column '{group_col}' missing from df.")

    # [F] keep only the features that are actually present and numeric
    feats = [f for f in features
             if f in df.columns and pd.api.types.is_numeric_dtype(df[f])]
    if not feats:
        raise ValueError("No usable numeric feature.")
    if len(feats) < len(features):
        print(f"[WARN] ignored features : {sorted(set(features) - set(feats))}")

    grp      = pd.to_numeric(df[group_col], errors="coerce")
    dat      = df.loc[grp.notna()].copy()
    dat[group_col] = grp.dropna().astype(int)

    mean_g   = dat.groupby(group_col)[feats].mean()
    counts_g = dat[group_col].value_counts().sort_index()
    unique_g = mean_g.index.to_numpy()
    pfx      = group_prefix(group_col)

    if len(unique_g) == 0:
        print("[WARN] no group.")
        return None, None

    # ── Min-max normalisation PER FEATURE, across groups ───────
    # Goal: make the radar axes comparable with each other. A feature
    # that is constant from one group to another is set to 0.5 (centre)
    # rather than 0, which would be interpreted as "minimum value".
    g_min = mean_g.min(axis=0)
    g_max = mean_g.max(axis=0)
    value_range = (g_max - g_min)
    mean_g_norm = mean_g.copy()
    for f in feats:
        mean_g_norm[f] = (0.5 if value_range[f] <= 0
                          else (mean_g[f] - g_min[f]) / value_range[f])

    # [A][B] colours: indexed by LABEL, never by position
    cols_g = {int(c): group_color(int(c), group_col) for c in unique_g}

    angles   = np.linspace(0, 2 * np.pi, len(feats), endpoint=False)
    ang_full = np.concatenate([angles, [angles[0]]])
    axis_labels  = [f.replace("_", "\n") for f in feats]

    # ══ FIGURE 1 — individual radars ══════════════════════════════════
    n_cols = min(n_cols_radar, len(unique_g))
    n_rows = int(np.ceil(len(unique_g) / n_cols))
    fig1   = plt.figure(figsize=(n_cols * 4, n_rows * 4.2))

    for i, c in enumerate(unique_g):
        c   = int(c)
        ax  = fig1.add_subplot(n_rows, n_cols, i + 1, polar=True)
        v   = mean_g_norm.loc[c].to_numpy()
        vals = np.concatenate([v, [v[0]]])
        col  = cols_g[c]

        ax.plot(ang_full, vals, linewidth=2.2, color=col, zorder=3)
        ax.fill(ang_full, vals, alpha=0.28, color=col, zorder=2)
        # Grey silhouette of the overall mean, to situate the group
        ref = mean_g_norm.mean(axis=0).to_numpy()
        ax.plot(ang_full, np.concatenate([ref, [ref[0]]]),
                lw=1, ls="--", color="#999999", alpha=0.8, zorder=1)

        ax.set_xticks(angles)
        ax.set_xticklabels(axis_labels, fontsize=7)
        ax.set_ylim(0, 1)
        ax.set_yticks([0.25, 0.5, 0.75])
        ax.set_yticklabels([], fontsize=6)
        ax.grid(color="#dddddd", lw=0.6)
        ax.set_title(f"{pfx}{c}  (n={counts_g.get(c, 0)})",
                     fontsize=11, pad=14,
                     color=_darken(col), fontweight="bold")   # [C]

    fig1.suptitle(f"Radar profiles — {group_col} (smooth={SELECTED_WINDOW})\n"
                  f"min-max per feature across groups · dashed = overall mean",
                  fontsize=12, y=1.005)
    plt.tight_layout()
    if save:
        plt.savefig(OUT_DIR / f"radar_smooth{SELECTED_WINDOW}_{group_col}.png",
                    dpi=150, bbox_inches="tight")
    plt.show()

    # ══ FIGURE 2 — [E] overlaid radar + counts ═════════════════════
    fig2 = plt.figure(figsize=(13, 5.6))
    axr  = fig2.add_subplot(1, 2, 1, polar=True)
    for c in unique_g:
        c = int(c)
        v = mean_g_norm.loc[c].to_numpy()
        axr.plot(ang_full, np.concatenate([v, [v[0]]]),
                 lw=2, color=cols_g[c], label=f"{pfx}{c}", alpha=0.9)
    axr.set_xticks(angles); axr.set_xticklabels(axis_labels, fontsize=7)
    axr.set_ylim(0, 1); axr.set_yticklabels([])
    axr.grid(color="#dddddd", lw=0.6)
    axr.set_title("Profile overlay", fontsize=11, pad=16)
    axr.legend(loc="upper right", bbox_to_anchor=(1.28, 1.10),
               fontsize=7, ncol=1 if len(unique_g) <= 10 else 2,
               frameon=True, framealpha=0.9)

    axb = fig2.add_subplot(1, 2, 2)
    labs_g = [f"{pfx}{int(c)}" for c in unique_g]
    vals_n = [counts_g.get(int(c), 0) for c in unique_g]
    axb.bar(labs_g, vals_n, color=[cols_g[int(c)] for c in unique_g],
            edgecolor="black", linewidth=0.5)
    for x, v in zip(labs_g, vals_n):
        axb.text(x, v, f"{v}", ha="center", va="bottom", fontsize=7)
    axb.set_ylabel("number of frames"); axb.set_title("Counts", fontsize=11)
    axb.tick_params(axis="x", labelrotation=45)
    _color_ticks(axb, labs_g, group_col, axis="x")           # [D]
    for sp in ("top", "right"):
        axb.spines[sp].set_visible(False)

    fig2.suptitle(f"Comparison of {group_col}s (smooth={SELECTED_WINDOW})",
                  fontsize=12)
    plt.tight_layout()
    if save:
        plt.savefig(OUT_DIR / f"overlay_smooth{SELECTED_WINDOW}_{group_col}.png",
                    dpi=150, bbox_inches="tight")
    plt.show()

    # ══ FIGURE 3 — heatmap + dendrogram ══════════════════════════════
    labels = [f"{pfx}{int(c)}" for c in unique_g]

    if len(unique_g) < 2:
        print("[INFO] fewer than 2 groups: heatmap/dendrogram cannot be computed.")
        return mean_g, mean_g_norm                              # [F]

    D_mat = squareform(pdist(mean_g_norm.to_numpy()))
    Z_lnk = linkage(mean_g_norm.to_numpy(), method="ward")

    fig3, axes = plt.subplots(1, 2, figsize=(15, 5.5))

    sns.heatmap(D_mat, cmap="viridis", ax=axes[0], square=True,
                xticklabels=labels, yticklabels=labels,
                annot=len(unique_g) <= 15, fmt=".2f",
                annot_kws={"fontsize": 6},
                cbar_kws={"label": "Euclidean distance", "shrink": 0.8})
    axes[0].set_title(f"Distance between {group_col} profiles", fontsize=11)
    _color_ticks(axes[0], labels, group_col, axis="x")        # [D]
    _color_ticks(axes[0], labels, group_col, axis="y")

    # color_threshold=0 -> disable scipy's automatic colouring
    # (which codes sub-trees, not groups) to keep ONLY our
    # colour code, carried by the leaf labels.
    dendrogram(Z_lnk, ax=axes[1], labels=labels, leaf_rotation=45,
               color_threshold=0, above_threshold_color="#999999")
    axes[1].set_title(f"Hierarchical dendrogram (Ward)", fontsize=11)
    axes[1].set_ylabel("merge distance")
    _color_ticks(axes[1], labels, group_col, axis="x")        # [D]
    for sp in ("top", "right"):
        axes[1].spines[sp].set_visible(False)

    # Reminder swatches under the leaves
    axes[1].legend(handles=[Line2D([0], [0], marker="s", color="none",
                                   markerfacecolor=cols_g[int(c)],
                                   markeredgecolor="black", markersize=7,
                                   label=f"{pfx}{int(c)}")
                            for c in unique_g],
                   fontsize=6, ncol=max(1, len(unique_g) // 8),
                   loc="upper right", framealpha=0.9, title=group_col,
                   title_fontsize=7)

    plt.suptitle(f"Structure of the {group_col}s (smooth={SELECTED_WINDOW})",
                 fontsize=12)
    plt.tight_layout()
    if save:
        plt.savefig(OUT_DIR / f"dist_dendro_smooth{SELECTED_WINDOW}_{group_col}.png",
                    dpi=150, bbox_inches="tight")
    plt.show()

    # ── Terminal summary ────────────────────────────────────────────────
    print(f"\n{group_col} — {len(unique_g)} groups, {len(feats)} features")
    closest_pair = np.unravel_index(
        np.argmin(D_mat + np.eye(len(D_mat)) * D_mat.max() * 2), D_mat.shape)
    print(f"  closest pair  : {labels[closest_pair[0]]} / "
          f"{labels[closest_pair[1]]}  (d={D_mat[closest_pair]:.3f})")
    farthest_pair = np.unravel_index(np.argmax(D_mat), D_mat.shape)
    print(f"  farthest pair : {labels[farthest_pair[0]]} / "
          f"{labels[farthest_pair[1]]}  (d={D_mat[farthest_pair]:.3f})")

    return mean_g, mean_g_norm


print("[OK] plot_cluster_analysis defined — usage :")
print("     mean_c, mean_c_n = plot_cluster_analysis(df, FEATURES, 'cluster')")
print("     mean_s, mean_s_n = plot_cluster_analysis(df, FEATURES, 'supercluster')")

In [ ]:
# plot cluster windows

import os
import seaborn as sns
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import dendrogram, linkage

# ── Call ──────────────────────────────────────────────────────
%matplotlib inline

plot_cluster_analysis(df, FEATURES, "cluster")
plot_cluster_analysis(df, FEATURES, "supercluster")

# Open full-size in the Windows image viewer
os.startfile(str(OUT_DIR / f"radar_smooth{SELECTED_WINDOW}_cluster.png"))
os.startfile(str(OUT_DIR / f"radar_smooth{SELECTED_WINDOW}_supercluster.png"))

In [ ]:

df              = results[SELECTED_WINDOW]["df"].copy()
X_scaled        = results[SELECTED_WINDOW]["X"]
Z               = results[SELECTED_WINDOW]["Z"]
n_clusters      = results[SELECTED_WINDOW]["n_clusters"]
n_superclusters = results[SELECTED_WINDOW]["n_superclusters"]

animal_ids  = sorted(df["animal_id"].unique())
n_animals   = len(animal_ids)
id_to_idx   = {aid: i for i, aid in enumerate(animal_ids)}

def animal_color(aid):
    return plt.cm.hsv(id_to_idx[aid] / n_animals)

animal_colors_per_point = [animal_color(aid) for aid in df["animal_id"]]

norm_cl = NORM_CLUSTER
norm_su = mpl.colors.BoundaryNorm(range(n_superclusters + 1), ncolors=n_superclusters)

print(f"[OK] Smoothing={SELECTED_WINDOW} | {n_clusters} clusters | {n_superclusters} superclusters")
print(f"     {n_animals} individuals | {len(df)} frames")

In [ ]:
# ===============================================================
#  CELL — UMAP  (unified colours + legends not overwritten)
#  PREREQUISITE: CELL 0 (colour registry).
#
#  Fixes vs previous version:
#   [A] supercluster: tab20 -> registry (S1 was orange in the
#       radar and light blue here)
#   [B] animal_color comes from the registry, no longer an external source
#   [C] both legends coexist (add_artist) instead of overwriting each other
#   [D] red "end" marker, no more invisible white-on-white
#   [E] highlight_animal resolved by ID (str/int) + .to_numpy() mask
#   [F] rainbow path as a LineCollection (1 artist instead of N)
#   [G] discrete colorbar with C0/C1… labels instead of a continuous scale
#   [H] labels placed on the MEDIAN (robust to crescent-shaped clouds)
# ===============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.collections import LineCollection


def _colorbar_discrete(sc, ax, n, prefix, label):
    """
    Discrete colorbar, one bin per group, with the real names.
    A continuous colorbar on integer labels is misleading:
    it suggests an order/distance between clusters that does not exist.
    """
    cb = plt.colorbar(sc, ax=ax, ticks=np.arange(n), fraction=0.046, pad=0.02)
    cb.ax.set_yticklabels([f"{prefix}{i}" for i in range(n)], fontsize=7)
    cb.set_label(label, fontsize=8)
    cb.outline.set_linewidth(0.5)
    return cb


def _annotate_groups(ax, emb, labels, n, group_col):
    """Coloured badge at the centre of each group, readable text guaranteed."""
    pfx = group_prefix(group_col)
    for c in range(n):
        m = (np.asarray(labels) == c)
        if not m.any():
            continue
        # [H] median, not mean: the centre stays INSIDE the cloud even
        #     if it is elongated, bent or bimodal.
        cx, cy = np.median(emb[m, 0]), np.median(emb[m, 1])
        bg = group_color(c, group_col)
        ax.text(cx, cy, f"{pfx}{c}",
                fontsize=8, fontweight="bold", ha="center", va="center",
                color=text_on(bg), zorder=8,
                bbox=dict(facecolor=bg, alpha=0.88, edgecolor="black",
                          linewidth=0.5, boxstyle="round,pad=0.28"))


def plot_umap(color_by="cluster", highlight_animal=None, overlay_animal=False,
              save=True, data=None, emb=None, s=3, figsize=(11, 8.5)):
    """
    color_by         : "cluster" | "supercluster" | "animal"
    highlight_animal : animal_id to highlight (rainbow path)
    overlay_animal   : True -> clusters in the background + individuals on top

    All colours come from group_color() / animal_color():
    strictly identical to plot_cluster_analysis, plot_animal_trajectory
    and the interactive viewer.
    """
    if "group_color" not in globals():
        raise RuntimeError("Run CELL 0 first (colour registry).")

    data = df if data is None else data      # globals by default
    emb  = Z  if emb  is None else emb
    emb  = np.asarray(emb)

    if len(emb) != len(data):
        raise ValueError(f"Z ({len(emb)}) and df ({len(data)}) do not have the "
                         f"same length — one of them has been filtered.")

    fig, ax = plt.subplots(figsize=figsize)
    legends = []          # [C] stack them, do not overwrite

    # ══ MODE 1 — overlay: clusters in the background + individuals on top ═════
    if overlay_animal:
        cmap_g, norm_g = group_cmap_norm("cluster")
        ax.scatter(emb[:, 0], emb[:, 1], c=data["cluster"],
                   cmap=cmap_g, norm=norm_g, s=s, alpha=0.22, zorder=1)
        for aid in animal_ids:
            m = (data["animal_id"] == aid).to_numpy()      # [E] .to_numpy()
            if m.any():
                ax.scatter(emb[m, 0], emb[m, 1], color=animal_color(aid),
                           s=s + 2, alpha=0.75, zorder=2, linewidths=0)

        legends.append(ax.legend(
            handles=[mpatches.Patch(color=animal_color(a), label=str(a))
                     for a in animal_ids],
            fontsize=6, ncol=4, loc="upper right", title="animal_id",
            title_fontsize=7, framealpha=0.9))
        _annotate_groups(ax, emb, data["cluster"].to_numpy(), n_clusters, "cluster")
        ax.set_title(f"UMAP overlay — clusters + {n_animals} individuals "
                     f"(smooth={SELECTED_WINDOW})")

    # ══ MODE 2 — coloured by cluster or supercluster ════════════════════
    elif color_by in ("cluster", "supercluster"):
        n_g = n_clusters if color_by == "cluster" else n_superclusters
        cmap_g, norm_g = group_cmap_norm(color_by)         # [A] registry

        sc = ax.scatter(emb[:, 0], emb[:, 1], c=data[color_by],
                        cmap=cmap_g, norm=norm_g, s=s, alpha=0.6, linewidths=0)

        _annotate_groups(ax, emb, data[color_by].to_numpy(), n_g, color_by)
        _colorbar_discrete(sc, ax, n_g, group_prefix(color_by), color_by)  # [G]
        ax.set_title(f"UMAP — {n_g} {color_by}s (smooth={SELECTED_WINDOW})")

    # ══ MODE 3 — coloured by individual ═══════════════════════════════════
    elif color_by == "animal":
        cols = data["animal_id"].map(animal_color).to_numpy()   # [B]
        ax.scatter(emb[:, 0], emb[:, 1], c=cols, s=s, alpha=0.6, linewidths=0)
        legends.append(ax.legend(
            handles=[mpatches.Patch(color=animal_color(a), label=str(a))
                     for a in animal_ids],
            fontsize=6, ncol=4, loc="upper right", title="animal_id",
            title_fontsize=7, framealpha=0.9))
        ax.set_title(f"UMAP — {n_animals} individuals (smooth={SELECTED_WINDOW})")

    else:
        raise ValueError(f"color_by='{color_by}' unknown — expected "
                         f"'cluster', 'supercluster' or 'animal'.")

    # ══ HIGHLIGHTING AN INDIVIDUAL ═════════════════════════════════════
    if highlight_animal is not None:
        aid_h = resolve_animal_id(data, highlight_animal)       # [E]
        if aid_h is None:
            available = sorted(str(a) for a in data["animal_id"].dropna().unique())
            print(f"[UMAP] animal {highlight_animal!r} not found — "
                  f"no highlighting. Available : {available[:15]}…")
        else:
            # .to_numpy() is essential: emb is a NumPy array indexed by
            # POSITION, whereas a pandas Series carries an index. If data was
            # filtered without reset_index, a pandas mask would
            # silently pick the wrong rows of emb.
            m_h = (data["animal_id"] == aid_h).to_numpy()
            Z_h = emb[m_h]

            if len(Z_h) > 1:
                # [F] LineCollection: 1 artist instead of N calls to plot()
                pts  = Z_h.reshape(-1, 1, 2)
                segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
                ax.add_collection(LineCollection(
                    segs, colors=plt.cm.rainbow(np.linspace(0, 1, len(segs))),
                    linewidths=1.6, alpha=0.9, zorder=5))

                ax.scatter(Z_h[0, 0], Z_h[0, 1], marker="o", s=110,
                           facecolor="#00e676", edgecolor="black",
                           linewidth=1.4, zorder=7)
                ax.scatter(Z_h[-1, 0], Z_h[-1, 1], marker="s", s=110,
                           facecolor="#ff1744", edgecolor="black",   # [D]
                           linewidth=1.4, zorder=7)
                # point at the middle of the sequence
                i_mid = len(Z_h) // 2
                ax.scatter(Z_h[i_mid, 0], Z_h[i_mid, 1], marker="D", s=95,
                           facecolor="#ffd600", edgecolor="black",
                           linewidth=1.4, zorder=7)

                # [C] pin the legends already placed before creating
                #     a new one — otherwise ax.legend() replaces them.
                for lg in legends:
                    ax.add_artist(lg)
                ax.legend(handles=[
                    Line2D([0], [0], marker="o", color="none",
                           markerfacecolor="#00e676", markeredgecolor="black",
                           markersize=9, label="start"),
                    Line2D([0], [0], marker="s", color="none",
                           markerfacecolor="#ff1744", markeredgecolor="black",
                           markersize=9, label="end"),
                    Line2D([0], [0], marker="D", color="none",
                           markerfacecolor="#ffd600", markeredgecolor="black",
                           markersize=8, label="mid-sequence"),
                    Line2D([0], [0], color="#b070d0", lw=2,
                           label="path (time →)"),
                ], fontsize=8, loc="lower left", framealpha=0.9,
                   title=f"Animal {aid_h}", title_fontsize=8)

                ax.set_title(f"{ax.get_title()}\n→ Animal {aid_h} "
                             f"({len(Z_h)} frames)")
            else:
                print(f"[UMAP] animal {aid_h!r} : {len(Z_h)} point(s), "
                      f"path cannot be drawn.")

    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_edgecolor("#cccccc")
    ax.set_xlabel("UMAP-1", fontsize=8); ax.set_ylabel("UMAP-2", fontsize=8)

    plt.tight_layout()

    if save:
        suffix = "_overlay" if overlay_animal else f"_{color_by}"
        if highlight_animal is not None:
            suffix += f"_animal{highlight_animal}"
        plt.savefig(OUT_DIR / f"umap_smooth{SELECTED_WINDOW}{suffix}.png",
                    dpi=150, bbox_inches="tight")
    plt.show()
    return fig, ax


# ═══════════════════════════════════════════════════════════════
#  COLOUR NON-REGRESSION TEST
# ═══════════════════════════════════════════════════════════════
def check_color_consistency():
    """
    Compares the colour that EACH function would assign to the same label.
    If a line shows DIVERGENT, a function is bypassing group_color().
    """
    print("=" * 58)
    print("COLOUR CONSISTENCY ACROSS PLOTS")
    print("=" * 58)
    ok_global = True
    for gc, n in [("cluster", n_clusters), ("supercluster", n_superclusters)]:
        cmap_g, norm_g = group_cmap_norm(gc)
        print(f"\n{gc} :")
        for l in range(n):
            ref      = group_color(l, gc)                       # reference
            via_cmap = mcolors.to_hex(cmap_g(norm_g(l)))        # scatter/UMAP
            status_str     = "OK" if ref == via_cmap else "DIVERGENT"
            ok_global &= (ref == via_cmap)
            print(f"  {group_prefix(gc)}{l:<3} {ref}  cmap(norm)={via_cmap}  {status_str}")
    print("\n" + ("Everything is consistent." if ok_global
                  else "⚠ Divergence detected — check CELL 0."))
    print("=" * 58)


check_color_consistency()
print("\n[OK] plot_umap defined — usage :")
print("     plot_umap('cluster')")
print("     plot_umap('supercluster', highlight_animal=12)")
print("     plot_umap(overlay_animal=True)")

In [ ]:
plot_umap(color_by="cluster")
plot_umap(color_by="supercluster")
# plot_umap(color_by="animal")
# plot_umap(overlay_animal=True)

In [ ]:
# specific animal
HIGHLIGHT_ANIMAL = resolve_animal_id(df, 42)

plot_umap(color_by="cluster",      highlight_animal=HIGHLIGHT_ANIMAL)
plot_umap(color_by="supercluster", highlight_animal=HIGHLIGHT_ANIMAL)

In [ ]:
# ===============================================================
#  CELL — Trajectory of one individual
#  Cuttlefish (trail coloured by group) + SHRIMP + distance
#  Requires CELL 0 (colour registry).
# ===============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D

# ── Naming conventions of the tracking points ──────────────────────
_CUTTLE_CANDS = ["TopMantle", "MidCuttleHead", "MidMantle", "MidBody"]
_PREY_CANDS   = ["MidShrimp", "HeadShrimp", "Prey"]

PREY_COLOR  = "#ffb300"     # amber — deliberately OUTSIDE the registry
LINK_COLOR  = "#9e9e9e"


def _pick_key(cands, cols):
    """First tracking point available among the candidates."""
    xset = {c[:-2] for c in cols if c.endswith("_x")}
    for c in cands:
        if c in xset:
            return c
    return None


def _runs(labels):
    """
    Compresses a sequence of labels into homogeneous runs.
    [0,0,0,1,1,2] -> [(0,3,0), (3,5,1), (5,6,2)]
    One axvspan per run instead of one per frame.
    """
    labels = np.asarray(labels)
    if len(labels) == 0:
        return []
    chg = np.flatnonzero(np.diff(labels)) + 1
    b   = np.concatenate([[0], chg, [len(labels)]])
    return [(int(b[k]), int(b[k + 1]), int(labels[b[k]])) for k in range(len(b) - 1)]


def _valid_segments(x, y):
    """
    Builds segments ONLY between consecutive points that are both
    valid. A LineCollection cannot "skip" a NaN: without this
    filtering, an untracked frame links two distant positions with a
    ghost line that never existed.

    Returns (segments, valid_points_mask, valid_pairs_mask).
    """
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    ok = np.isfinite(x) & np.isfinite(y)
    if len(x) < 2:
        return np.empty((0, 2, 2)), ok, np.zeros(0, bool)
    valid_pairs = ok[:-1] & ok[1:]
    pts    = np.column_stack([x, y]).reshape(-1, 1, 2)
    segs   = np.concatenate([pts[:-1], pts[1:]], axis=1)
    return segs[valid_pairs], ok, valid_pairs


def _resolve_aid(df, raw, id_col="animal_id"):
    """Input -> exact value present in the column (str/int/zeros)."""
    if "resolve_animal_id" in globals():
        return resolve_animal_id(df, raw, id_col)
    vals = df[id_col].dropna().unique()
    s = str(raw).strip()
    for v in vals:
        if str(v).strip() == s:
            return v
    try:
        num = float(s)
        for v in vals:
            try:
                if float(str(v).strip()) == num:
                    return v
            except ValueError:
                pass
    except ValueError:
        pass
    return None


def plot_animal_trajectory(animal_id, df, group_col="cluster", features=FEATURES,
                           point_key=None, prey_key=None,
                           show_prey=True, show_distance=True,
                           link_every=0, save=True, ax_legend_ncol=2):
    """
    Left panel  : 2D trajectory — cuttlefish coloured by group,
                  shrimp as dashed amber.
    Right panel : normalised features + cuttlefish/shrimp distance,
                  background coloured by group.

    show_prey     : draw the shrimp trajectory
    show_distance : add the cuttlefish–shrimp distance curve
    link_every    : if > 0, links cuttlefish and shrimp every N frames
                    (visualises the approach; 0 = disabled)
    """
    if "group_color" not in globals():
        raise RuntimeError("Run CELL 0 first (colour registry).")

    aid = _resolve_aid(df, animal_id)
    if aid is None:
        available = sorted(str(v) for v in df["animal_id"].dropna().unique())
        print(f"[ERROR] Animal {animal_id!r} not found. Available : {available[:15]}…")
        return
    cf = df[df["animal_id"] == aid].sort_values("frame").reset_index(drop=True)
    if cf.empty:
        print(f"[ERROR] Animal {aid!r} : no rows.")
        return
    if group_col not in cf.columns:
        print(f"[ERROR] Column '{group_col}' missing from df.")
        return

    cols = set(cf.columns)
    ckey = point_key or _pick_key(_CUTTLE_CANDS, cols)
    pkey = prey_key  or _pick_key(_PREY_CANDS,   cols)
    if ckey is None:
        print(f"[ERROR] No cuttlefish point among {_CUTTLE_CANDS}.")
        return
    if show_prey and pkey is None:
        print(f"[INFO] No shrimp point among {_PREY_CANDS} "
              f"-> prey trajectory not drawn.")

    labs = pd.to_numeric(cf[group_col], errors="coerce").fillna(0).to_numpy(int)
    x = cf[f"{ckey}_x"].to_numpy(float)
    y = cf[f"{ckey}_y"].to_numpy(float)
    pfx = group_prefix(group_col)

    has_prey = show_prey and pkey is not None
    if has_prey:
        px = cf[f"{pkey}_x"].to_numpy(float)
        py = cf[f"{pkey}_y"].to_numpy(float)
    else:
        px = py = None

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # ══ 1. TRAJECTORIES ════════════════════════════════════════════════
    # ── Cuttlefish: trail coloured by group ─────────────────────────────
    segs_c, ok_c, valid_pairs_c = _valid_segments(x, y)
    if len(segs_c) == 0:
        print(f"[ERROR] Animal {aid!r} : empty/NaN cuttlefish trajectory.")
        plt.close(fig)
        return
    cols_c = [group_color(l, group_col) for l in labs[:-1][valid_pairs_c]]
    axes[0].add_collection(LineCollection(segs_c, colors=cols_c, linewidths=2.2,
                                          capstyle="round", zorder=3))
    axes[0].scatter(x[ok_c], y[ok_c],
                    c=[group_color(l, group_col) for l in labs[ok_c]],
                    s=7, zorder=4, linewidths=0)

    i0c, i1c = np.flatnonzero(ok_c)[[0, -1]]
    axes[0].scatter(x[i0c], y[i0c], marker="o", s=130, facecolor="#00e676",
                    edgecolor="black", linewidth=1.5, zorder=8)
    axes[0].scatter(x[i1c], y[i1c], marker="s", s=130, facecolor="#ff1744",
                    edgecolor="black", linewidth=1.5, zorder=8)

    # ── Shrimp: amber, dashed, OUTSIDE the group colour code ──────
    # Deliberate choice: if the prey were coloured by cluster, the two
    # trajectories could no longer be told apart. Amber is reserved for
    # the prey throughout the notebook (same in the interactive viewer).
    coverage = np.nan
    if has_prey:
        segs_p, ok_p, _ = _valid_segments(px, py)
        coverage = 100 * ok_p.mean()
        if len(segs_p):
            axes[0].add_collection(LineCollection(
                segs_p, colors=PREY_COLOR, linewidths=1.6,
                linestyles=(0, (5, 2)), alpha=0.9, zorder=2))
            axes[0].scatter(px[ok_p], py[ok_p], s=5, c=PREY_COLOR,
                            alpha=0.55, zorder=2, linewidths=0)
            i0p, i1p = np.flatnonzero(ok_p)[[0, -1]]
            axes[0].scatter(px[i0p], py[i0p], marker="^", s=110,
                            facecolor=PREY_COLOR, edgecolor="black",
                            linewidth=1.3, zorder=7)
            axes[0].scatter(px[i1p], py[i1p], marker="v", s=110,
                            facecolor="#b26a00", edgecolor="black",
                            linewidth=1.3, zorder=7)
        else:
            print(f"[INFO] Shrimp never tracked for animal {aid}.")
            has_prey = False

    # ── Cuttlefish–shrimp links (visualise the approach) ────────────────────
    if has_prey and link_every > 0:
        k = np.arange(0, len(x), link_every)
        k = k[np.isfinite(x[k]) & np.isfinite(px[k])]
        if len(k):
            links = np.stack([np.column_stack([x[k], y[k]]),
                              np.column_stack([px[k], py[k]])], axis=1)
            axes[0].add_collection(LineCollection(
                links, colors=LINK_COLOR, linewidths=0.6, alpha=0.45, zorder=1))

    # ── Distance and frame of closest approach ──────────────────────
    dist = i_min = d_min = None
    if has_prey:
        dist = np.hypot(x - px, y - py)
        if np.isfinite(dist).any():
            i_min = int(np.nanargmin(dist))
            d_min = float(dist[i_min])
            axes[0].scatter(x[i_min], y[i_min], marker="*", s=320,
                            facecolor="#ffffff", edgecolor="black",
                            linewidth=1.4, zorder=9)

    # add_collection does NOT recompute the limits -> set them by hand,
    # on the union of both trajectories.
    xs_all = np.concatenate([x[ok_c]] + ([px[np.isfinite(px)]] if has_prey else []))
    ys_all = np.concatenate([y[ok_c]] + ([py[np.isfinite(py)]] if has_prey else []))
    value_range = max(np.ptp(xs_all), np.ptp(ys_all), 1.0)
    m = 0.06 * value_range
    axes[0].set_xlim(xs_all.min() - m, xs_all.max() + m)
    axes[0].set_ylim(ys_all.min() - m, ys_all.max() + m)
    axes[0].invert_yaxis()                       # image frame: y pointing down
    axes[0].set_aspect("equal", adjustable="datalim")
    axes[0].set_title(f"Animal {aid} — Trajectories ({group_col})", fontsize=11)
    axes[0].set_xlabel("X (px)"); axes[0].set_ylabel("Y (px)")

    # ── Legend ─────────────────────────────────────────────────────────
    h_grp = group_handles(labs[ok_c], group_col, lw=3)
    leg_g = axes[0].legend(handles=h_grp, fontsize=7.5, loc="upper left",
                           ncol=ax_legend_ncol, framealpha=0.92,
                           title=f"cuttlefish — {group_col}", title_fontsize=8)
    axes[0].add_artist(leg_g)          # otherwise the next legend() overwrites it

    h_mrk = [
        Line2D([0], [0], marker="o", color="none", markerfacecolor="#00e676",
               markeredgecolor="black", markersize=9, label="cuttlefish : start"),
        Line2D([0], [0], marker="s", color="none", markerfacecolor="#ff1744",
               markeredgecolor="black", markersize=9, label="cuttlefish : end"),
    ]
    if has_prey:
        h_mrk += [
            Line2D([0], [0], color=PREY_COLOR, lw=2, ls="--",
                   label=f"shrimp ({coverage:.0f}% tracked)"),
            Line2D([0], [0], marker="^", color="none", markerfacecolor=PREY_COLOR,
                   markeredgecolor="black", markersize=9, label="shrimp : start"),
            Line2D([0], [0], marker="v", color="none", markerfacecolor="#b26a00",
                   markeredgecolor="black", markersize=9, label="shrimp : end"),
        ]
        if d_min is not None:
            h_mrk.append(Line2D([0], [0], marker="*", color="none",
                                markerfacecolor="white", markeredgecolor="black",
                                markersize=13,
                                label=f"min dist. : {d_min:.0f} px (f{i_min})"))
    axes[0].legend(handles=h_mrk, fontsize=7.5, loc="lower right",
                   framealpha=0.92, title="markers", title_fontsize=8)

    # ══ 2. TIME SERIES ══════════════════════════════════════════
    label_runs = _runs(labs)
    for a, b, lab in label_runs:
        axes[1].axvspan(a, b, color=group_color(lab, group_col),
                        alpha=0.20, lw=0, zorder=0)

    def _minmax(v):
        """Min-max normalisation, robust to NaNs and constant series."""
        v = np.asarray(v, float)
        lo, hi = np.nanmin(v), np.nanmax(v)
        return (v - lo) / (hi - lo) if hi > lo else np.full_like(v, 0.5)

    for var in features:
        if var not in cf.columns:
            continue
        v = pd.to_numeric(cf[var], errors="coerce").to_numpy(float)
        if not np.isfinite(v).any():
            continue
        axes[1].plot(_minmax(v), label=var, alpha=0.8, lw=1.2, zorder=2)

    # Cuttlefish–shrimp distance: drawn on top, in thick black
    if show_distance and dist is not None and np.isfinite(dist).any():
        axes[1].plot(_minmax(dist), color="black", lw=2.2, zorder=5,
                     label=f"cuttlefish–shrimp distance "
                           f"[{np.nanmin(dist):.0f}–{np.nanmax(dist):.0f} px]")
        axes[1].axvline(i_min, color="black", ls=":", lw=1.2, zorder=4)
        axes[1].annotate(f"min {d_min:.0f} px", xy=(i_min, 0),
                         xytext=(i_min, -0.045), ha="center", fontsize=7,
                         color="black")

    axes[1].set_xlim(0, len(cf))
    axes[1].set_ylim(-0.08, 1.05)
    axes[1].set_title(f"Animal {aid} — min-max normalised series ({group_col})",
                      fontsize=11)
    axes[1].set_xlabel("Frame (index in the sequence)")
    axes[1].set_ylabel("normalised value")

    leg_f = axes[1].legend(fontsize=6.5, loc="upper left", framealpha=0.92,
                           ncol=2, title="series", title_fontsize=8)
    axes[1].add_artist(leg_f)
    axes[1].legend(handles=group_handles(labs[ok_c], group_col, style="patch"),
                   fontsize=7, loc="upper right", framealpha=0.92,
                   ncol=ax_legend_ncol, title=group_col, title_fontsize=8)

    sup = (f"Animal {aid} | smooth={SELECTED_WINDOW} | {group_col} | "
           f"{len(cf)} frames, {len(label_runs)} runs | cuttlefish={ckey}")
    if has_prey:
        sup += f", shrimp={pkey} ({coverage:.0f}%)"
    plt.suptitle(sup, fontsize=11)
    plt.tight_layout()

    if save:
        plt.savefig(OUT_DIR /
                    f"traj_smooth{SELECTED_WINDOW}_{group_col}_animal{aid}.png",
                    dpi=150, bbox_inches="tight")
    plt.show()

    if d_min is not None:
        print(f"[TRAJ] animal {aid} : min distance {d_min:.1f} px at frame "
              f"{cf['frame'].iloc[i_min]} (index {i_min}) — "
              f"{group_col} {pfx}{labs[i_min]}")
    return cf


print("[OK] plot_animal_trajectory defined — usage :")
print("     plot_animal_trajectory(12, df, 'cluster')")
print("     plot_animal_trajectory(12, df, 'supercluster', link_every=15)")

In [ ]:
ANIMAL_ID = animal_ids[112]-1   # ← or a specific ID such as 42
ANIMAL_ID_2 = animal_ids[117]-1

plot_animal_trajectory(ANIMAL_ID, df, group_col="supercluster")
plot_animal_trajectory(ANIMAL_ID_2, df, group_col="supercluster")

In [ ]:
# interactive umap plot

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.widgets import LassoSelector, Button, TextBox
from matplotlib.path import Path
from matplotlib.lines import Line2D

%matplotlib qt

def plot_umap_interactive_radar(df, Z, features, n_clusters, n_superclusters,
                                 color_by="cluster"):

    state = {"color_by": color_by, "lasso_active": True}

    global_min = df[features].min().values
    global_max = df[features].max().values
    angles     = np.linspace(0, 2 * np.pi, len(features), endpoint=False)
    ang_full   = np.concatenate([angles, [angles[0]]])
    Z_pts      = np.column_stack([Z[:, 0], Z[:, 1]])

    # Everything as str to avoid type mismatches
    df = df.copy()
    df["animal_id"] = df["animal_id"].astype(str)
    animal_ids = sorted(df["animal_id"].unique())
    n_animals  = len(animal_ids)
    id_to_idx  = {aid: i for i, aid in enumerate(animal_ids)}

    def get_col_data(cb):
        if cb == "supercluster":
            return (df["supercluster"].values, plt.cm.tab10,
                    mpl.colors.BoundaryNorm(range(n_superclusters+1), ncolors=n_superclusters), "S")
        elif cb == "cluster":
            return (df["cluster"].values, CMAP_CLUSTER, NORM_CLUSTER, "C")
        else:
            col = np.array([id_to_idx[a] for a in df["animal_id"]])
            return (col, plt.cm.hsv,
                    mpl.colors.Normalize(vmin=0, vmax=n_animals), "A")

    # ── Figure ──────────────────────────────────────────────────
    fig = plt.figure(figsize=(20, 10))
    fig.patch.set_facecolor("#1a1a2e")

    ax_umap  = fig.add_axes([0.02, 0.14, 0.46, 0.82])
    ax_radar = fig.add_axes([0.52, 0.52, 0.22, 0.42], polar=True)
    ax_traj  = fig.add_axes([0.52, 0.12, 0.22, 0.36])
    ax_feats = fig.add_axes([0.76, 0.12, 0.22, 0.78])

    for ax in [ax_umap, ax_radar, ax_traj, ax_feats]:
        ax.set_facecolor("#0d0d1a")
        for spine in ax.spines.values():
            spine.set_edgecolor("#333333")
        ax.tick_params(colors="#555555", labelsize=7)

    # ── Widgets ─────────────────────────────────────────────────
    ax_btn_mode   = fig.add_axes([0.02, 0.03, 0.10, 0.06])
    ax_btn_reset  = fig.add_axes([0.13, 0.03, 0.07, 0.06])
    ax_btn_lasso  = fig.add_axes([0.21, 0.03, 0.08, 0.06])
    ax_textbox    = fig.add_axes([0.31, 0.03, 0.10, 0.06])
    ax_btn_animal = fig.add_axes([0.42, 0.03, 0.10, 0.06])

    btn_mode   = Button(ax_btn_mode,  f"Mode: {color_by}", color="#222244", hovercolor="#333366")
    btn_reset  = Button(ax_btn_reset, "Reset",             color="#222222", hovercolor="#333333")
    btn_lasso  = Button(ax_btn_lasso, "Lasso: ON",         color="#442200", hovercolor="#663300")
    textbox    = TextBox(ax_textbox,  "Animal: ",          initial="",
                         color="#0d0d1a", hovercolor="#1a1a2e")
    btn_animal = Button(ax_btn_animal,"Show",              color="#224422", hovercolor="#336633")

    for btn in [btn_mode, btn_reset, btn_lasso, btn_animal]:
        btn.label.set_color("#cccccc")
        btn.label.set_fontsize(8)
    textbox.label.set_color("#aaaaaa")
    textbox.text_disp.set_color("#ffffff")

    # ── Scatter ─────────────────────────────────────────────────
    col_data, cmap, norm_g, _ = get_col_data(state["color_by"])
    sc        = ax_umap.scatter(Z_pts[:,0], Z_pts[:,1],
                                 c=cmap(norm_g(np.asarray(col_data))),   # Colours computed in one go (vectorised): 95k calls -> 1
                                 s=4, alpha=0.7, linewidths=0, zorder=2)
    hl_lasso  = ax_umap.scatter([], [], s=20, facecolors="none",
                                 edgecolors="white", linewidths=0.7, zorder=4)
    hl_animal = ax_umap.scatter([], [], s=15, zorder=5)
    hl_click  = ax_umap.scatter([], [], s=120, marker="*",
                                 facecolors="yellow", edgecolors="white",
                                 linewidths=0.7, zorder=6)

    # Path of the individual (like plot_umap) + start / middle / end markers
    from matplotlib.collections import LineCollection
    path_lc  = LineCollection([], linewidths=1.6, alpha=0.9, zorder=4)
    ax_umap.add_collection(path_lc)
    hl_start = ax_umap.scatter([], [], marker="o", s=110, facecolor="#00e676",
                               edgecolor="black", linewidth=1.4, zorder=7)
    hl_mid   = ax_umap.scatter([], [], marker="D", s=95,  facecolor="#ffd600",
                               edgecolor="black", linewidth=1.4, zorder=7)
    hl_end   = ax_umap.scatter([], [], marker="s", s=110, facecolor="#ff1744",
                               edgecolor="black", linewidth=1.4, zorder=7)

    ax_umap.set_facecolor("#0d0d1a")
    ax_umap.set_title(f"UMAP — {state['color_by']}", color="#aaaaaa", fontsize=10)
    for spine in ax_umap.spines.values():
        spine.set_edgecolor("#333333")

    info_text = ax_umap.text(
        0.02, 0.02,
        "Lasso ON → draw | Lasso OFF → click point | Dbl-click → reset",
        transform=ax_umap.transAxes, color="#777777", fontsize=8,
        bbox=dict(facecolor="#111111", alpha=0.7, edgecolor="none", pad=3)
    )

    # ── Radar ───────────────────────────────────────────────────
    def setup_radar():
        ax_radar.cla()
        ax_radar.set_facecolor("#0d0d1a")
        ax_radar.set_xticks(angles)
        ax_radar.set_xticklabels([f.replace("_", "\n") for f in features],
                                  color="#999999", fontsize=6)
        ax_radar.set_ylim(0, 1)
        ax_radar.set_yticks([0.5])
        ax_radar.set_yticklabels(["0.5"], color="#555555", fontsize=6)
        ax_radar.grid(color="#333333", linewidth=0.5)
        ax_radar.spines["polar"].set_color("#333333")
        ax_radar.set_title("Radar", color="#aaaaaa", fontsize=9, pad=10)

    def draw_radar(norm_vals, color, title):
        setup_radar()
        vf = np.concatenate([norm_vals, [norm_vals[0]]])
        ax_radar.plot(ang_full, vf, linewidth=2, color=color)
        ax_radar.fill(ang_full, vf, alpha=0.3, color=color)
        for a, vn in zip(angles, norm_vals):
            ax_radar.text(a, min(vn + 0.15, 1.1), f"{vn:.2f}",
                          ha="center", va="center", color="#dddddd", fontsize=6,
                          bbox=dict(facecolor="#111111", alpha=0.5, edgecolor="none", pad=1))
        ax_radar.set_title(title, color="#aaaaaa", fontsize=8, pad=10)
        fig.canvas.draw_idle()

    setup_radar()

    # ── Traj + feats ────────────────────────────────────────────
    def setup_traj():
        ax_traj.cla()
        ax_traj.set_facecolor("#0d0d1a")
        ax_traj.set_title("Trajectory", color="#aaaaaa", fontsize=9)
        for spine in ax_traj.spines.values():
            spine.set_edgecolor("#333333")
        ax_traj.tick_params(colors="#555555", labelsize=7)

    def setup_feats():
        ax_feats.cla()
        ax_feats.set_facecolor("#0d0d1a")
        ax_feats.set_title("Normalised variables", color="#aaaaaa", fontsize=9)
        for spine in ax_feats.spines.values():
            spine.set_edgecolor("#333333")
        ax_feats.tick_params(colors="#555555", labelsize=7)

    setup_traj()
    setup_feats()

    # ── Lasso ───────────────────────────────────────────────────
    def on_lasso(verts):
        if not state["lasso_active"]:
            return
        # Tiny lasso = simple click -> select the nearest point
        v  = np.asarray(verts)
        xr = ax_umap.get_xlim()
        if len(v) < 3 or np.ptp(v, axis=0).max() < 0.01 * (xr[1] - xr[0]):
            _select_point(*v[-1])
            return
        lasso_path   = Path(verts)
        inside       = lasso_path.contains_points(Z_pts)
        selected_idx = np.where(inside)[0]
        n_sel        = len(selected_idx)

        if n_sel == 0:
            info_text.set_text("Empty lasso")
            hl_lasso.set_offsets(np.empty((0, 2)))
            fig.canvas.draw_idle()
            return

        hl_lasso.set_offsets(Z_pts[selected_idx])
        raw_vals  = df[features].values[selected_idx].mean(axis=0)
        norm_vals = np.clip((raw_vals - global_min)/(global_max - global_min + 1e-9), 0, 1)

        cd, cm, ng, lpfx = get_col_data(state["color_by"])
        if state["color_by"] in ("cluster", "supercluster"):
            counts   = pd.Series(cd[selected_idx]).value_counts()
            dominant = counts.idxmax()
            color    = cm(ng(dominant))
            comp     = " | ".join([f"{lpfx}{g}:{n}" for g, n in counts.items()])
        else:
            color = "#44aaff"
            comp  = f"{pd.Series(df['animal_id'].values[selected_idx]).nunique()} individuals"

        info_text.set_text(f"{n_sel} pts — {comp}")
        draw_radar(norm_vals, color, f"Lasso n={n_sel}")

    lasso_selector = LassoSelector(ax_umap, on_lasso,
                                    props=dict(color="#FFD700", linewidth=1.5),
                                    useblit=True, button=1)

    # ── Click ───────────────────────────────────────────────────
    def _select_point(x, y):
        xy   = np.array([x, y])
        dist = np.linalg.norm(Z_pts - xy, axis=1)
        idx  = int(np.argmin(dist))
        xlim = ax_umap.get_xlim()
        if dist[idx] > (xlim[1] - xlim[0]) * 0.05:
            return

        hl_click.set_offsets([Z_pts[idx]])
        raw_vals  = df[features].values[idx]
        norm_vals = np.clip((raw_vals - global_min)/(global_max - global_min + 1e-9), 0, 1)
        cd, cm, ng, lpfx = get_col_data(state["color_by"])
        color = cm(ng(cd[idx]))
        label = f"{lpfx}{cd[idx]} | animal {df['animal_id'].iloc[idx]}"
        info_text.set_text(f"Point {idx} — {label}")
        draw_radar(norm_vals, color, f"Point {idx}\n{label}")

    def on_click(event):
        if event.inaxes != ax_umap:
            return
        if event.dblclick:
            reset(None); return
        if state["lasso_active"]:
            return  # the lasso handles clicks when active
        if event.button != 1 or event.xdata is None:
            return

        _select_point(event.xdata, event.ydata)

    fig.canvas.mpl_connect("button_press_event", on_click)

    # ── Lasso ON/OFF button ─────────────────────────────────────
    # Disabling the lasso frees the clicks for the TextBox and point clicking
    def toggle_lasso(event):
        state["lasso_active"] = not state["lasso_active"]
        if state["lasso_active"]:
            lasso_selector.set_active(True)
            btn_lasso.label.set_text("Lasso: ON")
            btn_lasso.ax.set_facecolor("#442200")
            info_text.set_text("Lasso ON — draw an area")
        else:
            lasso_selector.set_active(False)
            btn_lasso.label.set_text("Lasso: OFF")
            btn_lasso.ax.set_facecolor("#004422")
            info_text.set_text("Lasso OFF — click a point | type animal_id")
        fig.canvas.draw_idle()

    btn_lasso.on_clicked(toggle_lasso)

    # ── Mode button ─────────────────────────────────────────────
    modes = ["supercluster", "cluster", "animal"]

    def toggle_mode(event):
        idx = modes.index(state["color_by"])
        state["color_by"] = modes[(idx + 1) % len(modes)]
        btn_mode.label.set_text(f"Mode: {state['color_by']}")
        cd, cm, ng, _ = get_col_data(state["color_by"])
        sc.set_facecolors(cm(ng(np.asarray(cd))))
        ax_umap.set_title(f"UMAP — {state['color_by']}", color="#aaaaaa", fontsize=10)
        fig.canvas.draw_idle()

    btn_mode.on_clicked(toggle_mode)

    # ── Reset button ────────────────────────────────────────────
    def reset(event):
        hl_lasso.set_offsets(np.empty((0, 2)))
        hl_animal.set_offsets(np.empty((0, 2)))
        hl_click.set_offsets(np.empty((0, 2)))
        path_lc.set_segments([])
        for h in (hl_start, hl_mid, hl_end):
            h.set_offsets(np.empty((0, 2)))
        if ax_umap.get_legend() is not None:
            ax_umap.get_legend().remove()
        info_text.set_text("Reset")
        setup_radar()
        ax_radar.set_title("Radar", color="#aaaaaa", fontsize=9, pad=10)
        setup_traj()
        setup_feats()
        fig.canvas.draw_idle()

    btn_reset.on_clicked(reset)

    # ── Animal button ───────────────────────────────────────────
    def show_animal(event):
        raw = textbox.text.strip()
        if not raw:
            info_text.set_text(f"Example IDs : {', '.join(animal_ids[:6])}")
            fig.canvas.draw_idle()
            return

        aid = str(raw)
        if aid not in df["animal_id"].values:
            sample = ", ".join(animal_ids[:8])
            info_text.set_text(f"'{aid}' not found. Examples : {sample}")
            fig.canvas.draw_idle()
            return

        mask = df["animal_id"].values == aid
        idx  = np.where(mask)[0]
        n_a  = len(idx)

        # Rainbow UMAP highlighting
        colors_r = plt.cm.rainbow(np.linspace(0, 1, n_a))
        hl_animal.set_offsets(Z_pts[idx])
        hl_animal.set_facecolors(colors_r)
        hl_animal.set_sizes([15] * n_a)

        # Rainbow path between the points + start / middle / end
        pts_a = Z_pts[idx]
        if n_a > 1:
            segs = np.stack([pts_a[:-1], pts_a[1:]], axis=1)
            path_lc.set_segments(segs)
            path_lc.set_color(plt.cm.rainbow(np.linspace(0, 1, len(segs))))
        else:
            path_lc.set_segments([])
        hl_start.set_offsets(pts_a[[0]])
        hl_mid.set_offsets(pts_a[[n_a // 2]])
        hl_end.set_offsets(pts_a[[-1]])
        leg = ax_umap.legend(handles=[
            Line2D([0], [0], marker="o", color="none", markerfacecolor="#00e676",
                   markeredgecolor="black", markersize=9, label="start"),
            Line2D([0], [0], marker="D", color="none", markerfacecolor="#ffd600",
                   markeredgecolor="black", markersize=8, label="middle"),
            Line2D([0], [0], marker="s", color="none", markerfacecolor="#ff1744",
                   markeredgecolor="black", markersize=9, label="end"),
        ], loc="lower right", fontsize=8, facecolor="#111111",
           edgecolor="#333333", labelcolor="#cccccc", title=f"Animal {aid}")
        leg.get_title().set_color("#cccccc")

        # Mean radar
        raw_vals  = df[features].values[mask].mean(axis=0)
        norm_vals = np.clip((raw_vals - global_min)/(global_max - global_min + 1e-9), 0, 1)
        draw_radar(norm_vals, "#44aaff", f"Animal {aid} (n={n_a})")

        # Trajectory
        setup_traj()
        cf = df[mask].reset_index(drop=True)
        cl = df["cluster"].values[mask]
        _, cm_c, ng_c, _ = get_col_data("cluster")
        # A single LineCollection instead of one plot() per segment
        tx = cf["TopMantle_x"].to_numpy(float)
        ty = cf["TopMantle_y"].to_numpy(float)
        cols_t = cm_c(ng_c(np.asarray(cl)))
        if len(cf) > 1:
            pts_t = np.column_stack([tx, ty])
            ax_traj.add_collection(LineCollection(
                np.stack([pts_t[:-1], pts_t[1:]], axis=1),
                colors=cols_t[:-1], linewidths=0.8))
        ax_traj.scatter(tx, ty, c=cols_t, s=2, linewidths=0)
        ax_traj.autoscale_view()
        ax_traj.scatter(cf["TopMantle_x"].iloc[0],  cf["TopMantle_y"].iloc[0],
                        color="lime", s=40, zorder=5)
        ax_traj.scatter(cf["TopMantle_x"].iloc[-1], cf["TopMantle_y"].iloc[-1],
                        color="white", s=40, zorder=5)
        ax_traj.invert_yaxis()
        ax_traj.set_title(f"Animal {aid}", color="#aaaaaa", fontsize=9)

        # Time features
        setup_feats()
        t = np.arange(n_a)
        for feat in features:
            fmax = cf[feat].max()
            ax_feats.plot(t, cf[feat] / (fmax + 1e-9),
                          label=feat.replace("_"," "), alpha=0.8, linewidth=1)
        ax_feats.legend(fontsize=6, loc="upper right")
        ax_feats.set_xlabel("Frame", color="#555555", fontsize=7)
        ax_feats.set_title(f"Variables — animal {aid}", color="#aaaaaa", fontsize=9)

        info_text.set_text(f"Animal {aid} — {n_a} frames")
        fig.canvas.draw_idle()

    btn_animal.on_clicked(show_animal)

    # Pressing Enter in the textbox = same effect as the button
    textbox.on_submit(lambda text: show_animal(None))

    plt.suptitle("Lasso ON/OFF | Mode | Animal ID + Enter or Show | Dbl-click → reset",
                 color="#cccccc", fontsize=9)
    plt.show()
    return fig


plot_umap_interactive_radar(df, Z, FEATURES, n_clusters, n_superclusters,
                             color_by="cluster")

In [ ]:
%matplotlib inline

In [ ]:
import io, contextlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.widgets import Button, Slider, TextBox
from collections import Counter
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import pdist, squareform
from sklearn.preprocessing import StandardScaler


# ── Sequence extraction ──────────────────────────────────
def _extract_sequences(df, cluster_col, min_run_len):
    sequences, mono_ids, multi_ids = {}, [], []
    for aid, group in df.groupby("animal_id"):
        raw = group.sort_values("frame")[cluster_col].values
        blocks, lengths = [], []
        cur, start = raw[0], 0
        for i in range(1, len(raw)):
            if raw[i] != cur:
                blocks.append(int(cur)); lengths.append(i - start)
                cur, start = raw[i], i
        blocks.append(int(cur)); lengths.append(len(raw) - start)
        fb, fl = [], []
        for b, l in zip(blocks, lengths):
            if l >= min_run_len:
                if fb and fb[-1] == b: fl[-1] += l
                else: fb.append(b); fl.append(l)
        sequences[aid] = {"seq": fb, "lengths": fl,
                          "total": len(raw)}       # real duration (frames), short segments included
        (mono_ids if len(set(fb)) <= 1 else multi_ids).append(aid)
    return sequences, mono_ids, multi_ids


# ── State similarity, alignment and dynamics ───────────────
try:
    from numba import njit
except ImportError:                       # no numba: pure Python version (slower)
    def njit(*a, **k):
        return a[0] if a and callable(a[0]) else (lambda f: f)


def _state_kernel_half(state_dist):
    """Square root of the state similarity K = exp(-d²/2σ²), σ = median(d)/2.
    v @ K½ makes the Euclidean distance "soft": two close states
    (neighbouring prototypes) count as similar."""
    d = np.asarray(state_dist, float)
    sigma = np.median(d[d > 0]) / 2 if (d > 0).any() else 1.0
    K = np.exp(-d ** 2 / (2 * sigma ** 2))
    ev, evec = np.linalg.eigh(K)
    return evec @ np.diag(np.sqrt(np.clip(ev, 0, None))) @ evec.T


@njit(cache=False)
def _align_cost(a, b, sub, gap):
    """Global alignment (Needleman-Wunsch) of two state sequences.
    Substitution = distance between prototypes (0..1), insertion/deletion = gap.
    Normalised by the length of the longest sequence."""
    n, m = len(a), len(b)
    prev = np.empty(m + 1)
    for j in range(m + 1):
        prev[j] = j * gap
    for i in range(1, n + 1):
        cur = np.empty(m + 1)
        cur[0] = i * gap
        for j in range(1, m + 1):
            c = prev[j - 1] + sub[a[i - 1], b[j - 1]]
            if prev[j] + gap < c:
                c = prev[j] + gap
            if cur[j - 1] + gap < c:
                c = cur[j - 1] + gap
            cur[j] = c
        prev = cur
    return prev[m] / max(n, m, 1)


@njit(cache=False)
def _dtw_cost(x, y, radius):
    """Multivariate DTW (time × features) with a Sakoe-Chiba band:
    compares how the features evolve while tolerating a slight time shift."""
    n, m = x.shape[0], y.shape[0]
    D = np.full((n + 1, m + 1), 1e18)
    D[0, 0] = 0.0
    for i in range(1, n + 1):
        for j in range(max(1, i - radius), min(m, i + radius) + 1):
            d = 0.0
            for k in range(x.shape[1]):
                d += (x[i - 1, k] - y[j - 1, k]) ** 2
            best = D[i - 1, j - 1]
            if D[i - 1, j] < best:
                best = D[i - 1, j]
            if D[i, j - 1] < best:
                best = D[i, j - 1]
            D[i, j] = np.sqrt(d) + best
    return D[n, m]


def _pairwise(items, fn, *args):
    n = len(items)
    M = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            M[i, j] = M[j, i] = fn(items[i], items[j], *args)
    return M


# ── SeqVec ────────────────────────────────────────────────────
#   Each individual is described by 9 families, each answers one question:
#     hist     : how much time in each state?
#     temporal : which states in the 1st vs 2nd half (approach -> before the strike)?
#     lastapp  : which states just before the strike? (last appearance of each state)
#     duration : total duration of the sequence (1 column, log)
#     align    : is the order of the states similar? (close states = cheap substitution)
#     dynamics : do the features evolve the same way? (DTW on the features, without the states)
#     speed    : mean speed of the cuttlefish (1 column, log); with duration = distance travelled
#     path     : total distance travelled by the cuttlefish (mm, 1 column, log): did it move a lot?
#     start    : distance to the prey at the START of the sequence (mm, 1 column, log): does it start close?
#   Distance: D² = Σ_f w_f · (D_f / scale_f)² -> each family weighs w_f / Σw,
#   whatever its number of columns.
def _seqvec(sequences, ids, n_cl,
            w_hist=1.0, w_temporal=1.0, w_lastapp=1.0, w_duration=0.5, w_align=1.0,
            w_dynamics=1.0, w_speed=0.5, w_path=0.5, w_start=0.5, state_similarity=True,
            state_dist=None, align_gap=0.5, series=None, speed=None, path=None, start=None):

    def feat_hist(s, l, n):
        # share of time in each state (log1p: compresses extreme dominances)
        c = Counter()
        for cl, ln in zip(s, l):
            c[cl] += ln
        raw = np.array([c.get(i, 0) / sum(l) for i in range(n)])
        return np.log1p(raw * 10) / np.log1p(10)

    def feat_temporal(s, l, n):
        # share of time in each state in the 1st half (t1), 2nd half (t2)
        # and their difference (t2 - t1 > 0: state more present before the strike)
        total, half = sum(l), sum(l) / 2
        h1, h2, cum = np.zeros(n), np.zeros(n), 0
        for cl, ln in zip(s, l):
            if cl < n:
                in_h1 = max(0, min(ln, half - cum))
                h1[cl] += in_h1
                h2[cl] += ln - in_h1
            cum += ln
        t1, t2 = h1 / (half + 1e-9), h2 / (half + 1e-9)
        return np.concatenate([t1, t2, t2 - t1])

    def feat_lastapp(s, n):
        # presence (0/1) + position of the LAST appearance of each state
        # (1.0 = it is the final state, just before the strike)
        present, position = np.zeros(n), np.full(n, 0.5)
        for i in range(n):
            if i in s:
                present[i]  = 1.0
                position[i] = (len(s) - 1 - s[::-1].index(i)) / max(len(s) - 1, 1)
        return np.concatenate([present, position])

    # ── Raw vectors, grouped by family ────────────────────
    fam_sizes = {"hist": n_cl, "temporal": 3 * n_cl, "lastapp": 2 * n_cl, "duration": 1, "speed": 1, "path": 1,
                 "start": 1}
    weights   = {"hist": w_hist, "temporal": w_temporal, "lastapp": w_lastapp,
                 "duration": w_duration, "speed": w_speed if speed else 0.0,
                 "path": w_path if path else 0.0, "start": w_start if start else 0.0}
    blocks    = {f: [] for f in fam_sizes}
    for aid in ids:
        s, l = sequences[aid]["seq"], sequences[aid]["lengths"]
        if not s:                                   # empty sequence -> zero vector
            for f, k in fam_sizes.items():
                blocks[f].append(np.zeros(k))
            continue
        blocks["hist"].append(feat_hist(s, l, n_cl))
        blocks["temporal"].append(feat_temporal(s, l, n_cl))
        blocks["lastapp"].append(feat_lastapp(s, n_cl))
        # real duration of the sequence (not only the segments kept by min_run_len)
        blocks["duration"].append(np.array([np.log(sequences[aid].get("total", sum(l)))]))   # log: 100 -> 200 f = 1000 -> 2000 f
        blocks["speed"].append(np.array([np.log1p(speed.get(aid, 0.0)) if speed else 0.0]))   # mm/s, log
        blocks["path"].append(np.array([np.log1p(path.get(aid, 0.0)) if path else 0.0]))     # mm, log
        blocks["start"].append(np.array([np.log1p(start.get(aid, 0.0)) if start else 0.0]))   # mm, log
    blocks = {f: np.array(v, dtype=float) for f, v in blocks.items()}

    # ── Close states = similar ("soft" hist and temporal) ──
    if state_similarity and state_dist is not None:
        Kh = _state_kernel_half(state_dist)[:n_cl, :n_cl]
        blocks["hist"] = blocks["hist"] @ Kh
        blocks["temporal"] = (blocks["temporal"].reshape(len(ids), 3, n_cl) @ Kh
                              ).reshape(len(ids), 3 * n_cl)

    # ── Combination: each family brought to the same scale ─
    parts, used = [], {}
    for f in fam_sizes:
        if weights[f] <= 0:
            continue
        Xf = StandardScaler().fit_transform(blocks[f])
        scale = np.sqrt(np.mean(pdist(Xf) ** 2)) if len(Xf) > 1 else 0.0
        if scale < 1e-9:     # tolerance: a constant family varies by ~1e-15 (rounding)                              # constant family: adds nothing
            continue
        parts.append(Xf * np.sqrt(weights[f]) / scale)
        used[f] = weights[f]
    X  = np.hstack(parts) if parts else np.zeros((len(ids), 0))
    D2 = squareform(pdist(X)) ** 2 if parts else np.zeros((len(ids), len(ids)))

    # ── align: alignment distance of the state sequences ──────
    if w_align > 0:
        if state_dist is None:
            raise ValueError("w_align > 0 requires state_dist (provided by show_strategy_explorer)")
        sub = np.asarray(state_dist, float)[:n_cl, :n_cl]
        sub = sub / (sub.max() + 1e-12)
        sq  = [np.asarray(sequences[a]["seq"], dtype=np.int64) for a in ids]
        Da  = _pairwise(sq, _align_cost, sub, float(align_gap))
        sc  = np.sqrt(np.mean(Da[np.triu_indices(len(ids), 1)] ** 2))
        if sc > 1e-9:
            D2 += w_align * (Da / sc) ** 2
            used["align"] = w_align

    # ── dynamics: DTW on the feature series (N_DYN points per individual) ──
    if w_dynamics > 0:
        if series is None:
            raise ValueError("w_dynamics > 0 requires series (provided by show_strategy_explorer)")
        ser = [np.ascontiguousarray(series[a], dtype=float) for a in ids]
        rad = max(1, int(0.1 * ser[0].shape[0]))          # tolerated shift: 10 % of the time
        Dd  = _pairwise(ser, _dtw_cost, rad)
        sc  = np.sqrt(np.mean(Dd[np.triu_indices(len(ids), 1)] ** 2))
        if sc > 1e-9:
            D2 += w_dynamics * (Dd / sc) ** 2
            used["dynamics"] = w_dynamics

    if not used:
        raise ValueError("All SeqVec weights are 0 (or constant families).")
    D   = np.sqrt(D2)
    tot = sum(used.values())
    print("[SEQVEC] mean share of each family in D² : " +
          " | ".join(f"{f} {w / tot:.0%}" for f, w in used.items()))
    desc = f"SeqVec ({', '.join(used)})"
    print(f"[SEQVEC] {len(ids)} individuals | {desc}")
    return X, D, desc
# ── Hierarchical clustering ───────────────────────────────────
def _cluster_hier(D, ids, n_strat, linkage_method="ward"):
    Z      = linkage(squareform(np.clip(D, 0, None)), method=linkage_method)
    labels = fcluster(Z, n_strat, criterion="maxclust")
    n2l    = {n: str(i) for i, n in enumerate(sorted(set(labels)))}
    return {aid: n2l[lab] for aid, lab in zip(ids, labels)}, Z


def _ward_stability(D, k, linkage_method="ward", n_boot=30, seed=0):
    """Group stability: remove 20 % of the individuals, redo the
    clustering and measure the agreement (ARI) with the full clustering."""
    from sklearn.metrics import adjusted_rand_score
    cut = lambda M: fcluster(linkage(squareform(np.clip(M, 0, None), checks=False),
                                     method=linkage_method), k, criterion="maxclust")
    rng, full, aris = np.random.default_rng(seed), cut(D), []
    for _ in range(n_boot):
        sub = np.sort(rng.choice(len(D), int(.8 * len(D)), replace=False))
        aris.append(adjusted_rand_score(full[sub], cut(D[np.ix_(sub, sub)])))
    return float(np.mean(aris))


def _silhouette_optimal_k(D, k_range=range(2, 13), linkage_method="ward"):
    """K that maximises the silhouette of the hierarchical clustering on D.
    Silhouette: is each individual closer to its own group than to the
    neighbouring group? (1 = clean groups, 0 = overlap)."""
    from sklearn.metrics import silhouette_score
    Zl  = linkage(squareform(np.clip(D, 0, None), checks=False), method=linkage_method)
    res = {}
    print("[SILHOUETTE] K tested (Ward) :")
    for k in k_range:
        if k >= len(D):
            break
        lab = fcluster(Zl, k, criterion="maxclust")
        if len(set(lab)) < 2:
            continue
        res[k] = silhouette_score(D, lab, metric="precomputed")
        print("   K={k:<3} silhouette {s:+.3f} | smallest group {m}".format(k=k, s=res[k], m=int(np.bincount(lab)[1:].min())))
    k_best = max(res, key=res.get)
    print(f"[SILHOUETTE] K selected : {k_best}")
    return k_best, res


# ── Leiden to estimate the optimal K ────────────────────────────
def _leiden_optimal_k(D, ids, resolution_range=None):
    try:
        import igraph as ig
        import leidenalg
    except ImportError:
        print("[WARN] igraph/leidenalg not available")
        return None, {}
    n = len(ids)
    if n < 3: return None, {}
    D_max = D.max()
    if D_max == 0: return 1, {}
    sim       = 1 - D / D_max
    threshold = np.percentile(sim[sim > 0], 50)
    edges, weights = [], []
    for i in range(n):
        for j in range(i+1, n):
            if sim[i, j] > threshold:
                edges.append((i, j)); weights.append(float(sim[i, j]))
    if not edges: return 1, {}
    g = ig.Graph(n=n, edges=edges)
    g.es["weight"] = weights
    if resolution_range is None:
        resolution_range = [0.5, 0.8, 0.9, 1.0, 1.1, 1.2, 1.5, 2.0]
    results = {}
    for res in resolution_range:
        part = leidenalg.find_partition(
            g, leidenalg.RBConfigurationVertexPartition,
            weights="weight", resolution_parameter=res, seed=42)
        # standard modularity (comparable across resolutions, unlike
        # part.quality() which drops mechanically as the resolution rises)
        results[res] = {"k": len(set(part.membership)),
                        "modularity": g.modularity(part.membership, weights="weight")}
    ok = {r: v for r, v in results.items() if v["k"] >= 2}   # at least 2 groups
    if not ok: return 1, results
    best_res  = max(ok, key=lambda r: ok[r]["modularity"])
    k_optimal = results[best_res]["k"]
    print(f"\n[LEIDEN] optimal K = {k_optimal} (resolution={best_res:.2f})")
    for res, r in results.items():
        print(f"  res={res:.2f} → K={r['k']} modularity={r['modularity']:.3f}")
    return k_optimal, results


def _leiden_knn_groups(D, ids, knn_k=10, min_size=3,
                       resolutions=(0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.5)):
    """Groups = Leiden communities on the graph of the knn_k nearest neighbours.
    knn_k = number of NEIGHBOURS per individual (NOT the number of groups):
    Leiden decides the number of groups itself (resolution with the best modularity).
    Groups with < min_size individuals are attached to the closest group.
    Tested (test_k_methods.py): best method on synthetic and real data."""
    import igraph as ig
    import leidenalg
    n   = len(ids)
    sim = 1 - D / (D.max() + 1e-12)
    nn  = np.argsort(D, axis=1)[:, 1:knn_k + 1]
    edges = sorted({(min(i, j), max(i, j)) for i in range(n) for j in nn[i]})
    w = [float(sim[i, j]) for i, j in edges]
    g = ig.Graph(n=n, edges=edges)
    best, best_q, best_res = None, -np.inf, None
    for res in resolutions:
        p = leidenalg.find_partition(g, leidenalg.RBConfigurationVertexPartition,
                                     weights=w, resolution_parameter=res, seed=42)
        mem = np.array(p.membership)
        if len(set(mem)) < 2:
            continue
        q = g.modularity(mem, weights=w)
        if q > best_q:
            best, best_q, best_res = mem, q, res
    if best is None:
        return {a: "0" for a in ids}, 1
    lab = best.copy()
    while True:                                  # attach the small groups
        sizes = pd.Series(lab).value_counts()
        small, big = sizes[sizes < min_size].index, sizes[sizes >= min_size].index
        if len(small) == 0 or len(big) == 0:
            break
        for i in np.where(np.isin(lab, small))[0]:
            lab[i] = min(big, key=lambda c: D[i, lab == c].mean())
    order = pd.Series(lab).value_counts().index          # 0 = largest group
    l2s   = {c: str(r) for r, c in enumerate(order)}
    print(f"[LEIDEN kNN] {knn_k} neighbours | resolution {best_res} | modularity {best_q:.3f} "
          f"| {len(order)} groups")
    return {a: l2s[c] for a, c in zip(ids, lab)}, len(order)


def _knn_stability(D, knn_k=10, n_boot=30, seed=0):
    """Stability: remove 20 % of the individuals, redo the kNN groups and
    measure the agreement (ARI) with the full groups (> 0.75 good, < 0.5 fragile)."""
    from sklearn.metrics import adjusted_rand_score
    ids = list(range(len(D)))
    with contextlib.redirect_stdout(io.StringIO()):
        full = _leiden_knn_groups(D, ids, knn_k)[0]
        rng, aris = np.random.default_rng(seed), []
        for _ in range(n_boot):
            sub = np.sort(rng.choice(len(D), int(.8 * len(D)), replace=False))
            part = _leiden_knn_groups(D[np.ix_(sub, sub)], list(sub), knn_k)[0]
            aris.append(adjusted_rand_score([full[i] for i in sub], [part[i] for i in sub]))
    return float(np.mean(aris))


# ── Main window ────────────────────────────────────────
def show_strategy_explorer(
    df,
    cluster_col    = "cluster",
    n_strategies   = 4,
    min_run_len      = 3,
    use_leiden_k   = False,
    k_method       = None,            # "knn" (recommended) | "leiden" | "silhouette" | a number
    knn_k          = 10,              # k_method='knn': number of NEIGHBOURS per individual (not the number of groups)
    k_range        = range(2, 13),    # K tested by k_method='silhouette'
    linkage_method = "ward",
    # ── SeqVec: weights of the 9 families (see _seqvec) ──
    w_hist     = 1.0,     # time spent in each state
    w_temporal = 1.0,     # states at the start vs end of the sequence
    w_lastapp  = 1.0,     # states just before the strike
    w_duration = 0.5,     # total duration (1 column)
    w_align    = 1.0,     # order of the states (alignment)
    w_dynamics = 1.0,     # evolution of the features (DTW), without the states
    w_speed    = 0.5,     # mean speed (1 column); with duration = distance travelled
    w_path     = 0.5,     # total distance travelled (1 column)
    w_start    = 0.5,     # start distance to the prey (1 column): separates close-range strikes
    state_similarity = True,   # close states = similar
    features   = None,
):
    df = df.copy()
    df["animal_id"] = df["animal_id"].astype(str)

    # ── SeqVec inputs: state prototypes + feature series ──
    feats  = list(features) if features is not None else list(FEATURES)
    fw     = globals().get("FEATURE_WEIGHTS", {}) or {}
    zs_all = pd.DataFrame(((df[feats] - df[feats].mean()) / df[feats].std()).values
                          * np.array([fw.get(f, 1.0) for f in feats]), index=df.index)
    seqvec_inputs = {}
    N_DYN = 50          # points per individual for the DTW (duration is in the duration family)

    def _seqvec_inputs(col, n_cl):
        # state prototype = centroid in the standardised + weighted space
        cent = zs_all.groupby(df[col].values).mean().reindex(range(n_cl)).fillna(0).values
        # series of each individual: N_DYN successive means (normalised time)
        ser = {}
        for aid, g in df.groupby("animal_id"):
            v = zs_all.loc[g.sort_values("frame").index].values
            e = np.linspace(0, len(v), N_DYN + 1).astype(int)
            ser[aid] = np.vstack([v[e[i]:max(e[i + 1], e[i] + 1)].mean(0) for i in range(N_DYN)])
        # mean speed of each individual (speed family)
        spd = (df.groupby("animal_id")["speed_cuttle_mm_s"].mean().to_dict()
               if "speed_cuttle_mm_s" in df.columns else None)
        # total distance travelled by each individual (path family, mm)
        pth = ((df.groupby("animal_id")["speed_cuttle_mm_s"].sum() / 60).to_dict()
               if "speed_cuttle_mm_s" in df.columns else None)
        # distance to the prey at the start of the sequence (start family, mm)
        st0 = (df.sort_values("frame").groupby("animal_id")["distance_mm"].first().to_dict()
               if "distance_mm" in df.columns else None)
        return {"state_dist": squareform(pdist(cent)), "state_similarity": state_similarity,
                "series": ser,
                "speed": spd,
                "path": pth,
                "start": st0}

    def compute_all(col, n_s):
        n_cl = int(df[col].nunique())
        seqs, mono, multi = _extract_sequences(df, col, min_run_len)
        print(f"[INFO] {len(multi)} multi | {len(mono)} mono | n_cl={n_cl}")
        if len(multi) < 2:
            return seqs, mono, multi, None, None, {}, n_cl, "—", n_s
        seqvec_inputs.clear()
        seqvec_inputs.update(_seqvec_inputs(col, n_cl))
        X, D, desc = _seqvec(
            seqs, multi, n_cl,
            w_hist=w_hist, w_temporal=w_temporal, w_lastapp=w_lastapp,
            w_duration=w_duration, w_align=w_align, w_dynamics=w_dynamics,
            w_speed=w_speed, w_path=w_path, w_start=w_start, **seqvec_inputs,
        )
        method = (k_method if k_method is not None
                  else ("leiden" if use_leiden_k else n_s))
        if method == "knn":
            strat, n_s = _leiden_knn_groups(D, multi, knn_k)
            Z    = linkage(squareform(np.clip(D, 0, None), checks=False), method=linkage_method)  # dendrogram
            stab = _knn_stability(D, knn_k)
            print(f"[STABILITY] K={n_s} : bootstrap ARI {stab:.2f} (> 0.75 good, < 0.5 fragile)")
            print(f"[CLUSTER] {Counter(strat.values())} | {desc}")
            return seqs, mono, multi, D, Z, strat, n_cl, desc, n_s
        if method == "leiden":
            k_opt, _ = _leiden_optimal_k(D, multi)
            if k_opt is not None:
                n_s = k_opt
                print(f"[LEIDEN] K selected : {n_s}")
        elif method == "silhouette":
            n_s, _ = _silhouette_optimal_k(D, k_range, linkage_method)
        else:
            try:
                n_s = int(method)
            except (TypeError, ValueError):
                raise ValueError(f"k_method must be 'knn', 'leiden', 'silhouette' or a number (got {k_method!r})")
            if k_method is not None:
                print(f"[K] set by hand : {n_s}")
        strat, Z = _cluster_hier(D, multi, n_s, linkage_method)
        stab = _ward_stability(D, n_s, linkage_method)
        print(f"[STABILITY] K={n_s} : bootstrap ARI {stab:.2f} (> 0.75 good, < 0.5 fragile)")
        print(f"[CLUSTER] {Counter(strat.values())} | {desc}")
        return seqs, mono, multi, D, Z, strat, n_cl, desc, n_s

    seqs, mono, multi, D, Z, strat, n_cl, desc, n_s_eff = compute_all(
        cluster_col, n_strategies)

    cur = {"col": cluster_col, "n_s": n_s_eff,
           "seqs": seqs, "mono": mono, "multi": multi,
           "D": D, "Z": Z, "strat": strat, "n_cl": n_cl, "desc": desc}

    # ── Figure ────────────────────────────────────────────────
    fig = plt.figure(figsize=(24, 12))
    fig.patch.set_facecolor("#1a1a2e")

    ax_dend = fig.add_axes([0.02, 0.22, 0.19, 0.72])
    ax_heat = fig.add_axes([0.23, 0.22, 0.19, 0.72])
    ax_seq  = fig.add_axes([0.45, 0.60, 0.53, 0.34])
    zone_r  = [0.45, 0.22, 0.53, 0.34]

    for ax in [ax_dend, ax_heat, ax_seq]:
        ax.set_facecolor("#0d0d1a")
        for sp in ax.spines.values(): sp.set_edgecolor("#333333")
        ax.tick_params(colors="#555555", labelsize=7)

    ax_s_k   = fig.add_axes([0.02, 0.11, 0.19, 0.03])
    ax_info  = fig.add_axes([0.02, 0.03, 0.60, 0.05])
    ax_tb    = fig.add_axes([0.66, 0.11, 0.14, 0.04])
    ax_bshow = fig.add_axes([0.81, 0.11, 0.08, 0.04])
    ax_bleid = fig.add_axes([0.66, 0.04, 0.23, 0.04])

    slider   = Slider(ax_s_k, "K groups", 2,
                      max(3, min(12, len(multi)-1) if len(multi) > 2 else 12),
                      valinit=n_s_eff, valstep=1, color="#4466aa")
    textbox  = TextBox(ax_tb,    "Animal ID: ", initial="",
                       color="#111122", hovercolor="#1a1a2e")
    btn_show = Button(ax_bshow, "Show",  color="#224422", hovercolor="#336633")
    btn_leid = Button(ax_bleid, "Leiden kNN groups",
                      color="#442244", hovercolor="#663366")

    for btn in [btn_show, btn_leid]:
        btn.label.set_color("#cccccc"); btn.label.set_fontsize(9)
    slider.label.set_color("white"); slider.valtext.set_color("white")
    textbox.label.set_color("#aaaaaa"); textbox.text_disp.set_color("#ffffff")

    ax_info.axis("off")
    info_txt = ax_info.text(
        0, 0.5,
        f"col={cluster_col} | seqvec | min_run_len={min_run_len} | "
        f"K={n_s_eff} | {len(multi)} multi + {len(mono)} MONO",
        color="#888888", fontsize=8, va="center",
        transform=ax_info.transAxes, fontfamily="monospace")

    radar_axes = []

    def remove_radars():
        for ax in radar_axes: ax.remove()
        radar_axes.clear()

    def update_info():
        info_txt.set_text(
            f"col={cur['col']} | seqvec | min_run_len={min_run_len} | "
            f"K={cur['n_s']} | {len(cur['multi'])} multi + {len(cur['mono'])} MONO")

    def draw():
        seqs  = cur["seqs"]; mono  = cur["mono"]; multi = cur["multi"]
        D     = cur["D"];    Z     = cur["Z"];    strat = cur["strat"]
        n_cl  = cur["n_cl"]; n_s   = cur["n_s"]

        strategy_list = sorted(set(strat.values())) if strat else []
        pal = {s: STRATEGY_PALETTE[i % len(STRATEGY_PALETTE)]
               for i, s in enumerate(strategy_list)}
        pal["MONO"] = "#666666"

        # ── Dendrogram ──────────────────────────────────────
        ax_dend.cla(); ax_dend.set_facecolor("#0d0d1a")
        for sp in ax_dend.spines.values(): sp.set_edgecolor("#333333")
        ax_dend.tick_params(colors="#555555", labelsize=6)
        if Z is not None and len(multi) > 2:
            cut_h = Z[-(n_s-1), 2]*1.01 if len(Z) >= n_s else Z[-1, 2]
            dendrogram(Z, ax=ax_dend, no_labels=True,
                       color_threshold=cut_h,
                       above_threshold_color="#444455")
            ax_dend.axhline(cut_h, color="#FFD700", lw=1.2, ls="--",
                            label=f"K={n_s}")
            ax_dend.legend(fontsize=7, labelcolor="white",
                           loc="upper right").get_frame().set_facecolor("#111111")
        ax_dend.set_title(
            f"Dendrogram\n{len(multi)} multi + {len(mono)} MONO",
            color="#aaaaaa", fontsize=8, pad=6)
        ax_dend.set_xlabel("Individuals", color="#555555", fontsize=7)
        ax_dend.set_ylabel("Distance",  color="#555555", fontsize=7)

        # ── Heatmap ───────────────────────────────────────────
        ax_heat.cla(); ax_heat.set_facecolor("#0d0d1a")
        for sp in ax_heat.spines.values(): sp.set_edgecolor("#333333")
        if D is not None and len(multi) > 0:
            sorted_ids = sorted(multi, key=lambda x: strat.get(x, "?"))
            idx_map    = {aid: i for i, aid in enumerate(multi)}
            order      = [idx_map[aid] for aid in sorted_ids]
            Ds         = D[np.ix_(order, order)]
            im = ax_heat.imshow(Ds, cmap="viridis", aspect="auto")
            ax_heat.set_xticks([]); ax_heat.set_yticks([])
            prev_s, xp = None, 0
            for aid in sorted_ids:
                s = strat.get(aid, "?")
                if s != prev_s and prev_s is not None:
                    ax_heat.axhline(xp-.5, color="white", lw=0.8, alpha=0.5)
                    ax_heat.axvline(xp-.5, color="white", lw=0.8, alpha=0.5)
                prev_s = s; xp += 1
            plt.colorbar(im, ax=ax_heat, fraction=0.03, pad=0.02,
                         label="Dist.").ax.tick_params(colors="white", labelsize=5)
        patches = ([mpatches.Patch(color=pal[s],
                    label=f"{s} (n={sum(1 for v in strat.values() if v==s)})")
                    for s in strategy_list]
                   + [mpatches.Patch(color="#666666",
                      label=f"MONO (n={len(mono)})")])
        ax_heat.legend(handles=patches, fontsize=6, loc="lower right",
                       framealpha=0.5).get_frame().set_facecolor("#111111")
        ax_heat.set_title(f"Distances\n{cur['desc'][:35]}",
                          color="#aaaaaa", fontsize=8, pad=6)

        # ── Radars ────────────────────────────────────────────
        remove_radars()
        all_s = strategy_list + (["MONO"] if mono else [])
        nc = min(5, len(all_s)); nr = (len(all_s)+nc-1)//nc
        w_r = zone_r[2]/nc; h_r = zone_r[3]/nr

        for si, s in enumerate(all_s):
            members = mono if s == "MONO" else \
                      [aid for aid, lab in strat.items() if lab == s]
            profs = []
            for aid in members:
                sq = seqs[aid]["seq"]; lng = seqs[aid]["lengths"]
                total = sum(lng)
                if total == 0: continue
                h = np.zeros(n_cl)
                for cl, l in zip(sq, lng):
                    if cl < n_cl: h[cl] += l / total
                profs.append(h)
            if not profs: continue
            mp = np.mean(profs, axis=0)
            ci = si % nc; ri = si // nc
            rect = [zone_r[0]+ci*w_r, zone_r[1]+(nr-1-ri)*h_r,
                    w_r*0.90, h_r*0.88]
            ax_r = fig.add_axes(rect, polar=True)
            ax_r.set_facecolor("#0d0d1a"); radar_axes.append(ax_r)
            angs = np.linspace(0, 2*np.pi, n_cl, endpoint=False)
            af   = np.concatenate([angs, [angs[0]]])
            vf   = np.concatenate([mp, [mp[0]]])
            col  = pal.get(s, "#888888")
            ax_r.plot(af, vf, lw=2, color=col)
            ax_r.fill(af, vf, alpha=0.3, color=col)
            ax_r.set_xticks(angs)
            ax_r.set_xticklabels([f"C{c}" for c in range(n_cl)],
                                  fontsize=4, color="#aaaaaa")
            ax_r.set_ylim(0, max(mp.max()*1.2, 0.01))
            ax_r.set_yticks([])
            ax_r.grid(color="#333333", lw=0.5)
            ax_r.spines["polar"].set_color("#333333")
            ax_r.set_title(f"{s} n={len(members)}",
                           color=col, fontsize=7, pad=4)

        fig.suptitle(
            f"Strategies | col={cur['col']} | SeqVec+ | "
            f"K={n_s} | {len(multi)} multi + {len(mono)} MONO",
            color="#cccccc", fontsize=10, y=0.99)
        update_info()
        fig.canvas.draw_idle()

    draw()

    # ── Individual timeline ─────────────────────────────────────
    def show_indiv(event=None):
        raw   = textbox.text.strip()
        seqs  = cur["seqs"]; strat = cur["strat"]
        strategy_list = sorted(set(strat.values())) if strat else []
        pal = {s: STRATEGY_PALETTE[i % len(STRATEGY_PALETTE)]
               for i, s in enumerate(strategy_list)}
        pal["MONO"] = "#666666"

        aid = str(raw) if raw in seqs else None
        if aid is None and raw.isdigit():
            aid_int = str(int(raw))
            if aid_int in seqs: aid = aid_int

        ax_seq.cla(); ax_seq.set_facecolor("#0d0d1a")
        for sp in ax_seq.spines.values(): sp.set_edgecolor("#333333")

        if aid is None:
            sample = list(seqs.keys())[:8]
            ax_seq.text(0.5, 0.5,
                f"'{raw}' not found\nExamples : {sample}",
                ha="center", va="center", color="#ff4444",
                transform=ax_seq.transAxes, fontsize=9)
            fig.canvas.draw_idle(); return

        sq  = seqs[aid]["seq"]; lng = seqs[aid]["lengths"]
        s   = strat.get(aid, "MONO"); col = pal.get(s, "#888888")
        ax_seq.set_title(
            f"Animal {aid}  —  Strategy {s}  |  "
            f"{len(sq)} blocks, {sum(lng)} frames",
            color=col, fontsize=9)
        ax_seq.set_yticks([])
        ax_seq.tick_params(colors="#555555", labelsize=7)

        x = 0
        for cluster, length in zip(sq, lng):
            c = PALETTE[cluster % len(PALETTE)]
            ax_seq.barh(0, length, left=x, height=0.6,
                        color=c, edgecolor="#111122", lw=0.4)
            if length >= max(lng) * 0.06:
                ax_seq.text(x + length/2, 0, str(cluster),
                            ha="center", va="center",
                            fontsize=7, color="white", fontweight="bold")
            x += length

        ax_seq.set_xlim(0, x); ax_seq.set_ylim(-0.5, 0.5)
        ax_seq.set_xlabel("Frames", color="#555555", fontsize=7)
        ax_seq.grid(axis="x", alpha=0.15, color="#334455")
        patches = [mpatches.Patch(color=PALETTE[c % len(PALETTE)],
                   label=f"C{c}") for c in sorted(set(sq))]
        ax_seq.legend(handles=patches, fontsize=6, loc="upper right",
                      framealpha=0.5).get_frame().set_facecolor("#111111")
        fig.canvas.draw_idle()

    # ── Events ────────────────────────────────────────────
    def on_leiden(event):
        if cur["D"] is None: return
        print("[LEIDEN kNN] Computing...")
        strat, k_opt = _leiden_knn_groups(cur["D"], cur["multi"], knn_k)
        cur["n_s"] = k_opt
        slider.eventson = False; slider.set_val(min(k_opt, slider.valmax)); slider.eventson = True
        cur.update({"strat": strat})
        remove_radars(); draw()

    def on_slider(val):
        cur["n_s"] = int(val)
        if cur["D"] is not None:
            strat, Z = _cluster_hier(cur["D"], cur["multi"],
                                      cur["n_s"], linkage_method)
            cur.update({"strat": strat, "Z": Z})
        remove_radars(); draw()

    slider.on_changed(on_slider)
    btn_show.on_clicked(show_indiv)
    btn_leid.on_clicked(on_leiden)
    textbox.on_submit(lambda t: show_indiv())

    plt.show()

    # ── Return final_state ─────────────────────────────────
    # ── Return final_state ─────────────────────────────────
    strat_final = cur["strat"] if cur["strat"] else {}
    mono_final  = cur["mono"]

    # Convert numbers → letters (A, B, C...)
    unique_groups   = sorted(set(v for v in strat_final.values()))
    num_to_letter   = {g: chr(65+i) for i, g in enumerate(unique_groups)}
    mapping_letters = {aid: num_to_letter.get(g, "MONO")
                       for aid, g in strat_final.items()}
    for m in mono_final:
        mapping_letters[str(m)] = "MONO"

    return {
        "seqs"           : cur["seqs"],
        "strat"          : strat_final,
        "mono"           : mono_final,
        "multi"          : cur["multi"],
        "n_cl"           : cur["n_cl"],
        "col"            : cur["col"],   # ← added
        "D"              : cur["D"],
        "Z"              : cur["Z"],
        "n_s"            : cur["n_s"],
        "desc"           : cur["desc"],
        "mapping_letters": mapping_letters,
        "seqvec_inputs"  : seqvec_inputs,
        "seqvec_weights" : dict(w_hist=w_hist, w_temporal=w_temporal, w_lastapp=w_lastapp,
                                w_duration=w_duration, w_align=w_align, w_dynamics=w_dynamics,
                                w_speed=w_speed, w_path=w_path, w_start=w_start),
        "knn_k"          : knn_k,
        "min_run_len"      : min_run_len,
        }




In [ ]:
%matplotlib qt
# ── Call ─────────────────────────────────────────────────────
final_state = show_strategy_explorer(
    df             = df,
    cluster_col    = "cluster",
    n_strategies   = 4,
    min_run_len      = 15,    # segments < 15 frames (0.25 s) ignored: removes state flickering
    use_leiden_k   = True,
    k_method       = "knn",          # "knn" (Leiden on kNN graph, recommended) | "leiden" | "silhouette" | a number
    knn_k          = 10,             # number of NEIGHBOURS per individual in the graph (NOT the number of groups)
    linkage_method = "ward",
    # ── SeqVec: weights of the 9 families ──
    w_hist         = 1.0,     # time spent in each state
    w_temporal     = 1.0,     # states at the start vs end of the sequence
    w_lastapp      = 1.0,     # states just before the strike
    w_duration     = 0.5,     # total duration (1 column)
    w_align        = 1.0,     # order of the states (alignment, close states = low cost)
    w_dynamics     = 1.0,     # evolution of the features (DTW), without the states
    w_speed        = 0.5,     # mean speed of the cuttlefish (with duration = distance travelled)
    w_path         = 1.0,     # total distance travelled by the cuttlefish (did it move a lot?)
    w_start        = 1.0,     # start distance to the prey (does it start close?)
    state_similarity = True,  # close states = similar (hist, temporal)
)



In [ ]:
%matplotlib inline
# ===============================================================
#  CELL — Diagnostics of the SeqVec families
#  Tests each family ALONE, then combinations, to see
#  which one separates the groups and which one adds noise.
# ===============================================================
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics import silhouette_score


def diagnose_seqvec_families(final_state, k_range=(3, 4, 5, 6),
                             linkage_method="ward"):
    """
    For each isolated SeqVec family (weight 1, the others at 0), rebuilds
    D, clusters, and measures the best silhouette over k_range.
    _seqvec must be defined in the notebook.
    """
    seqs  = final_state["seqs"]
    multi = final_state["multi"]
    n_cl  = final_state["n_cl"]

    families = ["hist", "temporal", "lastapp", "duration", "align", "dynamics", "speed", "path", "start"]

    def best_sil(D):
        Zl = linkage(squareform(np.clip(D, 0, None)), method=linkage_method)
        best = -1
        for k in k_range:
            lab = fcluster(Zl, k, criterion="maxclust")
            if len(set(lab)) < 2:
                continue
            s = silhouette_score(D, lab, metric="precomputed")
            best = max(best, s)
        return best

    print("=" * 58)
    print("SILHOUETTE PER SEQVEC FAMILY (isolated)")
    print("=" * 58)
    print(f"  {'family':<12}{'silhouette':>12}  verdict")
    print("  " + "-" * 40)

    scores = {}
    for fam in families:
        w = {f"w_{x}": (1.0 if x == fam else 0.0) for x in families}
        try:
            _, D, _ = _seqvec(seqs, multi, n_cl, **w,
                        **final_state.get("seqvec_inputs", {}))
            s = best_sil(D)
            scores[fam] = s
            verdict = ("SEPARATES well" if s > 0.25 else
                       "a little" if s > 0.10 else "noise")
            print(f"  {fam:<12}{s:>+12.3f}  {verdict}")
        except Exception as e:
            print(f"  {fam:<12}{'ERROR':>12}  {e}")

    # reference: everything at 1
    w_all = {f"w_{x}": 1.0 for x in families}
    _, D_all, _ = _seqvec(seqs, multi, n_cl, **w_all,
                        **final_state.get("seqvec_inputs", {}))
    s_all = best_sil(D_all)
    print("  " + "-" * 40)
    print(f"  {'ALL (1,1..)':<12}{s_all:>+12.3f}  current reference")

    # combination of the 3 best families alone
    top3 = sorted(scores, key=scores.get, reverse=True)[:3]
    w_top = {f"w_{x}": (1.0 if x in top3 else 0.0) for x in families}
    _, D_top, _ = _seqvec(seqs, multi, n_cl, **w_top,
                        **final_state.get("seqvec_inputs", {}))
    s_top = best_sil(D_top)
    print(f"  {'top-3 only':<12}{s_top:>+12.3f}  ({', '.join(top3)})")

    print("=" * 58)
    print("\nHow to read :")
    print("  - 'noise' families (< 0.10) -> set their weight to 0")
    print("  - if 'top-3 only' > 'ALL' -> the discarded families were harmful")
    return scores, s_all, s_top


scores, s_all, s_top = diagnose_seqvec_families(final_state)

In [ ]:
# ===============================================================
#  CELL — Why these groups? (characteristics of each strategy)
#  Three complementary answers, for each group:
#   1. behaviour: median values (real units) + deviation from the mean (z score)
#   2. SeqVec families that separate the group from the others (score 0 -> 1)
#   3. frame states over-represented in the group
#  Groups: mapping_optimal if it exists, otherwise final_state (as the other cells).
# ===============================================================
import io
import contextlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kruskal
from sklearn.metrics import silhouette_samples

_EX_FAMILIES = ["hist", "temporal", "lastapp", "align", "dynamics", "duration", "speed", "path", "start"]
_EX_SHORT = {"distance_to_shrimp": "dist", "attack_side": "side", "bearing_to_prey": "bearing",
             "cuttle_approach_speed": "approach", "orbit_speed": "orbit", "who_moves": "who"}


def _ex_descriptors(df, ids, seqs):
    """Behaviour descriptors per individual, in real units (frames before the strike)."""
    d = df.copy()
    d["animal_id"] = d["animal_id"].astype(str)
    d = d[d["animal_id"].isin(ids)]
    if "time_to_strike_s" in d:
        d = d[d["time_to_strike_s"] <= 0]
    g = d.groupby("animal_id")
    cols = {}
    if "time_to_strike_s" in d:
        cols["duration (s)"] = -g["time_to_strike_s"].min()
    if "speed_cuttle_mm_s" in d:
        cols["distance travelled (mm)"] = g["speed_cuttle_mm_s"].sum() / 60
        cols["speed (mm/s)"] = g["speed_cuttle_mm_s"].mean()
        cols["% still"] = g["speed_cuttle_mm_s"].apply(lambda s: (s < 2).mean() * 100)
    if "distance_mm" in d:
        cols["start distance (mm)"] = g["distance_mm"].first()
        cols["mean distance (mm)"] = g["distance_mm"].median()
    if "cuttle_approach_speed_mm_s" in d:
        cols["approach (mm/s)"] = g["cuttle_approach_speed_mm_s"].mean()
    if "orbit_speed_deg_s" in d:
        cols["orbit (°/s)"] = g["orbit_speed_deg_s"].mean()
    if "who_moves" in d:
        cols["who_moves"] = g["who_moves"].mean()
    if "attack_side" in d:
        cols["side (°)"] = g["attack_side"].median()
        last = d[d["time_to_strike_s"] >= -1] if "time_to_strike_s" in d else d
        cols["side at strike (°)"] = last.groupby("animal_id")["attack_side"].median()
    if "bearing_to_prey" in d:
        cols["bearing (°)"] = g["bearing_to_prey"].median()
    out = pd.DataFrame(cols).reindex(ids)
    out["nb segments"] = [len(seqs[a]["seq"]) if a in seqs else np.nan for a in ids]
    return out


def _ex_state_labels(df, col, features):
    """Short description of each state: the 2 features that deviate most from the mean of the states."""
    feats = [f for f in features if f in df.columns]
    m = df.groupby(col)[feats].median()
    z = (m - m.mean()) / (m.std() + 1e-12)
    lab = {}
    for s, row in z.iterrows():
        top = row.abs().sort_values(ascending=False).index[:2]
        lab[s] = ", ".join(f"{_EX_SHORT.get(f, f[:8])} {'↑' if row[f] > 0 else '↓'}" for f in top)
    return lab


def explain_groups(final_state=None, df=None, groups=None, top=4, figure=True):
    """
    Explains the strategy groups: typical behaviour, SeqVec families
    that separate them, over-represented states, and a one-line summary per group.
    top: number of parameters cited in the summary of each group.
    """
    G  = globals()
    fs = G.get("final_state") if final_state is None else final_state
    df = G["df"] if df is None else df
    if groups is None:
        groups, src = ((G["mapping_optimal"], "mapping_optimal") if "mapping_optimal" in G
                       else (fs["mapping_letters"], "final_state"))
    else:
        src = "groups="
    groups = {str(a): g for a, g in groups.items() if g != "MONO"}
    ids    = [str(a) for a in fs["multi"] if str(a) in groups]
    lab    = np.array([groups[a] for a in ids])
    names  = sorted(set(lab))
    # ── Misclassified individuals : silhouette SeqVec de chaque individu ──
    pos = {str(a): i for i, a in enumerate(fs["multi"])}
    ix  = [pos[a] for a in ids]
    Dall = np.asarray(fs["D"])[np.ix_(ix, ix)]
    sil = silhouette_samples(Dall, lab, metric="precomputed") if len(names) > 1 else np.zeros(len(ids))
    closest = []
    for i in range(len(ids)):
        other = {g: Dall[i, lab == g].mean() for g in names if g != lab[i]}
        closest.append(min(other, key=other.get) if other else "")
    indiv = pd.DataFrame({"group": lab, "silhouette": sil, "closest group": closest}, index=ids)
    indiv["misclassified"] = indiv["silhouette"] < 0
    print(f"[GROUPS] groups : {src} | {len(ids)} individuals | {len(names)} groups : "
          + ", ".join(f"{g} ({(lab == g).sum()})" for g in names))

    # ── 1. Behaviour ──
    X = _ex_descriptors(df, ids, fs["seqs"])
    med = X.groupby(lab).median().loc[names]
    med.loc["ALL"] = X.median()
    med.insert(0, "n", list(pd.Series(lab).value_counts().loc[names]) + [len(ids)])
    n_mis = indiv.groupby("group")["misclassified"].sum().reindex(names).fillna(0).astype(int)
    med.insert(1, "misclassified", list(n_mis) + [int(indiv["misclassified"].sum())])
    med.insert(2, "silhouette", list(indiv.groupby("group")["silhouette"].mean().reindex(names)) + [sil.mean()])
    Z = (X - X.mean()) / (X.std() + 1e-12)
    zg = Z.groupby(lab).mean().loc[names]                 # mean deviation of the group from the mean of all, in standard deviations
    pval = {c: kruskal(*[X[c][lab == g].dropna() for g in names]).pvalue for c in X.columns}
    sig = [c for c in X.columns if pval[c] < 0.05]
    print("\n" + "═" * 100 + "\n1. BEHAVIOUR — medians per group (real units)\n" + "═" * 100)
    print(med.round(1).to_string())
    print("\nDeviation from the mean (z score: +1 = one standard deviation above the mean of all individuals):")
    print(zg.round(2).to_string())
    print("\nParameters that do NOT differ between groups (Kruskal-Wallis p >= 0.05) : " + (", ".join(c for c in X.columns if c not in sig) or "—"))

    # ── 2. SeqVec families ──
    fam_score = pd.DataFrame(index=names, dtype=float)
    W0 = {f"w_{f}": 0.0 for f in _EX_FAMILIES}
    inp = fs.get("seqvec_inputs", {})
    for fam in _EX_FAMILIES:
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                _, Df, _ = G["_seqvec"](fs["seqs"], ids, fs["n_cl"], **dict(W0, **{f"w_{fam}": 1.0}), **inp)
        except Exception:
            continue
        for g in names:
            m = lab == g
            within  = Df[np.ix_(m, m)][np.triu_indices(m.sum(), 1)].mean() if m.sum() > 1 else np.nan
            between = Df[np.ix_(m, ~m)].mean()
            fam_score.loc[g, fam] = (between - within) / (between + 1e-12)
    print("\n" + "═" * 100 + "\n2. SEQVEC FAMILIES — which ones separate each group?\n" + "═" * 100)
    print("Score = (distance to the others - internal distance) / distance to the others (0 = no separation, 1 = perfect):")
    print(fam_score.round(2).to_string())

    # ── 3. States ──
    col = fs.get("col", "cluster")
    st_lab = _ex_state_labels(df, col, list(G.get("FEATURES", [])))
    share = pd.DataFrame({g: pd.Series(np.concatenate([np.repeat(fs["seqs"][a]["seq"], fs["seqs"][a]["lengths"])
                                                       for a in np.array(ids)[lab == g]])).value_counts(normalize=True)
                          for g in names}).fillna(0).T
    overall = pd.Series(np.concatenate([np.repeat(fs["seqs"][a]["seq"], fs["seqs"][a]["lengths"]) for a in ids])
                        ).value_counts(normalize=True)
    print("\n" + "═" * 100 + "\n3. STATES — the most present in each group (and their description)\n" + "═" * 100)
    states_txt = {}
    for g in names:
        ratio = (share.loc[g] / overall.reindex(share.columns).replace(0, np.nan)).dropna()
        best = [s for s in share.loc[g].sort_values(ascending=False).index if share.loc[g, s] > 0.10][:3]
        over = [s for s in ratio.sort_values(ascending=False).index if ratio[s] > 1.5 and share.loc[g, s] > 0.05][:2]
        states_txt[g] = best
        print(f"  {g} : " + " | ".join(f"C{s} {share.loc[g, s]:.0%} ({st_lab.get(s, '')})" for s in best)
              + (f"   over-represented : " + ", ".join(f"C{s} x{ratio[s]:.1f}" for s in over) if over else ""))

    # ── 4. Misclassified individuals ──
    print("\n" + "═" * 100 + "\n4. MISCLASSIFIED INDIVIDUALS (silhouette < 0: closer to another group)\n" + "═" * 100)
    print("ID (silhouette → group it is closest to); sorted from the most to the least misclassified:")
    mis_txt = {}
    for g in names:
        m = indiv[(indiv["group"] == g) & indiv["misclassified"]].sort_values("silhouette")
        mis_txt[g] = ", ".join(f"{a} ({r['silhouette']:+.2f} → {r['closest group']})" for a, r in m.iterrows())
        print(f"  {g} ({len(m)}/{(lab == g).sum()}) : {mis_txt[g] or '—'}")

    # ── 5. Summary ──
    print("\n" + "═" * 100 + "\n5. SUMMARY — what characterises each group\n" + "═" * 100)
    summary = {}
    for g in names:
        zz = zg.loc[g, sig].dropna()
        key = zz.abs().sort_values(ascending=False).index[:top]
        parts = []
        for c in key:
            arrow = ("↑↑" if zz[c] > 1 else "↑") if zz[c] > 0 else ("↓↓" if zz[c] < -1 else "↓")
            parts.append(f"{c} {arrow} {med.loc[g, c]:.1f}")
        fams = fam_score.loc[g].dropna().sort_values(ascending=False).index[:2]
        summary[g] = " | ".join(parts)
        print(f"  {g} (n={(lab == g).sum()}, misclassified {n_mis[g]}) : {summary[g]}")
        print(f"        mostly separated by : {', '.join(fams)}   |   typical states : "
              + ", ".join(f"C{s} ({st_lab.get(s, '')})" for s in states_txt[g]))

    if figure:
        fig, axes = plt.subplots(1, 2, figsize=(18, 0.55 * len(names) + 3),
                                 gridspec_kw={"width_ratios": [len(X.columns), len(_EX_FAMILIES) * 0.8]})
        ax = axes[0]
        im = ax.imshow(zg.values, cmap="RdBu_r", vmin=-1.5, vmax=1.5, aspect="auto")
        ax.set_xticks(range(len(zg.columns)), zg.columns, rotation=40, ha="right", fontsize=8)
        ax.set_yticks(range(len(names)), [f"{g} (n={(lab == g).sum()})" for g in names])
        for i, g in enumerate(names):
            for j, c in enumerate(zg.columns):
                ax.text(j, i, f"{med.loc[g, c]:.0f}" if abs(med.loc[g, c]) >= 10 else f"{med.loc[g, c]:.1f}",
                        ha="center", va="center", fontsize=7,
                        color="white" if abs(zg.loc[g, c]) > 0.9 else "black",
                        weight="bold" if c in sig else "normal")
        ax.set_title("Behaviour: colour = deviation from the mean (z), text = median (bold = differs between groups)", fontsize=10)
        fig.colorbar(im, ax=ax, shrink=0.8, label="z score")
        ax = axes[1]
        im2 = ax.imshow(fam_score.values, cmap="Greens", vmin=0, vmax=max(0.3, np.nanmax(fam_score.values)),
                        aspect="auto")
        ax.set_xticks(range(len(fam_score.columns)), fam_score.columns, rotation=40, ha="right", fontsize=8)
        ax.set_yticks(range(len(names)), names)
        for i in range(len(names)):
            for j in range(len(fam_score.columns)):
                v = fam_score.values[i, j]
                ax.text(j, i, "—" if np.isnan(v) else f"{v:.2f}", ha="center", va="center", fontsize=7)
        ax.set_title("SeqVec families that separate each group", fontsize=10)
        fig.colorbar(im2, ax=ax, shrink=0.8)
        fig.tight_layout()
        plt.show()

    out_dir = G.get("OUT_DIR")
    if out_dir is not None:
        med.to_csv(out_dir / "groups_medians.csv")
        zg.to_csv(out_dir / "groups_zscores.csv")
        fam_score.to_csv(out_dir / "groups_families.csv")
        indiv.sort_values(["group", "silhouette"]).to_csv(out_dir / "groups_individuals.csv")
        print(f"\n[GROUPS] tables saved in {out_dir}")
    return {"medians": med, "zscores": zg, "families": fam_score, "states": share,
            "individuals": indiv, "misclassified": indiv[indiv["misclassified"]].sort_values(["group", "silhouette"]),
            "summary": summary, "pvalues": pval}


print("[OK] Group explanation defined — usage :  explication = explain_groups()")


In [ ]:
%matplotlib inline
# Why these groups? (run after the explorer, and the regrouping if used)
explication = explain_groups(
    top = 4,          # number of parameters cited in the summary of each group
)


In [ ]:
# ===============================================================
#  CELL — Visualisation of the SeqVec groups + biological audit
#  Replays EXACTLY the _seqvec of show_strategy_explorer, projects
#  in 2D, colours by group, and measures whether the groups hold.
# ===============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_samples, silhouette_score
from scipy.spatial.distance import pdist, squareform


def visualize_strategy_groups(final_state, method="mds",
                              recompute_seqvec=None, annotate=True):
    """
    final_state : output of show_strategy_explorer.
    method      : "mds"  -> projects the clustering D matrix (most faithful)
                  "umap" -> UMAP on the SeqVec features
                  "pca"  -> PCA
    recompute_seqvec : dict of weights {w_hist:..} to REPLAY _seqvec with
                       other weightings and see the effect on the groups.
                       None = reuses the already computed D.
    """
    seqs  = final_state["seqs"]
    multi = final_state["multi"]
    strat = final_state["strat"]
    n_cl  = final_state["n_cl"]
    ml    = final_state.get("mapping_letters", {})

    ids = [str(a) for a in multi]
    grp = np.array([ml.get(a, str(strat.get(a, "?"))) for a in ids])

    # ── Distance: existing D, or recomputed with other weights ─
    if recompute_seqvec is not None:
        # replays _seqvec (must be importable in the notebook)
        _, D, _ = _seqvec(seqs, multi, n_cl, **recompute_seqvec,
                        **final_state.get("seqvec_inputs", {}))
        title_w = "  ·  modified weights"
    else:
        D = np.asarray(final_state["D"])
        title_w = ""

    # ── 2D projection ──────────────────────────────────────────
    if method == "mds":
        from sklearn.manifold import MDS
        Z = MDS(n_components=2, dissimilarity="precomputed",
                random_state=42, normalized_stress="auto").fit_transform(D)
        tm = "MDS on D (actual clustering distance)"
        Xsil = D                     # silhouette on the precomputed distance
        sil_metric = "precomputed"
    elif method == "umap":
        import umap
        # starting from the features: rebuilding X from D is impossible,
        # so D is used as input with metric='precomputed'
        Z = umap.UMAP(n_components=2, metric="precomputed", random_state=42,
                      n_neighbors=min(15, len(D)-1)).fit_transform(D)
        tm = "UMAP on D"; Xsil = D; sil_metric = "precomputed"
    else:
        from sklearn.decomposition import PCA
        # PCA needs coordinates: classical MDS first
        from sklearn.manifold import MDS
        coords = MDS(n_components=min(10, len(D)-1), dissimilarity="precomputed",
                     random_state=42, normalized_stress="auto").fit_transform(D)
        Z = PCA(n_components=2).fit_transform(coords)
        tm = "PCA (via 10D MDS)"; Xsil = D; sil_metric = "precomputed"

    # ── Silhouette (do the groups hold?) ────────────────
    lab_num = pd.factorize(grp)[0]
    counts = pd.Series(lab_num).value_counts()
    valid = np.isin(lab_num, counts[counts >= 2].index)
    sil_ind = np.full(len(lab_num), np.nan)
    sil_glob = np.nan
    if valid.sum() >= 3 and len(set(lab_num[valid])) >= 2:
        sil_ind[valid] = silhouette_samples(Xsil[np.ix_(valid, valid)],
                                             lab_num[valid], metric=sil_metric)
        sil_glob = silhouette_score(Xsil[np.ix_(valid, valid)],
                                    lab_num[valid], metric=sil_metric)

    # ── Colours ───────────────────────────────────────────────
    gs = sorted(set(grp))
    pal = {g: STRATEGY_PALETTE[i % len(STRATEGY_PALETTE)] for i, g in enumerate(gs)}
    cols = [pal[g] for g in grp]

    # ══ FIGURE ═════════════════════════════════════════════════
    fig = plt.figure(figsize=(17, 6))
    fig.patch.set_facecolor("#1a1a2e")

    # P1 — coloured projection
    ax = fig.add_subplot(1, 3, 1); ax.set_facecolor("#0d0d1a")
    ax.scatter(Z[:, 0], Z[:, 1], c=cols, s=110, edgecolor="white",
               linewidth=0.6, zorder=3)
    if annotate:
        for (x, y), a in zip(Z, ids):
            ax.text(x, y, a, fontsize=6, color="#eee", ha="center", va="center")
    for g in gs:
        m = grp == g
        ax.scatter(Z[m, 0].mean(), Z[m, 1].mean(), marker="X", s=220,
                   c=pal[g], edgecolor="black", linewidth=1.5, zorder=5)
    ax.set_title(f"{tm}{title_w}", color="#ccc", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values(): sp.set_edgecolor("#333")
    ax.legend(handles=[mpatches.Patch(color=pal[g],
              label=f"{g} (n={int((grp==g).sum())})") for g in gs],
              fontsize=7, framealpha=0.3)

    # P2 — silhouette per individual, sorted by group
    ax2 = fig.add_subplot(1, 3, 2); ax2.set_facecolor("#0d0d1a")
    order = sorted(range(len(ids)),
                   key=lambda i: (grp[i], -(sil_ind[i] if not np.isnan(sil_ind[i]) else -9)))
    ax2.barh(range(len(order)), sil_ind[order],
             color=[cols[i] for i in order])
    ax2.axvline(0, color="#888", lw=1)
    if not np.isnan(sil_glob):
        ax2.axvline(sil_glob, color="#FFD700", lw=1.5, ls="--",
                    label=f"mean {sil_glob:.2f}")
        ax2.legend(fontsize=8, framealpha=0.3)
    ax2.set_yticks([]); ax2.set_xlim(-0.6, 1)
    ax2.set_xlabel("silhouette (negative = misclassified)", color="#aaa", fontsize=8)
    ax2.set_title("Cohesion per individual", color="#ccc", fontsize=9)
    ax2.tick_params(colors="#aaa")
    for sp in ax2.spines.values(): sp.set_edgecolor("#333")

    # P3 — time-per-cluster profile of each group
    ax3 = fig.add_subplot(1, 3, 3); ax3.set_facecolor("#0d0d1a")
    for g in gs:
        profs = []
        for a in np.array(ids)[grp == g]:
            sq, lg = seqs[a]["seq"], seqs[a]["lengths"]
            t = sum(lg)
            if t == 0: continue
            h = np.zeros(n_cl)
            for c, l in zip(sq, lg):
                if c < n_cl: h[c] += l / t
            profs.append(h)
        if profs:
            ax3.plot(range(n_cl), np.mean(profs, 0), "-o", color=pal[g],
                     label=g, lw=2, markersize=4)
    ax3.set_xticks(range(n_cl))
    ax3.set_xticklabels([f"C{c}" for c in range(n_cl)], fontsize=7, color="#aaa")
    ax3.set_ylabel("proportion of time", color="#aaa", fontsize=8)
    ax3.set_title("Mean profile per group", color="#ccc", fontsize=9)
    ax3.tick_params(colors="#aaa"); ax3.legend(fontsize=7, framealpha=0.3)
    for sp in ax3.spines.values(): sp.set_edgecolor("#333")

    plt.tight_layout(); plt.show()

    # ── Summary ─────────────────────────────────────────────────
    print(f"\n{'='*56}")
    print(f"QUALITY — {len(gs)} groups | global silhouette {sil_glob:+.3f} "
          f"({'solid' if sil_glob>0.25 else 'weak' if sil_glob>0 else 'inconsistent'})")
    print(f"{'='*56}")
    print(f"  {'group':<8}{'n':>4}{'silhouette':>12}  misclassified")
    for g in gs:
        m = grp == g
        s = sil_ind[m]; s = s[~np.isnan(s)]
        neg = int((s < 0).sum())
        print(f"  {g:<8}{int(m.sum()):>4}{(np.mean(s) if len(s) else 0):>+12.3f}"
              f"     {neg}/{int(m.sum())}")
    bad = sorted([(ids[i], grp[i], sil_ind[i]) for i in range(len(ids))
                  if not np.isnan(sil_ind[i]) and sil_ind[i] < 0],
                 key=lambda x: x[2])
    if bad:
        print(f"\n  Misclassified individuals (candidates for reclassifying/merging) :")
        for a, g, s in bad[:10]:
            print(f"    animal {a:<6} group {g:<4} sil={s:+.3f}")
    print("=" * 56)
    return Z, grp, sil_ind


# ── Usage ────────────────────────────────────────────────
# Z_mds, grp, sil = visualize_strategy_groups(final_state, method="mds")
#
# Test other weights WITHOUT rerunning the explorer:
# Z_mds, grp, sil = visualize_strategy_groups(final_state, method="mds",
#     recompute_seqvec=dict(w_hist=2.0, w_temporal=1.0, w_lastapp=1.0,
#                           w_duration=0.5, w_align=1.0, w_dynamics=1.0, w_speed=0.5,
#                           w_path=0.5, w_start=0.5))

In [ ]:
Z_mds, grp, sil = visualize_strategy_groups(final_state, method="mds")

## Continuous strategies — map of the individuals + Leiden groups

Alternative approach to SeqVec (the previous cells are unchanged):

1. **Profiles**: for each individual, % of time spent in each cluster. *Soft* version: two clusters that are close in feature space count as similar.
2. **A — Continuous map**: PCA of the profiles, 1 point = 1 individual, axes named after the clusters that vary along them.
3. **B — Groups**: Leiden (modularity) on the nearest-neighbour graph between individuals. The number of groups emerges by itself, the number of neighbours is chosen automatically by stability. No merging by size.
4. **D — Start / end**: what each group does in the 1st vs 2nd half of the sequence.
5. **Bridge**: `mapping_cont` has the same format as `mapping_optimal` to reuse the existing analyses.

Variables created: `profiles_cont`, `df_profiles`, `coords_cont`, `groups_cont`, `mapping_cont`, `info_cont`.

In [ ]:
# ===============================================================
#  CELL — Individual profiles (continuous map + Leiden groups)
#  1 row per individual: % of time spent in each cluster,
#  over the whole sequence, then separately at the start / end (halves).
#  "Soft" version: two clusters that are close in feature space
#  count as similar (C6 -> C8 is not a real change).
#  Prerequisites: df with the cluster column, FEATURES, FEATURE_WEIGHTS
# ===============================================================
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform

CONT_CLUSTER_COL = "cluster"   # "cluster" (fine) or "supercluster" (coarse)
CONT_SOFT        = True        # True = similarity between clusters taken into account


def build_individual_profiles(df, cluster_col="cluster", soft=True,
                              features=FEATURES, feature_weights=None):
    d = df.copy()
    d["animal_id"] = d["animal_id"].astype(str)
    d = d.sort_values(["animal_id", "frame"])
    labels = sorted(d[cluster_col].unique())

    def _comp(sub):
        return (pd.crosstab(sub["animal_id"], sub[cluster_col], normalize="index")
                  .reindex(columns=labels, fill_value=0.0))

    comp = _comp(d)
    # relative position of each frame in the sequence of its individual
    rel   = d.groupby("animal_id").cumcount() / d.groupby("animal_id")["frame"].transform("size")
    start = _comp(d[rel < 0.5]).reindex(comp.index, fill_value=0.0)
    end   = _comp(d[rel >= 0.5]).reindex(comp.index, fill_value=0.0)

    # ── Similarity between clusters: centroids in the standardised
    #    + weighted space (the same as Leiden/UMAP)
    w    = np.array([(feature_weights or {}).get(f, 1.0) for f in features])
    zs   = (d[features] - d[features].mean()) / d[features].std()
    cent = zs.groupby(d[cluster_col]).mean().loc[labels].values * w
    dc   = squareform(pdist(cent))
    if soft:
        sigma = np.median(dc[dc > 0]) / 2
        K = np.exp(-dc ** 2 / (2 * sigma ** 2))
    else:
        K = np.eye(len(labels))
    # square root of K: the soft distance (p-q)ᵀK(p-q) becomes Euclidean
    # on p @ K½, so PCA / kNN / Leiden work as they are
    ev, evec = np.linalg.eigh(K)
    K_half = evec @ np.diag(np.sqrt(np.clip(ev, 0, None))) @ evec.T

    # readable description of each cluster (raw means)
    cent_raw = d.groupby(cluster_col)[features].mean().loc[labels]
    pfx  = "C" if cluster_col == "cluster" else "S"
    name = {c: f"{pfx}{c}" for c in labels}

    n_frames = d.groupby("animal_id").size().loc[comp.index]
    print(f"[PROFILES] {len(comp)} individuals × {len(labels)} {cluster_col}s "
          f"| soft similarity={'yes' if soft else 'no'}"
          + (f" (σ={sigma:.2f})" if soft else ""))
    return dict(comp=comp, start=start, end=end, K=K, K_half=K_half,
                labels=labels, names=name, cent_raw=cent_raw,
                cluster_col=cluster_col, n_frames=n_frames, dist_clusters=dc)


profiles_cont = build_individual_profiles(df, CONT_CLUSTER_COL, CONT_SOFT,
                                          FEATURES, FEATURE_WEIGHTS)
df_profiles = profiles_cont["comp"].rename(columns=profiles_cont["names"]).mul(100).round(1)
df_profiles.insert(0, "n_frames", profiles_cont["n_frames"])
df_profiles.head()

In [ ]:
# ===============================================================
#  CELL — A. Continuous map of the individuals (PCA of the profiles)
#  Each point = one individual. The axes are named after the
#  clusters whose share of time varies most along the axis.
# ===============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

CONT_N_AXES = 3


def _cluster_desc(P, c):
    """E.g. 'C1 (v=0.93, angle=76°)' — mean speed and angle of the cluster."""
    r = P["cent_raw"].loc[c]
    v = r.get("speed_cuttle_smooth", r.get("speed_cuttle", np.nan))
    a = r.get("approach_angle", np.nan)
    return f"{P['names'][c]} (v={v:.2f}, angle={a:.0f}°)"


def continuous_map(P, n_axes=3):
    comp = P["comp"]
    E    = comp.values @ P["K_half"]            # profiles in the "soft" space
    pca  = PCA(n_components=n_axes).fit(E)
    S    = pca.transform(E)
    var  = pca.explained_variance_ratio_

    # correlation between each axis and the share of time of each cluster
    corr = pd.DataFrame({f"Axis {i + 1}": [np.corrcoef(S[:, i], comp[c])[0, 1]
                                          if comp[c].std() > 0 else 0.0
                                          for c in P["labels"]]
                         for i in range(n_axes)},
                        index=[_cluster_desc(P, c) for c in P["labels"]])
    # orientation: the most correlated cluster is always on the + side
    for i in range(n_axes):
        col = f"Axis {i + 1}"
        if corr[col].iloc[corr[col].abs().argmax()] < 0:
            S[:, i] *= -1; corr[col] *= -1

    axis_names = []
    for i in range(n_axes):
        c = corr[f"Axis {i + 1}"].sort_values()
        plus  = ", ".join(n.split(" ")[0] for n in c.index[::-1][:2])
        minus = ", ".join(n.split(" ")[0] for n in c.index[:2])
        axis_names.append(f"Axis {i + 1} ({var[i]:.0%}) : + {plus}  /  − {minus}")

    coords = pd.DataFrame(S, index=comp.index,
                          columns=[f"axis{i + 1}" for i in range(n_axes)])
    return coords, corr, axis_names, var


coords_cont, axes_corr, axes_names, axes_var = continuous_map(profiles_cont, CONT_N_AXES)
print("[MAP] explained variance :",
      " | ".join(f"axis{i + 1} {v:.0%}" for i, v in enumerate(axes_var)),
      f"(total {axes_var.sum():.0%})")
for n in axes_names:
    print("   ", n)

fig, ax = plt.subplots(1, 2, figsize=(16, 6.5), gridspec_kw={"width_ratios": [1.25, 1]})
fig.patch.set_facecolor("#1a1a2e")
for a in ax:
    a.set_facecolor("#0d0d1a"); a.tick_params(colors="#aaaaaa")
    for sp in a.spines.values(): sp.set_edgecolor("#333333")

nf = np.log10(profiles_cont["n_frames"].values)
sc = ax[0].scatter(coords_cont["axis1"], coords_cont["axis2"], c=nf, cmap="viridis",
                   s=45, edgecolor="white", linewidth=0.4)
for aid, (x, y) in coords_cont[["axis1", "axis2"]].iterrows():
    ax[0].text(x, y, aid, fontsize=5, color="#bbbbbb", ha="center", va="bottom")
cb = fig.colorbar(sc, ax=ax[0], fraction=0.04)
cb.set_label("log10(number of frames)", color="#aaaaaa"); cb.ax.tick_params(colors="#aaaaaa")
ax[0].set_xlabel(axes_names[0], color="#cccccc", fontsize=9)
ax[0].set_ylabel(axes_names[1], color="#cccccc", fontsize=9)
ax[0].set_title("Continuous map of the individuals (1 point = 1 individual)", color="#cccccc")

im = ax[1].imshow(axes_corr.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax[1].set_xticks(range(axes_corr.shape[1])); ax[1].set_xticklabels(axes_corr.columns, color="#cccccc")
ax[1].set_yticks(range(len(axes_corr))); ax[1].set_yticklabels(axes_corr.index, fontsize=7, color="#cccccc")
for i in range(axes_corr.shape[0]):
    for j in range(axes_corr.shape[1]):
        v = axes_corr.values[i, j]
        ax[1].text(j, i, f"{v:+.2f}", ha="center", va="center", fontsize=6.5,
                   color="white" if abs(v) > 0.5 else "black")
cb2 = fig.colorbar(im, ax=ax[1], fraction=0.04)
cb2.set_label("correlation axis ↔ % of time", color="#aaaaaa"); cb2.ax.tick_params(colors="#aaaaaa")
ax[1].set_title("What each axis represents", color="#cccccc")
plt.tight_layout(); plt.show()

In [ ]:
# ===============================================================
#  CELL — B. Groups of individuals by Leiden (automatic K)
#  Graph of the k nearest neighbours between individuals (soft profiles),
#  Leiden with classic modularity: no resolution to tune,
#  the number of groups emerges by itself. 50 seeds -> consensus
#  partition. No merging by size.
# ===============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import igraph as ig
import leidenalg
from scipy import sparse
from sklearn.neighbors import kneighbors_graph
from sklearn.metrics import adjusted_rand_score, silhouette_score

CONT_K_NEIGHBORS      = "auto"            # "auto" = the most stable among the candidates, or an integer
CONT_K_CANDIDATES     = (10, 15, 20, 30)  # graph neighbours: smaller = finer groups
CONT_INCLUDE_TEMPORAL = False             # True = the distance also accounts for start / end (D)
CONT_N_RUNS           = 50


def profile_matrix(P, include_temporal=False):
    """Individuals × dimensions matrix used for the distance."""
    Kh = P["K_half"]
    F  = P["comp"].values @ Kh
    if include_temporal:
        # global + start + end, the temporal part weighs as much as the global one
        F = np.hstack([F, P["start"].values @ Kh * np.sqrt(.5),
                          P["end"].values   @ Kh * np.sqrt(.5)])
    return F


def _leiden_on(F, k, seed):
    A = kneighbors_graph(F, n_neighbors=min(k, len(F) - 1), mode="distance")
    A = sparse.triu(A.maximum(A.T), k=1).tocoo()
    sig = np.median(A.data) + 1e-12
    g = ig.Graph(n=len(F), edges=np.column_stack([A.row, A.col]).tolist(), directed=False)
    g.es["weight"] = np.exp(-A.data ** 2 / (2 * sig ** 2)).tolist()
    return np.array(leidenalg.find_partition(
        g, leidenalg.ModularityVertexPartition, weights="weight", seed=seed).membership)


def leiden_groups(F, ids, k=10, n_runs=50, n_boot=30, seed=0):
    runs = [_leiden_on(F, k, seed + r) for r in range(n_runs)]
    # consensus partition = the one most similar to all the others
    ari  = np.array([[adjusted_rand_score(a, b) for b in runs] for a in runs])
    best = int(ari.mean(1).argmax())
    lab  = runs[best]

    # stability: remove 20 % of the individuals and check that the groups hold
    rng, boots = np.random.default_rng(seed), []
    for b in range(n_boot):
        sub = np.sort(rng.choice(len(F), int(.8 * len(F)), replace=False))
        boots.append(adjusted_rand_score(lab[sub], _leiden_on(F[sub], k, seed + 1000 + b)))

    # letters by decreasing size (A = the largest group)
    order  = pd.Series(lab).value_counts().index
    letter = {g: chr(65 + i) for i, g in enumerate(order)}
    groups = pd.Series([letter[g] for g in lab], index=ids, name="group")

    n_k  = [len(set(r)) for r in runs]
    sil  = silhouette_score(F, lab) if len(set(lab)) > 1 else np.nan
    info = dict(k_found=len(order), k_range=(min(n_k), max(n_k)),
                seed_agreement=float(ari.mean(1)[best]),
                bootstrap_ARI=float(np.mean(boots)), silhouette=float(sil))
    return groups, info


F_cont   = profile_matrix(profiles_cont, CONT_INCLUDE_TEMPORAL)
ids_cont = profiles_cont["comp"].index

# ── Number of neighbours: chosen automatically by stability ───
if CONT_K_NEIGHBORS == "auto":
    print("[GROUPS] choosing the number of neighbours (bootstrap stability) :")
    scan = []
    for k in CONT_K_CANDIDATES:
        g_k, i_k = leiden_groups(F_cont, ids_cont, k, n_runs=20, n_boot=20)
        scan.append((k, i_k))
        print(f"   k={k:<3} -> {i_k['k_found']:>2} groups | stability {i_k['bootstrap_ARI']:.2f}"
              f" | silhouette {i_k['silhouette']:.2f}")
    # the most stable; if tied (2 decimals), the finest
    k_used = max(scan, key=lambda t: (round(t[1]["bootstrap_ARI"], 2), -t[0]))[0]
    print(f"   -> k selected = {k_used}")
else:
    k_used = int(CONT_K_NEIGHBORS)

groups_cont, info_cont = leiden_groups(F_cont, ids_cont, k_used, CONT_N_RUNS)
info_cont["k_neighbors"] = k_used
mapping_cont = groups_cont.to_dict()            # same format as mapping_optimal
df_profiles["group"] = groups_cont
for c in coords_cont.columns:
    df_profiles[c] = coords_cont[c].round(3)

print(f"[GROUPS] Leiden (k={k_used} neighbours) : {info_cont['k_found']} groups found "
      f"(between {info_cont['k_range'][0]} and {info_cont['k_range'][1]} depending on the seed)")
print(f"  agreement between seeds : {info_cont['seed_agreement']:.2f}  (1 = always identical)")
print(f"  stability (bootstrap)   : {info_cont['bootstrap_ARI']:.2f}  (> 0.75 good, < 0.5 fragile)")
print(f"  silhouette              : {info_cont['silhouette']:.2f}")
print(f"  sizes                   : {groups_cont.value_counts().sort_index().to_dict()}")

# ── Figures: coloured map + mean profile of each group ────
gl  = sorted(groups_cont.unique())
pal = (STRATEGY_PALETTE if "STRATEGY_PALETTE" in globals()
       else [plt.cm.tab10(i) for i in range(10)])
gcol = {g: pal[i % len(pal)] for i, g in enumerate(gl)}

fig, ax = plt.subplots(1, 2, figsize=(17, 6.5), gridspec_kw={"width_ratios": [1.1, 1]})
fig.patch.set_facecolor("#1a1a2e")
for a in ax:
    a.set_facecolor("#0d0d1a"); a.tick_params(colors="#aaaaaa")
    for sp in a.spines.values(): sp.set_edgecolor("#333333")
for g in gl:
    m = (groups_cont == g).values
    ax[0].scatter(coords_cont["axis1"][m], coords_cont["axis2"][m], s=45, color=gcol[g],
                  edgecolor="white", linewidth=0.4, label=f"{g} (n={m.sum()})")
    ax[0].text(coords_cont["axis1"][m].mean(), coords_cont["axis2"][m].mean(), g,
               fontsize=14, fontweight="bold", color="white", ha="center", va="center",
               bbox=dict(facecolor=gcol[g], alpha=0.85, edgecolor="black", boxstyle="round"))
ax[0].set_xlabel(axes_names[0], color="#cccccc", fontsize=9)
ax[0].set_ylabel(axes_names[1], color="#cccccc", fontsize=9)
ax[0].set_title(f"Leiden groups on the continuous map — {info_cont['k_found']} groups, "
                f"stability {info_cont['bootstrap_ARI']:.2f}", color="#cccccc")
ax[0].legend(fontsize=8, facecolor="#111111", labelcolor="#cccccc", edgecolor="#333333")

prof = (profiles_cont["comp"].groupby(groups_cont).mean() * 100).loc[gl]
prof.columns = [_cluster_desc(profiles_cont, c) for c in profiles_cont["labels"]]
im = ax[1].imshow(prof.T.values, cmap="magma", aspect="auto")
ax[1].set_xticks(range(len(gl)))
ax[1].set_xticklabels([f"{g}\nn={(groups_cont == g).sum()}" for g in gl], color="#cccccc")
ax[1].set_yticks(range(prof.shape[1])); ax[1].set_yticklabels(prof.columns, fontsize=7, color="#cccccc")
for i in range(prof.shape[1]):
    for j in range(len(gl)):
        v = prof.values[j, i]
        ax[1].text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=7,
                   color="black" if v > prof.values.max() * 0.6 else "white")
cb = fig.colorbar(im, ax=ax[1], fraction=0.04)
cb.set_label("mean % of time", color="#aaaaaa"); cb.ax.tick_params(colors="#aaaaaa")
ax[1].set_title("Mean profile of each group", color="#cccccc")
plt.tight_layout(); plt.show()

In [ ]:
# ===============================================================
#  CELL — D. Start vs end of the sequence, per group
#  For each group: % of time in each cluster during the
#  1st half (approach) and the 2nd half (before the strike).
#  To include this split in the distance: CONT_INCLUDE_TEMPORAL = True
#  in cell B, then rerun B and D.
# ===============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

P   = profiles_cont
gl  = sorted(groups_cont.unique())
st  = (P["start"].groupby(groups_cont).mean() * 100).loc[gl]
en  = (P["end"].groupby(groups_cont).mean() * 100).loc[gl]
dif = en - st
names = [P["names"][c] for c in P["labels"]]

fig, ax = plt.subplots(1, 3, figsize=(18, max(3.5, len(gl) * 0.7 + 1.5)),
                       gridspec_kw={"width_ratios": [1, 1, 1]})
fig.patch.set_facecolor("#1a1a2e")
for a, M, title, cmap, sym in [(ax[0], st, "Start (1st half)", "magma", False),
                               (ax[1], en, "End (2nd half)", "magma", False),
                               (ax[2], dif, "End − start (% points)", "RdBu_r", True)]:
    a.set_facecolor("#0d0d1a")
    vmax = np.abs(M.values).max() if sym else max(st.values.max(), en.values.max())
    im = a.imshow(M.values, cmap=cmap, aspect="auto",
                  vmin=-vmax if sym else 0, vmax=vmax)
    a.set_xticks(range(len(names))); a.set_xticklabels(names, fontsize=7, color="#cccccc")
    a.set_yticks(range(len(gl)))
    a.set_yticklabels([f"{g} (n={(groups_cont == g).sum()})" for g in gl], color="#cccccc")
    for i in range(len(gl)):
        for j in range(len(names)):
            v = M.values[i, j]
            if abs(v) >= 1:
                a.text(j, i, f"{v:+.0f}" if sym else f"{v:.0f}", ha="center", va="center",
                       fontsize=6.5,
                       color=(("white" if abs(v) > vmax * 0.5 else "black") if sym
                              else ("black" if v > vmax * 0.6 else "white")))
    a.set_title(title, color="#cccccc")
    fig.colorbar(im, ax=a, fraction=0.04).ax.tick_params(colors="#aaaaaa")
fig.suptitle("What the groups do at the start vs the end of the sequence", color="#cccccc")
plt.tight_layout(); plt.show()

print("Largest changes start -> end, per group :")
for g in gl:
    s = dif.loc[g].sort_values()
    up   = ", ".join(f"{names[P['labels'].index(c)]} {v:+.0f}" for c, v in s[::-1][:2].items() if v > 2)
    down = ", ".join(f"{names[P['labels'].index(c)]} {v:+.0f}" for c, v in s[:2].items() if v < -2)
    print(f"  {g} : ↑ {up or '—'}   |   ↓ {down or '—'}")

In [ ]:
# ===============================================================
#  CELL — Bridge to the existing analyses (optional)
#  mapping_cont = {animal_id: "A"/"B"/...}: same format as
#  mapping_optimal. Uncomment to run the following cells
#  (profiles, power, comparative plots, catch_type…)
#  on the new groups. Keeps a copy of the old mapping.
# ===============================================================
# mapping_optimal_seqvec = dict(mapping_optimal)   # backup of the old one
# mapping_optimal        = dict(mapping_cont)

# Summary table: 1 row per individual (profile, coordinates, group)
# df_profiles.to_csv(OUT_DIR / "individual_profiles_continuous.csv")
df_profiles.sort_values("group").head(10)

In [ ]:
# CELL — show_power_analysis + optimal_regrouping
#  Computations on the SeqVec features (consistent with the clustering)
# ══════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from scipy.stats import mannwhitneyu
from itertools import combinations
from collections import Counter


def build_seqvec_features_per_individual(final_state, normalize=False):
    """
    Rebuilds the SeqVec features for each individual.
    Used for the statistical tests → normalize=False (raw values).
    Consistent with the _seqvec used for clustering.
    """
    seqs  = final_state["seqs"]
    n_cl  = final_state["n_cl"]
    ids   = list(seqs.keys())
    rows  = []

    for aid in ids:
        sq, lg = seqs[aid]["seq"], seqs[aid]["lengths"]
        total  = sum(lg)
        if total == 0: continue

        # Proportions
        hist = np.zeros(n_cl)
        for c, l in zip(sq, lg):
            if c < n_cl: hist[c] += l / total

        # Transitions
        trans = np.zeros((n_cl, n_cl))
        for a, b in zip(sq[:-1], sq[1:]):
            if 0 <= a < n_cl and 0 <= b < n_cl:
                trans[a, b] += 1
        rs   = trans.sum(axis=1, keepdims=True)
        rs[rs == 0] = 1
        trans_norm = (trans / rs).flatten()

        # Median durations
        durations = {c: [] for c in range(n_cl)}
        for c, l in zip(sq, lg):
            if c < n_cl: durations[c].append(l)
        runlen = np.array([np.median(durations[c]) if durations[c] else 0
                           for c in range(n_cl)])

        # First appearance
        firstapp = []
        for c in range(n_cl):
            try:    firstapp.append(sq.index(c) / len(sq))
            except: firstapp.append(1.0)
        firstapp = np.array(firstapp)

        # Global metrics
        n_used   = len(set(sq))
        entropy  = 0.0
        h_vals   = hist[hist > 0]
        if len(h_vals) > 1:
            entropy = -np.sum(h_vals * np.log(h_vals)) / np.log(n_cl)

        row = {}
        for c in range(n_cl):
            row[f"hist_C{c}"]     = hist[c]
            row[f"runlen_C{c}"]   = runlen[c]
            row[f"firstapp_C{c}"] = firstapp[c]
        for i in range(n_cl):
            for j in range(n_cl):
                row[f"trans_{i}_{j}"] = trans_norm[i * n_cl + j]
        row["entropy"]      = entropy
        row["n_cl_used"]    = n_used / n_cl
        row["inertia"]      = hist.max()
        row["total_frames"] = total
        rows.append({"animal_id": str(aid), **row})

    df_seq = pd.DataFrame(rows).set_index("animal_id")
    df_seq = df_seq.fillna(0)

    if normalize:
        from sklearn.preprocessing import StandardScaler
        df_seq = pd.DataFrame(
            StandardScaler().fit_transform(df_seq.values),
            index=df_seq.index, columns=df_seq.columns)

    print(f"[SEQ FEATURES] {df_seq.shape[0]} individuals × "
          f"{df_seq.shape[1]} features (raw)")
    return df_seq


def _bh_correction(p_values, alpha=0.05):
    """
    Benjamini-Hochberg correction (FDR).
    Returns a boolean array: True = significant after correction.
    """
    p   = np.array(p_values)
    n   = len(p)
    if n == 0: return np.array([], dtype=bool)
    ranked  = np.argsort(p)
    thresholds  = (np.arange(1, n+1) / n) * alpha
    passed  = p[ranked] < thresholds
    sig     = np.zeros(n, dtype=bool)
    if passed.any():
        last = np.where(passed)[0][-1]
        sig[ranked[:last+1]] = True
    return sig


def show_power_analysis(final_state, df,
                        features         = None,
                        use_raw_features = False,
                        top_pct_features = 1.0,    # 1.0 = keep all
                        alpha            = 0.05,
                        power_target     = 0.80):
    """
    Statistical power analysis.

    use_raw_features : True  → Cohen's d on raw behavioural features
                               (distance, speed...) — more interpretable
                       False → Cohen's d on SeqVec features (default)

    top_pct_features : proportion of the most discriminant features
                       to keep for the Cohen's d computation.
                       1.0 = all (default)
                       0.2 = top 20% most discriminant
                       → useful to reduce the noise of weakly
                         informative features that pull the median down
    """
    strat = final_state["strat"]
    mono  = final_state["mono"]

    # ── Feature choice ────────────────────────────────────
    df_w = df.copy()
    df_w["animal_id"] = df_w["animal_id"].astype(str)

    if use_raw_features and features is not None:
        # Raw behavioural features — mean per individual
        # More interpretable biologically, Cohen's d naturally
        # larger because the features are on real scales
        df_indiv  = df_w.groupby("animal_id")[features].mean()
        feat_cols = list(features)
        feat_label = f"raw features ({len(feat_cols)})"
    else:
        # SeqVec features — 184 abstract features
        df_indiv  = build_seqvec_features_per_individual(
                        final_state, normalize=False)
        feat_cols = list(df_indiv.columns)
        feat_label = f"raw SeqVec ({len(feat_cols)} features)"

    df_indiv["group"] = [strat.get(str(aid), "MONO")
                          for aid in df_indiv.index]
    df_indiv = df_indiv.dropna(subset=feat_cols)

    strategy_list = sorted(set(strat.values()))
    all_groups    = strategy_list + (["MONO"] if mono else [])

    # ── Discriminant feature filter ────────────────────────
    # Keeps only the top_pct_features% most discriminant ones
    # Criterion: ratio between-group variance / total variance (η²)
    # → prevents non-discriminant features from pulling Cohen's d
    #   down and underestimating the power
    if top_pct_features < 1.0 and len(feat_cols) > 5:
        scores = []
        grand_mean = df_indiv[feat_cols].mean()
        for feat in feat_cols:
            var_total = df_indiv[feat].var()
            if var_total < 1e-10:
                scores.append(0); continue
            var_inter = 0
            for g in strategy_list:
                mask = df_indiv["group"] == g
                if mask.sum() < 2: continue
                ng = mask.sum()
                mg = df_indiv.loc[mask, feat].mean()
                var_inter += ng * (mg - grand_mean[feat])**2
            var_inter /= max(len(df_indiv) - 1, 1)
            scores.append(var_inter / (var_total + 1e-10))

        threshold = np.percentile(scores, (1 - top_pct_features) * 100)
        feat_cols = [f for f, s in zip(feat_cols, scores)
                     if s >= threshold]
        feat_label += f" → top {int(top_pct_features*100)}%"
        print(f"[FILTER] {len(feat_cols)} features kept "
              f"(top {int(top_pct_features*100)}% discriminant)")

    pal         = {s: STRATEGY_PALETTE[i % len(STRATEGY_PALETTE)]
                   for i, s in enumerate(all_groups)}
    pal["MONO"] = "#666666"

    # ── Theoretical minimum N ──────────────────────────────────────
    z_alpha = stats.norm.ppf(1 - alpha/2)
    z_beta  = stats.norm.ppf(power_target)

    def n_min_for_d(d):
        if d <= 0: return np.inf
        return int(np.ceil((z_alpha + z_beta)**2 / (2 * d**2)))

    n_min_large  = n_min_for_d(0.8)
    n_min_medium = n_min_for_d(0.5)
    n_min_small  = n_min_for_d(0.2)

    group_sizes = {g: int((df_indiv["group"] == g).sum())
                   for g in all_groups}

    # ── Pairwise computation with BH ─────────────────────────────────
    pairs        = list(combinations(strategy_list, 2))
    pair_results = []

    for g1, g2 in pairs:
        mask1    = df_indiv["group"] == g1
        mask2    = df_indiv["group"] == g2
        n1, n2   = mask1.sum(), mask2.sum()

        if n1 < 2 or n2 < 2:
            pair_results.append({
                "g1": g1, "g2": g2, "n1": n1, "n2": n2,
                "cohens_d": 0, "power_obs": 0,
                "p_value": 1.0, "significant": False,
                "n_min_needed": np.inf,
            }); continue

        ds, ps = [], []
        for feat in feat_cols:
            x1 = df_indiv.loc[mask1, feat].dropna().values
            x2 = df_indiv.loc[mask2, feat].dropna().values
            if len(x1) < 2 or len(x2) < 2: continue
            ps_std = np.sqrt((x1.std()**2 + x2.std()**2) / 2)
            ds.append(abs(x1.mean() - x2.mean()) / (ps_std + 1e-9))
            try:
                _, p = mannwhitneyu(x1, x2, alternative="two-sided")
                ps.append(p)
            except:
                ps.append(1.0)

        mean_d = float(np.median(ds)) if ds else 0.0
        p_min  = float(np.min(ps))    if ps else 1.0
        n_obs  = min(n1, n2)
        n_need = n_min_for_d(mean_d)

        if n_need > 0 and n_need < np.inf:
            ratio     = n_obs / n_need
            power_obs = float(np.clip(stats.norm.cdf(
                z_beta * np.sqrt(ratio)
                - z_alpha * (1/np.sqrt(ratio) - 1)), 0, 1))
        else:
            power_obs = 0.0

        pair_results.append({
            "g1": g1, "g2": g2, "n1": n1, "n2": n2,
            "cohens_d":    round(mean_d, 3),
            "power_obs":   round(power_obs, 3),
            "p_value":     round(p_min, 4),
            "significant": False,
            "n_min_needed": n_need,
        })

    df_pairs = pd.DataFrame(pair_results)

    # BH correction
    if len(df_pairs) > 0:
        sig_bh = _bh_correction(df_pairs["p_value"].values, alpha)
        df_pairs["significant"] = sig_bh
        print(f"[BH] {sig_bh.sum()}/{len(sig_bh)} significant pairs "
              f"after Benjamini-Hochberg correction")

    # ── Scores ────────────────────────────────────────────────
    n_tot        = len(strategy_list)
    n_ok         = sum(1 for g, n in group_sizes.items()
                       if g != "MONO" and n >= n_min_medium)
    score_size   = n_ok / n_tot if n_tot > 0 else 0
    score_power  = df_pairs["power_obs"].mean()   if len(df_pairs) > 0 else 0
    score_sig    = df_pairs["significant"].mean() if len(df_pairs) > 0 else 0
    score_global = score_power * 0.5 + score_sig * 0.3 + score_size * 0.2

    pair_labels  = [f"{r.g1} ↔ {r.g2}" for r in df_pairs.itertuples()]

    # ════════════════════════════════════════════════════════
    # FIGURE 1 — Group sizes
    # ════════════════════════════════════════════════════════
    groups_plot = [g for g in all_groups if g != "MONO"]
    sizes_plot  = [group_sizes[g] for g in groups_plot]

    fig1, ax = plt.subplots(figsize=(max(8, len(groups_plot)*0.6), 5))
    fig1.patch.set_facecolor("#1a1a2e"); ax.set_facecolor("#0d0d1a")
    bars = ax.bar(groups_plot, sizes_plot,
                  color=[pal[g] for g in groups_plot],
                  edgecolor="#222", linewidth=0.8, alpha=0.9)
    for bar, n in zip(bars, sizes_plot):
        ax.text(bar.get_x()+bar.get_width()/2, n+0.05, str(n),
                ha="center", va="bottom", color="white",
                fontsize=9, fontweight="bold")
    ymax = max(sizes_plot + [n_min_small]) * 1.25
    ax.axhline(n_min_large,  color="#44ff44", lw=1.5, ls="--",
               label=f"Large effect d=0.8 → n≥{n_min_large}")
    ax.axhline(n_min_medium, color="#ffaa00", lw=1.5, ls="--",
               label=f"Medium effect d=0.5 → n≥{n_min_medium}")
    ax.axhline(n_min_small,  color="#ff4444", lw=1.5, ls="--",
               label=f"Small effect d=0.2 → n≥{n_min_small}")
    ax.axhspan(0,            n_min_large,  alpha=0.07, color="#ff4444")
    ax.axhspan(n_min_large,  n_min_medium, alpha=0.07, color="#ffaa00")
    ax.axhspan(n_min_medium, ymax,         alpha=0.04, color="#44ff44")
    ax.set_ylim(0, ymax)
    ax.set_title(f"Group size vs required minimum N\n"
                 f"(α={alpha}, target power={int(power_target*100)}%)",
                 color="#cccccc", fontsize=11)
    ax.set_ylabel("N individuals", color="#aaaaaa")
    ax.tick_params(colors="#aaaaaa", axis="x", rotation=45)
    ax.tick_params(colors="#aaaaaa", axis="y")
    for sp in ax.spines.values(): sp.set_edgecolor("#333333")
    leg = ax.legend(fontsize=8, loc="upper right")
    leg.get_frame().set_facecolor("#111111")
    for t in leg.get_texts(): t.set_color("white")
    plt.tight_layout(); plt.show()

    # ════════════════════════════════════════════════════════
    # FIGURE 2 — Cohen's d
    # ════════════════════════════════════════════════════════
    if len(df_pairs) > 0:
        ds_vals  = df_pairs["cohens_d"].values
        d_colors = ["#44ff44" if d >= 0.8
                    else "#ffaa00" if d >= 0.5
                    else "#ff4444" for d in ds_vals]

        fig2, ax = plt.subplots(figsize=(8, max(4, len(pair_labels)*0.35)))
        fig2.patch.set_facecolor("#1a1a2e"); ax.set_facecolor("#0d0d1a")
        ax.barh(range(len(pair_labels)), ds_vals, color=d_colors,
                edgecolor="#222", linewidth=0.5, alpha=0.9)
        for i, d in enumerate(ds_vals):
            ax.text(d+0.01, i, f"{d:.2f}", va="center",
                    color="white", fontsize=7)
        ax.axvline(0.8, color="#44ff44", lw=1.5, ls="--",
                   label="Large effect (0.8)")
        ax.axvline(0.5, color="#ffaa00", lw=1.5, ls="--",
                   label="Medium effect (0.5)")
        ax.axvline(0.2, color="#ff4444", lw=1.5, ls="--",
                   label="Small effect (0.2)")
        ax.set_yticks(range(len(pair_labels)))
        ax.set_yticklabels(pair_labels, fontsize=8, color="#cccccc")
        ax.set_xlabel(f"Median Cohen's d ({feat_label})",
                      color="#aaaaaa")
        ax.set_title("Effect size between each pair of groups",
                     color="#cccccc", fontsize=11)
        ax.tick_params(colors="#aaaaaa")
        for sp in ax.spines.values(): sp.set_edgecolor("#333333")
        leg = ax.legend(fontsize=8, loc="lower right")
        leg.get_frame().set_facecolor("#111111")
        for t in leg.get_texts(): t.set_color("white")
        plt.tight_layout(); plt.show()

    # ════════════════════════════════════════════════════════
    # FIGURE 3 — Power
    # ════════════════════════════════════════════════════════
    if len(df_pairs) > 0:
        powers   = df_pairs["power_obs"].values
        p_colors = ["#44ff44" if p >= power_target
                    else "#ffaa00" if p >= 0.5
                    else "#ff4444" for p in powers]

        fig3, ax = plt.subplots(figsize=(8, max(4, len(pair_labels)*0.35)))
        fig3.patch.set_facecolor("#1a1a2e"); ax.set_facecolor("#0d0d1a")
        ax.barh(range(len(pair_labels)), powers, color=p_colors,
                edgecolor="#222", linewidth=0.5, alpha=0.9)
        for i, p in enumerate(powers):
            ax.text(p+0.01, i, f"{p:.0%}", va="center",
                    color="white", fontsize=7)
        ax.axvline(power_target, color="#FFD700", lw=2, ls="--",
                   label=f"Target {int(power_target*100)}%")
        ax.axvline(0.5, color="#888888", lw=1, ls=":",
                   label="50% threshold")
        ax.set_xlim(0, 1.1)
        ax.set_yticks(range(len(pair_labels)))
        ax.set_yticklabels(pair_labels, fontsize=8, color="#cccccc")
        ax.set_xlabel("Estimated statistical power", color="#aaaaaa")
        ax.set_title(f"Power per pair — BH correction\n"
                     f"(green ≥ {int(power_target*100)}%, orange ≥ 50%, red < 50%)",
                     color="#cccccc", fontsize=11)
        ax.tick_params(colors="#aaaaaa")
        for sp in ax.spines.values(): sp.set_edgecolor("#333333")
        leg = ax.legend(fontsize=8)
        leg.get_frame().set_facecolor("#111111")
        for t in leg.get_texts(): t.set_color("white")
        plt.tight_layout(); plt.show()

    # ════════════════════════════════════════════════════════
    # FIGURE 4 — Table
    # ════════════════════════════════════════════════════════
    if len(df_pairs) > 0:
        headers    = ["G1","G2","n1","n2","Cohen d",
                      "Power","p-val (BH)","Sig ?","N missing"]
        table_data = []
        for r in df_pairs.itertuples():
            n_need   = r.n_min_needed
            n_missing = (max(0, n_need - min(r.n1, r.n2))
                        if n_need < np.inf else "∞")
            table_data.append([
                r.g1, r.g2, str(r.n1), str(r.n2),
                f"{r.cohens_d:.2f}",
                f"{r.power_obs:.0%}",
                f"{r.p_value:.4f}",
                "✓" if r.significant else "✗",
                str(int(n_missing)) if isinstance(n_missing, (int,float))
                else n_missing,
            ])

        fig4, ax = plt.subplots(
            figsize=(14, max(3, len(table_data)*0.4+1)))
        fig4.patch.set_facecolor("#1a1a2e")
        ax.set_facecolor("#0d0d1a"); ax.axis("off")
        tbl = ax.table(cellText=table_data, colLabels=headers,
                       loc="center", cellLoc="center")
        tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1, 1.6)
        for (row, col), cell in tbl.get_celld().items():
            cell.set_edgecolor("#333333")
            if row == 0:
                cell.set_facecolor("#1a1a3e")
                cell.set_text_props(color="white", fontweight="bold")
            else:
                cell.set_facecolor("#0d0d1a")
                cell.set_text_props(color="#cccccc")
                if col == 5:
                    try:
                        p = float(table_data[row-1][5].strip("%"))/100
                        cell.set_facecolor(
                            "#1a3a1a" if p >= power_target
                            else "#3a2a00" if p >= 0.5 else "#3a0000")
                    except: pass
                if col == 7:
                    cell.set_facecolor(
                        "#1a3a1a" if table_data[row-1][7]=="✓"
                        else "#3a0000")
                if col == 8:
                    try:
                        nm = int(table_data[row-1][8])
                        cell.set_facecolor(
                            "#1a3a1a" if nm == 0
                            else "#3a2a00" if nm <= 5 else "#3a0000")
                    except: pass
        ax.set_title(
            "Summary table — Benjamini-Hochberg correction\n"
            "'N missing' = additional individuals needed for the target power",
            color="#cccccc", fontsize=11, pad=15)
        plt.tight_layout(); plt.show()

    # ════════════════════════════════════════════════════════
    # FIGURE 5 — Global score
    # ════════════════════════════════════════════════════════
    fig5, ax = plt.subplots(figsize=(7, 5))
    fig5.patch.set_facecolor("#1a1a2e")
    ax.set_facecolor("#0d0d1a"); ax.axis("off")
    for sp in ax.spines.values(): sp.set_edgecolor("#333333")

    score_color = ("#44ff44" if score_global >= 0.7
                   else "#ffaa00" if score_global >= 0.4 else "#ff4444")
    verdict = ("✅ Statistically solid split"        if score_global >= 0.7
               else "🟡 Acceptable split, to be consolidated" if score_global >= 0.4
               else "🔴 Insufficient power — consider merging")

    ax.text(0.5, 0.92, f"Global score : {score_global:.0%}",
            ha="center", va="center", fontsize=24,
            color=score_color, fontweight="bold",
            transform=ax.transAxes)
    ax.text(0.5, 0.78, verdict, ha="center", va="center",
            fontsize=12, color=score_color, transform=ax.transAxes)

    for i, (label, val, col) in enumerate([
        ("Mean power between pairs (×0.5)",
         f"{score_power:.0%}",
         "#44ff44" if score_power >= 0.7 else "#ffaa00"
         if score_power >= 0.4 else "#ff4444"),
        ("Proportion of BH-significant pairs (×0.3)",
         f"{score_sig:.0%}",
         "#44ff44" if score_sig >= 0.7 else "#ffaa00"
         if score_sig >= 0.4 else "#ff4444"),
        ("Proportion of groups with OK size (×0.2)",
         f"{score_size:.0%}",
         "#44ff44" if score_size >= 0.7 else "#ffaa00"
         if score_size >= 0.4 else "#ff4444"),
    ]):
        y = 0.60 - i * 0.12
        ax.text(0.05, y, f"• {label}", ha="left", va="center",
                fontsize=9, color="#aaaaaa", transform=ax.transAxes)
        ax.text(0.95, y, val, ha="right", va="center",
                fontsize=11, color=col, fontweight="bold",
                transform=ax.transAxes)

    for i, line in enumerate([
        f"Min N large effect (d=0.8) : {n_min_large} individuals/group",
        f"Min N medium effect (d=0.5) : {n_min_medium} individuals/group",
        f"Min N small effect (d=0.2) : {n_min_small} individuals/group",
        f"Correction : Benjamini-Hochberg  |  α={alpha}",
        f"Features : {feat_label}",
    ]):
        ax.text(0.5, 0.22 - i*0.07, line, ha="center", va="center",
                fontsize=8, color="#666666", transform=ax.transAxes,
                fontfamily="monospace")

    ax.set_title(f"Summary — {n_tot} groups | α={alpha} | "
                 f"target power={int(power_target*100)}%",
                 color="#cccccc", fontsize=11)
    plt.tight_layout(); plt.show()

    print(f"\n{'='*55}")
    print(f"POWER — {n_tot} groups | α={alpha} | "
          f"target={int(power_target*100)}%")
    print(f"  Features  : {feat_label}")
    print(f"  Correction: Benjamini-Hochberg (FDR)")
    print(f"  Min N (d=0.8): {n_min_large} | (d=0.5): {n_min_medium} "
          f"| (d=0.2): {n_min_small}")
    print(f"  Sizes     : { {g: n for g, n in group_sizes.items()} }")
    print(f"  Score     : {score_global:.0%}  ({verdict})")

    return df_pairs




# ══════════════════════════════════════════════════════════════

def optimal_regrouping(
    final_state,
    df,
    alpha         = 0.05,
    power_target  = 0.80,
    target_effect = 0.5,
    min_groups    = 2,
    d_max_fusion  = 0.8,
    verbose       = True,
):
    """
    Iteratively merges the groups using the SeqVec D matrix
    — consistent with the initial clustering.

    d_max_fusion : maximum Cohen's d allowed for a forced merge.
                   If the closest pair has d > d_max_fusion,
                   we stop even if some groups are too small.
                   Avoids merging very different groups.
    """
    seqs = final_state["seqs"]
    D    = final_state["D"]
    ids  = final_state["multi"]   # list in the same order as D

    # Minimum N for the target effect
    z_alpha = stats.norm.ppf(1 - alpha/2)
    z_beta  = stats.norm.ppf(power_target)
    n_min   = int(np.ceil((z_alpha + z_beta)**2 / (2 * target_effect**2)))

    # Current mapping
    mapping = {str(aid): g for aid, g in final_state["strat"].items()}
    for m in final_state["mono"]:
        mapping[str(m)] = "MONO"

    # Index of the individuals in the D matrix
    id_to_idx = {str(aid): i for i, aid in enumerate(ids)}

    history = []

    def get_sizes(mapping):
        return dict(Counter(v for v in mapping.values() if v != "MONO"))

    def get_distance_D(g1, g2):
        """
        Mean distance in SeqVec space between the two groups.
        Uses the already computed D matrix → consistent with the clustering.
        """
        idx1 = [id_to_idx[aid] for aid in mapping
                if mapping[aid] == g1 and aid in id_to_idx]
        idx2 = [id_to_idx[aid] for aid in mapping
                if mapping[aid] == g2 and aid in id_to_idx]
        if not idx1 or not idx2: return np.inf
        sub = D[np.ix_(idx1, idx2)]
        return float(sub.mean())

    def all_ok(mapping, n_min, min_groups):
        sizes = get_sizes(mapping)
        if len(sizes) <= min_groups: return True
        return all(n >= n_min for n in sizes.values())

    if verbose:
        sizes_init = get_sizes(mapping)
        print(f"\n{'='*60}")
        print(f"OPTIMAL REGROUPING")
        print(f"  features     : SeqVec (D matrix of the clustering)")
        print(f"  n_min (d={target_effect}, power={int(power_target*100)}%) = {n_min}")
        print(f"  d_max_fusion : {d_max_fusion}")
        print(f"  Initial groups : {sizes_init}")
        print(f"{'='*60}")

        # Also show the recommended target_effect values
        if len(final_state.get("D", [])) > 0:
            df_pairs_preview = []
            non_mono = sorted(set(v for v in mapping.values() if v != "MONO"))
            for g1, g2 in combinations(non_mono, 2):
                d = get_distance_D(g1, g2)
                df_pairs_preview.append(d)
            if df_pairs_preview:
                print(f"  target_effect_auto   : "
                      f"{np.mean(df_pairs_preview):.3f}")
                print(f"  target_effect_median : "
                      f"{np.median(df_pairs_preview):.3f}")
                print(f"  target_effect_p25    : "
                      f"{np.percentile(df_pairs_preview, 25):.3f}")

    iteration = 0
    while not all_ok(mapping, n_min, min_groups):
        sizes    = get_sizes(mapping)
        non_mono = sorted(sizes.keys())

        if len(non_mono) <= min_groups:
            if verbose: print(f"[STOP] Minimum number of groups reached")
            break

        small_groups = {g for g in non_mono if sizes[g] < n_min}
        best_pair, best_score, best_d = None, np.inf, np.inf

        for g1, g2 in combinations(non_mono, 2):
            d            = get_distance_D(g1, g2)
            involves_sm  = g1 in small_groups or g2 in small_groups
            score        = d if involves_sm else d + 100
            if score < best_score:
                best_score = score
                best_pair  = (g1, g2)
                best_d     = d

        if best_pair is None: break

        # Check whether the merge is reasonable
        if best_d > d_max_fusion and not (best_pair[0] in small_groups
                                           or best_pair[1] in small_groups):
            if verbose:
                print(f"[STOP] Next merge too different "
                      f"(d={best_d:.3f} > {d_max_fusion})")
            break

        g1, g2    = best_pair
        reason    = ("insufficient size" if g1 in small_groups
                     or g2 in small_groups else "most similar groups")
        # New label = smallest label (alphabetical)
        # avoids always absorbing into the large group
        new_label = min(g1, g2)
        mapping   = {aid: (new_label if g in (g1, g2) else g)
                     for aid, g in mapping.items()}
        iteration += 1

        record = {
            "iteration": iteration, "g1": g1, "g2": g2,
            "new_label": new_label, "distance_D": round(best_d, 3),
            "n_g1": sizes[g1], "n_g2": sizes[g2],
            "n_merged": sizes[g1] + sizes[g2], "reason": reason,
        }
        history.append(record)

        if verbose:
            new_sizes = get_sizes(mapping)
            print(f"  [{iteration}] {g1}(n={sizes[g1]}) + "
                  f"{g2}(n={sizes[g2]}) → {new_label}"
                  f"(n={sizes[g1]+sizes[g2]})  "
                  f"d={best_d:.3f}  [{reason}]")
            print(f"       Remaining : {new_sizes}")

    if verbose:
        final_sizes = get_sizes(mapping)
        n_ok = sum(1 for n in final_sizes.values() if n >= n_min)
        print(f"\n[RESULT] {len(final_sizes)} groups | "
              f"{n_ok}/{len(final_sizes)} reach n≥{n_min}")
        print(f"  Final sizes : {final_sizes}")

    # ════════════════════════════════════════════════════════
    # FIGURE 1 — Before / After
    # ════════════════════════════════════════════════════════
    sizes_before = get_sizes({str(aid): g
                              for aid, g in final_state["strat"].items()})
    sizes_after = get_sizes(mapping)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.patch.set_facecolor("#1a1a2e")

    for ax, (sizes, title) in zip(axes, [
        (sizes_before, f"Before  ({len(sizes_before)} groups)"),
        (sizes_after, f"After  ({len(sizes_after)} groups)"),
    ]):
        ax.set_facecolor("#0d0d1a")
        for sp in ax.spines.values(): sp.set_edgecolor("#333333")
        ax.tick_params(colors="#aaaaaa")
        groups = sorted(sizes.keys())
        vals   = [sizes[g] for g in groups]
        colors = [STRATEGY_PALETTE[i % len(STRATEGY_PALETTE)]
                  for i in range(len(groups))]
        bars = ax.bar(groups, vals, color=colors,
                      edgecolor="#222", linewidth=0.8, alpha=0.9)
        for bar, n in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, n+0.05, str(n),
                    ha="center", va="bottom", color="white",
                    fontsize=9, fontweight="bold")
        ax.axhline(n_min, color="#FFD700", lw=2, ls="--",
                   label=f"n_min={n_min} (d={target_effect})")
        ax.axhspan(0, n_min, alpha=0.06, color="#ff4444")
        ax.axhspan(n_min, max(vals+[n_min])*1.3, alpha=0.04, color="#44ff44")
        ax.set_ylim(0, max(vals+[n_min])*1.3)
        ax.set_title(title, color="#cccccc", fontsize=11)
        ax.set_ylabel("N individuals", color="#aaaaaa")
        ax.tick_params(axis="x", rotation=45)
        leg = ax.legend(fontsize=8)
        leg.get_frame().set_facecolor("#111111")
        for t in leg.get_texts(): t.set_color("white")

    fig.suptitle(
        f"Optimal regrouping | n_min={n_min} | "
        f"{len(history)} merges | SeqVec distance",
        color="#cccccc", fontsize=12)
    plt.tight_layout(); plt.show()

    # ════════════════════════════════════════════════════════
    # FIGURE 2 — Merge log
    # ════════════════════════════════════════════════════════
    if history:
        df_hist = pd.DataFrame(history)
        fig2, axes2 = plt.subplots(1, 2,
                                    figsize=(14, max(3, len(history)*0.5+1)))
        fig2.patch.set_facecolor("#1a1a2e")

        labels_fus = [f"{r.g1}+{r.g2}→{r.new_label}"
                      for r in df_hist.itertuples()]

        # SeqVec distance of the merges
        ax = axes2[0]
        ax.set_facecolor("#0d0d1a")
        for sp in ax.spines.values(): sp.set_edgecolor("#333333")
        d_colors = ["#44ff44" if d < 2 else "#ffaa00" if d < 5 else "#ff4444"
                    for d in df_hist["distance_D"]]
        ax.barh(range(len(labels_fus)), df_hist["distance_D"],
                color=d_colors, edgecolor="#222", alpha=0.9)
        for i, (d, r) in enumerate(zip(df_hist["distance_D"],
                                        df_hist.itertuples())):
            ax.text(d+0.05, i, f"d={d:.2f}  [{r.reason}]",
                    va="center", color="#cccccc", fontsize=8)
        ax.set_yticks(range(len(labels_fus)))
        ax.set_yticklabels(labels_fus, fontsize=8, color="#cccccc")
        ax.set_xlabel("Mean SeqVec distance between the groups",
                      color="#aaaaaa")
        ax.set_title("Order and reason of the merges\n"
                     "(green=very close, red=forced by size)",
                     color="#cccccc", fontsize=10)
        ax.tick_params(colors="#aaaaaa")

        # Merged sizes
        ax2 = axes2[1]
        ax2.set_facecolor("#0d0d1a")
        for sp in ax2.spines.values(): sp.set_edgecolor("#333333")
        ax2.barh(range(len(df_hist)), df_hist["n_g1"],
                 color="#4466aa", alpha=0.8, label="n(G1)", edgecolor="#222")
        ax2.barh(range(len(df_hist)), df_hist["n_g2"],
                 left=df_hist["n_g1"],
                 color="#aa6644", alpha=0.8, label="n(G2)", edgecolor="#222")
        ax2.axvline(n_min, color="#FFD700", lw=2, ls="--",
                    label=f"n_min={n_min}")
        ax2.set_yticks(range(len(labels_fus)))
        ax2.set_yticklabels(labels_fus, fontsize=8, color="#cccccc")
        ax2.set_xlabel("Size of the merged groups", color="#aaaaaa")
        ax2.set_title("Size of the groups before merging",
                      color="#cccccc", fontsize=10)
        ax2.tick_params(colors="#aaaaaa")
        leg2 = ax2.legend(fontsize=8)
        leg2.get_frame().set_facecolor("#111111")
        for t in leg2.get_texts(): t.set_color("white")

        fig2.suptitle("Merge details", color="#cccccc", fontsize=11)
        plt.tight_layout(); plt.show()

    non_mono_groups = sorted(set(v for v in mapping.values() if v != "MONO"))
    num_to_letter   = {g: chr(65+i) for i, g in enumerate(non_mono_groups)}

    mapping_letters = {
        aid: (num_to_letter[g] if g != "MONO" else "MONO")
        for aid, g in mapping.items()
    }

    print(f"\n[LETTERS] {len(non_mono_groups)} groups → "
          f"{list(num_to_letter.values())}")

    return mapping_letters, history 


In [ ]:
# ── Calls ────────────────────────────────────────────────────
# ── Calls ────────────────────────────────────────────────────

# Option A — raw features (recommended for the power analysis)
df_pairs = show_power_analysis(
    final_state, df,
    features         = FEATURES,
    use_raw_features = True,
    top_pct_features = 1.0,
    alpha            = 0.05,
    power_target     = 0.80,
)


In [ ]:
# Depending on what you want to detect:
#target_effect = 0.8   # large effect  → n_min ≈ 26  → few merges
#target_effect = 0.5   # medium effect → n_min ≈ 64  → moderate merges
#target_effect = 0.2   # small effect  → n_min ≈ 394 → many merges

# For your data (200 individuals, ~20 initial groups)
# → 0.5 is a reasonable starting point
mapping_optimal, history = optimal_regrouping(
    final_state,
    df,
    alpha         = 0.05,
    power_target  = 0.95,
    target_effect = 0.75,   # ← fixed
    min_groups    = 2,
    d_max_fusion  = 0.8,
    verbose       = True,
)

In [ ]:
#def plot_strategy_profiles

from scipy.spatial.distance import cdist
import matplotlib.colors as mcolors


def plot_strategy_profiles(mapping_optimal, final_state, df, features=FEATURES,
                           cmap_cl=None, norm_cl=None):

    # Default values if not provided
    n_cl_local = final_state["n_cl"]
    if cmap_cl is None:
        cmap_cl = CMAP_CLUSTER
    if norm_cl is None:
        norm_cl = NORM_CLUSTER

    seqs = final_state["seqs"]
    n_cl = final_state["n_cl"]

    df_w = df.copy()
    df_w["animal_id"] = df_w["animal_id"].astype(str)

    unique_groups = sorted(set(v for v in mapping_optimal.values() if v != "MONO"))
    num_to_letter = {g: chr(65+i) for i, g in enumerate(unique_groups)}
    num_to_letter["MONO"] = "MONO"
    mapping_letters  = {aid: num_to_letter[g] for aid, g in mapping_optimal.items()}
    strategy_letters = sorted(set(v for v in mapping_letters.values() if v != "MONO"))
    n_strat          = len(strategy_letters)

    pal         = {s: STRATEGY_PALETTE[i % len(STRATEGY_PALETTE)]
                   for i, s in enumerate(strategy_letters)}
    pal["MONO"] = "#666666"

    # ── DTW for the medoid ──────────────────────────────────────
    def _dtw(seq_a, seq_b, n_cl):
        if not seq_a or not seq_b: return np.inf
        centers = np.eye(n_cl)
        tA = np.array([centers[min(c, n_cl-1)] for c in seq_a])
        tB = np.array([centers[min(c, n_cl-1)] for c in seq_b])
        TA, TB = len(tA), len(tB)
        cost   = cdist(tA, tB, metric='euclidean')
        D      = np.full((TA, TB), np.inf)
        D[0,0] = cost[0,0]
        for i in range(1, TA): D[i,0] = D[i-1,0] + cost[i,0]
        for j in range(1, TB): D[0,j] = D[0,j-1] + cost[0,j]
        for i in range(1, TA):
            for j in range(1, TB):
                D[i,j] = cost[i,j] + min(D[i-1,j], D[i,j-1], D[i-1,j-1])
        return D[TA-1, TB-1]

    def _medoid(members):
        n = len(members)
        if n == 1: return members[0], seqs[members[0]]["seq"]
        dists = np.zeros(n)
        for i in range(n):
            for j in range(n):
                if i != j:
                    dists[i] += _dtw(seqs[members[i]]["seq"],
                                     seqs[members[j]]["seq"], n_cl)
        best = int(np.argmin(dists))
        return members[best], seqs[members[best]]["seq"]

    angles   = np.linspace(0, 2*np.pi, n_cl, endpoint=False)
    ang_full = np.concatenate([angles, [angles[0]]])

    profiles = {}
    for s in strategy_letters:
        members         = [aid for aid, g in mapping_letters.items() if g == s]
        members_in_seqs = [m for m in members if m in seqs]
        if not members_in_seqs: continue

        hists = []
        for aid in members_in_seqs:
            sq, lg = seqs[aid]["seq"], seqs[aid]["lengths"]
            total  = sum(lg)
            if total == 0: continue
            h = np.zeros(n_cl)
            for c, l in zip(sq, lg):
                if c < n_cl: h[c] += l / total
            hists.append(h)
        mean_hist = np.mean(hists, axis=0) if hists else np.zeros(n_cl)

        print(f"  [MEDOID] Strategy {s} ({len(members_in_seqs)} ind)...", end=" ")
        medoid_aid, modal_compressed = _medoid(members_in_seqs)
        print(f"-> ID {medoid_aid}")

        durations = {c: [] for c in range(n_cl)}
        for aid in members_in_seqs:
            for c, l in zip(seqs[aid]["seq"], seqs[aid]["lengths"]):
                if c < n_cl: durations[c].append(l)
        median_durations = {c: np.median(durations[c]) if durations[c] else 0
                         for c in range(n_cl)}

        trans_count = {}
        for aid in members_in_seqs:
            sq = seqs[aid]["seq"]
            for a, b in zip(sq[:-1], sq[1:]):
                trans_count[(a,b)] = trans_count.get((a,b), 0) + 1
        top_trans = sorted(trans_count.items(), key=lambda x: -x[1])[:3]

        feat_means = {}
        if features:
            ids_str = [str(m) for m in members_in_seqs]
            mask    = df_w["animal_id"].isin(ids_str)
            if mask.any():
                feat_means = df_w[mask][features].mean().to_dict()

        profiles[s] = {
            "members"         : members_in_seqs,
            "all_members"     : members,
            "n"               : len(members_in_seqs),
            "mean_hist"       : mean_hist,
            "modal_seq"       : modal_compressed,
            "medoid_aid"      : medoid_aid,
            "median_durations"   : median_durations,
            "top_transitions" : top_trans,
            "dominant_cluster": int(np.argmax(mean_hist)),
            "feat_means"      : feat_means,
            "mean_seq_len"    : np.mean([len(seqs[a]["seq"])
                                         for a in members_in_seqs]),
            "mean_n_clusters" : np.mean([len(set(seqs[a]["seq"]))
                                         for a in members_in_seqs]),
        }
    print(f"[DEBUG] mapping_optimal sample : {list(mapping_optimal.items())[:5]}")
    print(f"[DEBUG] mapping_letters sample : {list(mapping_letters.items())[:5]}")
    print(f"[DEBUG] strategy_letters : {strategy_letters}")
    print(f"[DEBUG] seqs keys sample : {list(seqs.keys())[:5]}")

    for s in strategy_letters:
        members = [aid for aid, g in mapping_letters.items() if g == s]
        in_seqs = [m for m in members if m in seqs]
        print(f"  {s} : {len(members)} members, {len(in_seqs)} in seqs")
    # ════════════════════════════════════════════════════════
    # FIGURE 1 — Radars
    # ════════════════════════════════════════════════════════
    n_cols = min(4, n_strat)
    n_rows = (n_strat + n_cols - 1) // n_cols
    fig1   = plt.figure(figsize=(n_cols * 5, n_rows * 5))
    fig1.patch.set_facecolor("#1a1a2e")
    fig1.suptitle("Behavioural profiles of the strategies",
                  color="#cccccc", fontsize=13, y=1.01)

    for si, s in enumerate(strategy_letters):
        if s not in profiles: continue
        p   = profiles[s]
        col = pal[s]
        ax  = fig1.add_subplot(n_rows, n_cols, si+1, polar=True)
        ax.set_facecolor("#0d0d1a")
        vals_f = np.concatenate([p["mean_hist"], [p["mean_hist"][0]]])
        ax.plot(ang_full, vals_f, lw=2.5, color=col)
        ax.fill(ang_full, vals_f, alpha=0.3, color=col)
        ax.set_xticks(angles)
        ax.set_xticklabels([f"C{c}" for c in range(n_cl)],
                            fontsize=7, color="#aaaaaa")
        ax.set_ylim(0, max(p["mean_hist"].max()*1.2, 0.01))
        ax.set_yticks([])
        ax.grid(color="#333333", lw=0.5)
        ax.spines["polar"].set_color("#333333")
        modal_str = "→".join([f"C{c}" for c in p["modal_seq"][:6]])
        if len(p["modal_seq"]) > 6: modal_str += "…"
        ax.set_title(
            f"Strategy {s}  (n={p['n']})\n"
            f"Dominant : C{p['dominant_cluster']}\n"
            f"Medoid : ID {p['medoid_aid']}\n"
            f"{modal_str}",
            color=col, fontsize=9, pad=12
        )
    plt.tight_layout(); plt.show()

    # ════════════════════════════════════════════════════════
    # FIGURE 2 — MEDOID sequence (1 real individual)
    # Visual improvement: merged blocks, thickness by duration
    # ════════════════════════════════════════════════════════
    fig2, axes = plt.subplots(n_strat, 1,
                               figsize=(14, max(3, n_strat * 1.6)))
    fig2.patch.set_facecolor("#1a1a2e")
    fig2.suptitle(
        "Medoid sequence per strategy\n"
        "(most representative real individual — DTW medoid)\n"
        "width ∝ duration | consecutive identical blocks merged",
        color="#cccccc", fontsize=11
    )
    if n_strat == 1: axes = [axes]

    for si, (s, ax) in enumerate(zip(strategy_letters, axes)):
        if s not in profiles: continue
        p    = profiles[s]
        col  = pal[s]

        # Sequence of the medoid (real individual)
        seq_raw  = seqs[p["medoid_aid"]]["seq"]
        lgts_raw = seqs[p["medoid_aid"]]["lengths"]

        # ── Merge consecutive identical blocks ───────────
        # Avoids the visual repetition C0 C0 C0 → a single large C0
        seq_fused, lgts_fused = [], []
        for c, l in zip(seq_raw, lgts_raw):
            if seq_fused and seq_fused[-1] == c:
                lgts_fused[-1] += l
            else:
                seq_fused.append(c)
                lgts_fused.append(l)

        ax.set_facecolor("#0d0d1a")
        for sp in ax.spines.values(): sp.set_edgecolor("#333333")
        ax.set_yticks([])
        ax.tick_params(colors="#555555", labelsize=7)

        x    = 0
        dmax = max(lgts_fused) if lgts_fused else 1
        for c, dur in zip(seq_fused, lgts_fused):
            c_color = mcolors.to_hex(cmap_cl(norm_cl(c)))

            # Height proportional to duration → important blocks = thicker
            h = 0.35 + 0.55 * (dur / dmax)

            ax.barh(0, dur, left=x, height=h,
                    color=c_color, edgecolor="#0d0d1a", lw=0.8,
                    alpha=0.9)

            # Label only if the block is wide enough
            if dur >= dmax * 0.04:
                fontsize = min(9, max(5, int(7 * dur / dmax * 3)))
                ax.text(x + dur/2, 0, f"C{c}",
                        ha="center", va="center",
                        fontsize=fontsize, color="white",
                        fontweight="bold")
            x += dur

        ax.set_xlim(0, x)
        ax.set_ylim(-0.55, 0.55)

        # Group colour bar on the left
        ax.barh(0, x * 0.008, left=-x*0.015, height=1.1,
                color=col, alpha=0.9)

        total_frames = sum(lgts_fused)
        n_blocks_orig = len(seq_raw)
        n_blocks_fused  = len(seq_fused)

        trans_str = "  |  ".join(
            [f"C{a}→C{b} ({n}×)" for (a,b), n in p["top_transitions"]]
        )

        ax.set_ylabel(
            f"Strat. {s}\nn={p['n']}\nID {p['medoid_aid']}",
            color=col, fontsize=8, rotation=0,
            ha="right", va="center", labelpad=60
        )
        ax.text(x * 1.005, 0.25,
                f"{total_frames}f",
                va="center", ha="left", fontsize=6, color="#666666")
        ax.text(x * 1.005, -0.25,
                f"{n_blocks_fused} blocks ({n_blocks_orig} raw)",
                va="center", ha="left", fontsize=6, color="#555555")
        ax.text(x * 1.005, 0.0,
                f"↑ {trans_str}",
                va="center", ha="left", fontsize=6, color="#777777")

    plt.tight_layout(); plt.show()

    # ════════════════════════════════════════════════════════
    # FIGURE 3 — Summary table + list of individuals
    # ════════════════════════════════════════════════════════
    fig3, ax = plt.subplots(figsize=(18, max(3, n_strat * 0.9 + 1.5)))
    fig3.patch.set_facecolor("#1a1a2e")
    ax.set_facecolor("#0d0d1a"); ax.axis("off")

    headers = (["Strategy", "N", "Medoid", "Dominant cluster",
                "Mean blocks", "Distinct clusters", "Top transition",
                "Individuals (IDs)"]
               + [f.replace("_"," ")[:10] for f in features])

    table_data = []
    for s in strategy_letters:
        if s not in profiles: continue
        p = profiles[s]

        top_t = (f"C{p['top_transitions'][0][0][0]}→"
                 f"C{p['top_transitions'][0][0][1]}"
                 if p["top_transitions"] else "—")

        # List of individuals — sorted, comma-separated
        ids_str = ", ".join(sorted([str(m) for m in p["members"]]))
        # Truncate if too long for the table
        if len(ids_str) > 60:
            ids_str = ids_str[:57] + "…"

        row = [
            f"  {s}  ",
            str(p["n"]),
            str(p["medoid_aid"]),
            f"C{p['dominant_cluster']}  ({p['mean_hist'][p['dominant_cluster']]:.0%})",
            f"{p['mean_seq_len']:.1f}",
            f"{p['mean_n_clusters']:.1f}",
            top_t,
            ids_str,
        ]
        for feat in features:
            v = p["feat_means"].get(feat, np.nan)
            row.append(f"{v:.2f}" if not np.isnan(v) else "—")
        table_data.append(row)

    tbl = ax.table(cellText=table_data, colLabels=headers,
                   loc="center", cellLoc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(7)
    tbl.scale(1, 2.2)

    for (row, col_idx), cell in tbl.get_celld().items():
        cell.set_edgecolor("#333333")
        if row == 0:
            cell.set_facecolor("#1a1a3e")
            cell.set_text_props(color="white", fontweight="bold")
        else:
            s = strategy_letters[row-1] if row-1 < len(strategy_letters) else None
            cell.set_facecolor("#0d0d1a")
            cell.set_text_props(color="#cccccc")
            if col_idx == 0 and s:
                cell.set_facecolor(pal.get(s, "#0d0d1a") + "55")
                cell.set_text_props(color=pal.get(s, "white"),
                                    fontweight="bold")
            # Individuals column: lightly coloured
            if col_idx == 7 and s:
                cell.set_facecolor(pal.get(s, "#0d0d1a") + "22")
                cell.set_text_props(color="#aaaaaa", fontsize=6)

    ax.set_title("Summary table — DTW medoid + list of individuals",
                 color="#cccccc", fontsize=11, pad=15)
    plt.tight_layout(); plt.show()

    # ════════════════════════════════════════════════════════
    # FIGURE 4 — Detailed list of individuals per group
    # ════════════════════════════════════════════════════════
    fig4, ax4 = plt.subplots(figsize=(14, max(4, n_strat * 1.2 + 1)))
    fig4.patch.set_facecolor("#1a1a2e")
    ax4.set_facecolor("#0d0d1a"); ax4.axis("off")

    ax4.set_title("Individuals per strategy",
                  color="#cccccc", fontsize=12, pad=10)

    y_pos = 0.97
    step  = 1.0 / (n_strat + 1)

    for s in strategy_letters:
        if s not in profiles: continue
        p   = profiles[s]
        col = pal[s]

        ids_sorted = sorted([str(m) for m in p["members"]])
        ids_chunks = [ids_sorted[i:i+20] for i in range(0, len(ids_sorted), 20)]

        # Group label
        ax4.text(0.01, y_pos,
                 f"Strategy {s}  (n={p['n']})  —  medoid : {p['medoid_aid']}",
                 transform=ax4.transAxes,
                 color=col, fontsize=10, fontweight='bold', va='top')

        # IDs on one or more lines
        for li, chunk in enumerate(ids_chunks):
            ax4.text(0.12, y_pos - 0.015 * (li + 1),
                     "  ".join(chunk),
                     transform=ax4.transAxes,
                     color="#cccccc", fontsize=7, va='top',
                     fontfamily='monospace')

        y_pos -= step

    plt.tight_layout(); plt.show()

    # ── Terminal summary ───────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"PROFILES ({n_strat} groups) — DTW medoids")
    print(f"{'='*60}")
    for s in strategy_letters:
        if s not in profiles: continue
        p = profiles[s]
        ids_sorted = sorted([str(m) for m in p["members"]])
        print(f"\nStrategy {s}  (n={p['n']})  —  medoid : ID {p['medoid_aid']}")
        print(f"  Individuals : {', '.join(ids_sorted)}")
        print(f"  Dominant  : C{p['dominant_cluster']} "
              f"({p['mean_hist'][p['dominant_cluster']]:.0%})")
        print(f"  Mean blocks : {p['mean_seq_len']:.1f} | "
              f"Distinct clusters : {p['mean_n_clusters']:.1f}")
        if p["top_transitions"]:
            print(f"  Top trans : " +
                  ", ".join([f"C{a}→C{b}({n}×)"
                             for (a,b),n in p["top_transitions"]]))

    return profiles, mapping_letters

In [ ]:
# Cell 19 — must be run BEFORE cell 30
profiles, mapping_letters = plot_strategy_profiles(
    mapping_optimal,
    final_state,
    df,
    features = FEATURES,
)

In [ ]:
# ── Create final_state_optimal with the LETTERS of mapping_optimal ──
# ── Create final_state_optimal with the LETTERS of mapping_optimal ──
final_state_optimal = dict(final_state)

final_state_optimal["strat"] = {
    aid: g
    for aid, g in mapping_optimal.items()
    if g != "MONO"
}
final_state_optimal["mono"] = [
    aid for aid, g in mapping_optimal.items()
    if g == "MONO"
]

# ── ADD mapping_letters ────────────────────────────────────
# mapping_optimal already contains letters (A, B, C...)
# so mapping_letters = mapping_optimal directly
final_state_optimal["mapping_letters"] = dict(mapping_optimal)

# Check
print("strat groups     :", sorted(set(final_state_optimal["strat"].values())))
print("mapping_letters  :", sorted(set(final_state_optimal["mapping_letters"].values())))
print("n_individuals    :", len(final_state_optimal["strat"]))
# ── Calls ────────────────────────────────────────────────────

# Option A — raw features (recommended for the power analysis)
df_pairs = show_power_analysis(
    final_state_optimal, df,
    features         = FEATURES,
    use_raw_features = True,
    top_pct_features = 1.0,
    alpha            = 0.05,
    power_target     = 0.80,
)

# # Option B — SeqVec top 20% discriminant features
# df_pairs = show_power_analysis(
#     final_state_optimal, df,
#     use_raw_features = False,
#     top_pct_features = 0.20,
#     alpha            = 0.05,
#     power_target     = 0.80,
# )


In [ ]:
# ══════════════════════════════════════════════════════════════
#  EXAMPLE TRAJECTORIES PER GROUP — configurable version
# ══════════════════════════════════════════════════════════════

# ── Parameters to change here ──────────────────────────────────
FINAL_STATE_TO_USE = final_state_optimal  # ← final_state | final_state_optimal
DF_TO_USE          = df                   # ← df | df_shrimp | other
N_EXAMPLES         = 4                    # ← between 2 and 5

# Position columns used to draw the trajectory
# Cuttlefish : "body_cx" / "body_cy"  or  "TopMantle_x" / "TopMantle_y"
# Shrimp     : "HeadShrimp_x" / "HeadShrimp_y"
X_COL = "body_cx"        # ← change depending on the df
Y_COL = "body_cy"        # ← change depending on the df

# Output file
OUT_PATH = OUT_DIR / "example_trajectories_per_group.png"


# ══════════════════════════════════════════════════════════════
#  CODE — do not modify below
# ══════════════════════════════════════════════════════════════

strat_final     = FINAL_STATE_TO_USE["strat"]
mono_final      = FINAL_STATE_TO_USE["mono"]
seqs            = FINAL_STATE_TO_USE["seqs"]
n_cl            = FINAL_STATE_TO_USE["n_cl"]
mapping_letters = FINAL_STATE_TO_USE["mapping_letters"]

df_plot = DF_TO_USE.copy()
df_plot["animal_id"] = df_plot["animal_id"].astype(str)

# Fallback if the X/Y column does not exist
if X_COL not in df_plot.columns:
    fallbacks_x = ["body_cx", "TopMantle_x", "HeadShrimp_x"]
    fallbacks_y = ["body_cy", "TopMantle_y", "HeadShrimp_y"]
    X_COL = next((c for c in fallbacks_x if c in df_plot.columns), None)
    Y_COL = next((c for c in fallbacks_y if c in df_plot.columns), None)
    print(f"[AUTO] Position columns : {X_COL} / {Y_COL}")

# ── Group the individuals ─────────────────────────────────────
groups = {}
for aid, letter in mapping_letters.items():
    groups.setdefault(letter, []).append(str(aid))
groups_sorted = sorted(g for g in groups if g != "MONO")
if "MONO" in groups:
    groups_sorted.append("MONO")

print("Individuals per group :\n")
for g in groups_sorted:
    ids    = groups[g]
    n_show = min(N_EXAMPLES, len(ids))
    ids_sorted = sorted(
        ids,
        key=lambda a: abs(len(seqs.get(a, {}).get("seq", [])) -
                          np.median([len(seqs.get(x, {}).get("seq", []))
                                     for x in ids])))
    print(f"  Group {g} (n={len(ids)}) → examples : {ids_sorted[:n_show]}")

# ── Global limits ──────────────────────────────────────────
x_all = df_plot[X_COL].dropna()
y_all = df_plot[Y_COL].dropna()
x_min, x_max = x_all.quantile(0.01), x_all.quantile(0.99)
y_min, y_max = y_all.quantile(0.01), y_all.quantile(0.99)

# ── Group palette ───────────────────────────────────────────
pal_strat = {g: STRATEGY_PALETTE[i % len(STRATEGY_PALETTE)]
             for i, g in enumerate(groups_sorted)}
pal_strat["MONO"] = "#666666"

# ── Figure ────────────────────────────────────────────────────
n_groups = len(groups_sorted)
fig, axes = plt.subplots(
    n_groups, N_EXAMPLES,
    figsize=(N_EXAMPLES * 4, n_groups * 3.5),
    squeeze=False)   # squeeze=False → axes always 2D
fig.patch.set_facecolor("#1a1a2e")
fig.suptitle(
    f"Example trajectories per group\n"
    f"({X_COL}/{Y_COL} · {n_cl} clusters · "
    f"{len(strat_final)} multi individuals + {len(mono_final)} MONO)",
    color="#cccccc", fontsize=11, y=1.01)

for row_i, grp in enumerate(groups_sorted):
    ids      = groups[grp]
    col_grp  = pal_strat[grp]
    n_show   = min(N_EXAMPLES, len(ids))

    # Sort by representativeness (close to the median)
    ids_sorted = sorted(
        ids,
        key=lambda a: abs(len(seqs.get(a, {}).get("seq", [])) -
                          np.median([len(seqs.get(x, {}).get("seq", []))
                                     for x in ids])))
    examples = ids_sorted[:n_show]

    for col_i in range(N_EXAMPLES):
        ax = axes[row_i, col_i]
        ax.set_facecolor("#0a0a1c")
        for sp in ax.spines.values():
            sp.set_edgecolor("#333366")
        ax.tick_params(colors="#aaaaaa", labelsize=6)

        if col_i >= n_show:
            ax.set_visible(False)
            continue

        aid = examples[col_i]
        sub = df_plot[df_plot["animal_id"] == aid]

        if len(sub) == 0:
            ax.text(0.5, 0.5, f"ID {aid}\nnot found",
                    ha="center", va="center", color="#ff4444",
                    transform=ax.transAxes, fontsize=8)
            continue

        x = sub[X_COL].values.astype(float)
        y = sub[Y_COL].values.astype(float)

        # Filter out aberrant DLC jumps
        dist = np.sqrt(np.diff(x)**2 + np.diff(y)**2)
        jump = dist > 100
        x_clean = x.copy(); y_clean = y.copy()
        x_clean[1:][jump] = np.nan
        y_clean[1:][jump] = np.nan

        # Colour by cluster if available
        clusters = (sub["cluster"].values
                    if "cluster" in sub.columns else None)

        if clusters is not None:
            for i in range(len(x_clean) - 1):
                if np.isnan(x_clean[i]) or np.isnan(x_clean[i+1]):
                    continue
                c_idx = int(clusters[i]) % len(PALETTE)
                ax.plot(x_clean[i:i+2], y_clean[i:i+2],
                        color=PALETTE[c_idx],
                        alpha=0.7, linewidth=1.2, zorder=2)
        else:
            # No cluster → group colour
            ax.plot(x_clean, y_clean, color=col_grp,
                    alpha=0.7, linewidth=1.2, zorder=2)

        # Start / end points
        x_valid = x_clean[~np.isnan(x_clean)]
        y_valid = y_clean[~np.isnan(y_clean)]
        if len(x_valid) > 0:
            ax.scatter(x_valid[0],  y_valid[0],
                       color="#44ff88", s=40, zorder=5,
                       edgecolors="white", linewidths=0.5)
            ax.scatter(x_valid[-1], y_valid[-1],
                       color="#ff4444", s=40, zorder=5,
                       edgecolors="white", linewidths=0.5)

        # Sequence summary
        sq  = seqs.get(aid, {}).get("seq", [])
        lng = seqs.get(aid, {}).get("lengths", [])
        seq_str = "→".join(str(c) for c in sq[:8])
        if len(sq) > 8: seq_str += "…"

        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_max, y_min)
        ax.set_aspect("equal")
        ax.grid(alpha=0.1, color="#334466")
        ax.set_title(
            f"ID {aid}\n"
            f"{len(sq)} blocks · {sum(lng)} frames\n"
            f"{seq_str}",
            color=col_grp, fontsize=7, pad=3)

        if col_i == 0:
            ax.set_ylabel(
                f"Group {grp}\nn={len(ids)}",
                color=col_grp, fontsize=9,
                fontweight="bold", labelpad=8)

# Cluster legend
patches = [mpatches.Patch(color=PALETTE[c % len(PALETTE)], label=f"C{c}")
           for c in range(n_cl)]
fig.legend(handles=patches, loc="lower center",
           ncol=min(n_cl, 12), fontsize=7,
           facecolor="#0d0d1a", edgecolor="#334466",
           labelcolor="#cccccc",
           title="Clusters Leiden", title_fontsize=8,
           bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
plt.savefig(OUT_PATH, dpi=150, bbox_inches="tight", facecolor="#1a1a2e")
print(f"\n[OK] → {OUT_PATH}")
plt.show()

# ── Text summary ──────────────────────────────────────────────
print(f"\n{'='*55}")
print(f"IDs PER GROUP — {type(FINAL_STATE_TO_USE).__name__}")
print(f"{'='*55}")
for grp in groups_sorted:
    ids = groups[grp]
    ids_sorted = sorted(
        ids,
        key=lambda a: abs(len(seqs.get(a, {}).get("seq", [])) -
                          np.median([len(seqs.get(x, {}).get("seq", []))
                                     for x in ids])))
    all_ids = sorted(ids, key=lambda x: int(x) if str(x).isdigit() else 0)
    print(f"\n  Group {grp}  (n={len(ids)})")
    print(f"    Examples shown : {ids_sorted[:N_EXAMPLES]}")
    print(f"    All IDs        : {all_ids}")

##########################################################################################################
######### Analysis of the strategies and their composition ###################################################
#######################################################################################

In [ ]:
# ===============================================================
#  CELL — Example videos per group
#  For each group: 5 randomly drawn individuals, played side by
#  side (mosaic), group after group, to compare the behaviours
#  by eye. Each tile shows the ID and the current state.
#  Keys (video window): space = pause | n = next group
#                       r = replay the group | q = quit
# ===============================================================
import re
import random
import numpy as np
import cv2
from pathlib import Path

# VIDEO_DIR: defined in cell 0 depending on USE_FULL (videos_catch or 01_VIDEO_Cropped)
# REGROUPED groups (optimal_regrouping) if they exist, otherwise the explorer ones
if "mapping_optimal" in globals():
    VIDEO_GROUPS, _src = mapping_optimal, "mapping_optimal (regrouped)"
else:
    VIDEO_GROUPS, _src = final_state["mapping_letters"], "final_state (explorer, not regrouped)"
N_PER_GROUP  = 5
VIDEO_SPEED  = 2       # 1 = real time, 2 = twice as fast, ...
TILE_W       = 440     # size of a tile in pixels
CROP_ARENA   = True    # True = crop the central square (the arena) -> larger cuttlefish
RANDOM_SEED  = None    # an integer to get the same individuals again


def _video_index(video_dir):
    """{individual number: file}, e.g. video100_cut.mp4 -> 100."""
    idx = {}
    for f in Path(video_dir).iterdir():
        m = re.search(r"\d+", f.stem)
        if m and f.suffix.lower() in (".mp4", ".avi", ".mov", ".mkv"):
            idx[int(m.group())] = f
    return idx


def _text(img, txt, org, scale=0.7, color=(255, 255, 255)):
    cv2.putText(img, txt, org, cv2.FONT_HERSHEY_SIMPLEX, scale, (0, 0, 0), 4, cv2.LINE_AA)
    cv2.putText(img, txt, org, cv2.FONT_HERSHEY_SIMPLEX, scale, color, 2, cv2.LINE_AA)


def _play_mosaic(win, g, n_g, pick, vids, labels, pfx, speed, tile_w, tile_h, crop, ranges):
    caps = [cv2.VideoCapture(str(vids[int(a)])) for a in pick]
    # starts at the individual's 1st frame in df and stops at its last one (the strike)
    n_fr = []
    for c, a in zip(caps, pick):
        f0, f1 = ranges[a]
        c.set(cv2.CAP_PROP_POS_FRAMES, f0)
        n_fr.append(max(1, min(f1 - f0 + 1, int(c.get(cv2.CAP_PROP_FRAME_COUNT)) - f0)))
    fps  = caps[0].get(cv2.CAP_PROP_FPS) or 30
    last = [np.zeros((tile_h, tile_w, 3), np.uint8) for _ in pick]
    pos, paused, shown = [0] * len(pick), False, 0
    try:
        while True:
            if not paused:
                for k, cap in enumerate(caps):
                    if pos[k] >= n_fr[k]:
                        continue
                    for _ in range(speed - 1):          # skip frames
                        cap.grab()
                    ok, fr = cap.read()
                    if ok:
                        if crop:                            # central square = the arena
                            h, w = fr.shape[:2]; x0 = (w - h) // 2
                            fr = fr[:, max(x0, 0):max(x0, 0) + h]
                        last[k] = cv2.resize(fr, (tile_w, tile_h))
                    pos[k] = min(pos[k] + speed, n_fr[k])
            done = all(p >= n for p, n in zip(pos, n_fr))

            tiles = []
            for k, a in enumerate(pick):
                t   = last[k].copy()
                i   = max(pos[k] - 1, 0)
                lab = labels.get(a)
                st  = f"  {pfx}{lab[min(i, len(lab) - 1)]}" if lab is not None and len(lab) else ""
                _text(t, f"ID {a}{st}", (10, 30), 0.8)
                _text(t, f"{i + 1}/{n_fr[k]} f" + ("   FIN" if pos[k] >= n_fr[k] else ""),
                      (10, tile_h - 15), 0.6, (200, 255, 200) if pos[k] >= n_fr[k] else (255, 255, 255))
                tiles.append(t)
            info = np.full((tile_h, tile_w, 3), 30, np.uint8)
            _text(info, f"Group {g}  (n={n_g})", (20, 50), 1.2, (80, 220, 255))
            _text(info, f"{len(pick)} random individuals", (20, 95), 0.7)
            _text(info, "Group finished" if done else ("PAUSE" if paused else f"playing x{speed}"),
                  (20, 145), 0.8, (120, 255, 120) if done else (255, 255, 255))
            for j, line in enumerate(["space : pause", "n : next group",
                                      "r : replay", "q : quit"]):
                _text(info, line, (20, 200 + 32 * j), 0.65, (200, 200, 200))
            tiles.append(info)
            while len(tiles) < 6:
                tiles.append(np.zeros((tile_h, tile_w, 3), np.uint8))
            cv2.imshow(win, np.vstack([np.hstack(tiles[:3]), np.hstack(tiles[3:6])]))
            shown += 1

            key = cv2.waitKey(max(1, int(1000 / fps)) if not (paused or done) else 60) & 0xFF
            # window closed with the cross? (checked only after a few frames:
            # right after creation, Windows may report it "not visible")
            closed = shown > 5 and cv2.getWindowProperty(win, cv2.WND_PROP_VISIBLE) < 1
            if key == ord("q") or closed:
                return "quit"
            if key == ord("n"):
                return "next"
            if key == ord("r"):
                return "replay"
            if key == ord(" "):
                paused = not paused
    finally:
        for c in caps:
            c.release()


def play_group_videos(groups, df, video_dir=VIDEO_DIR, n_per_group=5, speed=2,
                      tile_w=440, seed=None, state_col=None, crop=True):
    vids = _video_index(video_dir)
    rng  = random.Random(seed)
    d    = df.copy()
    d["animal_id"] = d["animal_id"].astype(str)
    state_col = state_col or final_state.get("col", "cluster")
    pfx    = "C" if state_col == "cluster" else "S"
    labels = {a: g.sort_values("frame")[state_col].to_numpy() for a, g in d.groupby("animal_id")}
    ranges = {a: (int(g["frame"].min()), int(g["frame"].max())) for a, g in d.groupby("animal_id")}

    by_group = {}
    for aid, g in groups.items():
        if g != "MONO":
            by_group.setdefault(g, []).append(str(aid))

    tile_h = tile_w if crop else int(tile_w * 9 / 16)
    # unique name at each run: avoids reusing a window being closed
    # (otherwise the 2nd run stops immediately)
    import time
    cv2.destroyAllWindows()
    for _ in range(5):
        cv2.waitKey(1)
    win    = f"Videos per group - {time.strftime('%H:%M:%S')}"
    cv2.namedWindow(win, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(win, 3 * tile_w, 2 * tile_h)
    try:
        for g in sorted(by_group):
            avail = [a for a in by_group[g] if a.isdigit() and int(a) in vids]
            if not avail:
                print(f"Group {g} : no video found in {video_dir}")
                continue
            pick = rng.sample(avail, min(n_per_group, len(avail)))
            print(f"Group {g} (n={len(by_group[g])}) -> individuals {pick}")
            action = "replay"
            while action == "replay":
                action = _play_mosaic(win, g, len(by_group[g]), pick, vids,
                                      labels, pfx, speed, tile_w, tile_h, crop, ranges)
            if action == "quit":
                break
    finally:
        cv2.destroyAllWindows()
        for _ in range(10):          # lets Windows really close the window
            cv2.waitKey(1)


print(f"[VIDEOS] groups from {_src}: {sorted({g for g in VIDEO_GROUPS.values() if g != 'MONO'})}")
play_group_videos(VIDEO_GROUPS, df, VIDEO_DIR, N_PER_GROUP, VIDEO_SPEED, TILE_W,
                  RANDOM_SEED, crop=CROP_ARENA)

In [ ]:
#def plot_cluster_probability_over_time

import matplotlib as mpl

def plot_cluster_probability_over_time(profiles, seqs, n_cl,
                                        strategy_letters, pal,cmap_cl, norm_cl,
                                        n_bins=100, smooth_sigma=3,
                                        min_prob_threshold=0.05):
    """
    Two figures per run:

    Figure 1 — all clusters (complete stacked area chart)
    Figure 2 — filtered clusters: in each bin, the clusters whose
               probability < min_prob_threshold are set to 0
               then renormalised → shows only the clusters that are
               significant at each point in time

    min_prob_threshold : minimum probability threshold (default 0.05 = 5%)
                         clusters below it are hidden in fig 2
    """
    from scipy.ndimage import gaussian_filter1d

    n_strat = len(strategy_letters)
    t_axis  = np.linspace(0, 1, n_bins)

    # ── Probabilities for all strategies ────
    # Computed once, drawn as two figures
    all_probs = {}   # {strategy: array (n_cl, n_bins)}

    for s in strategy_letters:
        if s not in profiles: continue
        p       = profiles[s]
        members = [m for m in p["members"] if m in seqs]
        if not members: continue

        cluster_matrix = np.full((len(members), n_bins), -1, dtype=int)

        for mi, aid in enumerate(members):
            seq     = seqs[aid]["seq"]
            lengths = seqs[aid]["lengths"]
            total   = sum(lengths)
            if total == 0: continue

            # Frame-by-frame timeline from the blocks
            timeline = []
            for c, l in zip(seq, lengths):
                timeline.extend([c] * int(l))
            timeline = np.array(timeline)
            n_frames = len(timeline)

            for b in range(n_bins):
                i_start = int(b * n_frames / n_bins)
                i_end   = int((b + 1) * n_frames / n_bins)
                chunk   = timeline[i_start:i_end]
                if len(chunk) > 0:
                    cluster_matrix[mi, b] = int(
                        np.bincount(chunk, minlength=n_cl).argmax()
                    )

        # Raw probabilities
        prob = np.zeros((n_cl, n_bins))
        for b in range(n_bins):
            col_b = cluster_matrix[:, b]
            valid = col_b[col_b >= 0]
            if len(valid) == 0: continue
            for c in range(n_cl):
                prob[c, b] = np.sum(valid == c) / len(valid)

        # Smoothing
        if smooth_sigma > 0:
            for c in range(n_cl):
                prob[c] = gaussian_filter1d(prob[c], sigma=smooth_sigma,
                                             mode='nearest')
            col_sums = prob.sum(axis=0)
            col_sums[col_sums == 0] = 1
            prob /= col_sums

        all_probs[s] = prob

    # ══════════════════════════════════════════════════════════
    # Drawing function shared by both figures
    # ══════════════════════════════════════════════════════════
    def _draw_prob_fig(prob_dict, title_suffix, filter_label=""):

        fig, axes = plt.subplots(n_strat, 1,
                                  figsize=(14, max(3, n_strat * 2.2)))
        fig.patch.set_facecolor("#1a1a2e")
        fig.suptitle(
            f"Probability of being in each cluster over time\n"
            f"{title_suffix}",
            color="#cccccc", fontsize=12
        )
        if n_strat == 1: axes_list = [axes]
        else:            axes_list = list(axes)

        for si, (s, ax) in enumerate(zip(strategy_letters, axes_list)):
            if s not in profiles or s not in prob_dict: continue
            p   = profiles[s]
            col = pal[s]

            members = [m for m in p["members"] if m in seqs]
            prob    = prob_dict[s]

            # Stacked area chart
            ax.set_facecolor("#0d0d1a")
            for sp in ax.spines.values(): sp.set_edgecolor("#333333")
            ax.tick_params(colors="#aaaaaa", labelsize=8)

            bottom = np.zeros(n_bins)
            for c in range(n_cl):
                c_color = cmap_cl(norm_cl(c))
                vals    = prob[c]

                ax.fill_between(t_axis, bottom, bottom + vals,
                                color=c_color, alpha=0.85,
                                label=f"C{c}", zorder=2)
                ax.plot(t_axis, bottom + vals,
                        color="#0d0d1a", lw=0.4, alpha=0.6, zorder=3)
                bottom += vals

            # Midline
            ax.axvline(0.5, color="#ffffff", lw=0.8, ls=":",
                       alpha=0.3, zorder=4)

            # Dominant cluster annotations
            dominant  = np.argmax(prob, axis=0)
            prev_c    = dominant[0]
            run_start = 0
            for b in range(1, n_bins):
                if dominant[b] != prev_c or b == n_bins - 1:
                    run_len = b - run_start
                    if run_len >= n_bins * 0.05:
                        mid_b = (run_start + b) // 2
                        mid_x = t_axis[mid_b]
                        bot_c = prob[:prev_c, mid_b].sum()
                        top_c = bot_c + prob[prev_c, mid_b]
                        mid_y = (bot_c + top_c) / 2
                        if prob[prev_c, mid_b] > 0.08:
                            ax.text(mid_x, mid_y, f"C{prev_c}",
                                    ha="center", va="center",
                                    fontsize=7, color="white",
                                    fontweight="bold", zorder=5,
                                    alpha=0.9)
                    run_start = b
                    prev_c    = dominant[b]

            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)
            ax.set_ylabel(
                f"Strat. {s}\nn={len(members)}",
                color=col, fontsize=9, rotation=0,
                ha="right", va="center", labelpad=55
            )
            ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
            ax.set_yticklabels(["0%","25%","50%","75%","100%"],
                                color="#aaaaaa", fontsize=7)
            ax.grid(axis="x", color="#222233", alpha=0.5, lw=0.5)

            if si == n_strat - 1:
                ax.set_xlabel(
                    "Normalised time position  (0=start  1=end)",
                    color="#aaaaaa", fontsize=10)
            else:
                ax.set_xticklabels([])

            # Strategy colour bar on the left
            ax.barh(0.5, 0.008, left=-0.02, height=1.0,
                    color=col, alpha=0.9, transform=ax.transData,
                    clip_on=False, zorder=5)

            # Threshold annotation in the filtered figure
            if filter_label:
                ax.text(0.01, 0.97,
                        f"threshold = {int(min_prob_threshold*100)}%",
                        transform=ax.transAxes,
                        color="#777788", fontsize=7,
                        va="top", ha="left")

        # Legend
        legend_patches = [
            mpatches.Patch(color=cmap_cl(norm_cl(c)), label=f"C{c}")
            for c in range(n_cl)
        ]
        leg = axes_list[0].legend(
            handles=legend_patches,
            loc="upper right", ncol=min(n_cl, 6),
            facecolor="#0d0d1a", labelcolor="#cccccc",
            fontsize=8, title="Clusters", title_fontsize=8,
        )
        leg.get_frame().set_edgecolor("#444444")
        leg.get_title().set_color("#aaaaaa")

        plt.tight_layout()
        plt.show()

    # ══════════════════════════════════════════════════════════
    # FIGURE 1 — All clusters
    # ══════════════════════════════════════════════════════════
    _draw_prob_fig(
        prob_dict    = all_probs,
        title_suffix = "(all clusters — normalised time position)"
    )

    # ══════════════════════════════════════════════════════════
    # FIGURE 2 — Filtered clusters (prob < threshold → hidden)
    # In each bin: the clusters whose prob < min_prob_threshold
    # are set to 0, then the remaining probabilities are renormalised
    # → only the "active" clusters at each moment are shown
    # ══════════════════════════════════════════════════════════
    all_probs_filtered = {}

    for s, prob in all_probs.items():
        prob_f = prob.copy()

        # Mask: set clusters below the threshold to 0, bin by bin
        prob_f[prob_f < min_prob_threshold] = 0

        # Renormalise bin by bin so the sum stays = 1
        # → the freed space is redistributed proportionally
        col_sums = prob_f.sum(axis=0)
        col_sums[col_sums == 0] = 1
        prob_f /= col_sums

        all_probs_filtered[s] = prob_f

    _draw_prob_fig(
        prob_dict    = all_probs_filtered,
        title_suffix = (f"filtered clusters — threshold {int(min_prob_threshold*100)}% "
                        f"(clusters < {int(min_prob_threshold*100)}% hidden "
                        f"and redistributed)"),
        filter_label = True
    )


# ── Call ─────────────────────────────────────────────────────
# Prerequisite: profiles comes from plot_strategy_profiles
# profiles, mapping_letters = plot_strategy_profiles(...)

seqs             = final_state["seqs"]
n_cl             = final_state["n_cl"]
strategy_letters = sorted(profiles.keys())   # A, B, C... from profiles

pal         = {s: STRATEGY_PALETTE[i % len(STRATEGY_PALETTE)]
               for i, s in enumerate(strategy_letters)}
pal["MONO"] = "#666666"

# Cluster colours — same palette as everywhere else
cmap_cl = mpl.colors.ListedColormap(PALETTE)
norm_cl = mpl.colors.BoundaryNorm(range(len(PALETTE) + 1),
                                   ncolors=len(PALETTE))



In [ ]:
plot_cluster_probability_over_time(
    profiles          = profiles,
    seqs              = seqs,
    n_cl              = n_cl,
    strategy_letters  = strategy_letters,
    pal               = pal,
    cmap_cl           = cmap_cl,
    norm_cl           = norm_cl,
    n_bins            = 100,
    smooth_sigma      = 3,
    min_prob_threshold= 0.05,
)


In [ ]:
def plot_strategy_features_over_time(
    profiles, final_state, df,
    features      = None,
    n_bins        = 50,
    smooth_sigma  = 2,
    show_variance = True,
):
    """
    For each strategy: mean (± standard deviation) of each feature
    over normalised time [0=start, 1=end].

    Principle:
      1. For each individual, its frames are split into n_bins
         equal bins according to their normalised time position
      2. The mean of each feature is computed in each bin
      3. It is averaged over all individuals of the strategy
      4. The curve + variance band are displayed

    features      : list of columns to plot (default = FEATURES)
    n_bins        : time resolution (50 = fairly smooth)
    smooth_sigma  : Gaussian smoothing of the curves (0 = disabled)
    show_variance : show the standard deviation as a shaded band
    """
    from scipy.ndimage import gaussian_filter1d

    if features is None:
        features = FEATURES

    seqs             = final_state["seqs"]
    strategy_letters = sorted(profiles.keys())
    n_strat          = len(strategy_letters)
    n_feat           = len(features)
    t_axis           = np.linspace(0, 1, n_bins)

    pal         = {s: STRATEGY_PALETTE[i % len(STRATEGY_PALETTE)]
                   for i, s in enumerate(strategy_letters)}
    pal["MONO"] = "#666666"

    df_w = df.copy()
    df_w["animal_id"] = df_w["animal_id"].astype(str)

    # ── Computation of the time profiles ──────────────────────────
    # For each strategy and each feature:
    #   all_curves[s][feat] = array (n_members, n_bins)
    all_means = {}   # {s: array (n_feat, n_bins)}
    all_stds  = {}   # {s: array (n_feat, n_bins)}

    for s in strategy_letters:
        if s not in profiles: continue
        members = [m for m in profiles[s]["members"] if m in seqs]
        if not members: continue

        # Matrix (n_members, n_feat, n_bins)
        feat_matrix = np.full((len(members), n_feat, n_bins), np.nan)

        for mi, aid in enumerate(members):
            # Get the frames of this individual in df
            mask_ind = df_w["animal_id"] == str(aid)
            df_ind   = df_w[mask_ind].copy()

            if "frame" in df_ind.columns:
                df_ind = df_ind.sort_values("frame")

            n_frames = len(df_ind)
            if n_frames == 0: continue

            # Check that all features are present
            feats_ok = [f for f in features if f in df_ind.columns]
            if not feats_ok: continue

            # Split into n_bins according to time position
            for b in range(n_bins):
                i_start = int(b * n_frames / n_bins)
                i_end   = int((b + 1) * n_frames / n_bins)
                if i_end <= i_start:
                    i_end = i_start + 1
                i_end = min(i_end, n_frames)

                chunk = df_ind.iloc[i_start:i_end]
                if len(chunk) == 0: continue

                for fi, feat in enumerate(features):
                    if feat in chunk.columns:
                        feat_matrix[mi, fi, b] = chunk[feat].mean()

        # Mean and standard deviation over the individuals
        # nanmean/nanstd ignore bins without data
        mean_mat = np.nanmean(feat_matrix, axis=0)   # (n_feat, n_bins)
        std_mat  = np.nanstd(feat_matrix,  axis=0)   # (n_feat, n_bins)

        # Gaussian smoothing of the curves
        if smooth_sigma > 0:
            for fi in range(n_feat):
                mean_mat[fi] = gaussian_filter1d(
                    mean_mat[fi], sigma=smooth_sigma, mode='nearest')
                std_mat[fi]  = gaussian_filter1d(
                    std_mat[fi],  sigma=smooth_sigma, mode='nearest')

        all_means[s] = mean_mat
        all_stds[s]  = std_mat

        print(f"[FEATURES/TIME] Strategy {s} : "
              f"{len(members)} individuals × {n_feat} features × {n_bins} bins")

    if not all_means:
        print("[ERROR] No data computed.")
        return

    # ════════════════════════════════════════════════════════
    # FIGURE — one row per strategy, one column per feature
    # ════════════════════════════════════════════════════════
    fig, axes = plt.subplots(
        n_strat, n_feat,
        figsize=(n_feat * 3.5, n_strat * 2.5),
        sharex=True
    )
    fig.patch.set_facecolor("#1a1a2e")
    fig.suptitle(
        "Behavioural features over normalised time\n"
        "per strategy  (mean ± standard deviation over individuals)",
        color="#cccccc", fontsize=13, y=1.01
    )

    # Normalise axes to a 2D array even with a single row/column
    if n_strat == 1 and n_feat == 1:
        axes = np.array([[axes]])
    elif n_strat == 1:
        axes = axes[np.newaxis, :]
    elif n_feat == 1:
        axes = axes[:, np.newaxis]

    # Compute the global limits per feature
    # → same Y scale for all strategies of a feature
    ylims = {}
    for fi, feat in enumerate(features):
        all_vals = []
        for s in strategy_letters:
            if s not in all_means: continue
            m = all_means[s][fi]
            e = all_stds[s][fi]
            all_vals.extend((m - e).tolist())
            all_vals.extend((m + e).tolist())
        if all_vals:
            vmin = np.nanmin(all_vals)
            vmax = np.nanmax(all_vals)
            margin = (vmax - vmin) * 0.1
            ylims[fi] = (vmin - margin, vmax + margin)
        else:
            ylims[fi] = (0, 1)

    for si, s in enumerate(strategy_letters):
        if s not in all_means: continue
        col      = pal[s]
        mean_mat = all_means[s]
        std_mat  = all_stds[s]
        n_ind    = len(profiles[s]["members"])

        for fi, feat in enumerate(features):
            ax = axes[si, fi]
            ax.set_facecolor("#0d0d1a")
            for sp in ax.spines.values():
                sp.set_edgecolor("#333333")
            ax.tick_params(colors="#555555", labelsize=7)

            mean_v = mean_mat[fi]
            std_v  = std_mat[fi]

            # Mean curve
            ax.plot(t_axis, mean_v,
                    color=col, lw=2.0, alpha=0.95, zorder=3)

            # ± 1 standard deviation band
            if show_variance:
                ax.fill_between(
                    t_axis,
                    mean_v - std_v,
                    mean_v + std_v,
                    color=col, alpha=0.20, zorder=2
                )

            # Vertical line in the middle
            ax.axvline(0.5, color="#ffffff", lw=0.6,
                       ls=":", alpha=0.25, zorder=4)

            # Horizontal line at 0 if relevant
            if ylims[fi][0] < 0 < ylims[fi][1]:
                ax.axhline(0, color="#444455", lw=0.5,
                           ls="-", alpha=0.5, zorder=1)

            ax.set_xlim(0, 1)
            ax.set_ylim(ylims[fi])
            ax.grid(axis="x", color="#222233", alpha=0.4, lw=0.5)

            # Column title (feature) — only on the 1st row
            if si == 0:
                ax.set_title(
                    feat.replace("_", "\n"),
                    color="#aaaaaa", fontsize=9, pad=4
                )

            # Row label (strategy) — only on the 1st column
            if fi == 0:
                ax.set_ylabel(
                    f"Strat. {s}\nn={n_ind}",
                    color=col, fontsize=9,
                    rotation=0, ha="right",
                    va="center", labelpad=55
                )
            else:
                ax.set_ylabel("")

            # X axis — only on the last row
            if si == n_strat - 1:
                ax.set_xlabel("normalised t",
                              color="#555555", fontsize=7)
            else:
                ax.set_xticklabels([])

    plt.tight_layout()
    plt.show()


    # FIGURE 2 — All strategies overlaid per feature
    # ════════════════════════════════════════════════════════
    n_cols = min(3, n_feat)
    n_rows = (n_feat + n_cols - 1) // n_cols

    fig2, axes2 = plt.subplots(
        n_rows, n_cols,
        figsize=(n_cols * 4.5, n_rows * 3.5),
        sharex=True
    )
    fig2.patch.set_facecolor("#1a1a2e")
    fig2.suptitle(
        "Behavioural features — all strategies overlaid\n"
        "(mean ± standard deviation  |  normalised time position)",
        color="#cccccc", fontsize=13, y=1.01
    )

    # Normalise axes to a 2D array
    if n_rows == 1 and n_cols == 1:
        axes2 = np.array([[axes2]])
    elif n_rows == 1:
        axes2 = axes2[np.newaxis, :]
    elif n_cols == 1:
        axes2 = axes2[:, np.newaxis]

    for fi, feat in enumerate(features):
        ri = fi // n_cols
        ci = fi % n_cols
        ax = axes2[ri, ci]
        ax.set_facecolor("#0d0d1a")
        for sp in ax.spines.values(): sp.set_edgecolor("#333333")
        ax.tick_params(colors="#555555", labelsize=7)

        for s in strategy_letters:
            if s not in all_means: continue
            col      = pal[s]
            mean_v   = all_means[s][fi]
            std_v    = all_stds[s][fi]
            n_ind    = len(profiles[s]["members"])

            ax.plot(t_axis, mean_v,
                    color=col, lw=2.0, alpha=0.95,
                    label=f"{s} (n={n_ind})", zorder=3)

            if show_variance:
                ax.fill_between(
                    t_axis,
                    mean_v - std_v,
                    mean_v + std_v,
                    color=col, alpha=0.12, zorder=2
                )

        ax.axvline(0.5, color="#ffffff", lw=0.6,
                   ls=":", alpha=0.25, zorder=4)

        if ylims[fi][0] < 0 < ylims[fi][1]:
            ax.axhline(0, color="#444455", lw=0.5,
                       ls="-", alpha=0.5, zorder=1)

        ax.set_xlim(0, 1)
        ax.set_ylim(ylims[fi])
        ax.grid(axis="x", color="#222233", alpha=0.4, lw=0.5)
        ax.set_title(feat.replace("_", "\n"),
                     color="#aaaaaa", fontsize=10, pad=4)

        if ri == n_rows - 1:
            ax.set_xlabel("normalised t", color="#555555", fontsize=8)

        # Legend on the first subplot only
        if fi == 0:
            leg = ax.legend(
                fontsize=8, loc="upper right",
                facecolor="#111122", labelcolor="#cccccc",
                framealpha=0.8
            )
            leg.get_frame().set_edgecolor("#444444")

    # Hide the empty axes if n_feat < n_rows * n_cols
    for fi in range(n_feat, n_rows * n_cols):
        axes2[fi // n_cols, fi % n_cols].set_visible(False)

    plt.tight_layout()
    plt.show()

In [ ]:
# table of the different strategies per individual

global_strategies = pd.DataFrame() 
global_strategies["strat_base"] = final_state["strat"]
global_strategies["strat_opti"] = final_state_optimal["strat"]
global_strategies = global_strategies.iloc[:-2].copy()

counts      = global_strategies["strat_base"].value_counts()
counts_true = counts[counts >= 7].index          # Index: accepted by isin
mask        = global_strategies["strat_base"].isin(counts_true)
mapping_base = global_strategies.loc[mask, "strat_base"].to_dict()

profiles_base, mapping_letters = plot_strategy_profiles(
    mapping_base,
    final_state,
    df,
    features = FEATURES,
)


seqs             = final_state["seqs"]
n_cl             = final_state["n_cl"]
strategy_letters = sorted(profiles_base.keys())   # A, B, C... from profiles

pal         = {s: STRATEGY_PALETTE[i % len(STRATEGY_PALETTE)]
               for i, s in enumerate(strategy_letters)}
pal["MONO"] = "#666666"

# Cluster colours — same palette as everywhere else
cmap_cl = mpl.colors.ListedColormap(PALETTE)
norm_cl = mpl.colors.BoundaryNorm(range(len(PALETTE) + 1),
                                   ncolors=len(PALETTE))

plot_cluster_probability_over_time(
    profiles          = profiles_base,
    seqs              = seqs,
    n_cl              = n_cl,
    strategy_letters  = strategy_letters,
    pal               = pal,
    cmap_cl           = cmap_cl,
    norm_cl           = norm_cl,
    n_bins            = 100,
    smooth_sigma      = 3,
    min_prob_threshold= 0.05,
)

In [ ]:
counts 


In [ ]:
# ── Call ─────────────────────────────────────────────────────
plot_strategy_features_over_time(
    profiles      = profiles,
    final_state   = final_state,
    df            = df,
    features      = FEATURES,
    n_bins        = 50,
    smooth_sigma  = 2,
    show_variance = True,
)

In [ ]:
#  CELL — Comparative plots between strategies
#  Based on df (behavioural features) + mapping_letters
#  (strategies resulting from the optimal regrouping)
# ══════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib as mpl
from scipy import stats
from scipy.stats import mannwhitneyu, kruskal
from itertools import combinations
from collections import Counter

# ── Strategy palette ────────────────────────────────────────
strategy_letters_plot = sorted(
    set(v for v in mapping_letters.values() if v != "MONO")
)
all_groups_plot = strategy_letters_plot + (
    ["MONO"] if "MONO" in set(mapping_letters.values()) else []
)
STRAT_PAL = [
    "#FF4FB8","#C97E48","#8C6BB1","#F5E21E","#3A8891",
    "#E45756","#4CAF50","#1F77B4","#FF9F1C","#A05195",
    "#2CB1BC","#D62728","#9467BD","#8C564B","#E377C2",
]
pal_strat = {s: STRAT_PAL[i % len(STRAT_PAL)]
             for i, s in enumerate(all_groups_plot)}
pal_strat["MONO"] = "#666666"

n_strat = len(strategy_letters_plot)

# ── Build df individual × features + strategy ─────────────
df_w = df.copy()
df_w["animal_id"] = df_w["animal_id"].astype(str)

df_indiv = df_w.groupby("animal_id")[FEATURES].mean()
df_indiv["strategy"] = [
    mapping_letters.get(str(aid), "MONO")
    for aid in df_indiv.index
]
df_indiv = df_indiv.dropna(subset=FEATURES)

print(f"[OK] {len(df_indiv)} individuals | {n_strat} strategies")
print(f"     Distribution : {Counter(df_indiv['strategy'].values)}")


# ════════════════════════════════════════════════════════════
# FIGURE 1 — Boxplots per feature × strategy
# ════════════════════════════════════════════════════════════

n_feats = len(FEATURES)
fig, axes = plt.subplots(1, n_feats, figsize=(4 * n_feats, 6))
fig.patch.set_facecolor("#1a1a2e")
if n_feats == 1: axes = [axes]

for ax, feat in zip(axes, FEATURES):
    ax.set_facecolor("#0d0d1a")
    for sp in ax.spines.values(): sp.set_edgecolor("#333333")
    ax.tick_params(colors="#aaaaaa", labelsize=8)

    data_per_strat = [
        df_indiv.loc[df_indiv["strategy"] == s, feat].dropna().values
        for s in strategy_letters_plot
    ]
    labels = [f"{s}\nn={len(d)}" for s, d in zip(strategy_letters_plot, data_per_strat)]
    colors_bp = [pal_strat[s] for s in strategy_letters_plot]

    bp = ax.boxplot(
        data_per_strat, labels=labels, patch_artist=True,
        medianprops=dict(color="white", linewidth=2),
        whiskerprops=dict(linewidth=1.2, color="#aaaaaa"),
        capprops=dict(linewidth=1.2, color="#aaaaaa"),
        flierprops=dict(marker="o", markersize=2, alpha=0.4,
                        markerfacecolor="#aaaaaa", markeredgecolor="none"),
    )
    for patch, col in zip(bp["boxes"], colors_bp):
        patch.set_facecolor(col); patch.set_alpha(0.75)

    # Mean ± SEM overlaid
    for i, (vals, s) in enumerate(zip(data_per_strat, strategy_letters_plot), 1):
        if len(vals) == 0: continue
        mean = vals.mean()
        sem  = vals.std(ddof=1) / np.sqrt(len(vals)) if len(vals) > 1 else 0
        ax.errorbar(i, mean, yerr=sem, fmt="D", color="white",
                    markersize=5, capsize=4, linewidth=1.5, zorder=5)

    # Robust Y limits
    all_vals = np.concatenate([v for v in data_per_strat if len(v)])
    if len(all_vals):
        lo, hi = np.percentile(all_vals, [2, 98])
        margin = (hi - lo) * 0.15 or abs(hi) * 0.1 or 1
        ax.set_ylim(lo - margin, hi + margin)

    # Global Kruskal-Wallis test
    valid = [d for d in data_per_strat if len(d) >= 3]
    if len(valid) >= 2:
        try:
            _, p_kw = kruskal(*valid)
            sig = "***" if p_kw < 0.001 else "**" if p_kw < 0.01 else \
                  "*" if p_kw < 0.05 else "ns"
            ax.set_title(f"{feat.replace('_',' ')}\nKW p={p_kw:.3f} {sig}",
                         color="#cccccc", fontsize=9)
        except Exception:
            ax.set_title(feat.replace('_',' '), color="#cccccc", fontsize=9)
    else:
        ax.set_title(feat.replace('_',' '), color="#cccccc", fontsize=9)

    ax.tick_params(axis="x", rotation=45)

fig.suptitle("Distributions of the behavioural features per strategy",
             color="#cccccc", fontsize=12, y=1.01)
plt.tight_layout()
plt.show()


# ════════════════════════════════════════════════════════════
# FIGURE 2 — Heatmap of the normalised means per strategy
# ════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(max(8, n_strat * 0.8), 5))
fig.patch.set_facecolor("#1a1a2e")
ax.set_facecolor("#0d0d1a")

# Strategy × feature matrix (means normalised 0-1)
mat = np.zeros((n_strat, n_feats))
for i, s in enumerate(strategy_letters_plot):
    for j, feat in enumerate(FEATURES):
        vals = df_indiv.loc[df_indiv["strategy"] == s, feat].dropna().values
        mat[i, j] = vals.mean() if len(vals) > 0 else np.nan

# Min-max normalisation per feature
mat_norm = mat.copy()
for j in range(n_feats):
    col = mat[:, j]
    valid = col[~np.isnan(col)]
    if len(valid) > 0 and valid.max() != valid.min():
        mat_norm[:, j] = (col - valid.min()) / (valid.max() - valid.min())
    else:
        mat_norm[:, j] = 0.5

im = ax.imshow(mat_norm, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)

# Labels and annotations
ax.set_xticks(range(n_feats))
ax.set_xticklabels([f.replace("_","\n") for f in FEATURES],
                    color="#cccccc", fontsize=8)
ax.set_yticks(range(n_strat))
ax.set_yticklabels(
    [f"Strat. {s}  (n={int((df_indiv['strategy']==s).sum())})"
     for s in strategy_letters_plot],
    color="#cccccc", fontsize=9
)
# Colour the Y labels according to the strategy
for ytick, s in zip(ax.get_yticklabels(), strategy_letters_plot):
    ytick.set_color(pal_strat[s])

# Raw values in the cells
for i in range(n_strat):
    for j in range(n_feats):
        v = mat[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                    fontsize=8, color="black" if 0.2 < mat_norm[i,j] < 0.8 else "white",
                    fontweight="bold")

plt.colorbar(im, ax=ax, label="Normalised value (0=min  1=max)",
             fraction=0.03, pad=0.02).ax.tick_params(colors="white")
ax.set_title("Heatmap of the means per strategy × feature\n"
             "(colour normalised per feature | raw value shown)",
             color="#cccccc", fontsize=11)
for sp in ax.spines.values(): sp.set_edgecolor("#333333")
plt.tight_layout()
plt.show()


# ════════════════════════════════════════════════════════════
# FIGURE 3 — Overlaid comparative radars
# ════════════════════════════════════════════════════════════

angles   = np.linspace(0, 2*np.pi, n_feats, endpoint=False)
ang_full = np.concatenate([angles, [angles[0]]])

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
fig.patch.set_facecolor("#1a1a2e")
ax.set_facecolor("#0d0d1a")

global_min = np.nanmin(mat, axis=0)
global_max = np.nanmax(mat, axis=0)

for i, s in enumerate(strategy_letters_plot):
    row = mat[i]
    norm_row = np.where(
        global_max != global_min,
        (row - global_min) / (global_max - global_min + 1e-9),
        0.5
    )
    vals_f = np.concatenate([norm_row, [norm_row[0]]])
    col    = pal_strat[s]
    n_s    = int((df_indiv["strategy"] == s).sum())
    ax.plot(ang_full, vals_f, lw=2, color=col, label=f"Strat. {s} (n={n_s})")
    ax.fill(ang_full, vals_f, alpha=0.08, color=col)

ax.set_xticks(angles)
ax.set_xticklabels([f.replace("_","\n") for f in FEATURES],
                    color="#aaaaaa", fontsize=9)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75])
ax.set_yticklabels(["25%", "50%", "75%"], color="#555555", fontsize=7)
ax.grid(color="#333333", lw=0.5)
ax.spines["polar"].set_color("#333333")
ax.set_title("Comparative radar profiles\n(min-max normalised per feature)",
             color="#cccccc", fontsize=11, pad=20)

leg = ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1),
                fontsize=9, framealpha=0.5)
leg.get_frame().set_facecolor("#111111")
leg.get_frame().set_edgecolor("#333333")
for t in leg.get_texts(): t.set_color("white")

plt.tight_layout()
plt.show()


# ════════════════════════════════════════════════════════════
# FIGURE 4 — p-value matrix (pairwise Mann-Whitney)
# ════════════════════════════════════════════════════════════

n_s   = len(strategy_letters_plot)
fig, axes = plt.subplots(1, n_feats, figsize=(4 * n_feats, max(4, n_s * 0.6 + 1)))
fig.patch.set_facecolor("#1a1a2e")
if n_feats == 1: axes = [axes]

for ax, feat in zip(axes, FEATURES):
    ax.set_facecolor("#0d0d1a")
    for sp in ax.spines.values(): sp.set_edgecolor("#333333")

    # p-value matrix
    p_mat = np.ones((n_s, n_s))
    for i, s1 in enumerate(strategy_letters_plot):
        for j, s2 in enumerate(strategy_letters_plot):
            if i >= j: continue
            x1 = df_indiv.loc[df_indiv["strategy"]==s1, feat].dropna().values
            x2 = df_indiv.loc[df_indiv["strategy"]==s2, feat].dropna().values
            if len(x1) >= 3 and len(x2) >= 3:
                try:
                    _, p = mannwhitneyu(x1, x2, alternative="two-sided")
                    p_mat[i, j] = p
                    p_mat[j, i] = p
                except Exception:
                    pass

    # Log-transform for readability
    p_log = -np.log10(np.clip(p_mat, 1e-10, 1))
    np.fill_diagonal(p_log, 0)

    im = ax.imshow(p_log, cmap="Reds", aspect="auto", vmin=0, vmax=3)

    ax.set_xticks(range(n_s))
    ax.set_yticks(range(n_s))
    ax.set_xticklabels(strategy_letters_plot, color="#cccccc",
                        fontsize=8, rotation=45)
    ax.set_yticklabels(strategy_letters_plot, color="#cccccc", fontsize=8)

    for i in range(n_s):
        for j in range(n_s):
            if i == j: continue
            p_val = p_mat[i, j]
            sig   = "***" if p_val < 0.001 else "**" if p_val < 0.01 else \
                    "*"   if p_val < 0.05  else "ns"
            col_t = "white" if p_log[i,j] > 1.5 else "#cccccc"
            ax.text(j, i, sig, ha="center", va="center",
                    fontsize=7, color=col_t, fontweight="bold")

    ax.set_title(f"{feat.replace('_',' ')}\n-log10(p)",
                 color="#cccccc", fontsize=8)
    plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02
                 ).ax.tick_params(colors="white", labelsize=6)

fig.suptitle("Significance of the pairwise differences (Mann-Whitney)\n"
             "*** p<0.001  ** p<0.01  * p<0.05  ns = not significant",
             color="#cccccc", fontsize=11, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
#  Additional imports for trajectory analysis
import ast
from scipy.fft import fft
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score

# FEATURES and PALETTE already defined in cell 0
N_CLUSTERS = final_state["n_cl"]
print(f" Imports OK | N_CLUSTERS={N_CLUSTERS} | FEATURES={FEATURES}")

In [ ]:
#  Cluster centres in feature space
# Computed directly from df (already loaded in memory)
cluster_col = final_state["col"]   # "supercluster" or "cluster"
n_cl        = final_state["n_cl"]

cluster_ids        = sorted(df[cluster_col].unique())
cluster_stats_rows = []
for k in cluster_ids:
    mask = df[cluster_col] == k
    cluster_stats_rows.append(df.loc[mask, FEATURES].mean().values)

cluster_stats = pd.DataFrame(cluster_stats_rows, columns=FEATURES,
                              index=cluster_ids)
final_state["cluster_stats"] = cluster_stats

seqs_dict = final_state["seqs"]
n_mono    = sum(1 for s in seqs_dict.values() if len(s["seq"]) <= 1)

print(f"✅ cluster_stats : {cluster_stats.shape}")
print(cluster_stats.round(4))
print(f"\n✅ {len(seqs_dict)} individuals including {n_mono} MONO")
# mapping_letters is already defined (returned by plot_strategy_profiles)
print(f"✅ mapping_letters : {len(mapping_letters)} entries")

In [ ]:
# def plot 3D
from scipy.ndimage import gaussian_filter
from scipy.interpolate import griddata

def _compute_temporal_grid(df_w, resolution=100, smoothing=5,
                            normalize_by_individual=True):
    # ── Fix: reset the index to avoid IndexErrors ──
    # When df_w is a filtered sub-DataFrame, the pandas indices
    # no longer match the positions 0..n-1
    # → reset_index() guarantees that idx goes from 0 to len(df_w)-1
    df_w = df_w.reset_index(drop=True)

    xg = df_w["umap_x"].values
    yg = df_w["umap_y"].values

    if normalize_by_individual:
        zg = np.zeros(len(df_w))
        for aid, grp in df_w.groupby("animal_id"):
            idx = grp.index   # ← these indices are now really 0..n-1
            n   = len(grp)
            if n == 1:
                zg[idx] = 1.0
            else:
                zg[idx] = 1.0 - np.arange(n) / (n - 1)
    else:
        n  = len(df_w)
        zg = 1.0 - np.arange(n) / max(n - 1, 1)

    # the rest is unchanged...
    x_min, x_max = xg.min(), xg.max()
    y_min, y_max = yg.min(), yg.max()
    margin_x = (x_max - x_min) * 0.05
    margin_y = (y_max - y_min) * 0.05

    xi = np.linspace(x_min - margin_x, x_max + margin_x, resolution)
    yi = np.linspace(y_min - margin_y, y_max + margin_y, resolution)
    Xi, Yi = np.meshgrid(xi, yi)

    pts = np.column_stack([xg, yg])
    Zi  = griddata(pts, zg, (Xi, Yi), method='linear')

    mask_nan = np.isnan(Zi)
    if mask_nan.any():
        Zi_near      = griddata(pts, zg, (Xi, Yi), method='nearest')
        Zi[mask_nan] = Zi_near[mask_nan]

    if smoothing > 0:
        Zi = gaussian_filter(Zi, sigma=smoothing)

    Zi = np.clip(Zi, 0, 1)

    return Zi, Xi, Yi, xg, yg, zg

def plot_temporal_group_overlay(Z, df, mapping_letters,
                                 group_label,
                                 smoothing=5,
                                 resolution=100,
                                 normalize_by_individual=True,
                                 colormap="RdYlGn_r",
                                 stride=2):

    df_w = df.copy()
    df_w["animal_id"] = df_w["animal_id"].astype(str)
    df_w["umap_x"]    = Z[:, 0]
    df_w["umap_y"]    = Z[:, 1]
    df_w["group_label"]    = df_w["animal_id"].map(
        lambda a: mapping_letters.get(a, "MONO")
    )

    group_list = sorted(set(v for v in mapping_letters.values() if v != "MONO"))
    pal        = {g: PALETTE[i % len(PALETTE)] for i, g in enumerate(group_list)}
    col        = pal.get(group_label, "#4CAF50")
    n_ind      = df_w[df_w["group_label"] == group_label]["animal_id"].nunique()

    if group_label not in group_list:
        print(f"❌ Group '{group_label}' not found. Available : {group_list}")
        return

    cluster_col = ("supercluster" if "supercluster" in df_w.columns
                   else "cluster"  if "cluster"      in df_w.columns
                   else None)

    print(f"[OVERLAY] Computing global grid...")
    Zi_global, Xi, Yi, xg, yg, zg = _compute_temporal_grid(
        df_w, resolution, smoothing, normalize_by_individual
    )

    print(f"[OVERLAY] Computing grid for group {group_label} (n={n_ind})...")
    df_g = df_w[df_w["group_label"] == group_label]
    Zi_g, Xi_g, Yi_g, xg_, yg_, zg_ = _compute_temporal_grid(
        df_g, resolution, smoothing, normalize_by_individual
    )

    x_min, x_max = Xi.min(), Xi.max()
    y_min, y_max = Yi.min(), Yi.max()

    # ════════════════════════════════════════════════════════
    # FIGURE 1 — 3D
    # ════════════════════════════════════════════════════════
    fig1 = plt.figure(figsize=(12, 9))
    fig1.patch.set_facecolor("#1a1a2e")
    ax3d = fig1.add_subplot(111, projection='3d')
    ax3d.set_facecolor("#0d0d1a")

    ax3d.plot_surface(
        Xi[::stride*2, ::stride*2],
        Yi[::stride*2, ::stride*2],
        Zi_global[::stride*2, ::stride*2],
        cmap='gray', alpha=0.15,
        linewidth=0, antialiased=False, vmin=0, vmax=1,
    )
    surf = ax3d.plot_surface(
        Xi_g[::stride, ::stride],
        Yi_g[::stride, ::stride],
        Zi_g[::stride, ::stride],
        cmap=colormap, alpha=0.88,
        linewidth=0, antialiased=False, vmin=0, vmax=1,
    )
    ax3d.contourf(Xi_g, Yi_g, Zi_g, zdir='z', offset=-0.05,
                  cmap=colormap, alpha=0.25, levels=10)
    ax3d.set_zlim(-0.1, 1.1)
    ax3d.set_xlabel("UMAP D1",           color="#aaaaaa", labelpad=8)
    ax3d.set_ylabel("UMAP D2",           color="#aaaaaa", labelpad=8)
    ax3d.set_zlabel("Time position", color="#aaaaaa", labelpad=8)
    ax3d.tick_params(colors="#555555", labelsize=6)
    ax3d.xaxis.pane.fill = False
    ax3d.yaxis.pane.fill = False
    ax3d.zaxis.pane.fill = False
    ax3d.xaxis.pane.set_edgecolor("#333333")
    ax3d.yaxis.pane.set_edgecolor("#333333")
    ax3d.zaxis.pane.set_edgecolor("#333333")
    ax3d.grid(True, color="#333333", alpha=0.2)
    ax3d.set_title(f"3D — Group {group_label} (n={n_ind}) on global background\n"
                   f"σ={smoothing} | res={resolution}",
                   color=col, fontsize=11, pad=10)
    fig1.colorbar(surf, ax=ax3d, fraction=0.015, pad=0.05,
                  shrink=0.5, label="1=start  0=end"
                  ).ax.tick_params(colors="#aaaaaa", labelsize=7)
    fig1.suptitle(f"3D temporal topography — Strategy {group_label}",
                  color=col, fontsize=13)
    plt.tight_layout(); plt.show()

    # ════════════════════════════════════════════════════════
    # FIGURE 2 — 2D temporal overlay
    # ════════════════════════════════════════════════════════
    fig2, ax2d = plt.subplots(figsize=(11, 9))
    fig2.patch.set_facecolor("#1a1a2e")
    ax2d.set_facecolor("#0d0d1a"); ax2d.axis("off")

    ax2d.imshow(Zi_global, origin='lower',
                extent=[x_min, x_max, y_min, y_max],
                cmap='gray', vmin=0, vmax=1,
                aspect='auto', alpha=0.35, zorder=1)
    ax2d.imshow(Zi_g, origin='lower',
                extent=[Xi_g.min(), Xi_g.max(),
                        Yi_g.min(), Yi_g.max()],
                cmap=colormap, vmin=0, vmax=1,
                aspect='auto', alpha=0.80, zorder=2)
    ax2d.contour(Xi_g, Yi_g,
                 gaussian_filter(np.nan_to_num(Zi_g), sigma=2),
                 levels=8, colors=col, alpha=0.4,
                 linewidths=0.8, zorder=3)
    ax2d.scatter(xg_, yg_, c=zg_, cmap=colormap,
                 vmin=0, vmax=1, s=2, alpha=0.2, zorder=4)
    ax2d.contour(Xi, Yi,
                 gaussian_filter(np.nan_to_num(Zi_global), sigma=2),
                 levels=6, colors='white', alpha=0.10,
                 linewidths=0.5, zorder=3)

    im_p = plt.cm.ScalarMappable(cmap=colormap, norm=plt.Normalize(0,1))
    im_p.set_array([])
    fig2.colorbar(im_p, ax=ax2d, fraction=0.025, pad=0.01,
                  label="1=start  0=end"
                  ).ax.tick_params(colors="#aaaaaa", labelsize=7)
    ax2d.set_title(f"2D temporal topography — Group {group_label} / global background\n"
                   f"{'normalised/individual' if normalize_by_individual else 'raw'} "
                   f"| σ={smoothing}",
                   color=col, fontsize=11)
    fig2.suptitle(f"Temporal map — Strategy {group_label}", color=col, fontsize=13)
    plt.tight_layout(); plt.show()

    # ════════════════════════════════════════════════════════
    # FIGURE 3 — Behavioural clusters
    # ════════════════════════════════════════════════════════
    fig3, ax3 = plt.subplots(figsize=(11, 9))
    fig3.patch.set_facecolor("#1a1a2e")
    ax3.set_facecolor("#0d0d1a"); ax3.axis("off")

    if cluster_col:
        n_cl   = int(df_w[cluster_col].nunique())
        norm_c = plt.Normalize(vmin=0, vmax=n_cl-1)
        cmap_c = plt.cm.tab20

        # Background: all points greyed out
        ax3.scatter(df_w["umap_x"], df_w["umap_y"],
                    color="#222233", s=1.5, alpha=0.2, zorder=1)

        # Group points coloured by cluster
        ax3.scatter(df_g["umap_x"], df_g["umap_y"],
                    c=df_g[cluster_col].values,
                    cmap=cmap_c, norm=norm_c,
                    s=5, alpha=0.65, zorder=2)

        # Cluster labels at the global centroid
        for c in range(n_cl):
            mask_c = df_w[cluster_col] == c
            if not mask_c.any(): continue
            cx = df_w.loc[mask_c, "umap_x"].mean()
            cy = df_w.loc[mask_c, "umap_y"].mean()
            ax3.text(cx, cy, f"C{c}",
                     fontsize=7, fontweight="bold",
                     ha="center", va="center", color="white", zorder=3,
                     bbox=dict(facecolor=cmap_c(norm_c(c)),
                               alpha=0.75, edgecolor="none", pad=1))

        # Cluster colorbar
        sm_c = plt.cm.ScalarMappable(cmap=cmap_c, norm=norm_c)
        sm_c.set_array([])
        fig3.colorbar(sm_c, ax=ax3, fraction=0.025, pad=0.01,
                      label=cluster_col
                      ).ax.tick_params(colors="#aaaaaa", labelsize=7)

        # Group legend
        patches = [mpatches.Patch(color=col, label=f"Group {group_label} (n={n_ind})"),
                   mpatches.Patch(color="#222233", label="Others (background)")]
        leg = ax3.legend(handles=patches, facecolor="#0d0d1a",
                         labelcolor="#cccccc", fontsize=9, loc="upper right")
        leg.get_frame().set_edgecolor("#444444")

    else:
        ax3.text(0.5, 0.5, "Column 'cluster' or 'supercluster'\nnot found",
                 ha="center", va="center", color="#ff4444",
                 transform=ax3.transAxes, fontsize=12)

    ax3.set_title(f"Behavioural clusters — Group {group_label}\n"
                  f"(group points coloured by {cluster_col or 'N/A'} "
                  f"on greyed global background)",
                  color=col, fontsize=11)
    fig3.suptitle(f"Distribution in the clusters — Strategy {group_label}",
                  color=col, fontsize=13)
    plt.tight_layout(); plt.show()


# ══════════════════════════════════════════════════════════════

def plot_temporal_individual(Z, df, mapping_letters,
                              animal_id,
                              smoothing_global=5,
                              resolution=100,
                              normalize_by_individual=True,
                              colormap="RdYlGn_r",
                              stride=2):

    df_w = df.copy()
    df_w["animal_id"] = df_w["animal_id"].astype(str)
    df_w["umap_x"]    = Z[:, 0]
    df_w["umap_y"]    = Z[:, 1]

    aid    = str(animal_id)
    df_ind = df_w[df_w["animal_id"] == aid]

    if len(df_ind) == 0:
        print(f"❌ Individual '{aid}' not found.")
        return

    group_label = mapping_letters.get(aid, "MONO") if mapping_letters else "?"
    group_list = sorted(set(v for v in mapping_letters.values()
                            if v != "MONO")) if mapping_letters else []
    pal    = {g: PALETTE[i % len(PALETTE)] for i, g in enumerate(group_list)}
    col    = pal.get(group_label, "#4CAF50")
    n_frames = len(df_ind)

    cluster_col = ("supercluster" if "supercluster" in df_w.columns
                   else "cluster"  if "cluster"      in df_w.columns
                   else None)

    if "frame" in df_ind.columns:
        df_ind = df_ind.sort_values("frame")

    x_ind  = df_ind["umap_x"].values
    y_ind  = df_ind["umap_y"].values
    t_inv  = 1.0 - np.arange(n_frames) / max(n_frames-1, 1)
    cmap_obj = plt.cm.get_cmap(colormap)

    print(f"[INDIV] ID={aid} | Group={group_label} | {n_frames} frames")
    print(f"[INDIV] Computing global grid...")
    Zi_global, Xi, Yi, xg, yg, zg = _compute_temporal_grid(
        df_w, resolution, smoothing_global, normalize_by_individual
    )

    x_min, x_max = Xi.min(), Xi.max()
    y_min, y_max = Yi.min(), Yi.max()

    # ════════════════════════════════════════════════════════
    # FIGURE 1 — 3D
    # ════════════════════════════════════════════════════════
    fig1 = plt.figure(figsize=(12, 9))
    fig1.patch.set_facecolor("#1a1a2e")
    ax3d = fig1.add_subplot(111, projection='3d')
    ax3d.set_facecolor("#0d0d1a")

    ax3d.plot_surface(
        Xi[::stride*2, ::stride*2],
        Yi[::stride*2, ::stride*2],
        Zi_global[::stride*2, ::stride*2],
        cmap='gray', alpha=0.15,
        linewidth=0, antialiased=False, vmin=0, vmax=1,
    )
    for i in range(n_frames - 1):
        rgba = cmap_obj(t_inv[i])
        ax3d.plot(x_ind[i:i+2], y_ind[i:i+2], t_inv[i:i+2],
                  color=rgba, lw=2.0, alpha=0.85, zorder=6)
        ax3d.plot(x_ind[i:i+2], y_ind[i:i+2], [-0.05, -0.05],
                  color=rgba, lw=0.8, alpha=0.2, zorder=3)

    ax3d.scatter(x_ind[0],  y_ind[0],  t_inv[0],
                 color='lime',    s=100, zorder=8,
                 marker='o', edgecolors='white', linewidths=1.5,
                 depthshade=False)
    ax3d.scatter(x_ind[-1], y_ind[-1], t_inv[-1],
                 color='#ff4444', s=100, zorder=8,
                 marker='s', edgecolors='white', linewidths=1.5,
                 depthshade=False)

    ax3d.set_zlim(-0.1, 1.1)
    ax3d.set_xlabel("UMAP D1",           color="#aaaaaa", labelpad=8)
    ax3d.set_ylabel("UMAP D2",           color="#aaaaaa", labelpad=8)
    ax3d.set_zlabel("Time position", color="#aaaaaa", labelpad=8)
    ax3d.tick_params(colors="#555555", labelsize=6)
    ax3d.xaxis.pane.fill = False
    ax3d.yaxis.pane.fill = False
    ax3d.zaxis.pane.fill = False
    ax3d.xaxis.pane.set_edgecolor("#333333")
    ax3d.yaxis.pane.set_edgecolor("#333333")
    ax3d.zaxis.pane.set_edgecolor("#333333")
    ax3d.grid(True, color="#333333", alpha=0.2)
    ax3d.set_title(f"3D — Individual {aid} (Strategy {group_label})\n"
                   f"{n_frames} frames | on global background",
                   color=col, fontsize=11, pad=10)
    sm = plt.cm.ScalarMappable(cmap=colormap, norm=plt.Normalize(0,1))
    sm.set_array([])
    fig1.colorbar(sm, ax=ax3d, fraction=0.015, pad=0.05,
                  shrink=0.5, label="1=start  0=end"
                  ).ax.tick_params(colors="#aaaaaa", labelsize=7)
    fig1.suptitle(f"3D trajectory — Individual {aid} | Strategy {group_label}",
                  color=col, fontsize=13)
    plt.tight_layout(); plt.show()

    # ════════════════════════════════════════════════════════
    # FIGURE 2 — 2D temporal
    # ════════════════════════════════════════════════════════
    fig2, ax2d = plt.subplots(figsize=(11, 9))
    fig2.patch.set_facecolor("#1a1a2e")
    ax2d.set_facecolor("#0d0d1a"); ax2d.axis("off")

    ax2d.imshow(Zi_global, origin='lower',
                extent=[x_min, x_max, y_min, y_max],
                cmap='gray', vmin=0, vmax=1,
                aspect='auto', alpha=0.4, zorder=1)
    ax2d.contour(Xi, Yi,
                 gaussian_filter(np.nan_to_num(Zi_global), sigma=2),
                 levels=8, colors='white', alpha=0.12,
                 linewidths=0.5, zorder=2)

    for i in range(n_frames - 1):
        rgba = cmap_obj(t_inv[i])
        ax2d.plot(x_ind[i:i+2], y_ind[i:i+2],
                  color=rgba, lw=2.0, alpha=0.85, zorder=4)

    ax2d.scatter(x_ind[1:-1], y_ind[1:-1],
                 c=t_inv[1:-1], cmap=colormap,
                 vmin=0, vmax=1, s=8, alpha=0.5, zorder=5)
    ax2d.scatter(x_ind[0],  y_ind[0],
                 color='lime',    s=120, zorder=7,
                 marker='o', edgecolors='white', linewidths=1.5)
    ax2d.scatter(x_ind[-1], y_ind[-1],
                 color='#ff4444', s=120, zorder=7,
                 marker='s', edgecolors='white', linewidths=1.5)

    mid = n_frames // 2
    if mid + 1 < n_frames:
        ax2d.annotate("",
            xy=(x_ind[mid+1], y_ind[mid+1]),
            xytext=(x_ind[mid], y_ind[mid]),
            arrowprops=dict(arrowstyle="-|>", color=col,
                            lw=2.0, mutation_scale=18), zorder=6)

    im_p = plt.cm.ScalarMappable(cmap=colormap, norm=plt.Normalize(0,1))
    im_p.set_array([])
    fig2.colorbar(im_p, ax=ax2d, fraction=0.025, pad=0.01,
                  label="1=start  0=end"
                  ).ax.tick_params(colors="#aaaaaa", labelsize=7)
    ax2d.set_title(f"2D temporal trajectory — Individual {aid}\n"
                   f"{n_frames} frames | green=start | red=end",
                   color=col, fontsize=11)
    fig2.suptitle(f"Temporal map — Individual {aid} | Strategy {group_label}",
                  color=col, fontsize=13)
    plt.tight_layout(); plt.show()

    # ════════════════════════════════════════════════════════
    # FIGURE 3 — Behavioural clusters
    # ════════════════════════════════════════════════════════
    fig3, ax3 = plt.subplots(figsize=(11, 9))
    fig3.patch.set_facecolor("#1a1a2e")
    ax3.set_facecolor("#0d0d1a"); ax3.axis("off")

    if cluster_col:
        n_cl   = int(df_w[cluster_col].nunique())
        norm_c = plt.Normalize(vmin=0, vmax=n_cl-1)
        cmap_c = plt.cm.tab20

        # Background: all points greyed out
        ax3.scatter(df_w["umap_x"], df_w["umap_y"],
                    color="#222233", s=1.5, alpha=0.2, zorder=1)

        # Trajectory coloured by cluster
        if cluster_col in df_ind.columns:
            ax3.scatter(df_ind["umap_x"], df_ind["umap_y"],
                        c=df_ind[cluster_col].values,
                        cmap=cmap_c, norm=norm_c,
                        s=8, alpha=0.7, zorder=2)

        # Trajectory line in white to show the path
        ax3.plot(x_ind, y_ind,
                 color='white', lw=0.8, alpha=0.3, zorder=3)

        # Cluster labels at the global centroid
        for c in range(n_cl):
            mask_c = df_w[cluster_col] == c
            if not mask_c.any(): continue
            cx = df_w.loc[mask_c, "umap_x"].mean()
            cy = df_w.loc[mask_c, "umap_y"].mean()
            ax3.text(cx, cy, f"C{c}",
                     fontsize=7, fontweight="bold",
                     ha="center", va="center", color="white", zorder=4,
                     bbox=dict(facecolor=cmap_c(norm_c(c)),
                               alpha=0.75, edgecolor="none", pad=1))

        # Start and end
        ax3.scatter(x_ind[0],  y_ind[0],
                    color='lime',    s=120, zorder=6,
                    marker='o', edgecolors='white', linewidths=1.5,
                    label="Start")
        ax3.scatter(x_ind[-1], y_ind[-1],
                    color='#ff4444', s=120, zorder=6,
                    marker='s', edgecolors='white', linewidths=1.5,
                    label="End")

        # Direction arrow
        if mid + 1 < n_frames:
            ax3.annotate("",
                xy=(x_ind[mid+1], y_ind[mid+1]),
                xytext=(x_ind[mid], y_ind[mid]),
                arrowprops=dict(arrowstyle="-|>", color='white',
                                lw=1.5, mutation_scale=15), zorder=5)

        sm_c = plt.cm.ScalarMappable(cmap=cmap_c, norm=norm_c)
        sm_c.set_array([])
        fig3.colorbar(sm_c, ax=ax3, fraction=0.025, pad=0.01,
                      label=cluster_col
                      ).ax.tick_params(colors="#aaaaaa", labelsize=7)

        leg = ax3.legend(facecolor="#0d0d1a", labelcolor="#cccccc",
                         fontsize=8, loc="upper right")
        leg.get_frame().set_edgecolor("#444444")

    else:
        ax3.text(0.5, 0.5, "Column 'cluster' or 'supercluster'\nnot found",
                 ha="center", va="center", color="#ff4444",
                 transform=ax3.transAxes, fontsize=12)

    ax3.set_title(f"Behavioural clusters — Individual {aid}\n"
                  f"(points coloured by {cluster_col or 'N/A'} "
                  f"on greyed global background)",
                  color=col, fontsize=11)
    fig3.suptitle(f"Distribution in the clusters — Individual {aid} "
                  f"| Strategy {group_label}",
                  color=col, fontsize=13)
    plt.tight_layout(); plt.show()
def plot_3d_clusters(Z, df, mapping_letters=None,
                     resolution=100,
                     smoothing=5,
                     normalize_by_individual=True,
                     stride=2):
    """
    3D map coloured by behavioural cluster.
    Colours identical to plot_umap (tab20 + norm 0 to n-1).
    Height = time position (peak=start, trough=end).
    """
    df_w = df.copy()
    df_w["animal_id"] = df_w["animal_id"].astype(str)
    df_w["umap_x"]    = Z[:, 0]
    df_w["umap_y"]    = Z[:, 1]

    cluster_col = ("supercluster" if "supercluster" in df_w.columns
                   else "cluster"  if "cluster"      in df_w.columns
                   else None)
    if cluster_col is None:
        print("❌ Column 'cluster' or 'supercluster' not found.")
        return

    # ── Same colours as plot_umap ──────────────────────────
    n_cl   = int(df_w[cluster_col].nunique())
    norm_c = plt.Normalize(vmin=0, vmax=n_cl - 1)
    cmap_c = plt.cm.tab20

    x_pts  = df_w["umap_x"].values
    y_pts  = df_w["umap_y"].values
    c_pts  = df_w[cluster_col].values.astype(float)

    # ── Time grid for the HEIGHT ─────────────────────
    print(f"[3D CLUSTERS] Computing time height grid...")
    Zi_height, Xi, Yi, _, _, _ = _compute_temporal_grid(
        df_w, resolution, smoothing, normalize_by_individual
    )

    x_min, x_max = Xi.min(), Xi.max()
    y_min, y_max = Yi.min(), Yi.max()

    # ── Cluster grid for the COLOUR ────────────────────────
    # nearest → each cell takes the colour of the nearest
    # cluster → sharp borders between clusters
    print(f"[3D CLUSTERS] Computing cluster colour grid...")
    pts = np.column_stack([x_pts, y_pts])
    Ci  = griddata(pts, c_pts, (Xi, Yi), method='nearest')

    # Mask of areas without data
    density = griddata(pts, np.ones(len(x_pts)),
                       (Xi, Yi), method='linear')
    mask_empty = np.isnan(density)

    # ── Cluster → RGBA conversion ──────────────────────────────
    # plot_surface with facecolors expects an (n, m, 4) array
    Ci_norm    = norm_c(Ci)
    colors_rgb = cmap_c(Ci_norm)         # (res, res, 4)
    colors_rgb[mask_empty, 3] = 0.0       # transparent where there is no data

    # ── Figure ────────────────────────────────────────────────
    fig = plt.figure(figsize=(13, 10))
    fig.patch.set_facecolor("#1a1a2e")
    ax  = fig.add_subplot(111, projection='3d')
    ax.set_facecolor("#0d0d1a")

    # Surface: time height + cluster colour
    ax.plot_surface(
        Xi[::stride, ::stride],
        Yi[::stride, ::stride],
        Zi_height[::stride, ::stride],
        facecolors  = colors_rgb[::stride, ::stride],
        shade       = True,
        linewidth   = 0,
        antialiased = False,
    )

    # Ground-projected contour coloured by cluster
    ax.contourf(
        Xi, Yi, Ci,
        zdir   = 'z',
        offset = -0.05,
        cmap   = cmap_c,
        norm   = norm_c,
        alpha  = 0.30,
        levels = n_cl,
    )

    # Cluster labels at the centroid
    # X, Y position = centroid of the cluster in UMAP space
    # Z position    = mean height of this cluster on the surface
    for c in range(n_cl):
        mask_c = df_w[cluster_col] == c
        if not mask_c.any(): continue
        cx = df_w.loc[mask_c, "umap_x"].mean()
        cy = df_w.loc[mask_c, "umap_y"].mean()

        # Corresponding height on the smoothed grid
        dist   = (Xi - cx)**2 + (Yi - cy)**2
        i_near = np.unravel_index(np.nanargmin(dist), dist.shape)
        cz     = float(Zi_height[i_near]) if not np.isnan(Zi_height[i_near]) \
                 else 0.5

        ax.text(cx, cy, cz + 0.08,
                f"C{c}",
                color='white', fontsize=7, fontweight='bold',
                ha='center', va='bottom', zorder=10,
                bbox=dict(facecolor=cmap_c(norm_c(c)),
                          alpha=0.85, edgecolor='none', pad=1.5))

    # ── Style ─────────────────────────────────────────────────
    ax.set_zlim(-0.1, 1.1)
    ax.set_xlabel("UMAP D1",             color="#aaaaaa", labelpad=8)
    ax.set_ylabel("UMAP D2",             color="#aaaaaa", labelpad=8)
    ax.set_zlabel("Time position", color="#aaaaaa", labelpad=8)
    ax.tick_params(colors="#555555", labelsize=6)
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    ax.xaxis.pane.set_edgecolor("#333333")
    ax.yaxis.pane.set_edgecolor("#333333")
    ax.zaxis.pane.set_edgecolor("#333333")
    ax.grid(True, color="#333333", alpha=0.2)

    # Cluster colorbar — same ticks as plot_umap
    sm_c = plt.cm.ScalarMappable(cmap=cmap_c, norm=norm_c)
    sm_c.set_array([])
    cbar = fig.colorbar(sm_c, ax=ax, fraction=0.015, pad=0.05,
                        shrink=0.6, label=cluster_col)
    cbar.set_ticks(np.arange(n_cl))
    cbar.set_ticklabels([f"C{c}" for c in range(n_cl)])
    cbar.ax.tick_params(colors="#aaaaaa", labelsize=7)

    ax.set_title(
        f"3D UMAP surface — colour = {cluster_col} "
        f"(tab20, identical to plot_umap)\n"
        f"height = time position | "
        f"σ={smoothing} | res={resolution}",
        color="#cccccc", fontsize=11, pad=15
    )
    fig.suptitle(
        "Temporal topography coloured by behavioural cluster",
        color="#cccccc", fontsize=13
    )
    plt.tight_layout()
    plt.show()


def plot_individual_on_3d(Z, df, mapping_letters,
                           animal_id,
                           resolution=100,
                           smoothing=5,
                           normalize_by_individual=True,
                           colormap="RdYlGn_r",
                           stride=2):
    """
    Trajectory of ONE individual in 3D on the global surface.
    Global surface coloured by cluster (tab20, identical to plot_umap).
    Trajectory coloured by time position.
    """
    df_w = df.copy()
    df_w["animal_id"] = df_w["animal_id"].astype(str)
    df_w["umap_x"]    = Z[:, 0]
    df_w["umap_y"]    = Z[:, 1]

    aid    = str(animal_id)
    df_ind = df_w[df_w["animal_id"] == aid]

    if len(df_ind) == 0:
        print(f"❌ Individual '{aid}' not found.")
        print(f"   Examples : {df_w['animal_id'].unique()[:8].tolist()}")
        return

    if "frame" in df_ind.columns:
        df_ind = df_ind.sort_values("frame")

    group_label     = mapping_letters.get(aid, "MONO") if mapping_letters else "?"
    group_list = sorted(set(v for v in mapping_letters.values()
                            if v != "MONO")) if mapping_letters else []
    pal        = {g: PALETTE[i % len(PALETTE)] for i, g in enumerate(group_list)}
    col        = pal.get(group_label, "#4CAF50")
    n_frames   = len(df_ind)
    cmap_obj   = plt.cm.get_cmap(colormap)

    cluster_col = ("supercluster" if "supercluster" in df_w.columns
                   else "cluster"  if "cluster"      in df_w.columns
                   else None)

    # ── Same colours as plot_umap ──────────────────────────
    if cluster_col:
        n_cl   = int(df_w[cluster_col].nunique())
        norm_c = plt.Normalize(vmin=0, vmax=n_cl - 1)
        cmap_c = plt.cm.tab20

    x_ind  = df_ind["umap_x"].values
    y_ind  = df_ind["umap_y"].values
    t_inv  = 1.0 - np.arange(n_frames) / max(n_frames-1, 1)
    mid    = n_frames // 2

    # ── Global height grid ────────────────────────────────
    print(f"[INDIV 3D] Computing global grid...")
    Zi_global, Xi, Yi, xg, yg, zg = _compute_temporal_grid(
        df_w, resolution, smoothing, normalize_by_individual
    )

    # ── Cluster colour grid ────────────────────────────────
    if cluster_col:
        pts   = np.column_stack([xg, yg])
        c_pts = df_w[cluster_col].values.astype(float)
        Ci    = griddata(np.column_stack([df_w["umap_x"].values,
                                          df_w["umap_y"].values]),
                         c_pts, (Xi, Yi), method='nearest')
        density    = griddata(np.column_stack([df_w["umap_x"].values,
                                               df_w["umap_y"].values]),
                              np.ones(len(df_w)),
                              (Xi, Yi), method='linear')
        mask_empty  = np.isnan(density)
        Ci_norm    = norm_c(Ci)
        colors_rgb = cmap_c(Ci_norm)
        colors_rgb[mask_empty, 3] = 0.0

    # ── Figure ────────────────────────────────────────────────
    fig = plt.figure(figsize=(13, 10))
    fig.patch.set_facecolor("#1a1a2e")
    ax  = fig.add_subplot(111, projection='3d')
    ax.set_facecolor("#0d0d1a")

    # Global surface coloured by cluster (background)
    if cluster_col:
        ax.plot_surface(
            Xi[::stride*2, ::stride*2],
            Yi[::stride*2, ::stride*2],
            Zi_global[::stride*2, ::stride*2],
            facecolors  = colors_rgb[::stride*2, ::stride*2],
            shade       = True,
            alpha       = 0.30,
            linewidth   = 0,
            antialiased = False,
        )
    else:
        ax.plot_surface(
            Xi[::stride*2, ::stride*2],
            Yi[::stride*2, ::stride*2],
            Zi_global[::stride*2, ::stride*2],
            cmap='gray', alpha=0.20,
            linewidth=0, antialiased=False,
            vmin=0, vmax=1,
        )

    # Cluster labels on the surface
    if cluster_col:
        for c in range(n_cl):
            mask_c = df_w[cluster_col] == c
            if not mask_c.any(): continue
            cx = df_w.loc[mask_c, "umap_x"].mean()
            cy = df_w.loc[mask_c, "umap_y"].mean()
            dist   = (Xi - cx)**2 + (Yi - cy)**2
            i_near = np.unravel_index(np.nanargmin(dist), dist.shape)
            cz     = float(Zi_global[i_near]) \
                     if not np.isnan(Zi_global[i_near]) else 0.5
            ax.text(cx, cy, cz + 0.06,
                    f"C{c}",
                    color='white', fontsize=6, fontweight='bold',
                    ha='center', va='bottom', zorder=5,
                    bbox=dict(facecolor=cmap_c(norm_c(c)),
                              alpha=0.75, edgecolor='none', pad=1))

    # Trajectory of the individual coloured by time
    for i in range(n_frames - 1):
        rgba = cmap_obj(t_inv[i])
        ax.plot(x_ind[i:i+2], y_ind[i:i+2],
                t_inv[i:i+2],
                color=rgba, lw=2.5, alpha=0.95, zorder=8,
                solid_capstyle='round')

    # Ground projection
    for i in range(n_frames - 1):
        rgba = cmap_obj(t_inv[i])
        ax.plot(x_ind[i:i+2], y_ind[i:i+2],
                [-0.05, -0.05],
                color=rgba, lw=0.8, alpha=0.2, zorder=4)

    # Start / end plumb lines
    ax.plot([x_ind[0],  x_ind[0]],
            [y_ind[0],  y_ind[0]],
            [-0.05, t_inv[0]],
            color='lime', lw=0.8, alpha=0.4, ls=':', zorder=4)
    ax.plot([x_ind[-1], x_ind[-1]],
            [y_ind[-1], y_ind[-1]],
            [-0.05, t_inv[-1]],
            color='#ff4444', lw=0.8, alpha=0.4, ls=':', zorder=4)

    # Start and end
    ax.scatter(x_ind[0],  y_ind[0],  t_inv[0],
               color='lime',    s=120, zorder=10,
               marker='o', edgecolors='white', linewidths=1.5,
               depthshade=False)
    ax.scatter(x_ind[-1], y_ind[-1], t_inv[-1],
               color='#ff4444', s=120, zorder=10,
               marker='s', edgecolors='white', linewidths=1.5,
               depthshade=False)

    # Direction arrow
    if mid + 1 < n_frames:
        ax.quiver(x_ind[mid], y_ind[mid], t_inv[mid],
                  x_ind[mid+1]-x_ind[mid],
                  y_ind[mid+1]-y_ind[mid],
                  t_inv[mid+1]-t_inv[mid],
                  color=col, lw=2, arrow_length_ratio=0.4,
                  zorder=9)

    # ── Style ─────────────────────────────────────────────────
    ax.set_zlim(-0.1, 1.1)
    ax.set_xlabel("UMAP D1",             color="#aaaaaa", labelpad=8)
    ax.set_ylabel("UMAP D2",             color="#aaaaaa", labelpad=8)
    ax.set_zlabel("Time position", color="#aaaaaa", labelpad=8)
    ax.tick_params(colors="#555555", labelsize=6)
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    ax.xaxis.pane.set_edgecolor("#333333")
    ax.yaxis.pane.set_edgecolor("#333333")
    ax.zaxis.pane.set_edgecolor("#333333")
    ax.grid(True, color="#333333", alpha=0.2)

    # Strategy legend + start/end
    leg_els = [
        mpatches.Patch(color='lime',    label='○ Start'),
        mpatches.Patch(color='#ff4444', label='□ End'),
        mpatches.Patch(color=col,       label=f'Strategy {group_label}'),
    ]
    leg = ax.legend(handles=leg_els, facecolor="#0d0d1a",
                    labelcolor="#cccccc", fontsize=8, loc="upper left")
    leg.get_frame().set_edgecolor("#444444")

    # Time trajectory colorbar
    sm = plt.cm.ScalarMappable(cmap=colormap, norm=plt.Normalize(0,1))
    sm.set_array([])
    fig.colorbar(sm, ax=ax, fraction=0.015, pad=0.05,
                 shrink=0.5, label="Trajectory : 1=start  0=end"
                 ).ax.tick_params(colors="#aaaaaa", labelsize=7)

    # Cluster colorbar
    if cluster_col:
        sm_c = plt.cm.ScalarMappable(cmap=cmap_c, norm=norm_c)
        sm_c.set_array([])
        cb2 = fig.colorbar(sm_c, ax=ax, fraction=0.012, pad=0.10,
                           shrink=0.5, label=cluster_col)
        cb2.set_ticks(np.arange(n_cl))
        cb2.set_ticklabels([f"C{c}" for c in range(n_cl)])
        cb2.ax.tick_params(colors="#aaaaaa", labelsize=6)

    ax.set_title(
        f"Individual {aid} — Strategy {group_label} | {n_frames} frames\n"
        f"surface = {cluster_col} (tab20) | "
        f"trajectory = time position",
        color=col, fontsize=11, pad=15
    )
    fig.suptitle(
        f"3D trajectory — Individual {aid} | Strategy {group_label}",
        color=col, fontsize=13
    )
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Calls ────────────────────────────────────────────────────
%matplotlib qt

# 3D cluster map
plot_3d_clusters(
    Z                       = Z,
    df                      = df,
    resolution              = 100,
    smoothing               = 5,
    normalize_by_individual = True,
    stride                  = 2,
)

# Individual on the 3D cluster map
plot_individual_on_3d(
    Z                       = Z,
    df                      = df,
    mapping_letters         = mapping_letters,
    animal_id               = 116,        # ← your ID
    resolution              = 100,
    smoothing               = 5,
    normalize_by_individual = True,
    colormap                = "RdYlGn_r",
    stride                  = 2,
)

# ── Calls ────────────────────────────────────────────────────
plot_temporal_group_overlay(
    Z                       = Z,
    df                      = df,
    mapping_letters         = mapping_letters,
    group_label                  = "A",
    smoothing               = 5,
    resolution              = 100,
    normalize_by_individual = True,
    colormap                = "RdYlGn_r",
    stride                  = 2,
)

plot_temporal_individual(
    Z                       = Z,
    df                      = df,
    mapping_letters         = mapping_letters,
    animal_id               = 42,
    smoothing_global        = 5,
    resolution              = 100,
    normalize_by_individual = True,
    colormap                = "RdYlGn_r",
    stride                  = 2,
)

In [ ]:
%matplotlib inline

In [ ]:
def plot_strategy_trajectories_arena(df, mapping_letters,
                                      strategies_to_plot=None,
                                      arena_size=1500,
                                      smooth_mean=True,
                                      sigma_mean=15,
                                      align_on="shrimp"):
    """
    For each strategy: one figure with all the trajectories
    in the real arena + aligned trajectories + final overlay.

    align_on : "shrimp"  → X axis = cuttlefish→shrimp direction at the start
                           (the most biologically relevant)
               "motion"  → X axis = direction of the initial movement (10 frames)
               "end"     → X axis = start→end direction (old method)
    """
    from scipy.ndimage import gaussian_filter1d

    df_w = df.copy()
    df_w["animal_id"] = df_w["animal_id"].astype(int)

    has_shrimp  = (
        "HeadShrimp_x" in df_w.columns and
        "HeadShrimp_y" in df_w.columns
    )

    has_cluster = "cluster" in df_w.columns

    # ── Cluster palette ───────────────────────────────
    if has_cluster:

        valid_clusters = df_w["cluster"].dropna()

        if len(valid_clusters) > 0:
            n_clusters = int(valid_clusters.max()) + 1
        else:
            n_clusters = 1

        cmap_clusters = plt.cm.tab20(
            np.linspace(0, 1, max(n_clusters, 1))
        )

        clr_hex = [
            matplotlib.colors.to_hex(c)
            for c in cmap_clusters
        ]
    # ── Palette ───────────────────────────────────────────────
    strat_list  = sorted([s for s in set(mapping_letters.values())
                          if s != "MONO"])
    pal_s       = plt.cm.tab10(np.linspace(0, 0.8, max(len(strat_list), 1)))
    strat_color = {s: tuple(pal_s[i]) for i, s in enumerate(strat_list)}

    if strategies_to_plot is None:
        strategies_to_plot = strat_list

    N_RESAMPLE = 300
    t_common   = np.linspace(0, 1, N_RESAMPLE)

    # Storage of the mean trajectories for the overlay figure
    all_mean_trajs = {}

    # ════════════════════════════════════════════════════════
    # LOOP PER STRATEGY
    # ════════════════════════════════════════════════════════
    for strat in strategies_to_plot:

        members = [int(aid) for aid, g in mapping_letters.items()
                   if g == strat]
        if not members:
            print(f"[SKIP] Strategy {strat} — no member")
            continue

        col = strat_color.get(strat, "#4CAF50")

        # ── Alignment ────────────────────────────────────────
        trajs_aligned = []

        for aid in members:
            sub = df_w[df_w["animal_id"] == aid].copy()
            if len(sub) < 2: continue
            if "frame" in sub.columns:
                sub = sub.sort_values("frame")

            x = sub["TopMantle_x"].values.astype(float)
            y = sub["TopMantle_y"].values.astype(float)

            # ── Reference axis depending on align_on ───────────────
            if align_on == "shrimp" and has_shrimp:
                # Cuttlefish → shrimp direction at the start
                # Biologically: all cuttlefish start
                # "facing" the shrimp → common X axis
                sx0 = sub["HeadShrimp_x"].values[0]
                sy0 = sub["HeadShrimp_y"].values[0]
                dx  = sx0 - x[0]
                dy  = sy0 - y[0]

            elif align_on == "motion":
                # Direction of the initial movement (first 10 frames)
                # Less sensitive to on-the-spot rotations than start→end
                n_init = min(10, len(x) - 1)
                dx = x[n_init] - x[0]
                dy = y[n_init] - y[0]

            else:
                # Old method: start→end
                dx = x[-1] - x[0]
                dy = y[-1] - y[0]

            dist_ref = np.sqrt(dx**2 + dy**2)
            if dist_ref < 1e-6: continue

            # Aligned frame of reference
            ux =  dx / dist_ref   # X axis = towards the shrimp
            uy =  dy / dist_ref
            px = -uy              # Y axis = perpendicular
            py =  ux

            # Translation: origin = initial cuttlefish position
            tx = x - x[0]
            ty = y - y[0]

            # Projection (coordinates in pixels)
            proj_x = tx * ux + ty * uy
            proj_y = tx * px + ty * py

            # Cumulative distance for resampling by distance
            # (avoids distorting slow/fast trajectories)
            diffs    = np.sqrt(np.diff(proj_x)**2 + np.diff(proj_y)**2)
            cum_dist = np.concatenate([[0], np.cumsum(diffs)])
            total_d  = cum_dist[-1]
            if total_d < 1e-6: continue
            t_orig = cum_dist / total_d

            trajs_aligned.append({
                "aid"   : aid,
                "sub"   : sub,
                "traj"  : np.column_stack([proj_x, proj_y]),
                "t_orig": t_orig,
                "dist"  : np.sqrt((x[-1]-x[0])**2 + (y[-1]-y[0])**2),
            })

        if not trajs_aligned:
            print(f"[SKIP] Strategy {strat} — no valid trajectory")
            continue

        n_ind = len(trajs_aligned)
        print(f"[STRAT {strat}] {n_ind} individuals | "
              f"mean distance = "
              f"{np.mean([t['dist'] for t in trajs_aligned]):.0f} px")

        # ── Mean trajectory (resampled by distance) ───────
        trajs_rs = []
        for t in trajs_aligned:
            rx = np.interp(t_common, t["t_orig"], t["traj"][:, 0])
            ry = np.interp(t_common, t["t_orig"], t["traj"][:, 1])
            trajs_rs.append(np.column_stack([rx, ry]))

        mean_traj = np.mean(trajs_rs, axis=0)
        std_y     = np.std([t[:, 1] for t in trajs_rs], axis=0)

        if smooth_mean:
            mean_traj[:, 0] = gaussian_filter1d(mean_traj[:, 0],
                                sigma=sigma_mean, mode='nearest')
            mean_traj[:, 1] = gaussian_filter1d(mean_traj[:, 1],
                                sigma=sigma_mean, mode='nearest')
            std_y = gaussian_filter1d(std_y, sigma=sigma_mean, mode='nearest')

        all_mean_trajs[strat] = {"mean": mean_traj, "std": std_y,
                                  "color": col, "n": n_ind}

        # ── Aligned axes limits ──────────────────────────────
        all_x    = np.concatenate([t["traj"][:, 0] for t in trajs_aligned])
        all_y    = np.concatenate([t["traj"][:, 1] for t in trajs_aligned])
        x_min, x_max = all_x.min(), all_x.max()
        y_min, y_max = all_y.min(), all_y.max()
        margin_x = (x_max - x_min) * 0.08 + 20
        margin_y = (y_max - y_min) * 0.08 + 20

        # ════════════════════════════════════════════════════
        # FIGURE 1 — Real arena (left) + Aligned (right)
        # ════════════════════════════════════════════════════
        fig, axes_fig = plt.subplots(1, 2, figsize=(18, 8),
                                      gridspec_kw={"width_ratios": [1, 1.6]})
        fig.patch.set_facecolor("#1a1a2e")

        # ── LEFT AXIS: real arena ─────────────────────────
        ax_arena = axes_fig[0]
        ax_arena.set_facecolor("#0d0d1a")
        for sp in ax_arena.spines.values(): sp.set_edgecolor("#333333")
        ax_arena.tick_params(colors="#aaaaaa")
        ax_arena.set_xlim(0, arena_size)
        ax_arena.set_ylim(arena_size, 0)
        ax_arena.set_aspect("equal")
        ax_arena.add_patch(plt.Rectangle(
            (0, 0), arena_size, arena_size,
            fill=False, edgecolor="#334455", lw=1.5, ls="--"))

        for t in trajs_aligned:
            sub  = t["sub"]
            x_r  = sub["TopMantle_x"].values
            y_r  = sub["TopMantle_y"].values

            if has_cluster:
                # Colour by behavioural cluster
                cl_seq = (
                    sub["cluster"]
                    .fillna(-1)
                    .astype(int)
                    .values
                )

                for i in range(len(x_r) - 1):

                    cluster_id = cl_seq[i]

                    if cluster_id < 0:
                        color_seg = "#888888"
                    else:
                        color_seg = clr_hex[cluster_id % len(clr_hex)]

                    ax_arena.plot(
                        x_r[i:i+2],
                        y_r[i:i+2],
                        color=color_seg,
                        lw=0.8,
                        alpha=0.45,
                        zorder=2
                    )
            else:
                ax_arena.plot(x_r, y_r, color=col,
                              lw=0.7, alpha=0.25, zorder=2)

            ax_arena.scatter(x_r[0],  y_r[0],
                             color='lime',    s=15, alpha=0.5,
                             zorder=3, marker='o')
            ax_arena.scatter(x_r[-1], y_r[-1],
                             color='#ff4444', s=15, alpha=0.5,
                             zorder=3, marker='s')

            # Shrimp at the start
            if has_shrimp:
                sx = sub["HeadShrimp_x"].values
                sy = sub["HeadShrimp_y"].values
                ax_arena.scatter(sx[0], sy[0],
                                 color="yellow", s=30, marker="*",
                                 alpha=0.6, zorder=4)
                ax_arena.plot([x_r[0], sx[0]], [y_r[0], sy[0]],
                              color="yellow", lw=0.4, alpha=0.2,
                              ls=":", zorder=2)

        # Cluster legend if available
        if has_cluster:
            n_cl_plot = min(n_clusters, 10)
            patches_cl = [mpatches.Patch(color=clr_hex[c],
                           label=f"C{c}") for c in range(n_cl_plot)]
            leg_cl = ax_arena.legend(handles=patches_cl,
                                      facecolor="#111", fontsize=6,
                                      loc="lower right", ncol=2)
            leg_cl.get_frame().set_edgecolor("#444")
            for t in leg_cl.get_texts(): t.set_color("white")

        ax_arena.set_title(
            f"Strategy {strat} — real positions\n"
            f"(colours = clusters | ★ = initial shrimp)",
            color=col, fontsize=10)
        ax_arena.set_xlabel("X (px)", color="#aaaaaa")
        ax_arena.set_ylabel("Y (px)", color="#aaaaaa")
        ax_arena.grid(True, color="#1a2535", alpha=0.6)

        # ── RIGHT AXIS: aligned trajectories ─────────────────
        ax_al = axes_fig[1]
        ax_al.set_facecolor("#0d0d1a")
        for sp in ax_al.spines.values(): sp.set_edgecolor("#333333")
        ax_al.tick_params(colors="#aaaaaa")
        ax_al.axhline(0, color="#334455", lw=1.0, ls="--", zorder=0)
        ax_al.axvline(0, color="#334455", lw=0.6, ls=":", zorder=0)

        # Aligned initial shrimp (always on the X axis)
        if align_on == "shrimp" and has_shrimp:
            dist_shrimp_mean = np.mean([
                np.sqrt((t["sub"]["HeadShrimp_x"].values[0] -
                         t["sub"]["TopMantle_x"].values[0])**2 +
                        (t["sub"]["HeadShrimp_y"].values[0] -
                         t["sub"]["TopMantle_y"].values[0])**2)
                for t in trajs_aligned
            ])
            ax_al.scatter(dist_shrimp_mean, 0,
                          color="yellow", s=200, marker="*",
                          zorder=6, label=f"Mean shrimp ({dist_shrimp_mean:.0f}px)")

        # Individual trajectories
        cmap_time = plt.cm.plasma
        for t in trajs_aligned:
            traj = t["traj"]
            n    = len(traj)
            for i in range(n - 1):
                alpha = 0.05 + 0.12 * (i / n)
                ax_al.plot(traj[i:i+2, 0], traj[i:i+2, 1],
                           color=col, lw=0.9, alpha=alpha,
                           zorder=2, solid_capstyle='round')
            ax_al.scatter(traj[0, 0],  traj[0, 1],
                          color='lime',    s=12, alpha=0.4,
                          zorder=3, marker='o')
            ax_al.scatter(traj[-1, 0], traj[-1, 1],
                          color='#ff4444', s=12, alpha=0.4,
                          zorder=3, marker='s')

        # ±1 std envelope
        ax_al.fill_between(
            mean_traj[:, 0],
            mean_traj[:, 1] - std_y,
            mean_traj[:, 1] + std_y,
            color=col, alpha=0.12, zorder=3, label="±1 std")

        # White outline
        ax_al.plot(mean_traj[:, 0], mean_traj[:, 1],
                   color='white', lw=6, alpha=0.12,
                   zorder=4, solid_capstyle='round')

        # Mean trajectory coloured by time
        for i in range(N_RESAMPLE - 1):
            rgba = cmap_time(i / N_RESAMPLE)
            ax_al.plot(mean_traj[i:i+2, 0], mean_traj[i:i+2, 1],
                       color=rgba, lw=3.5, alpha=0.95,
                       zorder=5, solid_capstyle='round')

        # Direction arrow
        mid = N_RESAMPLE // 2
        ax_al.annotate("",
            xy    =(mean_traj[mid+2, 0], mean_traj[mid+2, 1]),
            xytext=(mean_traj[mid,   0], mean_traj[mid,   1]),
            arrowprops=dict(arrowstyle="-|>", color="white",
                            lw=2.0, mutation_scale=20), zorder=6)

        # Start/end points
        ax_al.scatter(mean_traj[0, 0],  mean_traj[0, 1],
                      color='lime',    s=180, zorder=8,
                      marker='o', edgecolors='white', lw=2,
                      label="Mean start")
        ax_al.scatter(mean_traj[-1, 0], mean_traj[-1, 1],
                      color='#ff4444', s=180, zorder=8,
                      marker='s', edgecolors='white', lw=2,
                      label="Mean end")

        mean_dist = np.mean([t["dist"] for t in trajs_aligned])

        ax_al.set_xlim(x_min - margin_x, x_max + margin_x)
        ax_al.set_ylim(y_min - margin_y, y_max + margin_y)
        ax_al.set_aspect("equal")

        align_label = {"shrimp": "cuttlefish→shrimp",
                       "motion": "initial movement",
                       "end":    "start→end"}[align_on]
        ax_al.set_xlabel(f"{align_label} axis (px)",
                          color="#aaaaaa", fontsize=10)
        ax_al.set_ylabel("Perpendicular axis (px)",
                          color="#aaaaaa", fontsize=10)
        ax_al.set_title(
            f"Strategy {strat} — aligned trajectories\n"
            f"(X axis = {align_label} | resampled by distance)",
            color=col, fontsize=10)
        ax_al.grid(True, color="#1a2535", alpha=0.5)

        # Time colorbar
        sm = plt.cm.ScalarMappable(cmap=cmap_time,
                                    norm=plt.Normalize(0, 1))
        sm.set_array([])
        cb = fig.colorbar(sm, ax=ax_al, fraction=0.02, pad=0.02,
                          label="Time progression (0=start 1=end)")
        cb.ax.tick_params(colors="#aaaaaa", labelsize=8)
        cb.ax.yaxis.label.set_color("#aaaaaa")

        # Legend
        leg_els = [
            mpatches.Patch(color=col, alpha=0.35,
                           label=f"Individuals (n={n_ind})"),
            mpatches.Patch(color=col, alpha=0.12,
                           label="±1 std"),
            mpatches.Patch(color='lime',    label="○ Start"),
            mpatches.Patch(color='#ff4444', label="□ End"),
        ]
        if align_on == "shrimp" and has_shrimp:
            leg_els.append(mpatches.Patch(color="yellow",
                           label="★ Mean initial shrimp"))
        leg = ax_al.legend(handles=leg_els, facecolor="#0d0d1a",
                            labelcolor="#cccccc", fontsize=8,
                            loc="upper left")
        leg.get_frame().set_edgecolor("#444444")

        fig.suptitle(
            f"Strategy {strat}  |  {n_ind} individuals  |  "
            f"mean distance = {mean_dist:.0f} px",
            color=col, fontsize=13, y=1.01)
        plt.tight_layout()
        plt.show()

    # ════════════════════════════════════════════════════════
    # FINAL FIGURE — Overlay of all strategies
    # ════════════════════════════════════════════════════════
    if len(all_mean_trajs) > 1:
        fig, ax = plt.subplots(figsize=(12, 9))
        fig.patch.set_facecolor("#1a1a2e")
        ax.set_facecolor("#0d0d1a")
        for sp in ax.spines.values(): sp.set_edgecolor("#333333")
        ax.tick_params(colors="#aaaaaa")

        ax.axhline(0, color="#334455", lw=1.0, ls="--", zorder=0,
                   label="Reference axis")
        ax.axvline(0, color="#334455", lw=0.6, ls=":", zorder=0)

        for strat, data in all_mean_trajs.items():
            mt  = data["mean"]
            std = data["std"]
            col = data["color"]
            n   = data["n"]

            # Envelope
            ax.fill_between(mt[:, 0],
                            mt[:, 1] - std,
                            mt[:, 1] + std,
                            color=col, alpha=0.10, zorder=2)
            # Mean trajectory
            ax.plot(mt[:, 0], mt[:, 1],
                    color=col, lw=3, alpha=0.95,
                    zorder=3, label=f"Strat. {strat} (n={n})",
                    solid_capstyle='round')
            # Start and end
            ax.scatter(mt[0, 0],  mt[0, 1],
                       color=col, s=120, zorder=5,
                       marker='o', edgecolors='white', lw=1.5)
            ax.scatter(mt[-1, 0], mt[-1, 1],
                       color=col, s=120, zorder=5,
                       marker='s', edgecolors='white', lw=1.5)
            # Arrow
            mid = N_RESAMPLE // 2
            ax.annotate("",
                xy    =(mt[mid+2, 0], mt[mid+2, 1]),
                xytext=(mt[mid,   0], mt[mid,   1]),
                arrowprops=dict(arrowstyle="-|>", color=col,
                                lw=1.5, mutation_scale=16), zorder=4)

        if align_on == "shrimp" and has_shrimp:
            ax.scatter(0, 0, color="yellow", s=300, marker="*",
                       zorder=6, label="Initial shrimp (axis)")

        ax.set_aspect("equal")
        align_label = {"shrimp": "cuttlefish→shrimp",
                       "motion": "initial movement",
                       "end":    "start→end"}[align_on]
        ax.set_xlabel(f"{align_label} axis (px)",
                       color="#aaaaaa", fontsize=11)
        ax.set_ylabel("Perpendicular axis (px)",
                       color="#aaaaaa", fontsize=11)
        ax.set_title(
            "Overlay — mean trajectories per strategy\n"
            f"(X axis = {align_label} | ±1 std shown transparent)",
            color="#cccccc", fontsize=12)
        ax.grid(True, color="#1a2535", alpha=0.5)

        leg = ax.legend(facecolor="#0d0d1a", labelcolor="#cccccc",
                         fontsize=10, loc="upper left")
        leg.get_frame().set_edgecolor("#444444")

        fig.suptitle("Comparison of the mean trajectories per strategy",
                     color="#cccccc", fontsize=14)
        plt.tight_layout()
        plt.show()




In [ ]:
import matplotlib

%matplotlib inline

plot_strategy_trajectories_arena(
    df=df,
    mapping_letters=mapping_letters,
    strategies_to_plot=None,
    arena_size=1500,
    smooth_mean=True,
    sigma_mean=15,
    align_on="shrimp"
)

In [ ]:
%matplotlib inline


In [ ]:
# ══════════════════════════════════════════════════════════════
#  POLAR PLOTS + KDR — inline notebook version
# ══════════════════════════════════════════════════════════════

%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ── Parameters ────────────────────────────────────────────────
COMPARATIVE_DATA_PATH = DATA_DIR / "Data_cuttlfish_strike_complet.xlsx"
HIT_DATA_PATH         = r""   # ← fill in if a hit file is available

DIST_COL  = "Distance_strike_px"
ANGLE_COL = "angle_strike_360"
CATCH_COL = "catch_type"
NBINS     = 16
CAP_PCT   = 0.90

# ── Build df_summary from mapping_optimal ──────────────
df_summary = pd.DataFrame([
    {"ID": int(aid), "strategy": g}
    for aid, g in mapping_optimal.items()
    if g != "MONO"
])
print(f"[OK] {len(df_summary)} individuals | "
      f"strategies : {sorted(df_summary['strategy'].unique())}")

# ── Loading + merge ───────────────────────────────────────
df_data   = pd.read_excel(COMPARATIVE_DATA_PATH)
df_merged = df_data.merge(
    df_summary[["ID","strategy"]],
    left_on="video_id", right_on="ID", how="inner"
).dropna(subset=["strategy"])

strategies = sorted(df_merged["strategy"].unique())
n_strat    = len(strategies)
print(f"[OK] {len(df_merged)} individuals after merge | "
      f"{n_strat} strategies : {strategies}")

# Palette consistent with the notebook
pal_strat = {s: STRATEGY_PALETTE[i % len(STRATEGY_PALETTE)]
             for i, s in enumerate(strategies)}

def _hex_rgb(h):
    h = h.lstrip("#")
    return tuple(int(h[i:i+2],16)/255 for i in (0,2,4))

def _theta(series):
    return np.deg2rad(series % 360)

def _setup_polar(ax, r_lim):
    ax.set_theta_zero_location("S")
    ax.set_theta_direction(1)
    ax.set_ylim(0, r_lim)
    ax.grid(alpha=0.3, color="#334466")
    ax.set_rlabel_position(180)
    ax.tick_params(labelsize=7, colors="#aaaaaa")
    ax.set_facecolor("#0d0d1a")

def _draw_density(ax, theta_vals, nbins, r_lim):
    hist, edges = np.histogram(theta_vals, bins=nbins, range=(0, 2*np.pi))
    norm = hist / hist.max() if hist.max() > 0 else hist
    cmap = plt.cm.Reds
    for left, width, h in zip(edges[:-1], np.diff(edges), norm):
        ax.bar(left, r_lim, width=width, bottom=0.0,
               color=cmap(h*0.7), edgecolor="none",
               align="edge", alpha=0.85)

def _circular_mean(angles_rad):
    m = np.arctan2(np.sin(angles_rad).sum(), np.cos(angles_rad).sum())
    return m if m >= 0 else m + 2*np.pi


# ════════════════════════════════════════════════════════════
# FIGURE 1 — Global polar plot
# ════════════════════════════════════════════════════════════
if DIST_COL in df_merged.columns and ANGLE_COL in df_merged.columns:

    cap_val = df_merged[DIST_COL].quantile(CAP_PCT)
    df_merged["_dist_plot"] = np.minimum(
        df_merged[DIST_COL].astype(float), float(cap_val))
    r_lim_global = df_merged["_dist_plot"].max() * 1.05

    fig = plt.figure(figsize=(7,7))
    fig.patch.set_facecolor("#1a1a2e")
    ax  = fig.add_subplot(111, polar=True)
    _setup_polar(ax, r_lim_global)

    theta_all = _theta(df_merged[ANGLE_COL].dropna())
    dist_all  = df_merged.loc[df_merged[ANGLE_COL].notna(), "_dist_plot"]
    _draw_density(ax, theta_all, NBINS, r_lim_global)
    ax.scatter(theta_all, dist_all, marker="x", s=20,
               alpha=0.4, color="white", zorder=2)
    ax.set_title("GLOBAL — Angular heatmap (all strategies)",
                 color="#cccccc", pad=14)
    ax.text(0,      r_lim_global, "Back",  fontsize=12,
            fontweight="bold", ha="center", va="bottom", color="#cccccc")
    ax.text(np.pi,  r_lim_global, "Front", fontsize=12,
            fontweight="bold", ha="center", va="top",   color="#cccccc")
    plt.tight_layout(); plt.show()


    # ════════════════════════════════════════════════════════
    # FIGURE 2 — Polar plot per strategy
    # ════════════════════════════════════════════════════════
    nc   = min(3, n_strat)
    nr   = int(np.ceil(n_strat / nc))
    fig, axes = plt.subplots(nr, nc, subplot_kw=dict(polar=True),
                              figsize=(nc*4.5, nr*4))
    fig.patch.set_facecolor("#1a1a2e")
    fig.suptitle("Angular heatmap per strategy",
                 color="#cccccc", fontsize=12)
    axes_flat = np.array(axes).flatten() if nr*nc > 1 else [axes]

    for i, strat in enumerate(strategies):
        ax  = axes_flat[i]
        sub = df_merged[df_merged["strategy"]==strat
                        ].dropna(subset=[ANGLE_COL])
        r_lim_s = sub["_dist_plot"].max()*1.05 if len(sub) else 1.0
        _setup_polar(ax, r_lim_s)
        if len(sub):
            th = _theta(sub[ANGLE_COL])
            _draw_density(ax, th, NBINS, r_lim_s)
            ax.scatter(th, sub["_dist_plot"], marker="x", s=20,
                       alpha=0.4, color="white", zorder=2)
            ax.text(0,     r_lim_s, "Back",  fontsize=9,
                    fontweight="bold", ha="center", va="bottom",
                    color="#cccccc")
            ax.text(np.pi, r_lim_s, "Front", fontsize=9,
                    fontweight="bold", ha="center", va="top",
                    color="#cccccc")
        ax.set_title(f"Strategy {strat} (n={len(sub)})",
                     color=pal_strat.get(strat,"#ffffff"), pad=8)

    for j in range(n_strat, len(axes_flat)):
        axes_flat[j].set_visible(False)
    plt.tight_layout(); plt.show()


    # ════════════════════════════════════════════════════════
    # FIGURE 3 — Circular means ± SD
    # ════════════════════════════════════════════════════════
    fig = plt.figure(figsize=(7,7))
    fig.patch.set_facecolor("#1a1a2e")
    ax  = fig.add_subplot(111, polar=True)
    _setup_polar(ax, r_lim_global)

    for strat in strategies:
        sub = df_merged[df_merged["strategy"]==strat
                        ].dropna(subset=[ANGLE_COL,"_dist_plot"])
        if len(sub) == 0: continue
        theta_mean = _circular_mean(np.deg2rad(sub[ANGLE_COL]))
        r_mean     = sub["_dist_plot"].mean()
        r_std      = sub["_dist_plot"].std()
        col_s      = _hex_rgb(pal_strat.get(strat,"#ffffff"))
        ax.errorbar(theta_mean, r_mean, yerr=r_std, fmt="o",
                    color=col_s, capsize=4, markersize=8,
                    label=f"Strat {strat}")

    ax.set_title("Circular means ± SD per strategy",
                 color="#cccccc", pad=14)
    leg = ax.legend(loc="upper center", bbox_to_anchor=(0.5,-0.12),
                    ncol=3, fontsize=8)
    leg.get_frame().set_facecolor("#1a1a2e")
    leg.get_frame().set_edgecolor("#334466")
    for t in leg.get_texts(): t.set_color("#cccccc")
    plt.tight_layout(); plt.show()

    df_merged.drop(columns=["_dist_plot"], inplace=True)


    # ════════════════════════════════════════════════════════
    # FIGURE 4 — catch_type × strategy heatmaps
    # ════════════════════════════════════════════════════════
    catch_col_found = None
    for c in ["catch_type", "catch_type_redefined", "catch_type_found",
               "Catch_type"]:
        if c in df_merged.columns:
            catch_col_found = c
            break

    if catch_col_found:
        df_counts = (
            df_merged.dropna(subset=[catch_col_found,"strategy"])
            .groupby(["strategy", catch_col_found])
            .size().reset_index(name="count")
        )

        # Proportion of catch within strategy
        df_pc         = df_counts.copy()
        df_pc["prop"] = (df_pc["count"] /
                         df_pc.groupby("strategy")["count"].transform("sum"))
        pivot = df_pc.pivot(index="strategy", columns=catch_col_found,
                            values="prop").fillna(0)

        fig, ax = plt.subplots(
            figsize=(max(6, len(pivot.columns)*1.2), max(4, len(pivot))))
        fig.patch.set_facecolor("#1a1a2e")
        ax.set_facecolor("#0d0d1a")
        sns.heatmap(pivot, annot=True, fmt=".2f", cmap="YlGnBu", ax=ax,
                    cbar_kws={"label":"Proportion"})
        ax.set_title("Proportion of catch_type in each strategy",
                     color="#cccccc")
        ax.tick_params(colors="#aaaaaa")
        plt.tight_layout(); plt.show()

        # Proportion of strategy within catch
        df_ps         = df_counts.copy()
        df_ps["prop"] = (df_ps["count"] /
                         df_ps.groupby(catch_col_found)["count"].transform("sum"))
        pivot2 = df_ps.pivot(index=catch_col_found, columns="strategy",
                             values="prop").fillna(0)

        fig, ax = plt.subplots(
            figsize=(max(6, len(pivot2.columns)*1.2), max(4, len(pivot2))))
        fig.patch.set_facecolor("#1a1a2e")
        ax.set_facecolor("#0d0d1a")
        sns.heatmap(pivot2, annot=True, fmt=".2f", cmap="Purples", ax=ax,
                    cbar_kws={"label":"Proportion"})
        ax.set_title("Proportion of strategies in each catch_type",
                     color="#cccccc")
        ax.tick_params(colors="#aaaaaa")
        plt.tight_layout(); plt.show()
    else:
        print("[SKIP] No catch_type column found")

else:
    print(f"[SKIP] Missing columns : {DIST_COL} / {ANGLE_COL}")

# ════════════════════════════════════════════════════════
    # FIGURE 4 — Heatmaps for each catch_type variant
    # ════════════════════════════════════════════════════════
    for catch_col in ["catch_type", "catch_type_redefined", "catch_type_found"]:
        if catch_col not in df_merged.columns:
            print(f"[SKIP] {catch_col} missing")
            continue

        df_counts = (
            df_merged.dropna(subset=[catch_col, "strategy"])
            .groupby(["strategy", catch_col])
            .size().reset_index(name="count")
        )
        if len(df_counts) == 0:
            print(f"[SKIP] {catch_col} — no data")
            continue

        # Proportion of catch within strategy
        df_pc         = df_counts.copy()
        df_pc["prop"] = (df_pc["count"] /
                         df_pc.groupby("strategy")["count"].transform("sum"))
        pivot = df_pc.pivot(index="strategy", columns=catch_col,
                            values="prop").fillna(0)

        fig, ax = plt.subplots(
            figsize=(max(6, len(pivot.columns)*1.2), max(4, len(pivot))))
        fig.patch.set_facecolor("#1a1a2e"); ax.set_facecolor("#0d0d1a")
        sns.heatmap(pivot, annot=True, fmt=".2f", cmap="YlGnBu", ax=ax,
                    cbar_kws={"label":"Proportion"})
        ax.set_title(f"Proportion of {catch_col} in each strategy",
                     color="#cccccc")
        ax.tick_params(colors="#aaaaaa")
        plt.tight_layout(); plt.show()

        # Proportion of strategy within catch
        df_ps         = df_counts.copy()
        df_ps["prop"] = (df_ps["count"] /
                         df_ps.groupby(catch_col)["count"].transform("sum"))
        pivot2 = df_ps.pivot(index=catch_col, columns="strategy",
                             values="prop").fillna(0)

        fig, ax = plt.subplots(
            figsize=(max(6, len(pivot2.columns)*1.2), max(4, len(pivot2))))
        fig.patch.set_facecolor("#1a1a2e"); ax.set_facecolor("#0d0d1a")
        sns.heatmap(pivot2, annot=True, fmt=".2f", cmap="Purples", ax=ax,
                    cbar_kws={"label":"Proportion"})
        ax.set_title(f"Proportion of strategies in each {catch_col}",
                     color="#cccccc")
        ax.tick_params(colors="#aaaaaa")
        plt.tight_layout(); plt.show()

# ════════════════════════════════════════════════════════════
# FIGURE 5-8 — KDR (relative distance of the strike point)
# ════════════════════════════════════════════════════════════
hit_path = DATA_DIR / "Classeur1.xlsx"

if hit_path.exists():
    print("\n[OK] KDR plots...")
    df_hit = pd.read_excel(hit_path)

    # Merge on video → video_id to get Length_Shrimp + strategy
    df_hit = df_hit.merge(
        df_merged[["video_id", "Length_Shrimp", "strategy"]],
        left_on="video", right_on="video_id", how="left"
    )

    print(f"  After merge : {len(df_hit)} rows")
    print(f"  Matched with strategy : {df_hit['strategy'].notna().sum()}")

    df_hit["distance_relative"] = (
        df_hit["distance_yeux_tentacule"].astype(float) /
        df_hit["Length_Shrimp"].astype(float)
    ) * 100

    df_hit     = df_hit.dropna(subset=["distance_relative","strategy"])
    strats_hit = sorted(df_hit["strategy"].unique())
    print(f"  {len(df_hit)} valid individuals | strategies : {strats_hit}")

    # ── Global KDE ───────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor("#1a1a2e"); ax.set_facecolor("#0d0d1a")
    for sp in ax.spines.values(): sp.set_edgecolor("#333333")
    for strat in strats_hit:
        sub   = df_hit[df_hit["strategy"]==strat]["distance_relative"]
        col_s = _hex_rgb(pal_strat.get(strat,"#ffffff"))
        sns.kdeplot(sub, ax=ax, fill=False, lw=2,
                    color=col_s, label=f"Strat {strat}")
    ax.axvline(50, color="#ff4444", lw=1.5, ls="--",
               label="Random reference (50%)")
    ax.set_xlim(0, 100)
    ax.set_xlabel("Relative distance (%)", color="#aaaaaa")
    ax.set_ylabel("KDE density",           color="#aaaaaa")
    ax.set_title("KDR — KDE curves per strategy", color="#cccccc")
    ax.tick_params(colors="#aaaaaa")
    leg = ax.legend(fontsize=8)
    leg.get_frame().set_facecolor("#111122")
    for t in leg.get_texts(): t.set_color("#cccccc")
    plt.tight_layout(); plt.show()

    # ── KDR boxplot ───────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(max(8, len(strats_hit)*1.4), 5))
    fig.patch.set_facecolor("#1a1a2e"); ax.set_facecolor("#0d0d1a")
    for sp in ax.spines.values(): sp.set_edgecolor("#333333")
    pal_hit = {s: pal_strat.get(s,"#888888") for s in strats_hit}
    sns.boxplot(x="strategy", y="distance_relative", data=df_hit,
                order=strats_hit, palette=pal_hit, ax=ax)
    sns.stripplot(x="strategy", y="distance_relative", data=df_hit,
                  order=strats_hit, color="white",
                  alpha=0.4, size=3, ax=ax)
    ax.axhline(50, color="#ff4444", ls="--", lw=1.5,
               label="Random reference (50%)")
    ax.set_xticklabels([f"Strat {s}" for s in strats_hit], color="#aaaaaa")
    ax.set_title("KDR — Distribution per strategy", color="#cccccc")
    ax.set_ylabel("Relative distance (%)", color="#aaaaaa")
    ax.tick_params(colors="#aaaaaa")
    leg = ax.legend(fontsize=8)
    leg.get_frame().set_facecolor("#111122")
    for t in leg.get_texts(): t.set_color("#cccccc")
    plt.tight_layout(); plt.show()

    # ── Mean ± SD horizontal line ────────────────────────
    means      = [df_hit[df_hit["strategy"]==s]["distance_relative"].mean()
                  for s in strats_hit]
    stds       = [df_hit[df_hit["strategy"]==s]["distance_relative"].std()
                  for s in strats_hit]
    means_norm = np.array(means)/100
    stds_norm  = np.array(stds)/100

    fig, ax = plt.subplots(figsize=(10, 2))
    fig.patch.set_facecolor("#1a1a2e"); ax.set_facecolor("#0d0d1a")
    ax.plot([0,1],[0,0], color="#555555", lw=1)
    ax.axvline(0.5, color="#ff4444", ls="--", lw=2, label="50% reference")
    for i, strat in enumerate(strats_hit):
        m, s  = means_norm[i], stds_norm[i]
        col_s = _hex_rgb(pal_strat.get(strat,"#888888"))
        ax.hlines(0, m-s, m+s, color=col_s, lw=6, alpha=0.6)
        ax.scatter(m, 0, color=col_s, s=100,
                   edgecolor="white", zorder=3, label=f"Strat {strat}")
    ax.set_yticks([])
    ax.set_xlim(0, 1)
    ax.set_xlabel("Normalised relative distance (0–1)", color="#aaaaaa")
    ax.set_title("KDR — Mean position ± SD per strategy", color="#cccccc")
    ax.tick_params(colors="#aaaaaa")
    for sp in ax.spines.values(): sp.set_edgecolor("#333333")
    leg = ax.legend(loc="upper center", bbox_to_anchor=(0.5,-0.3),
                    ncol=len(strats_hit), fontsize=8)
    leg.get_frame().set_facecolor("#111122")
    for t in leg.get_texts(): t.set_color("#cccccc")
    plt.tight_layout(); plt.show()

    # ── Mean ± SD errorbar ─────────────────────────────────
    x   = np.arange(len(strats_hit))
    fig, ax = plt.subplots(figsize=(max(8, len(strats_hit)*1.4), 5))
    fig.patch.set_facecolor("#1a1a2e"); ax.set_facecolor("#0d0d1a")
    for sp in ax.spines.values(): sp.set_edgecolor("#333333")
    ax.errorbar(x, means, yerr=stds, fmt="o", markersize=8,
                capsize=5, color="white", ecolor="#888888", elinewidth=2)
    for i, strat in enumerate(strats_hit):
        col_s = _hex_rgb(pal_strat.get(strat,"#888888"))
        ax.scatter(x[i], means[i], color=col_s, s=120, zorder=3)
    ax.axhline(50, color="#ff4444", ls="--", lw=1.5,
               label="Random reference (50%)")
    ax.set_xticks(x)
    ax.set_xticklabels([f"Strat {s}" for s in strats_hit], color="#aaaaaa")
    ax.set_ylabel("Mean relative distance (%)", color="#aaaaaa")
    ax.set_title("KDR — Mean ± SD per strategy", color="#cccccc")
    ax.tick_params(colors="#aaaaaa")
    ax.grid(alpha=0.15, color="#334455")
    leg = ax.legend(fontsize=8)
    leg.get_frame().set_facecolor("#111122")
    for t in leg.get_texts(): t.set_color("#cccccc")
    plt.tight_layout(); plt.show()

else:
    print(f"[SKIP] Hit file not found : {hit_path}")


In [ ]:
# ════════════════════════════════════════════════════════════
#  CATCH TYPE × STRATEGY COMPARISON
# ════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, fisher_exact
from itertools import combinations

fig_bg  = "#1a1a2e"
ax_bg   = "#0d0d1a"
txt_col = "#cccccc"

def _style(ax):
    ax.set_facecolor(ax_bg)
    for sp in ax.spines.values(): sp.set_edgecolor("#333333")
    ax.tick_params(colors="#aaaaaa")

# ════════════════════════════════════════════════════════════
# FIGURE 1 — Distribution of catch_type per strategy
#            (normalised stacked bars)
# ════════════════════════════════════════════════════════════
for catch_col, title_suffix in [
    ("catch_type",          "detailed (6 categories)"),
    ("catch_type_redefined","simplified (Pursuit / No Pursuit)"),
]:
    ct = pd.crosstab(df_merged["strategy"], df_merged[catch_col],
                     normalize="index") * 100

    fig, ax = plt.subplots(figsize=(max(10, len(ct)*0.9), 5))
    fig.patch.set_facecolor(fig_bg); _style(ax)

    bottom = np.zeros(len(ct))
    cats   = ct.columns.tolist()
    cmap   = plt.cm.tab10

    for i, cat in enumerate(cats):
        vals = ct[cat].values
        bars = ax.bar(ct.index, vals, bottom=bottom,
                      color=cmap(i/len(cats)), label=cat,
                      edgecolor="#0d0d1a", lw=0.5, alpha=0.9)
        # Label if wide enough
        for j, (v, b) in enumerate(zip(vals, bottom)):
            if v > 5:
                ax.text(j, b + v/2, f"{v:.0f}%",
                        ha="center", va="center",
                        fontsize=7, color="white", fontweight="bold")
        bottom += vals

    ax.set_ylim(0, 100)
    ax.set_ylabel("% individuals", color="#aaaaaa")
    ax.set_xlabel("Strategy",      color="#aaaaaa")
    ax.set_title(f"Distribution of catch_type {title_suffix}\nper strategy",
                 color=txt_col, fontsize=11)
    ax.tick_params(axis="x", colors="#aaaaaa")
    ax.tick_params(axis="y", colors="#aaaaaa")
    leg = ax.legend(loc="upper right", fontsize=8,
                    facecolor="#111122", edgecolor="#334466")
    for t in leg.get_texts(): t.set_color(txt_col)
    plt.tight_layout(); plt.show()


# ════════════════════════════════════════════════════════════
# FIGURE 2 — Distribution of strategies per catch_type
#            (reverse: who does what?)
# ════════════════════════════════════════════════════════════
for catch_col, title_suffix in [
    ("catch_type",          "detailed"),
    ("catch_type_redefined","simplified"),
]:
    ct = pd.crosstab(df_merged[catch_col], df_merged["strategy"],
                     normalize="index") * 100

    fig, ax = plt.subplots(figsize=(max(8, len(ct)*1.2), 5))
    fig.patch.set_facecolor(fig_bg); _style(ax)

    bottom = np.zeros(len(ct))
    for i, strat in enumerate(ct.columns):
        vals = ct[strat].values
        col_s = pal_strat.get(strat, "#888888")
        r,g,b = tuple(int(col_s.lstrip("#")[j:j+2],16)/255 for j in (0,2,4))
        ax.bar(ct.index, vals, bottom=bottom,
               color=(r,g,b), label=f"Strat {strat}",
               edgecolor="#0d0d1a", lw=0.5, alpha=0.9)
        for j, (v, b) in enumerate(zip(vals, bottom)):
            if v > 6:
                ax.text(j, b + v/2, strat,
                        ha="center", va="center",
                        fontsize=7, color="white", fontweight="bold")
        bottom += vals

    ax.set_ylim(0, 100)
    ax.set_ylabel("% individuals", color="#aaaaaa")
    ax.set_xlabel(catch_col,     color="#aaaaaa")
    ax.set_title(f"Distribution of strategies per catch_type {title_suffix}",
                 color=txt_col, fontsize=11)
    ax.tick_params(axis="x", colors="#aaaaaa", rotation=20)
    ax.tick_params(axis="y", colors="#aaaaaa")
    leg = ax.legend(loc="upper right", fontsize=7, ncol=2,
                    facecolor="#111122", edgecolor="#334466")
    for t in leg.get_texts(): t.set_color(txt_col)
    plt.tight_layout(); plt.show()


# ════════════════════════════════════════════════════════════
# FIGURE 3 — Heatmap counts + proportions
# ════════════════════════════════════════════════════════════
for catch_col in ["catch_type", "catch_type_redefined"]:
    ct_count = pd.crosstab(df_merged["strategy"], df_merged[catch_col])
    ct_prop  = ct_count.div(ct_count.sum(axis=1), axis=0).round(2)

    # Annotations: "n (xx%)"
    annot = ct_count.astype(str) + "\n(" + (ct_prop*100).round(0).astype(int).astype(str) + "%)"

    fig, ax = plt.subplots(
        figsize=(max(6, len(ct_count.columns)*1.5),
                 max(4, len(ct_count)*0.7)))
    fig.patch.set_facecolor(fig_bg)
    sns.heatmap(ct_prop, annot=annot, fmt="", cmap="YlOrRd",
                ax=ax, linewidths=0.5, linecolor="#333333",
                cbar_kws={"label":"Proportion"})
    ax.set_title(f"Strategy × {catch_col}\n(n and proportion per row)",
                 color=txt_col, fontsize=11)
    ax.set_xlabel(catch_col,   color="#aaaaaa")
    ax.set_ylabel("Strategy", color="#aaaaaa")
    ax.tick_params(colors="#aaaaaa")
    plt.tight_layout(); plt.show()


# ════════════════════════════════════════════════════════════
# FIGURE 4 — Chi² test + standardised residuals
#            Which strategy/catch associations are
#            significantly over- or under-represented?
# ════════════════════════════════════════════════════════════
for catch_col, label in [
    ("catch_type",          "detailed catch"),
    ("catch_type_redefined","Pursuit/No Pursuit"),
]:
    ct_count = pd.crosstab(df_merged["strategy"], df_merged[catch_col])

    chi2, p, dof, expected = chi2_contingency(ct_count)
    print(f"\n{'='*50}")
    print(f"Chi² ({label}) : χ²={chi2:.2f}  p={p:.4f}  df={dof}")
    if p < 0.001:  print("  *** p < 0.001 — highly significant association")
    elif p < 0.01: print("  **  p < 0.01  — significant association")
    elif p < 0.05: print("  *   p < 0.05  — significant association")
    else:          print("  ns  p ≥ 0.05  — no significant association")

    # Standardised residuals: (observed - expected) / sqrt(expected)
    # > +2 = over-representation  |  < -2 = under-representation
    resid = (ct_count.values - expected) / np.sqrt(expected)
    df_resid = pd.DataFrame(resid,
                            index=ct_count.index,
                            columns=ct_count.columns)

    fig, ax = plt.subplots(
        figsize=(max(6, len(ct_count.columns)*1.5),
                 max(4, len(ct_count)*0.7)))
    fig.patch.set_facecolor(fig_bg)
    sns.heatmap(df_resid, annot=True, fmt=".1f",
                cmap="RdBu_r", center=0, vmin=-4, vmax=4,
                ax=ax, linewidths=0.5, linecolor="#333333",
                cbar_kws={"label":"Standardised residual"})
    ax.set_title(
        f"Standardised residuals — Strategy × {label}\n"
        f"χ²={chi2:.1f}  p={p:.4f}  "
        f"{'(sig.)' if p < 0.05 else '(n.s.)'}\n"
        f"> +2 = over-represented  |  < -2 = under-represented",
        color=txt_col, fontsize=10)
    ax.set_xlabel(catch_col,   color="#aaaaaa")
    ax.set_ylabel("Strategy", color="#aaaaaa")
    ax.tick_params(colors="#aaaaaa")
    plt.tight_layout(); plt.show()



In [ ]:
# ══════════════════════════════════════════════════════════════
#  CORRESPONDENCE MATRIX — Cuttlefish × Shrimp strategies
# ══════════════════════════════════════════════════════════════

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import chi2_contingency

# ── Paths to the saved mappings ─────────────────────
PATH_CUTTLE  = OUT_DIR / "mapping_optimal_cuttle.json"
PATH_SHRIMP  = DATA_DIR / "mapping_optimal_shrimp.json"

# ── Loading ────────────────────────────────────────────────
map_cuttle = json.load(open(PATH_CUTTLE))   # {id: cuttlefish letter}
map_shrimp = json.load(open(PATH_SHRIMP))   # {id: shrimp letter}

# Normalise the keys as strings
map_cuttle = {str(k): v for k, v in map_cuttle.items()}
map_shrimp = {str(k): v for k, v in map_shrimp.items()}

# ── Common individuals ──────────────────────────────────────────
common_ids = set(map_cuttle.keys()) & set(map_shrimp.keys())
print(f"[OK] {len(common_ids)} common individuals")
print(f"     Cuttlefish : {len(map_cuttle)} individuals | "
      f"groups = {sorted(set(map_cuttle.values()))}")
print(f"     Shrimp     : {len(map_shrimp)} individuals | "
      f"groups = {sorted(set(map_shrimp.values()))}")

if len(common_ids) == 0:
    raise ValueError("No common individual — check that the animal_id values are identical")

# ── Correspondence DataFrame ───────────────────────────────
df_match = pd.DataFrame([
    {"animal_id":      aid,
     "strat_cuttle":   map_cuttle[aid],
     "strat_shrimp":   map_shrimp[aid]}
    for aid in sorted(common_ids)
])

# Exclude MONO if needed
df_match = df_match[
    (df_match["strat_cuttle"] != "MONO") &
    (df_match["strat_shrimp"] != "MONO")
]
print(f"[OK] {len(df_match)} individuals (MONO excluded)")
print(df_match.head())

# ── Contingency table: counts ──────────────────────────
ct_count = pd.crosstab(
    df_match["strat_cuttle"],
    df_match["strat_shrimp"],
    margins=False)
ct_count.index.name   = "Cuttlefish strategy"
ct_count.columns.name = "Shrimp strategy"

# ── Normalisation ─────────────────────────────────────────────
# Per row = proportion of shrimp for each cuttlefish
ct_row = ct_count.div(ct_count.sum(axis=1), axis=0).round(3)
# Per column = proportion of cuttlefish for each shrimp
ct_col = ct_count.div(ct_count.sum(axis=0), axis=1).round(3)

# ── Chi² test ─────────────────────────────────────────────────
chi2, p, dof, expected = chi2_contingency(ct_count)
print(f"\n[CHI²] χ²={chi2:.2f}  p={p:.4f}  df={dof}")
print(f"       {'*** highly significant association' if p < 0.001 else '** significant' if p < 0.01 else '* significant' if p < 0.05 else 'n.s. — no significant association'}")

# Standardised residuals
resid = (ct_count.values - expected) / np.sqrt(expected)
df_resid = pd.DataFrame(resid,
                         index=ct_count.index,
                         columns=ct_count.columns)

# ══════════════════════════════════════════════════════════════
# FIGURE 1 — Heatmap of raw counts
# ══════════════════════════════════════════════════════════════
fig, ax = plt.subplots(
    figsize=(max(6, len(ct_count.columns)*1.2),
             max(4, len(ct_count)*0.8)))
fig.patch.set_facecolor("#1a1a2e")
sns.heatmap(ct_count, annot=True, fmt="d", cmap="YlOrRd",
            ax=ax, linewidths=0.5, linecolor="#333333",
            cbar_kws={"label": "Number of individuals"})
ax.set_title(f"Cuttlefish × Shrimp correspondence — raw counts\n"
             f"χ²={chi2:.1f}  p={p:.4f}  "
             f"({'sig.' if p < 0.05 else 'n.s.'})",
             color="#cccccc", fontsize=11)
ax.set_xlabel("Shrimp strategy",     color="#aaaaaa")
ax.set_ylabel("Cuttlefish strategy", color="#aaaaaa")
ax.tick_params(colors="#aaaaaa")
plt.tight_layout(); plt.show()

# ══════════════════════════════════════════════════════════════
# FIGURE 2 — Heatmap proportion per cuttlefish (row)
#            "For each cuttlefish strategy, which shrimp?"
# ══════════════════════════════════════════════════════════════
annot_row = ct_count.astype(str) + "\n(" + (ct_row*100).round(0).astype(int).astype(str) + "%)"

fig, ax = plt.subplots(
    figsize=(max(6, len(ct_count.columns)*1.2),
             max(4, len(ct_count)*0.8)))
fig.patch.set_facecolor("#1a1a2e")
sns.heatmap(ct_row, annot=annot_row, fmt="", cmap="Blues",
            ax=ax, linewidths=0.5, linecolor="#333333",
            vmin=0, vmax=1,
            cbar_kws={"label": "Proportion (per row = per cuttlefish)"})
ax.set_title("For each CUTTLEFISH strategy → distribution of SHRIMP strategies\n"
             "(each row sums to 100%)",
             color="#cccccc", fontsize=11)
ax.set_xlabel("Shrimp strategy",     color="#aaaaaa")
ax.set_ylabel("Cuttlefish strategy", color="#aaaaaa")
ax.tick_params(colors="#aaaaaa")
plt.tight_layout(); plt.show()

# ══════════════════════════════════════════════════════════════
# FIGURE 3 — Heatmap proportion per shrimp (column)
#            "For each shrimp strategy, which cuttlefish?"
# ══════════════════════════════════════════════════════════════
annot_col = ct_count.astype(str) + "\n(" + (ct_col*100).round(0).astype(int).astype(str) + "%)"

fig, ax = plt.subplots(
    figsize=(max(6, len(ct_count.columns)*1.2),
             max(4, len(ct_count)*0.8)))
fig.patch.set_facecolor("#1a1a2e")
sns.heatmap(ct_col, annot=annot_col, fmt="", cmap="Greens",
            ax=ax, linewidths=0.5, linecolor="#333333",
            vmin=0, vmax=1,
            cbar_kws={"label": "Proportion (per column = per shrimp)"})
ax.set_title("For each SHRIMP strategy → distribution of CUTTLEFISH strategies\n"
             "(each column sums to 100%)",
             color="#cccccc", fontsize=11)
ax.set_xlabel("Shrimp strategy",     color="#aaaaaa")
ax.set_ylabel("Cuttlefish strategy", color="#aaaaaa")
ax.tick_params(colors="#aaaaaa")
plt.tight_layout(); plt.show()

# ══════════════════════════════════════════════════════════════
# FIGURE 4 — Standardised residuals
#            Which associations are over/under-represented?
# ══════════════════════════════════════════════════════════════
fig, ax = plt.subplots(
    figsize=(max(6, len(ct_count.columns)*1.2),
             max(4, len(ct_count)*0.8)))
fig.patch.set_facecolor("#1a1a2e")
sns.heatmap(df_resid, annot=True, fmt=".1f",
            cmap="RdBu_r", center=0, vmin=-4, vmax=4,
            ax=ax, linewidths=0.5, linecolor="#333333",
            cbar_kws={"label": "Standardised residual"})
ax.set_title(
    f"Standardised residuals — Cuttlefish × Shrimp\n"
    f"χ²={chi2:.1f}  p={p:.4f}  "
    f"({'sig.' if p < 0.05 else 'n.s.'})\n"
    f"> +2 = over-represented association  |  < -2 = under-represented",
    color="#cccccc", fontsize=10)
ax.set_xlabel("Shrimp strategy",     color="#aaaaaa")
ax.set_ylabel("Cuttlefish strategy", color="#aaaaaa")
ax.tick_params(colors="#aaaaaa")
plt.tight_layout(); plt.show()

# ── Text summary ──────────────────────────────────────────────
print(f"\n{'='*55}")
print(f"NOTABLE ASSOCIATIONS (residual > +2 or < -2)")
print(f"{'='*55}")
for i in df_resid.index:
    for j in df_resid.columns:
        r = df_resid.loc[i, j]
        if abs(r) >= 2:
            n = ct_count.loc[i, j]
            direction = "OVER-represented" if r > 0 else "UNDER-represented"
            print(f"  Cuttlefish {i} × Shrimp {j} : "
                  f"residual={r:.1f}  n={n}  → {direction}")

Individual plot interface

In [ ]:
# ══════════════════════════════════════════════════════════════
#  BEHAVIOURAL SIMULATOR — standalone notebook version
# ══════════════════════════════════════════════════════════════

%matplotlib inline   # tkinter window (not matplotlib): no need for qt

import io, math, contextlib
import numpy as np
import pandas as pd
import tkinter as tk
from tkinter import ttk
from pathlib import Path
from collections import Counter
from itertools import groupby
from datetime import datetime

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patches as mpatches

try:
    from scipy.cluster.hierarchy import dendrogram as scipy_dendrogram, fcluster, linkage as sp_linkage
    from sklearn.preprocessing import StandardScaler
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False

try:
    from PIL import Image, ImageTk, ImageDraw
    HAS_PIL = True
except ImportError:
    HAS_PIL = False

# ── Layout constants ─────────────────────────────────────────
BG         = "#0a0a1c"
BG2        = "#0d0d22"
SEQ_H      = 56
SEQ_CELL_W = 40
ARENA_W    = 480
ARENA_H    = 480
ARENA_R    = 210
DEFAULT_ARENA_REAL_RADIUS = 330
SIM_FPS    = 60       # frames / s
SIM_TICK   = 15       # 1 window of the strip = 15 frames = 0.25 s


# ════════════════════════════════════════════════════════════════
#  COLOUR HELPERS
# ════════════════════════════════════════════════════════════════

def _hex_to_rgb01(h):
    h = h.lstrip("#")
    return tuple(int(h[i:i+2], 16)/255 for i in (0,2,4))

def _hex_to_rgb255(h):
    h = h.lstrip("#")
    return tuple(int(h[i:i+2], 16) for i in (0,2,4))

def _col_mpl(cid, pal):
    return _hex_to_rgb01(pal[cid % len(pal)])


# ════════════════════════════════════════════════════════════════
#  LOADING DATA FROM THE NOTEBOOK
# ════════════════════════════════════════════════════════════════

def _build_model_data(df, final_state, features, palette):
    """
    Builds the 'data' dict used by the simulator
    directly from the notebook variables.

    df          : DataFrame with 'cluster' columns + features
    final_state : result of show_strategy_explorer
    features    : list of features (e.g. FEATURES)
    palette     : list of hex colours (e.g. PALETTE)
    """
    K    = final_state["n_cl"]
    seqs = final_state["seqs"]

    df_w = df.copy()
    df_w["animal_id"] = df_w["animal_id"].astype(str)

    # ── Cluster profiles: mean of the features per cluster ───────────────
    cluster_profiles = np.zeros((K, len(features)))
    for c in range(K):
        mask = df_w["cluster"] == c
        if mask.any():
            cluster_profiles[c] = df_w.loc[mask, features].mean().values

    prof_min = cluster_profiles.min(axis=0)
    prof_max = cluster_profiles.max(axis=0)

    # ── (K+2)×(K+2) transition matrix from the sequences ───────────
    SIZE  = K + 2
    START = 0
    END   = K + 1
    trans_mat = np.zeros((SIZE, SIZE), dtype=float)

    n_built = 0
    for aid, info in seqs.items():
        seq = [int(c) for c in info["seq"] if 0 <= int(c) < K]
        if not seq:
            continue
        trans_mat[START, seq[0]+1] += 1
        for t in range(len(seq)-1):
            trans_mat[seq[t]+1, seq[t+1]+1] += 1
        trans_mat[seq[-1]+1, END] += 1
        n_built += 1

    print(f"[SIMU] Matrix from {n_built} sequences")

    # Normalisation
    for i in range(SIZE-1):
        s = trans_mat[i].sum()
        if s > 0:
            trans_mat[i] /= s

    p_start = trans_mat[START, 1:K+1]
    print(f"[SIMU] P(START) = {[f'C{c}:{p_start[c]:.2f}' for c in range(K)]}")

    # ── Segment durations per state (frames): probability of STAYING ──
    #   The matrix only contains state CHANGES (compressed sequences):
    #   the duration of each segment is drawn among the real durations
    #   of that state (semi-Markov model).
    durations = {c: [] for c in range(K)}
    for info in seqs.values():
        for c, l in zip(info["seq"], info["lengths"]):
            if 0 <= int(c) < K:
                durations[int(c)].append(int(l))
    all_l = [l for v in durations.values() for l in v] or [SIM_FPS]
    durations = {c: np.array(v if v else all_l) for c, v in durations.items()}
    print(f"[SIMU] Median segment duration : " + " | ".join(
        f"C{c} {np.median(v) / SIM_FPS:.1f}s" for c, v in durations.items()))

    # ── State prototypes (standardised + weighted space, as in SeqVec) ──
    fw    = globals().get("FEATURE_WEIGHTS", {}) or {}
    zs    = ((df_w[features] - df_w[features].mean()) / df_w[features].std()
             * np.array([fw.get(f, 1.0) for f in features]))
    proto = zs.groupby(df_w["cluster"].values).mean().reindex(range(K)).fillna(0).to_numpy()

    return {
        "durations":        durations,
        "proto":            proto,
        "state_dist_mm":    (df_w.groupby("cluster")["distance_mm"].median()   # median distance to the prey in each state (start family)
                             .reindex(range(K)).fillna(0).to_numpy()
                             if "distance_mm" in df_w.columns else None),
        "state_speed":      (df_w.groupby("cluster")["speed_cuttle_mm_s"].mean()   # mean speed of each state (SeqVec speed family)
                             .reindex(range(K)).fillna(0).to_numpy()
                             if "speed_cuttle_mm_s" in df_w.columns else None),
        "final_state":      final_state,
        "trans_mat":        trans_mat,
        "cluster_profiles": cluster_profiles,
        "prof_min":         prof_min,
        "prof_max":         prof_max,
        "K":                K,
        "START":            START,
        "END":              END,
        "features":         features,
        "palette":          palette[:K],
    }


def _build_dendro_data(final_state):
    """
    Builds the dendrogram data from final_state.
    Uses the already computed Z (linkage) and seqs.
    """
    if not HAS_SCIPY:
        return None
    try:
        Z    = final_state.get("Z")
        seqs = final_state.get("seqs", {})
        n_cl = final_state.get("n_cl", 1)
        n_s  = final_state.get("n_s", 4)

        if Z is None:
            print("[SIMU] Z missing from final_state")
            return None

        ids = list(seqs.keys())

        # Simple feature matrix: histogram of the clusters
        rows = []
        for aid in ids:
            sq = seqs[aid]["seq"]
            lg = seqs[aid]["lengths"]
            total = sum(lg)
            if total == 0:
                rows.append(np.zeros(n_cl)); continue
            hist = np.zeros(n_cl)
            for c, l in zip(sq, lg):
                if c < n_cl: hist[c] += l / total
            rows.append(hist)

        feature_matrix = pd.DataFrame(
            np.array(rows), index=ids,
            columns=[f"hist_C{c}" for c in range(n_cl)])

        scaler   = StandardScaler()
        X_scaled = scaler.fit_transform(feature_matrix.values.astype(float))

        h_min    = float(Z[:, 2].min()) * 0.5
        h_max    = float(Z[-1, 2])      * 1.05
        init_cut = float(Z[-1, 2])      * 0.5

        # Find the height corresponding to n_s groups
        heights_desc = sorted(Z[:, 2], reverse=True)
        for k in range(2, 15):
            idx = k - 2
            if idx < len(heights_desc):
                h_above = heights_desc[idx]
                h_below = heights_desc[idx+1] if idx+1 < len(heights_desc) else 0.0
                h = (h_above + h_below) / 2.0
                try:
                    n = len(set(fcluster(Z, h, criterion="distance")))
                    if n == n_s:
                        init_cut = h
                        break
                except:
                    pass

        print(f"[SIMU] Dendro OK : {len(ids)} individuals, init_cut={init_cut:.2f}")

        return {
            "Z_linkage":      Z,
            "feature_matrix": feature_matrix,
            "groups_map":        final_state.get("strat", {}),
            "scaler":         scaler,
            "X_scaled":       X_scaled,
            "column_names":   list(feature_matrix.columns),
            "h_min":          h_min,
            "h_max":          h_max,
            "init_cut":       init_cut,
        }
    except Exception as e:
        import traceback
        print(f"[SIMU] Dendro : {e}")
        traceback.print_exc()
        return None


# ════════════════════════════════════════════════════════════════
#  SIMULATION ENGINE
# ════════════════════════════════════════════════════════════════

def simulate_sequence(data, mode="random", start_cluster=None,
                      max_steps=100, allow_end=True, seed=None, return_segments=False):
    """SEMI-MARKOV simulation.
    - next state: transition matrix (state changes + END)
    - duration of each segment: drawn among the real durations of the state
    max_steps: max number of windows (1 window = SIM_TICK frames)
    Returns the sequence per window (+ the segments (state, frames))."""
    rng   = np.random.default_rng(seed)
    mat   = data["trans_mat"]
    K     = data["K"]
    START = data["START"]
    END   = data["END"]

    if mode == "random" or start_cluster is None:
        probs = mat[START, 1:K+1].copy()
        s     = probs.sum()
        probs = probs / s if s > 0 else np.ones(K)/K
        current = int(rng.choice(K, p=probs))
    else:
        current = int(start_cluster)

    segs, n_win = [], 0
    while True:
        L = int(rng.choice(data["durations"][current]))
        w = max(1, int(round(L / SIM_TICK)))
        if n_win + w >= max_steps:                       # max duration reached: segment truncated
            w = max_steps - n_win
            if w > 0:
                segs.append((current, w * SIM_TICK))
            break
        segs.append((current, L))
        n_win += w
        row = mat[current+1].copy()
        if not allow_end:
            row[END] = 0.0
        s = row.sum()
        if s <= 0:
            break
        nxt = int(rng.choice(len(row), p=row / s))
        if nxt == END:
            break
        current = nxt - 1

    sequence = [c for c, L in segs for _ in range(max(1, int(round(L / SIM_TICK))))]
    return (sequence, segs) if return_segments else sequence


def assign_strategy(segments_list, data, k=None, batch=20):
    """Strategy group of each simulated sequence, computed as in the
    explorer: SeqVec distance between the simulation and all real
    individuals, then vote of the k nearest neighbours (k = explorer knn_k).
    Groups: mapping_optimal if it exists, otherwise the explorer ones."""
    G  = globals()
    fs = data.get("final_state")
    if fs is None or "_seqvec" not in G or "seqvec_inputs" not in fs:
        return [None] * len(segments_list)
    groups = G.get("mapping_optimal") or fs.get("mapping_letters", {})
    groups = {str(a): g for a, g in groups.items()}
    real = [str(a) for a in fs["multi"] if groups.get(str(a), "MONO") != "MONO"]
    k    = min(k or fs.get("knn_k", 10), len(real))
    w    = fs.get("seqvec_weights") or dict(w_hist=1.0, w_temporal=1.0, w_lastapp=1.0,
                                            w_duration=0.5, w_align=1.0, w_dynamics=1.0, w_speed=0.5,
                                            w_path=0.5, w_start=0.5)
    inp  = fs["seqvec_inputs"]
    ser0 = inp.get("series") or {}
    spd0 = inp.get("speed") or {}
    pth0 = inp.get("path") or {}
    sta0 = inp.get("start") or {}
    n_dyn = next(iter(ser0.values())).shape[0] if ser0 else 50
    out = []
    for b0 in range(0, len(segments_list), batch):
        chunk  = segments_list[b0:b0 + batch]
        seqs   = {a: fs["seqs"][a] for a in real}
        series = dict(ser0)
        spd    = dict(spd0)
        pth    = dict(pth0)
        sta    = dict(sta0)
        sim_ids = []
        for j, segs in enumerate(chunk):
            sid = f"SIM{b0 + j}"
            st  = [int(c) for c, L in segs]
            ln  = [int(L) for c, L in segs]
            seqs[sid] = {"seq": st, "lengths": ln, "total": sum(ln)}
            v = np.repeat(data["proto"][st], ln, axis=0)        # feature series of the simulation: state prototypes, 1 row per frame
            e = np.linspace(0, len(v), n_dyn + 1).astype(int)
            series[sid] = np.vstack([v[e[i]:max(e[i + 1], e[i] + 1)].mean(0) for i in range(n_dyn)])
            if data.get("state_speed") is not None:              # mean speed of the simulation = mean of the states weighted by their duration
                spd[sid] = float(np.dot(ln, data["state_speed"][st]) / max(sum(ln), 1))
                pth[sid] = float(np.dot(ln, data["state_speed"][st]) / SIM_FPS)   # distance travelled by the simulation = speed of each state x its duration
            if data.get("state_dist_mm") is not None:            # start distance of the simulation = typical distance of its 1st state
                sta[sid] = float(data["state_dist_mm"][st[0]])
            sim_ids.append(sid)
        with contextlib.redirect_stdout(io.StringIO()):
            _, D, _ = G["_seqvec"](seqs, real + sim_ids, fs["n_cl"], **w,
                                   state_dist=inp["state_dist"],
                                   state_similarity=inp.get("state_similarity", True),
                                   series=series, speed=spd or None,
                                   path=pth or None, start=sta or None)
        nr = len(real)
        for j in range(len(sim_ids)):
            d  = D[nr + j, :nr]
            nn = np.argsort(d)[:k]
            votes = Counter(groups[real[i]] for i in nn)
            g, nv = votes.most_common(1)[0]
            out.append({"group": g, "conf": nv / len(nn), "votes": dict(votes),
                        "nearest": real[nn[0]], "nearest_group": groups[real[nn[0]]]})
    return out


def detect_cluster_from_position(cx, cy, cangle, px, py, data,
                                  prey_angle=0,
                                  arena_real_radius=DEFAULT_ARENA_REAL_RADIUS):
    dx = px - cx
    dy = py - cy
    dist_display  = math.sqrt(dx**2 + dy**2)
    scale         = arena_real_radius / ARENA_R
    distance_real = dist_display * scale

    dir_to_prey_rad = math.atan2(-dy, dx)
    diff_axes       = (cangle - prey_angle + 180) % 360 - 180
    approach_angle  = abs(diff_axes)

    # vector built from the feature NAMES (works whatever FEATURES is):
    # distance and angle known, the others at 0 as before
    feats    = list(data["features"])
    features = np.zeros(len(feats))
    if "distance_to_shrimp" in feats:
        features[feats.index("distance_to_shrimp")] = distance_real
    if "approach_angle" in feats:
        features[feats.index("approach_angle")] = approach_angle
    profiles   = data["cluster_profiles"]
    pmin, pmax = data["prof_min"], data["prof_max"]

    if profiles is None:
        return 0, {}

    feat_norm  = np.clip((features - pmin) / (pmax - pmin + 1e-9), 0, 1)
    prof_norm  = np.clip((profiles  - pmin) / (pmax - pmin + 1e-9), 0, 1)
    dists      = np.linalg.norm(prof_norm - feat_norm, axis=1)
    best       = int(np.argmin(dists))

    return best, {
        "distance_real":  round(distance_real, 1),
        "approach_angle": round(approach_angle, 1),
        "cluster_detected": best,
        "dists":          dists.round(3).tolist(),
    }


# ════════════════════════════════════════════════════════════════
#  PIL RENDERING
# ════════════════════════════════════════════════════════════════

def _render_seq_bar(sequence, palette, current_idx=None):
    n   = len(sequence)
    w   = max(n * SEQ_CELL_W, 100)
    img = Image.new("RGB", (w, SEQ_H), (6, 6, 20))
    drw = ImageDraw.Draw(img)
    for i, lab in enumerate(sequence):
        x0 = i * SEQ_CELL_W + 2
        x1 = (i+1) * SEQ_CELL_W - 2
        r, g, b = _hex_to_rgb255(palette[lab % len(palette)])
        outline = (255,255,255) if i == current_idx else (34,51,68)
        lw = 3 if i == current_idx else 1
        drw.rectangle([x0,4,x1,SEQ_H-4], fill=(r,g,b), outline=outline, width=lw)
        drw.text((x0+(x1-x0)//2-3, SEQ_H//2-6), str(lab), fill=(255,255,255))
    return img


def _render_radar(vals_norm, feature_names, color, title, size_in=2.8, dpi=90):
    SHORT = {"distance_to_shrimp":"dist","approach_angle":"angle",
             "orientation_variation":"orient.","speed_cuttle_smooth":"speed",
             "jerk_cuttle":"jerk","accel_cuttle":"accel",
             "speed_cuttle_log":"speed","speed_shrimp_log":"shrimp v",
             "cuttle_approach_speed":"approach","shrimp_retreat_speed":"retreat",
             "approach_angle_variation":"angle var.","bearing_to_prey":"bearing",
             "attack_side":"side","orbit_speed":"orbit","who_moves":"who moves"}
    fig = Figure(figsize=(size_in, size_in), facecolor=BG)
    ax  = fig.add_subplot(111, polar=True, facecolor="#0a0a1c")
    n   = len(feature_names)
    ang = np.linspace(0, 2*np.pi, n, endpoint=False).tolist()
    ang += ang[:1]
    vals = list(vals_norm) + [vals_norm[0]]
    ax.plot(ang, vals, lw=2, color=color, zorder=3)
    ax.fill(ang, vals, alpha=0.25, color=color, zorder=2)
    ax.set_xticks(ang[:-1])
    ax.set_xticklabels([SHORT.get(f,f) for f in feature_names],
                       fontsize=5.5, color="#cccccc")
    ax.set_yticks([0.25, 0.5, 0.75]); ax.set_yticklabels(["","",""])
    ax.set_ylim(0, 1)
    ax.set_title(title, fontsize=7, pad=4, color="#dddddd", fontweight="bold")
    ax.spines["polar"].set_color("#334466")
    ax.grid(color="#334466", linewidth=0.5)
    fig.tight_layout(pad=0.2)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=dpi, facecolor=BG)
    buf.seek(0)
    img = Image.open(buf).copy()
    plt.close(fig); buf.close()
    return img


def _render_cluster_grid(data, active_cluster=None, ncols=4, dpi=80):
    K       = data["K"]
    feats   = data["features"]
    palette = data["palette"]
    profs   = data["cluster_profiles"]
    pmin    = data["prof_min"]
    pmax    = data["prof_max"]

    nrows = math.ceil(K / ncols)
    fig   = Figure(figsize=(ncols*1.1, nrows*1.1+0.2), facecolor=BG)
    fig.text(0.5, 0.98, "Clusters", ha="center", va="top",
             fontsize=6, color="#aaaacc", fontweight="bold")
    SHORT = {"distance_to_shrimp":"dist","approach_angle":"angle",
             "orientation_variation":"orient.","speed_cuttle_smooth":"speed",
             "jerk_cuttle":"jerk","accel_cuttle":"accel",
             "speed_cuttle_log":"speed","speed_shrimp_log":"shrimp v",
             "cuttle_approach_speed":"approach","shrimp_retreat_speed":"retreat",
             "approach_angle_variation":"angle var.","bearing_to_prey":"bearing",
             "attack_side":"side","orbit_speed":"orbit","who_moves":"who moves"}
    n   = len(feats)
    ang = np.linspace(0, 2*np.pi, n, endpoint=False).tolist() + [0]

    for c in range(K):
        row, col = divmod(c, ncols)
        ax = fig.add_axes([
            col/ncols+0.01, (nrows-row-1)/nrows*0.93+0.01,
            1/ncols-0.02,   1/nrows*0.93-0.04], polar=True)
        is_active = (active_cluster is not None and c == active_cluster)
        ax.set_facecolor("#1a1a3a" if is_active else "#0a0a1c")
        color = _col_mpl(c, palette)
        v = (np.clip((profs[c]-pmin)/(pmax-pmin+1e-9),0,1)
             if profs is not None else np.full(n, 0.5))
        vals = list(v)+[v[0]]
        ax.plot(ang, vals, lw=2.0 if is_active else 1.4, color=color, zorder=3)
        ax.fill(ang, vals, alpha=0.30 if is_active else 0.20, color=color, zorder=2)
        ax.set_xticks(ang[:-1])
        ax.set_xticklabels([SHORT.get(f,f) for f in feats], fontsize=3.5, color="#aaaacc")
        ax.set_yticks([]); ax.set_ylim(0,1)
        ax.spines["polar"].set_color("#ffffff" if is_active else "#334466")
        ax.spines["polar"].set_linewidth(1.6 if is_active else 0.7)
        ax.grid(color="#334466", linewidth=0.4)
        ax.set_title(f"C{c}", fontsize=5.5, pad=2,
                     color="#ffffff" if is_active else "#cccccc",
                     fontweight="bold")

    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=dpi, facecolor=BG, bbox_inches="tight")
    buf.seek(0)
    img = Image.open(buf).copy()
    plt.close(fig); buf.close()
    return img


def _render_transition_matrix(data, dpi=90):
    mat    = data["trans_mat"]
    K      = data["K"]
    pal    = data["palette"]
    SIZE   = K + 2
    START  = data["START"]
    END    = data["END"]
    labels = ["START"] + [f"C{c}" for c in range(K)] + ["END"]

    cmap  = LinearSegmentedColormap.from_list(
        "trans", ["#0a0a2e","#1a3a6e","#2266aa","#55aadd",
                  "#ffffff","#ffcc88","#ff8844","#cc3311"], N=256)
    fig_s = max(5, SIZE*0.75)
    fig   = Figure(figsize=(fig_s+1.2, fig_s), facecolor=BG)
    ax    = fig.add_subplot(111, facecolor=BG2)
    im    = ax.imshow(mat, cmap=cmap, vmin=0, vmax=1, aspect="auto")

    for j in range(SIZE):
        ax.add_patch(plt.Rectangle((j-.5,START-.5),1,1,fill=True,
                                    color="#00aa44",alpha=0.12,zorder=0))
    for i in range(SIZE):
        ax.add_patch(plt.Rectangle((END-.5,i-.5),1,1,fill=True,
                                    color="#cc2222",alpha=0.12,zorder=0))
    for k in range(1,K+1):
        ax.add_patch(plt.Rectangle((k-.5,k-.5),1,1,fill=False,
                                    edgecolor="white",linewidth=1.5,
                                    linestyle="--",zorder=3))
    ax.axhline(0.5, color="#44ff88",lw=1.3,alpha=0.7)
    ax.axvline(K+0.5,color="#ff6644",lw=1.3,alpha=0.7)
    for i in range(SIZE):
        for j in range(SIZE):
            v = mat[i,j]
            if v < 0.005: continue
            ax.text(j,i,f"{v:.2f}",ha="center",va="center",
                    fontsize=7,color="black" if v>0.55 else "white",
                    fontweight="bold",zorder=4)
    ax.set_xticks(range(SIZE)); ax.set_yticks(range(SIZE))
    ax.set_xticklabels(labels,fontsize=8,rotation=45,ha="right")
    ax.set_yticklabels(labels,fontsize=8)
    for k in range(K):
        ax.get_xticklabels()[k+1].set_color(pal[k%len(pal)])
        ax.get_yticklabels()[k+1].set_color(pal[k%len(pal)])
    ax.get_xticklabels()[0].set_color("#44ff88")
    ax.get_yticklabels()[0].set_color("#44ff88")
    ax.get_xticklabels()[-1].set_color("#ff6644")
    ax.get_yticklabels()[-1].set_color("#ff6644")
    ax.set_xlabel("To →",fontsize=9,color="#aaaacc",labelpad=6)
    ax.set_ylabel("From ↓",fontsize=9,color="#aaaacc",labelpad=6)
    ax.set_title("Enriched transition matrix",fontsize=8,color="#dddddd",pad=8)
    ax.tick_params(colors="#aaaacc")
    for sp in ax.spines.values(): sp.set_edgecolor("#334466")
    fig.colorbar(im,ax=ax,fraction=0.028,pad=0.02
                 ).ax.yaxis.set_tick_params(color="#aaaacc",labelcolor="#aaaacc")
    fig.tight_layout(pad=0.8)
    buf = io.BytesIO()
    fig.savefig(buf,format="png",dpi=dpi,facecolor=BG,bbox_inches="tight")
    buf.seek(0)
    img = Image.open(buf).copy()
    plt.close(fig); buf.close()
    return img


def _render_arena(cx, cy, cangle, px, py, sequence, current_idx, palette,
                  prey_angle=0):
    img = Image.new("RGB", (ARENA_W, ARENA_H), (15,15,35))
    drw = ImageDraw.Draw(img)
    acx, acy = ARENA_W//2, ARENA_H//2
    drw.ellipse([acx-ARENA_R,acy-ARENA_R,acx+ARENA_R,acy+ARENA_R],
                fill=(28,28,48),outline=(80,80,120),width=3)
    for r in [ARENA_R//3, 2*ARENA_R//3]:
        drw.ellipse([acx-r,acy-r,acx+r,acy+r],outline=(40,40,70),width=1)
    drw.line([(acx-ARENA_R,acy),(acx+ARENA_R,acy)],fill=(40,40,70),width=1)
    drw.line([(acx,acy-ARENA_R),(acx,acy+ARENA_R)],fill=(40,40,70),width=1)

    if px is not None:
        ipx, ipy = int(px), int(py)
        drw.ellipse([ipx-7,ipy-7,ipx+7,ipy+7],
                    fill=(255,180,50),outline=(255,255,255),width=1)
        prad = math.radians(prey_angle)
        epx  = int(ipx+18*math.cos(prad))
        epy  = int(ipy-18*math.sin(prad))
        drw.line([(ipx,ipy),(epx,epy)],fill=(255,210,80),width=2)
        drw.text((ipx+10,ipy-6),"prey",fill=(255,210,80))

    if cx is not None:
        isx, isy = int(cx), int(cy)
        lab = (sequence[current_idx]
               if sequence and current_idx is not None
               and current_idx < len(sequence) else -1)
        col = (_hex_to_rgb255(palette[lab%len(palette)]) if lab>=0
               else (100,200,255))
        drw.ellipse([isx-11,isy-11,isx+11,isy+11],
                    fill=col,outline=(255,255,255),width=2)
        crad = math.radians(cangle)
        ecx  = int(isx+28*math.cos(crad))
        ecy  = int(isy-28*math.sin(crad))
        drw.line([(isx,isy),(ecx,ecy)],fill=(255,255,255),width=3)
        drw.text((isx+13,isy-17),"cuttlefish",fill=(200,220,255))

    if cx is not None and px is not None:
        isx,isy = int(cx),int(cy)
        ipx,ipy = int(px),int(py)
        drw.line([(isx,isy),(ipx,ipy)],fill=(200,200,100),width=1)
        crad = math.radians(cangle)
        prad = math.radians(prey_angle)
        arc_len = 44
        diff_signed = (cangle - prey_angle + 180) % 360 - 180
        ap_display  = abs(diff_signed)
        ax1 = int(isx+arc_len*math.cos(crad))
        ay1 = int(isy-arc_len*math.sin(crad))
        drw.line([(isx,isy),(ax1,ay1)],fill=(120,210,120),width=1)
        ax2 = int(isx+arc_len*math.cos(prad))
        ay2 = int(isy-arc_len*math.sin(prad))
        drw.line([(isx,isy),(ax2,ay2)],fill=(255,190,60),width=1)
        mid_rad = crad + math.radians(-diff_signed/2)
        lx = int(isx+(arc_len+14)*math.cos(mid_rad))
        ly = int(isy-(arc_len+14)*math.sin(mid_rad))
        drw.text((lx-12,ly-6),f"{ap_display:.0f}°",fill=(150,230,150))
    return img


def _render_stats(sequence, data):
    if not sequence:
        return Image.new("RGB", (420, 220), (10,10,28))
    K = data["K"]; pal = data["palette"]
    counts = np.zeros(K, dtype=int)
    for c in sequence:
        if 0 <= c < K: counts[c] += 1
    total = max(counts.sum(), 1)
    props = counts / total
    p     = props[props > 0]
    ent   = float(-np.sum(p * np.log2(p)))
    comp  = [sequence[0]]
    for v in sequence[1:]:
        if v != comp[-1]: comp.append(v)
    fig = Figure(figsize=(4.4, 2.8), facecolor=BG)
    ax  = fig.add_subplot(111, facecolor="#0a0a1c")
    ax.bar(range(K), props,
           color=[_col_mpl(c,pal) for c in range(K)],
           edgecolor="white",linewidth=0.5)
    for c, pv in enumerate(props):
        if pv > 0.01:
            ax.text(c, pv+0.01, f"{pv:.0%}",
                    ha="center",fontsize=7,color="white")
    ax.set_xticks(range(K))
    ax.set_xticklabels([f"C{c}" for c in range(K)],color="#cccccc",fontsize=8)
    ax.set_ylabel("Proportion",color="#cccccc",fontsize=8)
    ax.set_ylim(0, max(props.max()*1.3, 0.1))
    ax.tick_params(colors="#cccccc")
    ax.set_title(f"{len(sequence)} win.  ·  {len(comp)} blocks  ·  H={ent:.2f}b",
                 fontsize=7,color="#dddddd",pad=4)
    for sp in ax.spines.values(): sp.set_edgecolor("#334466")
    ax.grid(axis="y",alpha=0.2,color="#334466")
    fig.tight_layout(pad=0.4)
    buf = io.BytesIO()
    fig.savefig(buf,format="png",dpi=90,facecolor=BG)
    buf.seek(0)
    img = Image.open(buf).copy()
    plt.close(fig); buf.close()
    return img


# ════════════════════════════════════════════════════════════════
#  TKINTER APPLICATION
# ════════════════════════════════════════════════════════════════

class SimulatorApp:

    def __init__(self, root, data, dendro_data, out_dir):
        self.root        = root
        self.data        = data
        self.dendro_data = dendro_data
        self.out_dir     = Path(out_dir)

        self.sequence      = []
        self.all_sequences = []
        self.current_idx   = 0

        self.cuttle_x     = ARENA_W//2 - 70
        self.cuttle_y     = ARENA_H//2
        self.cuttle_angle = 0
        self.prey_x       = ARENA_W//2 + 90
        self.prey_y       = ARENA_H//2
        self.prey_angle   = 180
        self.dragging     = None

        self._cluster_grid_imgs = {}
        self._trans_matrix_img  = None
        self._dendro_after_id   = None

        self._build_ui()
        self._init_data()

    # ── Init ─────────────────────────────────────────────────────────────────

    def _init_data(self):
        K = self.data["K"]
        self.start_spin.configure(to=K-1)

        self._set_status("Pre-computing cluster grid...")
        self.root.update_idletasks()
        for active in [None] + list(range(K)):
            self._cluster_grid_imgs[active] = _render_cluster_grid(
                self.data, active_cluster=active)

        self._set_status("Pre-computing matrix...")
        self.root.update_idletasks()
        self._trans_matrix_img = _render_transition_matrix(self.data)

        self._refresh_cluster_grid(None)
        self._refresh_matrix()

        # Dendrogram
        if self.dendro_data:
            Z      = self.dendro_data["Z_linkage"]
            h_min  = self.dendro_data["h_min"]
            h_max  = self.dendro_data["h_max"]
            init_h = self.dendro_data["init_cut"]
            self.dendro_slider.configure(
                from_=h_min, to=h_max,
                resolution=round((h_max-h_min)/500, 4))
            self.dendro_cut_var.set(init_h)
            n_init = len(set(fcluster(Z, init_h, criterion="distance")))
            self.dendro_cut_lbl.configure(text=f"h={init_h:.2f}")
            self.dendro_nstrat_lbl.configure(text=f"→ {n_init} strats")

        self._set_status(
            f"Ready — {K} clusters  ·  "
            f"P(START)={self.data['trans_mat'][0,1:K+1].round(2).tolist()}")
        self._redraw_arena()

    # ── UI ───────────────────────────────────────────────────────────────────

    def _build_ui(self):
        self.root.title("Behavioural simulator — Notebook")
        self.root.configure(bg=BG)
        self.root.resizable(True, True)

        tk.Label(self.root, text="Behavioural simulator",
                 font=("Helvetica",14,"bold"), fg="#f0c040", bg=BG
                 ).pack(side="top", pady=(8,2))

        main = tk.Frame(self.root, bg=BG)
        main.pack(side="top", fill="both", expand=True, padx=8, pady=4)

        left = tk.Frame(main, bg=BG)
        left.pack(side="left", fill="y")
        self._build_params(left)
        self._build_arena_panel(left)

        right = tk.Frame(main, bg=BG)
        right.pack(side="left", fill="both", expand=True, padx=(10,0))
        self._build_tabs(right)

        self.status_lbl = tk.Label(
            self.root, text="Loading...",
            font=("Helvetica",8), fg="#888899", bg=BG, anchor="w")
        self.status_lbl.pack(side="bottom", fill="x", padx=8, pady=2)

    def _build_params(self, parent):
        pf = tk.LabelFrame(parent, text="Parameters",
                            bg=BG, fg="#9999bb", font=("Helvetica",9,"bold"))
        pf.pack(side="top", fill="x", pady=(0,6))

        mf = tk.Frame(pf, bg=BG)
        mf.pack(fill="x", padx=6, pady=4)
        tk.Label(mf, text="Mode :", font=("Helvetica",9),
                 fg="#cccccc", bg=BG).pack(side="left")
        self.mode_var = tk.StringVar(value="random")
        for val, txt in [("random","Random"),
                          ("fixed_start","Fixed cluster"),
                          ("positional","Positional")]:
            tk.Radiobutton(mf, text=txt, variable=self.mode_var, value=val,
                           font=("Helvetica",9), fg="#cccccc", bg=BG,
                           selectcolor="#1a1a40", activebackground=BG,
                           activeforeground="#cccccc",
                           command=self._on_mode_change
                           ).pack(side="left", padx=3)

        sf = tk.Frame(pf, bg=BG)
        sf.pack(fill="x", padx=6, pady=2)
        tk.Label(sf, text="Start cluster :", font=("Helvetica",9),
                 fg="#cccccc", bg=BG).pack(side="left")
        self.start_cluster_var = tk.IntVar(value=0)
        self.start_spin = tk.Spinbox(
            sf, from_=0, to=20, textvariable=self.start_cluster_var,
            width=4, bg="#1a1a40", fg="#e0e0e0",
            insertbackground="white", state="disabled")
        self.start_spin.pack(side="left", padx=4)

        df = tk.Frame(pf, bg=BG)
        df.pack(fill="x", padx=6, pady=2)
        self.fixed_len_var = tk.BooleanVar(value=False)
        tk.Checkbutton(df, text="Fixed duration :",
                       variable=self.fixed_len_var,
                       font=("Helvetica",9), fg="#cccccc", bg=BG,
                       selectcolor="#1a1a40", activebackground=BG,
                       activeforeground="#cccccc"
                       ).pack(side="left")
        self.n_steps_var = tk.IntVar(value=30)
        tk.Spinbox(df, from_=1, to=500, textvariable=self.n_steps_var,
                   width=5, bg="#1a1a40", fg="#e0e0e0",
                   insertbackground="white").pack(side="left", padx=4)
        tk.Label(df, text="windows (0.25 s)", font=("Helvetica",8),
                 fg="#667788", bg=BG).pack(side="left")

        nf = tk.Frame(pf, bg=BG)
        nf.pack(fill="x", padx=6, pady=2)
        tk.Label(nf, text="No. of simulations :", font=("Helvetica",9),
                 fg="#cccccc", bg=BG).pack(side="left")
        self.n_simu_var = tk.IntVar(value=1)
        tk.Spinbox(nf, from_=1, to=200, textvariable=self.n_simu_var,
                   width=5, bg="#1a1a40", fg="#e0e0e0",
                   insertbackground="white").pack(side="left", padx=4)

        seedf = tk.Frame(pf, bg=BG)
        seedf.pack(fill="x", padx=6, pady=2)
        self.use_seed_var = tk.BooleanVar(value=False)
        tk.Checkbutton(seedf, text="Seed :",
                       variable=self.use_seed_var,
                       font=("Helvetica",9), fg="#cccccc", bg=BG,
                       selectcolor="#1a1a40", activebackground=BG,
                       activeforeground="#cccccc").pack(side="left")
        self.seed_var = tk.IntVar(value=42)
        tk.Spinbox(seedf, from_=0, to=9999, textvariable=self.seed_var,
                   width=6, bg="#1a1a40", fg="#e0e0e0",
                   insertbackground="white").pack(side="left", padx=4)

        bf = tk.Frame(pf, bg=BG)
        bf.pack(fill="x", padx=6, pady=6)
        bk = dict(font=("Helvetica",10,"bold"), bg="#1e1e5a",
                  fg="#e0e0e0", relief="flat",
                  activebackground="#3333aa", activeforeground="white",
                  padx=10, pady=5)
        tk.Button(bf, text="Simulate", command=self._run, **bk
                  ).pack(side="left", padx=4)
        tk.Button(bf, text="Reset",    command=self._reset,
                  **{**bk,"bg":"#2a2a2a"}).pack(side="left", padx=4)
        tk.Button(bf, text="Save",     command=self._save,
                  **{**bk,"bg":"#1a3a1a"}).pack(side="left", padx=4)

    def _build_arena_panel(self, parent):
        af = tk.LabelFrame(parent, text="Arena (click=move, right click=orient)",
                            bg=BG, fg="#9999bb", font=("Helvetica",8,"bold"))
        af.pack(side="top", fill="x", pady=(0,6))

        self.arena_canvas = tk.Canvas(
            af, width=ARENA_W, height=ARENA_H,
            bg="#0a0a1c", highlightthickness=1,
            highlightbackground="#223366")
        self.arena_canvas.pack(padx=6, pady=4)
        self.arena_canvas.bind("<Button-1>",       self._arena_click)
        self.arena_canvas.bind("<B1-Motion>",       self._arena_drag)
        self.arena_canvas.bind("<ButtonRelease-1>", self._arena_release)
        self.arena_canvas.bind("<Button-3>",        self._arena_right_click)

        af2 = tk.Frame(af, bg=BG); af2.pack(fill="x", padx=6, pady=(0,1))
        tk.Label(af2, text="Cuttlefish (°) :",
                 font=("Helvetica",8), fg="#cccccc", bg=BG).pack(side="left")
        self.angle_var = tk.IntVar(value=0)
        tk.Spinbox(af2, from_=0, to=359, textvariable=self.angle_var,
                   width=5, bg="#1a1a40", fg="#e0e0e0",
                   insertbackground="white",
                   command=self._redraw_arena).pack(side="left", padx=4)

        af3 = tk.Frame(af, bg=BG); af3.pack(fill="x", padx=6, pady=(0,2))
        tk.Label(af3, text="Prey       (°) :",
                 font=("Helvetica",8), fg="#e0c060", bg=BG).pack(side="left")
        self.prey_angle_var = tk.IntVar(value=180)
        tk.Spinbox(af3, from_=0, to=359, textvariable=self.prey_angle_var,
                   width=5, bg="#2a1a00", fg="#ffe080",
                   insertbackground="white",
                   command=self._redraw_arena).pack(side="left", padx=4)

        self.detected_lbl = tk.Label(
            af, text="Detected cluster : —",
            font=("Helvetica",9,"bold"), fg="#f0c040", bg=BG)
        self.detected_lbl.pack(pady=(0,4))
        self._redraw_arena()

    def _build_tabs(self, parent):
        style = ttk.Style()
        style.theme_use("default")
        style.configure("Dark.TNotebook", background=BG, borderwidth=0)
        style.configure("Dark.TNotebook.Tab",
                        background="#1a1a40", foreground="#cccccc",
                        padding=[10,4], font=("Helvetica",9))
        style.map("Dark.TNotebook.Tab",
                  background=[("selected","#334488")],
                  foreground=[("selected","#ffffff")])

        nb = ttk.Notebook(parent, style="Dark.TNotebook")
        nb.pack(fill="both", expand=True)
        self.notebook = nb

        # Simulation tab
        t1 = tk.Frame(nb, bg=BG); nb.add(t1, text="  Simulation  ")
        self._build_simu_tab(t1)

        # Clusters tab
        t2 = tk.Frame(nb, bg=BG); nb.add(t2, text="  Clusters  ")
        tk.Label(t2, text="Cluster profiles (active one highlighted)",
                 font=("Helvetica",8), fg="#9999bb", bg=BG).pack(pady=(6,2))
        self.lbl_cluster_grid = tk.Label(t2, bg=BG)
        self.lbl_cluster_grid.pack(pady=4)

        # Matrix tab
        t3 = tk.Frame(nb, bg=BG); nb.add(t3, text="  Matrix  ")
        tk.Label(t3, text="Enriched transition matrix",
                 font=("Helvetica",8), fg="#9999bb", bg=BG).pack(pady=(6,2))
        self.lbl_trans_matrix = tk.Label(t3, bg=BG)
        self.lbl_trans_matrix.pack(pady=4)

        # Dendrogram tab
        t4 = tk.Frame(nb, bg=BG); nb.add(t4, text="  Dendrogram  ")
        self._build_dendro_tab(t4)

    def _build_simu_tab(self, parent):
        tk.Label(parent, text="Simulated sequence",
                 font=("Helvetica",9,"bold"), fg="#9999bb", bg=BG
                 ).pack(anchor="w", padx=4)
        seqf = tk.Frame(parent, bg=BG)
        seqf.pack(fill="x", pady=(2,6), padx=4)
        self.seq_canvas = tk.Canvas(
            seqf, height=SEQ_H, width=480,
            bg="#060614", highlightthickness=1, highlightbackground="#223366")
        self.seq_canvas.pack(side="top", fill="x", expand=True)
        hsb = ttk.Scrollbar(seqf, orient="horizontal",
                            command=self.seq_canvas.xview)
        hsb.pack(side="bottom", fill="x")
        self.seq_canvas.configure(xscrollcommand=hsb.set)
        self.seq_canvas.bind("<Button-1>", self._on_seq_click)
        self.strat_lbl = tk.Label(parent, text="", font=("Helvetica", 10, "bold"),
                                  fg="#f0c040", bg=BG, anchor="w", justify="left")
        self.strat_lbl.pack(fill="x", padx=4, pady=(0, 4))

        rf = tk.Frame(parent, bg=BG)
        rf.pack(fill="x", pady=(0,2), padx=4)
        tk.Label(rf, text="Start :", font=("Helvetica",7),
                 fg="#888888", bg=BG).pack(side="left")
        self.lbl_rs = tk.Label(rf, bg=BG); self.lbl_rs.pack(side="left",padx=2)
        tk.Label(rf, text="  End :", font=("Helvetica",7),
                 fg="#888888", bg=BG).pack(side="left")
        self.lbl_re = tk.Label(rf, bg=BG); self.lbl_re.pack(side="left",padx=2)

        self.lbl_rc    = tk.Label(parent, bg=BG); self.lbl_rc.pack(pady=(0,4))
        self.lbl_stats = tk.Label(parent, bg=BG); self.lbl_stats.pack(pady=(0,4))

        tk.Label(parent, text="Simulations (click = show)",
                 font=("Helvetica",8,"bold"), fg="#9999bb", bg=BG
                 ).pack(anchor="w", padx=4)
        listf = tk.Frame(parent, bg=BG)
        listf.pack(fill="both", expand=True, padx=4)
        self.seq_listbox = tk.Listbox(
            listf, bg="#060614", fg="#e0e0e0",
            selectbackground="#2233aa", font=("Courier",8), height=6)
        self.seq_listbox.pack(side="left", fill="both", expand=True)
        vsb = ttk.Scrollbar(listf, orient="vertical",
                            command=self.seq_listbox.yview)
        vsb.pack(side="right", fill="y")
        self.seq_listbox.configure(yscrollcommand=vsb.set)
        self.seq_listbox.bind("<<ListboxSelect>>", self._on_list_select)

    def _build_dendro_tab(self, parent):
        self.dendro_strategy_lbl = tk.Label(
            parent, text="— simulate first —",
            font=("Helvetica",10,"bold"), fg="#f0c040", bg=BG)
        self.dendro_strategy_lbl.pack(pady=(6,2))

        ctrl = tk.Frame(parent, bg=BG)
        ctrl.pack(fill="x", padx=10, pady=(0,4))
        tk.Label(ctrl, text="Cut height :",
                 font=("Helvetica",9), fg="#cccccc", bg=BG).pack(side="left")

        self.dendro_cut_var = tk.DoubleVar(value=0.0)
        self.dendro_slider  = tk.Scale(
            ctrl, variable=self.dendro_cut_var,
            from_=0.0, to=1.0, resolution=0.001,
            orient="horizontal", length=340,
            bg=BG, fg="#e0e0e0", troughcolor="#1a1a40",
            activebackground="#3333aa", highlightthickness=0,
            showvalue=False,
            command=self._on_dendro_slider_move)
        self.dendro_slider.pack(side="left", padx=6)

        self.dendro_cut_lbl = tk.Label(
            ctrl, text="h=—", width=10,
            font=("Helvetica",9,"bold"), fg="#ff7777", bg=BG)
        self.dendro_cut_lbl.pack(side="left")
        self.dendro_nstrat_lbl = tk.Label(
            ctrl, text="", font=("Helvetica",9), fg="#aaaacc", bg=BG)
        self.dendro_nstrat_lbl.pack(side="left", padx=(4,0))

        self.lbl_dendro = tk.Label(parent, bg=BG)
        self.lbl_dendro.pack(pady=2, fill="both", expand=True)

    # ── Callbacks ────────────────────────────────────────────────────────────

    def _on_mode_change(self):
        mode = self.mode_var.get()
        self.start_spin.configure(
            state="normal" if mode == "fixed_start" else "disabled")

    def _redraw_arena(self):
        if not HAS_PIL: return
        self.cuttle_angle = self.angle_var.get()
        self.prey_angle   = self.prey_angle_var.get()
        img = _render_arena(
            self.cuttle_x, self.cuttle_y, self.cuttle_angle,
            self.prey_x,   self.prey_y,
            self.sequence if self.sequence else None,
            self.current_idx if self.sequence else None,
            self.data["palette"],
            prey_angle=self.prey_angle)
        self._arena_tk = ImageTk.PhotoImage(image=img)
        self.arena_canvas.delete("all")
        self.arena_canvas.create_image(0,0,anchor="nw",image=self._arena_tk)
        if self.mode_var.get() == "positional":
            self._update_detected()

    def _arena_click(self, event):
        ds = math.hypot(event.x-self.cuttle_x, event.y-self.cuttle_y)
        dp = math.hypot(event.x-self.prey_x,   event.y-self.prey_y)
        self.dragging = "cuttle" if ds < 20 else ("prey" if dp < 18 else None)

    def _arena_drag(self, event):
        if self.dragging == "cuttle":
            self.cuttle_x = max(0, min(ARENA_W, event.x))
            self.cuttle_y = max(0, min(ARENA_H, event.y))
            self._redraw_arena()
        elif self.dragging == "prey":
            self.prey_x = max(0, min(ARENA_W, event.x))
            self.prey_y = max(0, min(ARENA_H, event.y))
            self._redraw_arena()

    def _arena_release(self, event): self.dragging = None

    def _arena_right_click(self, event):
        ds = math.hypot(event.x-self.cuttle_x, event.y-self.cuttle_y)
        dp = math.hypot(event.x-self.prey_x,   event.y-self.prey_y)
        if dp < ds:
            dx = event.x-self.prey_x; dy = self.prey_y-event.y
            self.prey_angle_var.set(int(math.degrees(math.atan2(dy,dx))%360))
        else:
            dx = event.x-self.cuttle_x; dy = self.cuttle_y-event.y
            self.angle_var.set(int(math.degrees(math.atan2(dy,dx))%360))
        self._redraw_arena()

    def _update_detected(self):
        cl, fd = detect_cluster_from_position(
            self.cuttle_x, self.cuttle_y, self.angle_var.get(),
            self.prey_x,   self.prey_y,   self.data,
            prey_angle=self.prey_angle_var.get())
        pal = self.data["palette"]
        self.detected_lbl.configure(
            text=f"Detected cluster : {cl}  ·  "
                 f"dist={fd.get('distance_real','?')}px  ·  "
                 f"approach={fd.get('approach_angle','?')}°",
            fg=pal[cl % len(pal)])
        self._refresh_cluster_grid(cl)

    def _refresh_cluster_grid(self, active):
        if not self._cluster_grid_imgs: return
        img = self._cluster_grid_imgs.get(active,
                                           self._cluster_grid_imgs.get(None))
        if img:
            self._tk_grid = ImageTk.PhotoImage(image=img)
            self.lbl_cluster_grid.configure(image=self._tk_grid)

    def _refresh_matrix(self):
        if self._trans_matrix_img:
            self._tk_mat = ImageTk.PhotoImage(image=self._trans_matrix_img)
            self.lbl_trans_matrix.configure(image=self._tk_mat)

    def _on_dendro_slider_move(self, val):
        if self.dendro_data is None: return
        cut_h = float(val)
        try:
            n = len(set(fcluster(self.dendro_data["Z_linkage"],
                                 cut_h, criterion="distance")))
        except: n = "?"
        self.dendro_cut_lbl.configure(text=f"h={cut_h:.2f}")
        self.dendro_nstrat_lbl.configure(text=f"→ {n} strats")
        if self._dendro_after_id:
            self.root.after_cancel(self._dendro_after_id)
        if self.sequence:
            self._dendro_after_id = self.root.after(
                350, lambda: self._update_dendro(self.sequence))

    def _update_dendro(self, sequence):
        if not HAS_SCIPY or self.dendro_data is None or not sequence:
            return
        try:
            K      = self.data["K"]
            Z      = self.dendro_data["Z_linkage"]
            cut_h  = float(self.dendro_cut_var.get())
            scaler = self.dendro_data["scaler"]
            Xs     = self.dendro_data["X_scaled"]
            cols   = self.dendro_data["column_names"]

            # Histogram vector of the simulation
            counts = Counter(sequence)
            total  = len(sequence)
            hist   = np.array([counts.get(c,0)/total for c in range(K)])
            # Align on the feature_matrix columns
            n_cols = len(cols)
            vec    = np.zeros(n_cols)
            for i, col in enumerate(cols):
                try:
                    c = int(col.split("_C")[1])
                    if c < K: vec[i] = hist[c]
                except: pass

            vec_sc  = scaler.transform(vec.reshape(1,-1))[0]
            dists   = np.linalg.norm(Xs - vec_sc, axis=1)
            nn_idx  = int(np.argmin(dists))
            nn_id   = self.dendro_data["feature_matrix"].index[nn_idx]
            labels  = fcluster(Z, cut_h, criterion="distance")
            grp     = int(labels[nn_idx])
            n_strat = len(set(labels))

            # Simple dendrogram
            n_ind = self.dendro_data["feature_matrix"].shape[0]
            fig, ax = plt.subplots(figsize=(13, 5), facecolor=BG)
            ax.set_facecolor(BG2)
            dend = scipy_dendrogram(Z, ax=ax, no_labels=True,
                                    color_threshold=cut_h,
                                    above_threshold_color="lightgray")
            leaves   = dend["leaves"]
            leaf_xs  = {li: (pos*10+5) for pos, li in enumerate(leaves)}
            ax.axhline(cut_h, color="crimson", lw=2.5, ls="--", alpha=0.9,
                       label=f"h={cut_h:.2f} → {n_strat} groups")
            if nn_idx in leaf_xs:
                nx = leaf_xs[nn_idx]
                ax.plot(nx, 0, "o", color="red", markersize=11,
                        markeredgecolor="white", markeredgewidth=1.2, zorder=6)
                ax.annotate(f"Group {grp}",
                            xy=(nx,0), xytext=(nx, ax.get_ylim()[1]*0.08),
                            fontsize=8, color="red", fontweight="bold",
                            ha="center",
                            arrowprops=dict(arrowstyle="->",color="red",lw=1.5))
            ax.legend(fontsize=8, loc="upper left",
                      facecolor="#111133", edgecolor="#334466", labelcolor="white")
            ax.set_title(f"Dendrogram — {n_strat} groups",
                         fontsize=10, color="#e0e0ff", pad=6)
            ax.set_ylabel("Distance (Ward)", fontsize=9, color="#9999bb")
            ax.tick_params(colors="#9999bb", labelsize=7)
            for sp in ax.spines.values(): sp.set_color("#334466")
            ax.set_xlim(-5, n_ind*10+5)
            fig.tight_layout(pad=0.6)

            buf = io.BytesIO()
            fig.savefig(buf, format="png", dpi=90, facecolor=BG,
                        bbox_inches="tight")
            buf.seek(0)
            img = Image.open(buf).copy()
            plt.close(fig); buf.close()

            self._tk_dendro = ImageTk.PhotoImage(image=img)
            self.lbl_dendro.configure(image=self._tk_dendro)
            self.dendro_strategy_lbl.configure(
                text=f"Group {grp} / {n_strat}  "
                     f"(neighbour : {nn_id}, dist={dists[nn_idx]:.2f})",
                fg="#ff8888")
        except Exception as e:
            import traceback
            print(f"[SIMU] Dendro : {e}")
            traceback.print_exc()

    # ── Simulation ───────────────────────────────────────────────────────────

    def _run(self):
        mode      = self.mode_var.get()
        n_simu    = self.n_simu_var.get()
        max_steps = self.n_steps_var.get() if self.fixed_len_var.get() else 500
        allow_end = not self.fixed_len_var.get()
        seed      = self.seed_var.get() if self.use_seed_var.get() else None

        self.all_sequences = []; self.all_segments = []; self.all_groups = []
        self.seq_listbox.delete(0, "end")
        self._set_status("Simulation...")
        self.root.update_idletasks()

        for i in range(n_simu):
            s = (seed + i) if seed is not None else None
            if mode == "positional":
                sc, _ = detect_cluster_from_position(
                    self.cuttle_x, self.cuttle_y, self.angle_var.get(),
                    self.prey_x,   self.prey_y,   self.data,
                    prey_angle=self.prey_angle_var.get())
                seq, segs = simulate_sequence(
                    self.data, mode="fixed", start_cluster=sc,
                    max_steps=max_steps, allow_end=allow_end, seed=s, return_segments=True)
            elif mode == "fixed_start":
                sc  = self.start_cluster_var.get()
                seq, segs = simulate_sequence(
                    self.data, mode="fixed", start_cluster=sc,
                    max_steps=max_steps, allow_end=allow_end, seed=s, return_segments=True)
            else:
                seq, segs = simulate_sequence(
                    self.data, mode="random",
                    max_steps=max_steps, allow_end=allow_end, seed=s, return_segments=True)

            self.all_sequences.append(seq)
            self.all_segments.append(segs)

        self._set_status("Assigning groups (SeqVec)...")
        self.all_groups = assign_strategy(self.all_segments, self.data)
        for i, (seq, segs) in enumerate(zip(self.all_sequences, self.all_segments)):
            res  = self.all_groups[i]
            comp = [c for c, _ in segs]
            dur  = sum(L for _, L in segs) / SIM_FPS
            g    = res["group"] if res else "?"
            conf = f"{res['conf']:.0%}" if res else ""
            self.seq_listbox.insert("end", f"#{i+1:3d} {dur:5.1f}s  {g:>4s} {conf:>4s}  "
                f"{'→'.join(str(c) for c in comp[:12])}{'…' if len(comp) > 12 else ''}")

        if self.all_sequences:
            self._display(0)
        mean_len = np.mean([len(s) for s in self.all_sequences])
        mean_s   = np.mean([sum(L for _, L in g) for g in self.all_segments]) / SIM_FPS
        n_grp    = Counter(r["group"] for r in self.all_groups if r)
        self._set_status(f"{n_simu} simulation(s) — mean duration {mean_s:.1f} s ({mean_len:.1f} win.)"
                         + (" | groups : " + ", ".join(f"{g} {n}" for g, n in n_grp.most_common()) if n_grp else ""))

    def _display(self, idx):
        if not self.all_sequences or idx >= len(self.all_sequences): return
        self.sequence    = self.all_sequences[idx]
        self.current_idx = 0
        K = self.data["K"]; pal = self.data["palette"]
        feats = self.data["features"]; profs = self.data["cluster_profiles"]
        pmin  = self.data["prof_min"];  pmax  = self.data["prof_max"]

        def _pnorm(c):
            if profs is None: return np.full(len(feats), 0.5)
            return np.clip((profs[c]-pmin)/(pmax-pmin+1e-9), 0, 1)

        seq_img = _render_seq_bar(self.sequence, pal, 0)
        tw = len(self.sequence) * SEQ_CELL_W
        self._seq_tk = ImageTk.PhotoImage(image=seq_img)
        self.seq_canvas.delete("all")
        self.seq_canvas.create_image(0,0,anchor="nw",image=self._seq_tk)
        self.seq_canvas.configure(scrollregion=(0,0,tw,SEQ_H))

        cs = self.sequence[0]
        self._tk_rs = ImageTk.PhotoImage(
            image=_render_radar(_pnorm(cs), feats, _col_mpl(cs,pal),
                                f"Start C{cs}", size_in=1.8, dpi=85))
        self.lbl_rs.configure(image=self._tk_rs)

        ce = self.sequence[-1]
        self._tk_re = ImageTk.PhotoImage(
            image=_render_radar(_pnorm(ce), feats, _col_mpl(ce,pal),
                                f"End C{ce}", size_in=1.8, dpi=85))
        self.lbl_re.configure(image=self._tk_re)

        self._update_radar_cur()
        self._tk_stats = ImageTk.PhotoImage(
            image=_render_stats(self.sequence, self.data))
        self.lbl_stats.configure(image=self._tk_stats)
        self._refresh_cluster_grid(self.sequence[0])
        self._redraw_arena()
        self._update_dendro(self.sequence)
        self._show_strategy(idx)

    def _show_strategy(self, idx):
        groups = getattr(self, "all_groups", [])
        res = groups[idx] if idx < len(groups) else None
        if not res:
            self.strat_lbl.configure(text="Strategy: — (run the explorer before the simulator)")
            return
        segs  = self.all_segments[idx]
        dur   = sum(L for _, L in segs) / SIM_FPS
        n_v   = sum(res["votes"].values())
        votes = "  ".join(f"{g}:{n}" for g, n in sorted(res["votes"].items(), key=lambda x: -x[1]))
        self.strat_lbl.configure(text=f"Strategy: {res['group']}  ({res['conf']:.0%} of the {n_v} SeqVec neighbours)   |   "
                 f"votes {votes}   |   closest: ID {res['nearest']} ({res['nearest_group']})   |   "
                 f"{dur:.1f} s, {len(segs)} segments")

    def _update_radar_cur(self):
        if not self.sequence: return
        feats = self.data["features"]; profs = self.data["cluster_profiles"]
        pmin  = self.data["prof_min"];  pmax  = self.data["prof_max"]
        pal   = self.data["palette"]
        c     = self.sequence[self.current_idx]
        ref   = (np.clip((profs[c]-pmin)/(pmax-pmin+1e-9), 0, 1)
                 if profs is not None else np.full(len(feats), 0.5))
        img   = _render_radar(ref, feats, _col_mpl(c,pal),
                              f"Current C{c} (win.{self.current_idx+1}/"
                              f"{len(self.sequence)})",
                              size_in=2.8, dpi=90)
        self._tk_rc = ImageTk.PhotoImage(image=img)
        self.lbl_rc.configure(image=self._tk_rc)
        self._refresh_cluster_grid(c)

    def _on_seq_click(self, event):
        idx = max(0, min(event.x//SEQ_CELL_W, len(self.sequence)-1))
        self.current_idx = idx
        self._seq_tk = ImageTk.PhotoImage(
            image=_render_seq_bar(self.sequence, self.data["palette"], idx))
        self.seq_canvas.delete("all")
        self.seq_canvas.create_image(0,0,anchor="nw",image=self._seq_tk)
        self._update_radar_cur()

    def _on_list_select(self, event):
        sel = self.seq_listbox.curselection()
        if sel: self._display(sel[0])

    def _save(self):
        if not self.all_sequences:
            self._set_status("Nothing to save"); return
        out = self.out_dir / "simulations"
        out.mkdir(parents=True, exist_ok=True)
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        rows = []
        for i, seq in enumerate(self.all_sequences):
            comp = [seq[0]]
            for v in seq[1:]:
                if v != comp[-1]: comp.append(v)
            p = np.zeros(self.data["K"])
            for c in seq:
                if 0 <= c < self.data["K"]: p[c] += 1
            p /= max(p.sum(), 1)
            ent = float(-np.sum(p[p>0]*np.log2(p[p>0])))
            rows.append({"simulation":i+1,"n_windows":len(seq),
                         "n_blocks":len(comp),"entropy":round(ent,4),
                         "sequence":str(comp)})
        path = out / f"simulated_{stamp}.csv"
        pd.DataFrame(rows).to_csv(path, index=False)
        self._set_status(f"Saved → {path}")

    def _reset(self):
        self.sequence = []; self.all_sequences = []; self.current_idx = 0
        self.seq_canvas.delete("all"); self.seq_listbox.delete(0,"end")
        for lbl in [self.lbl_rs,self.lbl_re,self.lbl_rc,self.lbl_stats]:
            lbl.configure(image="")
        self._refresh_cluster_grid(None)
        self.lbl_dendro.configure(image="")
        self.dendro_strategy_lbl.configure(text="— simulate first —",fg="#f0c040")
        self._set_status("Reset")

    def _set_status(self, msg):
        self.status_lbl.configure(text=msg)
        self.root.update_idletasks()




In [ ]:
%matplotlib inline
# ════════════════════════════════════════════════════════════════
#  LAUNCH
# ════════════════════════════════════════════════════════════════

print(f"[SIMU] Preparing...")
print(f"  Clusters  : {final_state['n_cl']}")
print(f"  Individuals : {len(final_state['seqs'])}")
print(f"  Features  : {FEATURES}")

# Build the data
simu_data    = _build_model_data(df, final_state, FEATURES, PALETTE)
simu_dendro  = _build_dendro_data(final_state)

# Launch
root = tk.Tk()
app  = SimulatorApp(root, simu_data, simu_dendro, str(OUT_DIR))
root.protocol("WM_DELETE_WINDOW", root.destroy)
root.mainloop()

In [ ]:
%matplotlib inline

Global save and restore

In [ ]:
# ===============================================================
#  CELL — Interactive viewer per individual
#  Video + cuttlefish trail coloured by group + SHRIMP TRAIL
#  + cuttlefish/shrimp link + distance curve + legend
#
#  PREREQUISITE: CELL 0 (colour registry).
#
#  Fixes and additions:
#   [1]  Robust resolution of animal_id (str/int, leading zeros)
#   [2]  Wrapped callbacks: exceptions are visible
#   [3]  Status message shown INSIDE the figure
#   [4]  Radio built from the columns present
#   [5]  Widgets stored in `state` -> no garbage collection
#   [6]  Loading lock -> no re-entrance
#   [7]  Enter key + ◀ ▶ buttons
#   [8]  Colours delegated to the single registry
#   [9]  Trails via LineCollection (1 artist, not N)
#  [10]  Dynamic legend
#  [11]  SHRIMP: black outlined trail, markers, link at the frame
#  [12]  Cuttlefish–shrimp distance curve + star at the minimum
#  [13]  NaN filtering: no more ghost line when the tracking jumps
#  [14]  Check boxes to show/hide each layer
# ===============================================================

import re
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patheffects as mpe
from matplotlib.widgets import Slider, Button, TextBox, RadioButtons, CheckButtons
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D

try:
    import cv2
    _HAS_CV2 = True
except ImportError:
    _HAS_CV2 = False
    print("[VIEWER] opencv-python not installed -> pip install opencv-python "
          "(degraded mode without video)")


# ── Tracking points ─────────────────────────────────────────────────
_CUTTLE_CANDS = ["TopMantle", "MidCuttleHead", "MidMantle", "MidBody"]
_PREY_CANDS   = ["MidShrimp", "HeadShrimp", "Prey"]

_CHECK_LABELS = ["shrimp", "link", "trail"]


def _cf_pick_key(cands, cols):
    xset = {c[:-2] for c in cols if c.endswith("_x")}
    for c in cands:
        if c in xset:
            return c
    return None


def _valid_segments(x, y):
    """
    [13] Segments between consecutive points that are BOTH valid.
    A LineCollection cannot skip a NaN: without this mask, an
    untracked frame links two distant positions with a line that
    never existed.

    Returns (FULL_segments, points_mask, pairs_mask).
    The full (unfiltered) segments are kept so that the trail can be
    cut frame by frame: segs[:i][mask[:i]].
    """
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    ok = np.isfinite(x) & np.isfinite(y)
    if len(x) < 2:
        return np.empty((0, 2, 2)), ok, np.zeros(0, bool)
    pairs = ok[:-1] & ok[1:]
    pts   = np.column_stack([x, y]).reshape(-1, 1, 2)
    segs  = np.concatenate([pts[:-1], pts[1:]], axis=1)
    return segs, ok, pairs


def _cf_find_video(video_dir, animal_id):
    """Returns (Path|None, error_message|None) — prints nothing."""
    video_dir = Path(video_dir)
    if not video_dir.exists():
        return None, f"Video folder not found : {video_dir}"

    exts  = {".mp4", ".avi", ".mov", ".mkv", ".MP4", ".AVI", ".MOV", ".MKV"}
    all_v = [f for f in video_dir.iterdir() if f.is_file() and f.suffix in exts]
    digits = "".join(filter(str.isdigit, str(animal_id)))

    def _exact(stem, num):
        return bool(re.search(r"(?<![0-9])" + re.escape(num) + r"(?![0-9])", stem))

    cands = [str(animal_id)]
    if digits:
        n = int(digits)
        cands += [str(n), f"{n:02d}", f"{n:03d}"]

    for num in dict.fromkeys(cands):
        for f in all_v:
            if _exact(f.stem, num):
                return f, None

    if digits:
        for f in all_v:
            d = "".join(filter(str.isdigit, f.stem))
            if d and int(d) == int(digits):
                return f, None

    return None, f"No video for animal_id={animal_id}"


# ═══════════════════════════════════════════════════════════════
#  [1] ID resolution
# ═══════════════════════════════════════════════════════════════
def _build_id_resolver(df, id_col="animal_id"):
    ids       = df[id_col].dropna().unique()
    by_text = {str(v).strip(): v for v in ids}

    by_number = {}
    for v in ids:
        try:
            by_number[float(str(v).strip())] = v
        except (TypeError, ValueError):
            pass

    def sort_key(a):
        s = str(a).strip()
        try:
            return (0, float(s), "")
        except ValueError:
            return (1, 0.0, s)

    def resolve(raw):
        s = str(raw).strip()
        if s in by_text:
            return by_text[s]
        try:
            return by_number.get(float(s))
        except (TypeError, ValueError):
            return None

    return resolve, sorted(by_text.keys(), key=sort_key)


# ═══════════════════════════════════════════════════════════════
#  VIEWER
# ═══════════════════════════════════════════════════════════════
def open_cuttlefish_viewer(df, animal_id, video_dir,
                           cluster_col="supercluster",
                           id_col="animal_id",
                           label_cols=("cluster", "supercluster"),
                           trail_lw=1.8,
                           prey_color="black", prey_halo="#ffffff",
                           prey_lw=1.6,
                           min_run_len=15):            # "SeqVec" mode: segments < min_run_len frames greyed out (as in _extract_sequences)
    """
    Video + cuttlefish trail (group colour) + shrimp trail
    + instantaneous link + distance curve.

    prey_color : colour of the shrimp line. "black" is readable
                 thanks to the `prey_halo` outline; use "#ffb300" for
                 the amber of plot_animal_trajectory.

    Returns `state`. KEEP IT:  vw = open_cuttlefish_viewer(...)
    otherwise the garbage collector destroys the widgets.
    """
    if "group_color" not in globals():
        raise RuntimeError("Run CELL 0 first (colour registry).")
    _col = group_color
    _pfx = group_prefix

    available = [c for c in label_cols if c in df.columns]
    if not available:
        raise KeyError(f"No column among {list(label_cols)} in df.")
    if cluster_col not in available:
        print(f"[VIEWER] '{cluster_col}' missing -> switching to '{available[0]}'")
        cluster_col = available[0]
    if id_col not in df.columns:
        raise KeyError(f"Column '{id_col}' missing from df.")
    if "frame" not in df.columns:
        raise KeyError("Column 'frame' missing from df.")

    # cleanly close an already open viewer (otherwise 2 timers -> crash)
    for _old in (globals().get("viewer_state"),):
        if isinstance(_old, dict) and _old.get("widgets"):
            try:
                _old["widgets"].get("timer") and _old["widgets"]["timer"].stop()
                plt.close(_old["widgets"]["fig"])
            except Exception:
                pass
    resolve_id, available_ids = _build_id_resolver(df, id_col)

    state = {"df_i": None, "xs": None, "ys": None, "pxs": None, "pys": None,
             "labels": None, "frame_nums": None,
             "csegs": None, "cmask": None, "ccols": None,
             "psegs": None, "pmask": None, "pok": None,
             "dist": None, "i_min": None, "d_min": None,
             "cap": None, "col": cluster_col, "aid": None,
             "playing": False, "loading": False, "legend": None,
             "show": {"shrimp": True, "link": True, "trail": True},
             "widgets": {}}

    # ── Layout ────────────────────────────────────────────────────
    fig     = plt.figure(figsize=(9.8, 11.6), facecolor="#0d0d1a")
    ax_vid  = fig.add_axes([0.06, 0.50,  0.88, 0.47])
    ax_seq  = fig.add_axes([0.06, 0.455, 0.88, 0.035])
    ax_dist = fig.add_axes([0.06, 0.38,  0.88, 0.065])
    ax_feat = fig.add_axes([0.06, 0.265, 0.88, 0.105])
    for a in (ax_vid, ax_seq, ax_dist, ax_feat):
        a.set_facecolor("#0d0d1a")
        for sp in a.spines.values():
            sp.set_edgecolor("#333333")

    ax_vid.set_axis_off()
    im_artist = ax_vid.imshow(np.zeros((10, 10, 3), dtype=np.uint8))

    # [9] Cuttlefish trail — a single artist
    traj_lc = LineCollection([], linewidths=trail_lw, capstyle="round",
                             zorder=3, alpha=0.95)
    ax_vid.add_collection(traj_lc)

    # [11] Shrimp trail — black, outlined to stay readable on
    #      a dark background as well as on a bright video.
    prey_lc = LineCollection([], linewidths=prey_lw, colors=prey_color,
                             linestyles=(0, (5, 2)), zorder=2, alpha=0.95)
    prey_lc.set_path_effects([mpe.Stroke(linewidth=prey_lw + 1.8,
                                         foreground=prey_halo, alpha=0.85),
                              mpe.Normal()])
    ax_vid.add_collection(prey_lc)

    link_line, = ax_vid.plot([], [], color="#cccccc", lw=0.9, ls=":",
                             alpha=0.8, zorder=4)

    start_dot  = ax_vid.scatter([], [], marker="o", s=70, facecolor="#00e676",
                                edgecolor="black", linewidth=1.2, zorder=6)
    prey_start = ax_vid.scatter([], [], marker="^", s=70, facecolor=prey_color,
                                edgecolor=prey_halo, linewidth=1.2, zorder=6)
    min_star   = ax_vid.scatter([], [], marker="*", s=260, facecolor="white",
                                edgecolor="black", linewidth=1.2, zorder=7)
    pos_dot    = ax_vid.scatter([], [], s=75, c="white",
                                edgecolor="black", linewidth=1.2, zorder=9)
    prey_dot   = ax_vid.scatter([], [], marker="D", s=40, c=prey_color,
                                edgecolor=prey_halo, linewidth=1.0, zorder=9)
    title_txt  = ax_vid.set_title("", color="white", fontsize=11)

    # Sequence strip
    ax_seq.set_yticks([]); ax_seq.set_xticks([])
    seq_im      = ax_seq.imshow(np.zeros((1, 10, 3)), aspect="auto",
                                interpolation="nearest")
    cursor_line = ax_seq.axvline(0, color="red", lw=1.5)

    # [12] Distance curve
    ax_dist.tick_params(colors="#666666", labelsize=6)
    ax_dist.set_xticks([])
    dist_line,   = ax_dist.plot([], [], color="#e0e0e0", lw=1.2, zorder=2)
    dist_cursor  = ax_dist.axvline(0, color="red", lw=1.2, zorder=3)
    dist_minmark, = ax_dist.plot([], [], ls="none", marker="*", ms=11,
                                 color="white", mec="black", mew=0.8, zorder=4)
    dist_txt = ax_dist.text(0.005, 0.92, "", transform=ax_dist.transAxes,
                            color="#888888", fontsize=6.5, va="top",
                            family="monospace")

    # ── Evolution of the parameters (clustering features), frame by frame ──
    #   curves normalised over all the data (p1-p99) -> comparable between animals
    #   values shown in real units when the column exists
    feat_list = [f for f in globals().get("FEATURES", []) if f in df.columns]
    feat_rng  = {f: (np.nanpercentile(df[f], 1), np.nanpercentile(df[f], 99)) for f in feat_list}
    feat_cols = [mcolors.to_hex(plt.cm.tab10(k % 10)) for k in range(len(feat_list))]
    _FSHORT = {"distance_to_shrimp": "dist", "attack_side": "side", "bearing_to_prey": "bearing",
               "cuttle_approach_speed": "approach", "orbit_speed": "orbit", "who_moves": "who",
               "speed_cuttle_log": "speed", "orientation_variation": "orient. var",
               "approach_angle": "angle", "speed_cuttle_smooth": "speed", "jerk_cuttle": "jerk"}
    _FRAW = {"distance_to_shrimp": ("distance_mm", "mm"), "orbit_speed": ("orbit_speed_deg_s", "°/s"),
             "cuttle_approach_speed": ("cuttle_approach_speed_mm_s", "mm/s"),
             "attack_side": ("attack_side", "°"), "bearing_to_prey": ("bearing_to_prey", "°"),
             "speed_cuttle_log": ("speed_cuttle_mm_s", "mm/s")}
    fname = lambda f: _FSHORT.get(f, f[:10])
    ax_feat.set_xticks([]); ax_feat.set_yticks([])
    feat_cursor = ax_feat.axvline(0, color="red", lw=1.2, zorder=3)
    feat_txt    = ax_feat.text(0.005, 0.96, "", transform=ax_feat.transAxes, color="#dddddd",
                               fontsize=6.5, va="top", family="monospace", zorder=5,
                               bbox=dict(facecolor="#0d0d1a", alpha=0.6, edgecolor="none", pad=1))
    ax_feat.text(0.995, 0.04, "parameters (normalised p1–p99)", transform=ax_feat.transAxes, color="#666666",
                 fontsize=6, ha="right", va="bottom")

    # ── Live radar: current frame (state colour) vs mean profile of the state (dashed) ──
    ax_radar = fig.add_axes([0.755, 0.515, 0.18, 0.18], projection="polar")
    ax_radar.set_facecolor((0.05, 0.05, 0.1, 0.75))
    _ang = np.linspace(0, 2 * np.pi, max(len(feat_list), 1), endpoint=False)
    _angc = np.r_[_ang, _ang[:1]]
    ax_radar.set_xticks(_ang)
    ax_radar.set_xticklabels([fname(f) for f in feat_list], fontsize=6.5, color="white")
    for _l in ax_radar.get_xticklabels():
        _l.set_path_effects([mpe.withStroke(linewidth=2.2, foreground="black")])
    ax_radar.set_ylim(0, 1); ax_radar.set_yticks([0.5, 1.0]); ax_radar.set_yticklabels([])
    ax_radar.tick_params(pad=1)
    ax_radar.grid(color="#555577", lw=0.5)
    ax_radar.spines["polar"].set_color("#555577")
    radar_poly  = ax_radar.fill(_angc, np.zeros(len(_angc)), alpha=0.45, lw=1.5)[0]
    radar_proto, = ax_radar.plot(_angc, np.zeros(len(_angc)), ls="--", lw=1, color="white", alpha=0.8)
    ax_radar.set_visible(len(feat_list) >= 3)
    _protos = {}                                   # mean profile of each state (cached per column)

    def _proto(col, lab):
        if col not in _protos:
            m = df.groupby(col)[feat_list].mean()
            for f in feat_list:
                lo, hi = feat_rng[f]
                m[f] = np.clip((m[f] - lo) / (hi - lo + 1e-12), 0, 1)
            _protos[col] = m
        return _protos[col].loc[lab].to_numpy(float) if lab in _protos[col].index else None

    status_txt = fig.text(0.06, 0.252, "", color="#888888", fontsize=8,
                          va="top", ha="left", family="monospace")

    def _status(msg, ok=True):
        status_txt.set_text(msg)
        status_txt.set_color("#7fbf7f" if ok else "#ff7676")
        fig.canvas.draw_idle()
        print(f"[VIEWER] {msg}")

    # ── Controls ───────────────────────────────────────────────────────
    ax_slider = fig.add_axes([0.06, 0.175, 0.62, 0.028], facecolor="#141428")
    slider    = Slider(ax_slider, "Frame", 0, 1, valinit=0, valstep=1, color="#4fc3f7")
    slider.label.set_color("white"); slider.valtext.set_color("white")

    ax_play  = fig.add_axes([0.72, 0.165, 0.08, 0.048])
    btn_play = Button(ax_play, "▶ Play", color="#224422", hovercolor="#336633")
    btn_play.label.set_color("white")

    ax_tb   = fig.add_axes([0.06, 0.090, 0.14, 0.048])
    textbox = TextBox(ax_tb, "Animal ID: ", initial=str(animal_id),
                      color="#111122", hovercolor="#1a1a2e")
    textbox.label.set_color("#aaaaaa"); textbox.text_disp.set_color("white")

    ax_go  = fig.add_axes([0.21, 0.090, 0.08, 0.048])
    btn_go = Button(ax_go, "Go", color="#224422", hovercolor="#336633")
    btn_go.label.set_color("white")

    ax_prev  = fig.add_axes([0.300, 0.090, 0.030, 0.048])
    ax_next  = fig.add_axes([0.335, 0.090, 0.030, 0.048])
    btn_prev = Button(ax_prev, "◀", color="#222244", hovercolor="#333366")
    btn_next = Button(ax_next, "▶", color="#222244", hovercolor="#333366")
    for b in (btn_prev, btn_next):
        b.label.set_color("white")

    ax_radio = fig.add_axes([0.40, 0.055, 0.17, 0.095], facecolor="#141428")
    radio    = RadioButtons(ax_radio, available, active=available.index(cluster_col),
                            activecolor="#4fc3f7")
    for lbl in radio.labels:
        lbl.set_color("white"); lbl.set_fontsize(8)

    # [14] Toggleable layers
    ax_check = fig.add_axes([0.60, 0.055, 0.20, 0.095], facecolor="#141428")
    check    = CheckButtons(ax_check, _CHECK_LABELS, [True, True, True])
    for lbl in check.labels:
        lbl.set_color("white"); lbl.set_fontsize(8)

    state["seq_mode"] = "total"
    ax_seqb  = fig.add_axes([0.82, 0.090, 0.13, 0.048])
    btn_seq  = Button(ax_seqb, "Sequence: full", color="#222244", hovercolor="#333366")
    btn_seq.label.set_color("white"); btn_seq.label.set_fontsize(7.5)

    state["widgets"] = {"slider": slider, "btn_play": btn_play, "btn_seq": btn_seq,
                        "textbox": textbox, "btn_go": btn_go,
                        "btn_prev": btn_prev, "btn_next": btn_next,
                        "radio": radio, "check": check, "fig": fig}

    # ── [2] Exception decorator ─────────────────────────────────────
    def safe(fn):
        def wrapper(*a, **kw):
            try:
                return fn(*a, **kw)
            except Exception as exc:
                _status(f"ERROR {type(exc).__name__}: {exc}", ok=False)
                traceback.print_exc()
        wrapper.__name__ = getattr(fn, "__name__", "callback")
        return wrapper

    # ── [10] Dynamic legend ──────────────────────────────────────────
    def _build_legend(labels, col, has_prey, coverage):
        if state["legend"] is not None:
            state["legend"].remove()
            state["legend"] = None

        uniq = sorted({int(l) for l in labels})
        h = [Line2D([0], [0], color=_col(l, col), lw=3.5, label=f"{_pfx(col)}{l}")
             for l in uniq]
        h += [Line2D([0], [0], marker="o", color="none", markerfacecolor="white",
                     markeredgecolor="black", markersize=8, label="cuttlefish"),
              Line2D([0], [0], marker="o", color="none", markerfacecolor="#00e676",
                     markeredgecolor="black", markersize=8, label="start")]
        if has_prey:
            h += [Line2D([0], [0], color=prey_color, lw=2, ls="--",
                         path_effects=[mpe.Stroke(linewidth=3.5,
                                                  foreground=prey_halo, alpha=0.85),
                                       mpe.Normal()],
                         label=f"shrimp ({coverage:.0f}%)"),
                  Line2D([0], [0], marker="*", color="none", markerfacecolor="white",
                         markeredgecolor="black", markersize=12, label="min dist.")]

        leg = ax_vid.legend(handles=h, loc="upper right", fontsize=7,
                            ncol=2 if len(uniq) > 6 else 1,
                            framealpha=0.85, facecolor="#141428",
                            edgecolor="#444444", labelcolor="white",
                            title=col, title_fontsize=7.5, borderpad=0.6)
        leg.get_title().set_color("#aaaaaa")
        leg.set_zorder(12)
        state["legend"] = leg

    # Sequence strip: "full" = state of each frame; "SeqVec" = what SeqVec
    # sees (segments < min_run_len frames removed -> grey)
    def _draw_features(df_i):
        for ln in state.get("feat_lines", []):
            ln.remove()
        state["feat_lines"], state["feat_raw"] = [], None
        if state.get("feat_leg") is not None:
            state["feat_leg"].remove()
            state["feat_leg"] = None
        if not feat_list:
            feat_txt.set_text("FEATURES missing: run cell 0")
            return
        x = np.arange(len(df_i))
        raw, norm = [], []
        for k, f in enumerate(feat_list):
            lo, hi = feat_rng[f]
            y = np.clip((df_i[f].to_numpy(float) - lo) / (hi - lo + 1e-12), 0, 1)
            ln, = ax_feat.plot(x, y, color=feat_cols[k], lw=1.0, alpha=0.9, label=fname(f))
            norm.append(y)
            state["feat_lines"].append(ln)
            rc, unit = _FRAW.get(f, (f, ""))
            raw.append((df_i[rc] if rc in df_i.columns else df_i[f]).to_numpy(float))
        state["feat_raw"] = np.column_stack(raw)
        state["feat_norm"] = np.column_stack(norm)
        ax_feat.set_xlim(0, len(df_i)); ax_feat.set_ylim(-0.04, 1.04)
        state["feat_leg"] = ax_feat.legend(loc="upper right", fontsize=6, ncol=len(feat_list),
                                           framealpha=0.6, facecolor="#141428", edgecolor="#444444",
                                           labelcolor="white", handlelength=1.2, columnspacing=0.8)

    def _kept_mask(labels):
        """True for the frames in a segment >= min_run_len frames."""
        lab = np.asarray(labels)
        if len(lab) == 0:
            return np.zeros(0, bool)
        cut = np.flatnonzero(lab[1:] != lab[:-1]) + 1
        lens = np.diff(np.r_[0, cut, len(lab)])
        return np.repeat(lens >= min_run_len, lens)

    def _draw_seq_band(col):
        labels = state["labels"]
        rgb = np.array([mcolors.to_rgb(_col(l, col)) for l in labels])
        if state["seq_mode"] == "seqvec" and len(rgb):
            keep = _kept_mask(labels)
            rgb[~keep] = mcolors.to_rgb("#555555")
            pct = 100 * (1 - keep.mean())
            note = "" if col == "cluster" else " | SeqVec uses 'cluster'"
            _status(f"{pct:.0f} % of the frames removed by SeqVec (segments < {min_run_len} f)" + note)
        seq_im.set_data(rgb[np.newaxis, :, :])
        seq_im.set_extent([0, len(labels), 0, 1])
        ax_seq.set_xlim(0, len(labels))
        fig.canvas.draw_idle()

    # ── Loading an individual ────────────────────────────────────────
    def _load_animal(raw_id, col):
        aid = resolve_id(raw_id)                       # [1]
        if aid is None:
            _status(f"ID '{raw_id}' not found. Available : "
                    f"{', '.join(available_ids[:12])}…", ok=False)
            return False
        if col not in df.columns:
            _status(f"Column '{col}' missing from df.", ok=False)
            return False

        df_i = df[df[id_col] == aid].sort_values("frame").reset_index(drop=True)
        if df_i.empty:
            _status(f"animal_id={aid!r} : no rows.", ok=False)
            return False

        cols = set(df_i.columns)
        ckey = _cf_pick_key(_CUTTLE_CANDS, cols)
        pkey = _cf_pick_key(_PREY_CANDS, cols)
        if ckey is None:
            _status("No cuttlefish tracking point found.", ok=False)
            return False

        labs  = pd.to_numeric(df_i[col], errors="coerce")
        n_bad = int(labs.isna().sum())
        if n_bad:
            _status(f"{n_bad} non-numeric '{col}' label(s) -> 0.", ok=False)
            labs = labs.fillna(0)

        verr, coverage = None, 0.0
        state["loading"] = True                        # [6]
        try:
            state["df_i"]   = df_i
            state["aid"]    = aid
            state["col"]    = col
            state["xs"]     = df_i[f"{ckey}_x"].to_numpy(float)
            state["ys"]     = df_i[f"{ckey}_y"].to_numpy(float)
            state["labels"] = labs.to_numpy(int)
            state["frame_nums"] = pd.to_numeric(df_i["frame"], errors="coerce") \
                                    .fillna(0).to_numpy(int)

            # [13] Cuttlefish: full segments + validity mask
            csegs, cok, cmask = _valid_segments(state["xs"], state["ys"])
            state["csegs"] = csegs
            state["cmask"] = cmask
            # object array -> boolean indexing works on it
            state["ccols"] = np.array(
                [_col(l, col) for l in state["labels"][:-1]], dtype=object) \
                if len(state["labels"]) > 1 else np.empty(0, dtype=object)
            state["cok"] = cok

            # [11] Shrimp
            has_prey = pkey is not None
            if has_prey:
                state["pxs"] = df_i[f"{pkey}_x"].to_numpy(float)
                state["pys"] = df_i[f"{pkey}_y"].to_numpy(float)
                psegs, pok, pmask = _valid_segments(state["pxs"], state["pys"])
                state["psegs"], state["pok"], state["pmask"] = psegs, pok, pmask
                coverage = 100 * pok.mean()
                has_prey = bool(pok.any())
            else:
                state["pxs"] = state["pys"] = None
                state["psegs"] = state["pok"] = state["pmask"] = None

            # [12] Cuttlefish–shrimp distance
            if has_prey:
                d = np.hypot(state["xs"] - state["pxs"],
                             state["ys"] - state["pys"])
                state["dist"] = d
                if np.isfinite(d).any():
                    state["i_min"] = int(np.nanargmin(d))
                    state["d_min"] = float(d[state["i_min"]])
                    n = len(d)
                    ax_dist.set_visible(True)
                    dist_line.set_data(np.arange(n), d)
                    ax_dist.set_xlim(0, n)
                    lo, hi = np.nanmin(d), np.nanmax(d)
                    ax_dist.set_ylim(lo - 0.05 * (hi - lo + 1),
                                     hi + 0.10 * (hi - lo + 1))
                    dist_minmark.set_data([state["i_min"]], [state["d_min"]])
                    dist_txt.set_text(f"cuttlefish–shrimp distance (px) — "
                                      f"min {state['d_min']:.0f} @ f{state['i_min']}")
                    min_star.set_offsets([[state["xs"][state["i_min"]],
                                           state["ys"][state["i_min"]]]])
                    min_star.set_visible(True)
                else:
                    has_prey = False
            if not has_prey:
                state["dist"] = state["i_min"] = state["d_min"] = None
                dist_line.set_data([], [])
                dist_minmark.set_data([], [])
                dist_txt.set_text("shrimp not tracked — no distance")
                min_star.set_visible(False)

            # Sequence strip — SAME colour source
            _draw_seq_band(col)

            _build_legend(state["labels"], col, has_prey, coverage)   # [10]
            _draw_features(df_i)

            # Video
            if state["cap"] is not None:
                state["cap"].release()
                state["cap"] = None
            vpath, verr = _cf_find_video(video_dir, aid)
            state["cap"] = cv2.VideoCapture(str(vpath)) if (vpath and _HAS_CV2) else None

            n = len(df_i)
            slider.valmin = 0
            slider.valmax = max(1, n - 1)
            slider.ax.set_xlim(0, slider.valmax)
            slider.val = 0
        finally:
            state["loading"] = False

        _display_frame(0)
        n_grp = len(set(state["labels"]))
        msg = f"animal {aid} — {len(df_i)} f, {n_grp} {col}s"
        msg += f", shrimp {coverage:.0f}%" if state["pxs"] is not None else ", without shrimp"
        if state["cap"] is None:
            msg += f"  ({verr or 'without video'})"
        _status(msg, ok=True)
        return True

    # ── Displaying a frame ───────────────────────────────────────────
    def _display_frame(i):
        df_i = state["df_i"]
        if df_i is None or len(df_i) == 0:
            return
        i = int(max(0, min(i, len(df_i) - 1)))
        show = state["show"]

        # Background: video or framing on both trajectories
        if state["cap"] is not None:
            fn = int(state["frame_nums"][i])
            state["cap"].set(cv2.CAP_PROP_POS_FRAMES, fn)
            ret, frame = state["cap"].read()
            if ret:
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                im_artist.set_data(rgb)
                im_artist.set_extent([0, rgb.shape[1], rgb.shape[0], 0])
                ax_vid.set_xlim(0, rgb.shape[1])
                ax_vid.set_ylim(rgb.shape[0], 0)
        else:
            h, w = 480, 640
            im_artist.set_data(np.full((h, w, 3), 15, dtype=np.uint8))
            im_artist.set_extent([0, w, h, 0])
            xa = [state["xs"][state["cok"]]]
            ya = [state["ys"][state["cok"]]]
            if state["pxs"] is not None and state["pok"].any():
                xa.append(state["pxs"][state["pok"]])
                ya.append(state["pys"][state["pok"]])
            xa, ya = np.concatenate(xa), np.concatenate(ya)
            if len(xa):
                m = 0.06 * max(np.ptp(xa), np.ptp(ya), 1.0)
                ax_vid.set_xlim(xa.min() - m, xa.max() + m)
                ax_vid.set_ylim(ya.max() + m, ya.min() - m)

        # [9][13] Cuttlefish trail: cut at i THEN filter the gaps
        if i > 0 and show["trail"]:
            m = state["cmask"][:i]
            traj_lc.set_segments(state["csegs"][:i][m])
            traj_lc.set_color(list(state["ccols"][:i][m]))
        else:
            traj_lc.set_segments([])
        traj_lc.set_visible(show["trail"])

        # [11] Shrimp trail, same logic
        has_p = state["pxs"] is not None
        if has_p and i > 0 and show["shrimp"]:
            mp = state["pmask"][:i]
            prey_lc.set_segments(state["psegs"][:i][mp])
        else:
            prey_lc.set_segments([])
        prey_lc.set_visible(has_p and show["shrimp"])

        # Markers
        start_dot.set_offsets([[state["xs"][0], state["ys"][0]]])
        pos_dot.set_offsets([[state["xs"][i], state["ys"][i]]])

        if has_p:
            idx_ok = np.flatnonzero(state["pok"])
            if len(idx_ok):
                prey_start.set_offsets([[state["pxs"][idx_ok[0]],
                                         state["pys"][idx_ok[0]]]])
            prey_start.set_visible(show["shrimp"] and len(idx_ok) > 0)
            vis_now = show["shrimp"] and np.isfinite(state["pxs"][i])
            if vis_now:
                prey_dot.set_offsets([[state["pxs"][i], state["pys"][i]]])
            prey_dot.set_visible(bool(vis_now))

            # Instantaneous cuttlefish <-> shrimp link
            if show["link"] and vis_now:
                link_line.set_data([state["xs"][i], state["pxs"][i]],
                                   [state["ys"][i], state["pys"][i]])
                link_line.set_visible(True)
            else:
                link_line.set_visible(False)
        else:
            prey_start.set_visible(False)
            prey_dot.set_visible(False)
            link_line.set_visible(False)

        min_star.set_visible(state["i_min"] is not None and show["shrimp"])

        # Title
        lab     = int(state["labels"][i])
        col_hex = _col(lab, state["col"])
        pos_dot.set_facecolor(col_hex)
        txt = (f"Animal {state['aid']}  |  frame {state['frame_nums'][i]} "
               f"({i+1}/{len(df_i)})  |  {state['col']} = "
               f"{_pfx(state['col'])}{lab}")
        if state["dist"] is not None and np.isfinite(state["dist"][i]):
            txt += f"  |  d = {state['dist'][i]:.0f} px"
        title_txt.set_text(txt)
        title_txt.set_color(col_hex)

        # Cursors
        cursor_line.set_xdata([i, i])
        dist_cursor.set_xdata([i, i])
        feat_cursor.set_xdata([i, i])
        if state.get("feat_raw") is not None:
            v = state["feat_raw"][i]
            feat_txt.set_text("  ".join(f"{fname(f)} {v[k]:.1f}{_FRAW.get(f, ('', ''))[1]}"
                                        for k, f in enumerate(feat_list)))
        if state.get("feat_norm") is not None and len(feat_list) >= 3:
            r = state["feat_norm"][i]
            radar_poly.set_xy(np.column_stack([_angc, np.r_[r, r[:1]]]))
            radar_poly.set_facecolor(col_hex); radar_poly.set_edgecolor(col_hex)
            p = _proto(state["col"], state["labels"][i])
            radar_proto.set_data(_angc, np.r_[p, p[:1]] if p is not None else np.zeros(len(_angc)))
        slider.eventson = False
        slider.set_val(i)
        slider.eventson = True
        fig.canvas.draw_idle()

    # ── Callbacks ───────────────────────────────────────────────────────
    def _stop_playback():
        state["playing"] = False
        btn_play.label.set_text("▶ Play")
        timer.stop()

    @safe
    def on_slider(val):
        if state["loading"]:
            return
        _display_frame(int(val))

    @safe
    def on_go(event=None):
        _stop_playback()
        _load_animal(textbox.text, state["col"])

    @safe
    def on_submit(text):
        on_go()

    @safe
    def on_radio(label):
        _stop_playback()
        cur = state["aid"] if state["aid"] is not None else textbox.text
        _load_animal(cur, label)

    @safe
    def on_check(label):
        state["show"] = dict(zip(_CHECK_LABELS, check.get_status()))
        _display_frame(int(slider.val))

    @safe
    def _step_animal(delta):
        cur = str(state["aid"]) if state["aid"] is not None else textbox.text.strip()
        try:
            k = available_ids.index(cur)
        except ValueError:
            k = 0
        textbox.set_val(available_ids[(k + delta) % len(available_ids)])

    @safe
    def _tick():
        if not plt.fignum_exists(fig.number):        # window closed (or destroyed by a backend switch) -> stop the timer
            timer.stop()
            return
        if not state["playing"]:
            return
        df_i = state["df_i"]
        i = int(slider.val)
        if df_i is None or i >= len(df_i) - 1:
            _stop_playback()
            return
        _display_frame(i + 1)

    timer = fig.canvas.new_timer(interval=40)
    timer.add_callback(_tick)
    state["widgets"]["timer"] = timer

    @safe
    def on_play(event):
        state["playing"] = not state["playing"]
        btn_play.label.set_text("⏸ Pause" if state["playing"] else "▶ Play")
        timer.start() if state["playing"] else timer.stop()

    @safe
    def on_close(event):
        timer.stop()
        if state["cap"] is not None:
            state["cap"].release()
            state["cap"] = None

    slider.on_changed(on_slider)
    btn_go.on_clicked(on_go)
    textbox.on_submit(on_submit)
    btn_prev.on_clicked(lambda e: _step_animal(-1))
    btn_next.on_clicked(lambda e: _step_animal(+1))
    btn_play.on_clicked(on_play)
    radio.on_clicked(on_radio)
    check.on_clicked(on_check)

    @safe
    def on_seq_mode(event):
        state["seq_mode"] = "seqvec" if state["seq_mode"] == "total" else "total"
        btn_seq.label.set_text("Sequence: SeqVec" if state["seq_mode"] == "seqvec" else "Sequence: full")
        if state["labels"] is not None:
            _draw_seq_band(state["col"])
    btn_seq.on_clicked(on_seq_mode)
    fig.canvas.mpl_connect("close_event", on_close)

    _load_animal(animal_id, cluster_col)
    plt.show()
    return state


# ═══════════════════════════════════════════════════════════════
#  DIAGNOSTIC
# ═══════════════════════════════════════════════════════════════
def diagnose_viewer(df, animal_id, video_dir, id_col="animal_id"):
    print("=" * 64)
    print("DIAGNOSTIC VIEWER")
    print("=" * 64)

    print(f"df[{id_col!r}].dtype        : {df[id_col].dtype}")
    ex = df[id_col].dropna().unique()[:8]
    print(f"example values             : {list(ex)}")
    print(f"requested animal_id        : {animal_id!r} ({type(animal_id).__name__})")

    raw_match = df[df[id_col] == animal_id]
    print(f"RAW match                  : {len(raw_match)} row(s)"
          f"{'   <-- BROKEN' if raw_match.empty else '   OK'}")
    resolve, _ = _build_id_resolver(df, id_col)
    aid = resolve(animal_id)
    print(f"RESOLVED match             : {aid!r}")

    cols = set(df.columns)
    ckey = _cf_pick_key(_CUTTLE_CANDS, cols)
    pkey = _cf_pick_key(_PREY_CANDS, cols)
    print(f"cuttlefish point           : {ckey or 'NONE among ' + str(_CUTTLE_CANDS)}")
    print(f"shrimp point               : {pkey or 'NONE among ' + str(_PREY_CANDS)}")

    if aid is not None and pkey:
        sub = df[df[id_col] == aid]
        ok = np.isfinite(sub[f"{pkey}_x"].to_numpy(float)) & \
             np.isfinite(sub[f"{pkey}_y"].to_numpy(float))
        print(f"shrimp coverage            : {100*ok.mean():.1f}% "
              f"({int(ok.sum())}/{len(ok)} frames)")

    for c in ("cluster", "supercluster", "frame"):
        print(f"column {c!r:<16} : {'present' if c in cols else 'MISSING'}")
    print(f"colour registry            : "
          f"{'OK' if 'group_color' in globals() else 'MISSING — run CELL 0'}")
    print(f"opencv (cv2)               : {'yes' if _HAS_CV2 else 'NO'}")
    v, err = _cf_find_video(video_dir, animal_id)
    print(f"video                      : {v.name if v else err}")
    print(f"backend matplotlib         : {plt.get_backend()}")
    if "inline" in plt.get_backend().lower():
        print("   ⚠ 'inline' backend -> NO widget will respond. "
              "Use %matplotlib widget")
    print("=" * 64)


print("[OK] Viewer defined — usage :")
print("     diagnose_viewer(df, 12, VIDEO_DIR)")
print("     vw = open_cuttlefish_viewer(df, 12, VIDEO_DIR, cluster_col='supercluster')")
print("     vw = open_cuttlefish_viewer(..., prey_color='#ffb300')  # ambre")

In [ ]:
# VIDEO_DIR: defined in cell 0 depending on USE_FULL (videos_catch or 01_VIDEO_Cropped)

%matplotlib qt

viewer_state = open_cuttlefish_viewer(
    df,
    animal_id=animal_ids[0],      # starting ID — then change via the text box + "Go"
    video_dir=VIDEO_DIR,
    cluster_col="supercluster",   # or "cluster" (SeqVec uses "cluster")
    min_run_len=15,               # "Sequence: SeqVec" button: same threshold as the explorer
)


In [ ]:
# ===============================================================
#  CELL — Video browser by strategy
#  Browses the videos of a strategy group, several at a time.
#  - group: list on the left or ◀ ▶ | number of videos: 1, 2, 4, 6, 9
#  - previous / next page within the group, pause, restart, speed
#  - order: typical first (closest to the group centre), random or ID
#  - click on a tile = detailed viewer of the individual
#  Groups: mapping_optimal (regrouped) if it exists, otherwise final_state.
# ===============================================================
import re
import random
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.widgets import Button, RadioButtons, Slider, CheckButtons
from matplotlib.collections import LineCollection
import matplotlib.patheffects as mpe

try:
    import cv2
except ImportError:
    cv2 = None

_SB_LAYOUT = {1: (1, 1), 2: (1, 2), 4: (2, 2), 6: (2, 3), 9: (3, 3)}   # vidéos -> (lignes, colonnes)
_SB_ORDERS = ("typical", "random", "id")
_SB_COLS   = ("cluster", "supercluster")
_SB_SHORT  = {"distance_to_shrimp": "dist", "attack_side": "side", "bearing_to_prey": "bearing",
              "cuttle_approach_speed": "approach", "orbit_speed": "orbit", "who_moves": "who",
              "speed_cuttle_log": "speed", "orientation_variation": "orient. var",
              "approach_angle": "angle", "speed_cuttle_smooth": "speed", "jerk_cuttle": "jerk"}


def _sb_video_index(video_dir):
    """{individual number: Path}, ex. video100_cut.mp4 -> 100."""
    idx = {}
    if not Path(video_dir).is_dir():
        raise FileNotFoundError(f"Video folder not found: {video_dir} — put the videos there (see README) or change VIDEO_ROOT in cell 0")
    for f in Path(video_dir).iterdir():
        m = re.search(r"\d+", f.stem)
        if m and f.suffix.lower() in (".mp4", ".avi", ".mov", ".mkv"):
            idx[int(m.group())] = f
    return idx


def _sb_order(members, how, final_state, seed=0):
    """Order of the individuals of a group: typical first (lowest mean SeqVec
    distance to the other members), random or ID number."""
    by_id = sorted(members, key=lambda a: (0, int(a)) if a.isdigit() else (1, a))
    if how == "random":
        out = list(by_id)
        random.Random(seed).shuffle(out)
        return out
    if how == "typical" and final_state is not None and final_state.get("D") is not None:
        pos = {str(a): i for i, a in enumerate(final_state["multi"])}
        inside = [a for a in by_id if a in pos]
        if len(inside) > 1:
            D = np.asarray(final_state["D"])
            ix = [pos[a] for a in inside]
            score = D[np.ix_(ix, ix)].mean(1)          # mean distance to the other members
            inside = [a for _, a in sorted(zip(score, inside))]
        return inside + [a for a in by_id if a not in pos]
    return by_id


def open_strategy_browser(groups=None, df=None, video_dir=None, final_state=None,
                          n_show=4, speed=2, order="typical", crop=True,
                          tile_w=360, cluster_col="cluster", include_mono=False,
                          min_run_len=15):
    """
    Video browser by strategy group: several videos played at the
    same time, page by page, with group switching.

    groups      : {animal_id: group letter} — default: mapping_optimal otherwise final_state
    n_show      : videos shown at the same time (1, 2, 4, 6 or 9)
    speed       : 1 = real time, 2 = twice as fast, ...
    order       : "typical" | "random" | "id"
    cluster_col : "cluster" | "supercluster" (button in the window)
    crop        : True = crop the central square (the arena)
    min_run_len : "SeqVec" strip: segments < min_run_len frames greyed out

    Returns `state`: KEEP IT (sb = open_strategy_browser(...)), otherwise the widgets die.
    """
    if cv2 is None:
        raise ImportError("opencv-python required: pip install opencv-python")
    G  = globals()
    # cleanly close an already open browser (otherwise 2 timers -> crash)
    for _old in (G.get("sb"),):
        if isinstance(_old, dict) and _old.get("widgets"):
            try:
                _old["widgets"]["timer"].stop()
                plt.close(_old["widgets"]["fig"])
            except Exception:
                pass
    df = G["df"] if df is None else df
    video_dir   = G.get("VIDEO_DIR") if video_dir is None else video_dir
    final_state = G.get("final_state") if final_state is None else final_state
    if groups is None:
        if "mapping_optimal" in G:
            groups, src = G["mapping_optimal"], "mapping_optimal (regrouped)"
        elif final_state is not None:
            groups, src = final_state["mapping_letters"], "final_state (explorer)"
        else:
            raise RuntimeError("No groups: run the explorer (final_state) or pass groups=")
    else:
        src = "groups="

    vids = _sb_video_index(video_dir)
    by_group = {}
    for a, gr in groups.items():
        a = str(a)
        if (gr == "MONO" and not include_mono) or not (a.isdigit() and int(a) in vids):
            continue
        by_group.setdefault(gr, []).append(a)
    if not by_group:
        raise RuntimeError(f"No video found for these groups in {video_dir}")
    names = sorted(by_group)

    # ── Parameters (features) normalised p1-p99 over all the data ──
    feat_list = [f for f in G.get("FEATURES", []) if f in df.columns]
    feat_rng  = {f: (np.nanpercentile(df[f], 1), np.nanpercentile(df[f], 99)) for f in feat_list}
    feat_cols = [mcolors.to_hex(plt.cm.tab10(k % 10)) for k in range(len(feat_list))]

    def fnorm(X):
        if not feat_list:
            return None
        return np.column_stack([np.clip((X[:, k] - feat_rng[f][0]) / (feat_rng[f][1] - feat_rng[f][0] + 1e-12), 0, 1)
                                for k, f in enumerate(feat_list)])

    _protos = {}

    def _proto(col, lab):
        if col not in _protos:
            m = df.groupby(col)[feat_list].mean()
            _protos[col] = pd.DataFrame(fnorm(m.to_numpy(float)), index=m.index)
        return _protos[col].loc[lab].to_numpy(float) if lab in _protos[col].index else None

    # ── Per-individual info (frames, states, positions) ──
    d = df[df["animal_id"].astype(str).isin({a for m in by_group.values() for a in m})]
    info = {}
    for a, gi in d.groupby(d["animal_id"].astype(str)):
        gi = gi.sort_values("frame")
        info[a] = {
            "f0": int(gi["frame"].iloc[0]), "n": len(gi),
            "labs": {c: gi[c].to_numpy() for c in _SB_COLS if c in gi},
            "cx": gi["TopMantle_x"].to_numpy(float) if "TopMantle_x" in gi else None,
            "cy": gi["TopMantle_y"].to_numpy(float) if "TopMantle_y" in gi else None,
            "px": gi["MidShrimp_x"].to_numpy(float) if "MidShrimp_x" in gi else None,
            "py": gi["MidShrimp_y"].to_numpy(float) if "MidShrimp_y" in gi else None,
            "dur": (-gi["time_to_strike_s"].min() if "time_to_strike_s" in gi else len(gi) / 60),
            "feat": fnorm(gi[feat_list].to_numpy(float)) if feat_list else None,
        }
        ia = info[a]
        if "distance_mm" in gi:                                      # cuttlefish–shrimp distance
            ia["dist"], ia["dunit"] = gi["distance_mm"].to_numpy(float), "mm"
        elif ia["cx"] is not None and ia["px"] is not None:
            ia["dist"], ia["dunit"] = np.hypot(ia["cx"] - ia["px"], ia["cy"] - ia["py"]), "px"
        else:
            ia["dist"], ia["dunit"] = None, ""
        ia["kepts"] = {}                                             # frames kept by SeqVec (segments >= min_run_len)
        for c_, lab in ia["labs"].items():
            cut = np.flatnonzero(lab[1:] != lab[:-1]) + 1
            lens = np.diff(np.r_[0, cut, len(lab)])
            ia["kepts"][c_] = np.repeat(lens >= min_run_len, lens)

    cols = [c for c in _SB_COLS if c in df.columns] or [cluster_col]
    if cluster_col not in cols:
        cluster_col = cols[0]

    def _use_col(col):
        """Switches the displayed states (cluster / supercluster)."""
        for ia in info.values():
            ia["lab"], ia["kept"] = ia["labs"].get(col), ia["kepts"].get(col)

    _use_col(cluster_col)

    pal  = G.get("STRATEGY_PALETTE") or [mcolors.to_hex(plt.cm.tab10(i)) for i in range(10)]
    gcol = {gr: pal[i % len(pal)] for i, gr in enumerate(names)}
    scol = (lambda l: G["group_color"](l, st["col"])) if "group_color" in G else (lambda l: "white")
    pfx  = (lambda: G["group_prefix"](st["col"])) if "group_prefix" in G else (lambda: "C" if st["col"] == "cluster" else "S")

    st = {"group": names[0], "n_show": n_show if n_show in _SB_LAYOUT else 4, "page": 0,
          "order": order if order in _SB_ORDERS else "typical", "speed": int(speed), "col": cluster_col,
          "playing": True, "show_pos": True, "trail": True, "seqvec": False, "loop": False,
          "tiles": [], "detail": None, "widgets": {}, "groups": by_group, "source": src}

    # ── Layout ──
    fig = plt.figure(figsize=(15, 9.5), facecolor="#0d0d1a")
    ZONE = (0.17, 0.155, 0.82, 0.765)                              # x0, y0, largeur, hauteur
    title  = fig.text(0.58, 0.955, "", ha="center", fontsize=14, weight="bold")
    status = fig.text(0.17, 0.025, "", color="#999999", fontsize=8.5, family="monospace")

    def _panel(rect, label):
        ax = fig.add_axes(rect, facecolor="#141428")
        ax.set_title(label, color="#aaaaaa", fontsize=9, loc="left")
        return ax

    gl = [f"{g}  (n={len(by_group[g])})" for g in names]
    h_g = min(0.40, 0.035 * len(names) + 0.02)
    ax_grp = _panel([0.01, 0.93 - h_g, 0.14, h_g], "Group")
    radio_g = RadioButtons(ax_grp, gl, active=0, activecolor="#4fc3f7")
    for k, lbl in enumerate(radio_g.labels):
        lbl.set_color(gcol[names[k]]); lbl.set_fontsize(9)
    y = 0.93 - h_g - 0.055
    b_gprev = Button(fig.add_axes([0.01, y, 0.068, 0.04]), "◀ group", color="#222244", hovercolor="#333366")
    b_gnext = Button(fig.add_axes([0.082, y, 0.068, 0.04]), "group ▶", color="#222244", hovercolor="#333366")

    y -= 0.20
    ax_n = _panel([0.01, y, 0.14, 0.16], "Videos shown")
    radio_n = RadioButtons(ax_n, [str(k) for k in _SB_LAYOUT], active=list(_SB_LAYOUT).index(st["n_show"]),
                           activecolor="#4fc3f7")
    y -= 0.13
    ax_o = _panel([0.01, y, 0.14, 0.10], "Order")
    radio_o = RadioButtons(ax_o, list(_SB_ORDERS), active=_SB_ORDERS.index(st["order"]), activecolor="#4fc3f7")
    y -= 0.14
    ax_c = _panel([0.01, y, 0.14, 0.12], "Display")
    check = CheckButtons(ax_c, ["positions", "trail", "SeqVec (grey)", "loop"], [True, True, False, False])
    for w in (radio_g, radio_n, radio_o, check):
        for lbl in w.labels:
            if w is not radio_g:
                lbl.set_color("white"); lbl.set_fontsize(9)

    b_prev  = Button(fig.add_axes([0.17, 0.065, 0.11, 0.045]), "◀ previous", color="#222244", hovercolor="#333366")
    b_next  = Button(fig.add_axes([0.29, 0.065, 0.11, 0.045]), "next ▶", color="#222244", hovercolor="#333366")
    b_play  = Button(fig.add_axes([0.43, 0.065, 0.08, 0.045]), "Pause", color="#224422", hovercolor="#336633")
    b_again = Button(fig.add_axes([0.52, 0.065, 0.11, 0.045]), "Restart", color="#224422", hovercolor="#336633")
    ax_col  = fig.add_axes([0.645, 0.045, 0.095, 0.065], facecolor="#141428")
    ax_col.set_title("States", color="#aaaaaa", fontsize=8, loc="left")
    radio_c = RadioButtons(ax_col, cols, active=cols.index(cluster_col), activecolor="#4fc3f7")
    for lbl in radio_c.labels:
        lbl.set_color("white"); lbl.set_fontsize(8.5)
    ax_sp   = fig.add_axes([0.79, 0.075, 0.17, 0.025], facecolor="#141428")
    s_speed = Slider(ax_sp, "Speed", 1, 8, valinit=st["speed"], valstep=1, color="#4fc3f7")
    for b in (b_gprev, b_gnext, b_prev, b_next, b_play, b_again):
        b.label.set_color("white"); b.label.set_fontsize(9)
    s_speed.label.set_color("white"); s_speed.valtext.set_color("white")

    x_ = 0.17                                                     # colour legend of the parameters
    for k_, f in enumerate(feat_list):
        nm = _SB_SHORT.get(f, f[:10])
        fig.text(x_, 0.131, f"— {nm}", color=feat_cols[k_], fontsize=8.5, weight="bold")
        x_ += 0.014 + 0.0062 * len(nm)
    if feat_list:
        fig.text(x_ + 0.01, 0.131, "(parameters normalised p1–p99; radar: dashed = mean profile of the state)", color="#777777", fontsize=7.5)

    def _status(msg, ok=True):
        status.set_text(msg)
        status.set_color("#7fbf7f" if ok else "#ff7676")
        fig.canvas.draw_idle()

    def safe(fn):
        def wrapper(*a, **kw):
            try:
                return fn(*a, **kw)
            except Exception as e:
                traceback.print_exc()
                _status(f"ERROR {type(e).__name__}: {e}", ok=False)
        return wrapper

    # ── Tiles ──
    def _members():
        return _sb_order(by_group[st["group"]], st["order"], final_state)

    def _clear_tiles():
        for t in st["tiles"]:
            if t.get("cap") is not None:
                t["cap"].release()
            for ax in t["axes"]:
                ax.remove()
        st["tiles"] = []

    def _band(t):
        """Sequence strip (colour of the state of each frame), as in the viewer."""
        ia = info.get(t["aid"])
        if not ia or ia["lab"] is None:
            return
        rgb = np.array([mcolors.to_rgb(scol(l)) for l in ia["lab"]])
        if st["seqvec"]:
            rgb[~ia["kept"]] = mcolors.to_rgb("#555555")
        t["band"].set_data(rgb[np.newaxis, :, :])
        t["band"].set_extent([0, len(rgb), 0, 1])
        t["ax_seq"].set_xlim(0, len(rgb))

    def _trail_segments(t):
        """Trail segments in tile coordinates (computed once)."""
        ia = info[t["aid"]]
        out = {}
        for key, xs, ys in (("c", ia["cx"], ia["cy"]), ("p", ia["px"], ia["py"])):
            if xs is None:
                out[key] = None
                continue
            X, Y = (xs - t["xo"]) * t["s"], ys * t["s"]
            pts = np.column_stack([X, Y]).reshape(-1, 1, 2)
            ok = np.isfinite(X) & np.isfinite(Y)
            out[key] = (np.concatenate([pts[:-1], pts[1:]], axis=1), ok[:-1] & ok[1:])
        t["segs"] = out
        if ia["lab"] is not None:
            t["scols"] = np.array([scol(l) for l in ia["lab"][:-1]], dtype=object)

    def _read(t, step):
        """Reads the next frame (skipping step-1 frames), crops and resizes."""
        if t["pos"] >= t["n"]:
            return
        for _ in range(step - 1):
            t["cap"].grab()
        ok, fr = t["cap"].read()
        t["pos"] = min(t["pos"] + step, t["n"]) if ok else t["n"]
        if not ok:
            return
        h, w = fr.shape[:2]
        xo = max((w - h) // 2, 0) if crop else 0
        if crop:
            fr = fr[:, xo:xo + h]
        s = tile_w / fr.shape[1]
        fr = cv2.resize(fr, (tile_w, int(round(fr.shape[0] * s))))
        t["im"].set_data(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
        if t.get("s") is None:
            t["im"].set_extent([0, fr.shape[1], fr.shape[0], 0])
            t["ax"].set_xlim(0, fr.shape[1]); t["ax"].set_ylim(fr.shape[0], 0)
        t["xo"], t["s"] = xo, s

    def _draw(t):
        a, ia = t["aid"], info.get(t["aid"])
        i = max(t["pos"] - 1, 0)
        end = t["pos"] >= t["n"]
        j = min(i, ia["n"] - 1) if ia else 0
        lab = ia["lab"][j] if ia and ia["lab"] is not None else None
        txt = f"ID {a}  #{t['rank']}"
        if lab is not None:
            txt += f"  |  {pfx()}{int(lab)}"
        txt += f"  |  {i + 1}/{t['n']} f" + ("  END" if end else "")
        t["txt"].set_text(txt)
        t["txt"].set_color("#7fbf7f" if end else gcol[st["group"]])
        vis = st["show_pos"] and ia is not None and t.get("s") is not None
        if vis and ia["cx"] is not None and np.isfinite(ia["cx"][j]):
            t["dc"].set_offsets([[(ia["cx"][j] - t["xo"]) * t["s"], ia["cy"][j] * t["s"]]])
            t["dc"].set_facecolor(scol(lab) if lab is not None else "white")
            t["dc"].set_visible(True)
        else:
            t["dc"].set_visible(False)
        if vis and ia["px"] is not None and np.isfinite(ia["px"][j]):
            t["dp"].set_offsets([[(ia["px"][j] - t["xo"]) * t["s"], ia["py"][j] * t["s"]]])
            t["dp"].set_visible(True)
        else:
            t["dp"].set_visible(False)
        # trails (cuttlefish coloured by state, shrimp dashed) up to frame j
        if st["trail"] and ia is not None and t.get("s") is not None:
            if t.get("segs") is None:
                _trail_segments(t)
            sc = t["segs"]["c"]
            if sc is not None and j > 0:
                m = sc[1][:j]
                t["tc"].set_segments(sc[0][:j][m])
                if t.get("scols") is not None:
                    t["tc"].set_color(list(t["scols"][:j][m]))
            else:
                t["tc"].set_segments([])
            sp = t["segs"]["p"]
            t["tp"].set_segments(sp[0][:j][sp[1][:j]] if sp is not None and j > 0 else [])
        else:
            t["tc"].set_segments([]); t["tp"].set_segments([])
        # cursors under the video
        t["cur_seq"].set_xdata([j, j])
        if t.get("cur_d") is not None:
            t["cur_d"].set_xdata([j, j])
        if t.get("cur_f") is not None:
            t["cur_f"].set_xdata([j, j])
        if t.get("radar") is not None and ia is not None and ia.get("feat") is not None:
            angc, poly, proto = t["radar"]
            r = ia["feat"][j]
            poly.set_xy(np.column_stack([angc, np.r_[r, r[:1]]]))
            c_ = scol(lab) if lab is not None else "white"
            poly.set_facecolor(c_); poly.set_edgecolor(c_)
            p = _proto(st["col"], lab) if lab is not None else None
            proto.set_data(angc, np.r_[p, p[:1]] if p is not None else np.zeros(len(angc)))

    def _restart(t):
        t["pos"] = 0
        t["cap"].set(cv2.CAP_PROP_POS_FRAMES, t["f0"])
        _read(t, 1)
        _draw(t)

    def _build_page():
        st["bg"] = None
        _clear_tiles()
        mem, n = _members(), st["n_show"]
        n_pages = max(1, -(-len(mem) // n))
        st["page"] %= n_pages
        pick = mem[st["page"] * n:(st["page"] + 1) * n]
        r, c = _SB_LAYOUT[n]
        x0, y0, W, H = ZONE
        for k in range(r * c):
            i, j = divmod(k, c)
            cx, cy, cw, ch = (x0 + j * W / c + 0.004, y0 + (r - 1 - i) * H / r + 0.004,
                              W / c - 0.008, H / r - 0.035)
            if k >= len(pick):
                continue
            # video (top) | sequence strip | distance curve (bottom)
            ax  = fig.add_axes([cx, cy + 0.37 * ch, cw, 0.63 * ch])
            axs = fig.add_axes([cx, cy + 0.305 * ch, cw, 0.05 * ch])
            axd = fig.add_axes([cx, cy + 0.19 * ch, cw, 0.105 * ch])
            axf = fig.add_axes([cx, cy, cw, 0.18 * ch])
            axr = fig.add_axes([cx + 0.79 * cw, cy + 0.40 * ch, 0.21 * cw, 0.30 * ch], projection="polar")
            ax.set_axis_off()
            for a_ in (axs, axd, axf):
                a_.set_facecolor("#0d0d1a")
                a_.set_xticks([]); a_.set_yticks([])
                for sp in a_.spines.values():
                    sp.set_edgecolor("#333333")
            a  = pick[k]
            ia = info.get(a)
            cap = cv2.VideoCapture(str(vids[int(a)]))
            f0  = ia["f0"] if ia else 0
            n_v = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) - f0
            tc = LineCollection([], linewidths=1.6, capstyle="round", alpha=0.95, zorder=2)
            tp = LineCollection([], linewidths=1.2, colors="black", linestyles=(0, (4, 2)), zorder=2)
            ax.add_collection(tc); ax.add_collection(tp)
            t = {"axes": [ax, axs, axd, axf, axr], "ax": ax, "ax_seq": axs, "aid": a, "cap": cap, "f0": f0,
                 "pos": 0, "s": None, "xo": 0, "segs": None, "scols": None,
                 "n": max(1, min(ia["n"], n_v) if ia else n_v), "rank": mem.index(a) + 1,
                 "im": ax.imshow(np.zeros((10, 10, 3), np.uint8), interpolation="nearest"),
                 "dc": ax.scatter([], [], s=55, c="white", edgecolor="black", lw=1.2, zorder=4),
                 "dp": ax.scatter([], [], s=30, marker="D", c="black", edgecolor="white", lw=1, zorder=4),
                 "tc": tc, "tp": tp, "txt": ax.set_title("", fontsize=9, loc="left"),
                 "band": axs.imshow(np.zeros((1, 2, 3)), aspect="auto", interpolation="nearest"),
                 "cur_seq": axs.axvline(0, color="red", lw=1.3), "cur_d": None, "cur_f": None,
                 "radar": None}
            _band(t)
            if ia and ia["dist"] is not None:
                dd = ia["dist"]
                axd.plot(np.arange(len(dd)), dd, color="#e0e0e0", lw=1)
                if np.isfinite(dd).any():
                    im_ = int(np.nanargmin(dd))
                    axd.plot([im_], [dd[im_]], ls="none", marker="*", ms=8, color="white", mec="black", mew=0.6)
                    lo, hi = np.nanmin(dd), np.nanmax(dd)
                    axd.set_ylim(lo - 0.05 * (hi - lo + 1), hi + 0.1 * (hi - lo + 1))
                axd.set_xlim(0, len(dd))
                axd.text(0.005, 0.92, f"distance ({ia['dunit']})", transform=axd.transAxes,
                         color="#888888", fontsize=6.5, va="top")
                t["cur_d"] = axd.axvline(0, color="red", lw=1.1)
            if ia and ia.get("feat") is not None:                       # parameter curves + cursor
                xs_ = np.arange(len(ia["feat"]))
                for k_ in range(len(feat_list)):
                    axf.plot(xs_, ia["feat"][:, k_], color=feat_cols[k_], lw=0.8, alpha=0.9)
                axf.set_xlim(0, len(xs_)); axf.set_ylim(-0.04, 1.04)
                t["cur_f"] = axf.axvline(0, color="red", lw=1.1)
            if len(feat_list) >= 3:                                     # live radar: frame (state colour) vs mean profile of the state (dashed)
                ang = np.linspace(0, 2 * np.pi, len(feat_list), endpoint=False)
                angc = np.r_[ang, ang[:1]]
                axr.set_facecolor((0.05, 0.05, 0.1, 0.75))
                axr.set_xticks(ang)
                axr.set_xticklabels([_SB_SHORT.get(f, f[:8]) for f in feat_list] if n <= 4 else [],
                                    fontsize=5.5, color="white")
                for l_ in axr.get_xticklabels():
                    l_.set_path_effects([mpe.withStroke(linewidth=2, foreground="black")])
                axr.set_ylim(0, 1); axr.set_yticks([0.5, 1.0]); axr.set_yticklabels([])
                axr.tick_params(pad=0)
                axr.grid(color="#555577", lw=0.4)
                axr.spines["polar"].set_color("#555577")
                t["radar"] = (angc, axr.fill(angc, np.zeros(len(angc)), alpha=0.45, lw=1.2)[0],
                              axr.plot(angc, np.zeros(len(angc)), ls="--", lw=0.8, color="white", alpha=0.8)[0])
            else:
                axr.set_visible(False)
            dyn = [t["im"], t["dc"], t["dp"], t["tc"], t["tp"], t["txt"], t["cur_seq"]]
            dyn += [x for x in (t["cur_d"], t["cur_f"]) if x is not None]
            if t["radar"] is not None:
                dyn += [t["radar"][1], t["radar"][2]]
            for x in dyn:
                x.set_animated(True)
            t["dyn"] = dyn
            st["tiles"].append(t)
            _restart(t)
        dur = [info[a]["dur"] for a in pick if a in info]
        title.set_text(f"Group {st['group']}  —  {len(mem)} individuals  —  page {st['page'] + 1}/{n_pages}"
                       f"  —  order : {st['order']}")
        title.set_color(gcol[st["group"]])
        _status(f"groups : {st['source']}  |  shown : {', '.join(pick)}"
                + (f"  |  mean duration {np.mean(dur):.1f} s" if dur else "") + "  |  click = detailed viewer")

    # ── Playback ──
    def _tick():
        if not plt.fignum_exists(fig.number):        # window closed (or destroyed by a backend switch) -> stop the timer
            timer.stop()
            return
        if not st["playing"]:
            return
        live = [t for t in st["tiles"] if t["aid"] and t["pos"] < t["n"]]
        if not live:
            if st["loop"] and any(t["aid"] for t in st["tiles"]):
                for t in st["tiles"]:
                    if t["aid"]:
                        _restart(t)
                fig.canvas.draw_idle()
            return
        for t in live:
            _read(t, st["speed"])
            _draw(t)
        _refresh()

    # ── Blitting: only the moving elements are redrawn at each step ──
    st["bg"] = None

    def _blit_dynamic():
        for t in st["tiles"]:
            for x in t.get("dyn", []):
                x.axes.draw_artist(x)

    def _on_draw(event):
        st["bg"] = fig.canvas.copy_from_bbox(fig.bbox)
        _blit_dynamic()

    def _refresh():
        if st["bg"] is None:
            fig.canvas.draw_idle()
            return
        fig.canvas.restore_region(st["bg"])
        _blit_dynamic()
        fig.canvas.blit(fig.bbox)

    fig.canvas.mpl_connect("draw_event", _on_draw)

    timer = fig.canvas.new_timer(interval=40)
    timer.add_callback(safe(_tick))

    # ── Callbacks ──
    @safe
    def on_group(label):
        st["group"], st["page"] = label.split("  (")[0], 0
        _build_page()

    @safe
    def step_group(delta):
        k = (names.index(st["group"]) + delta) % len(names)
        radio_g.set_active(k)                               # -> on_group

    @safe
    def on_n(label):
        first = st["page"] * st["n_show"]
        st["n_show"] = int(label)
        st["page"] = first // st["n_show"]
        _build_page()

    @safe
    def on_order(label):
        st["order"], st["page"] = label, 0
        _build_page()

    @safe
    def step_page(delta):
        st["page"] += delta
        _build_page()

    @safe
    def on_play(event):
        st["playing"] = not st["playing"]
        b_play.label.set_text("Pause" if st["playing"] else "▶ Play")
        fig.canvas.draw_idle()

    @safe
    def on_again(event):
        for t in st["tiles"]:
            if t["aid"]:
                _restart(t)
        st["playing"] = True
        b_play.label.set_text("Pause")
        fig.canvas.draw_idle()

    @safe
    def on_speed(val):
        st["speed"] = int(val)

    @safe
    def on_check(label):
        st["show_pos"], st["trail"], st["seqvec"], st["loop"] = check.get_status()
        for t in st["tiles"]:
            if t["aid"]:
                _band(t)
                _draw(t)
        fig.canvas.draw_idle()

    @safe
    def on_col(label):
        st["col"] = label
        _use_col(label)
        for t in st["tiles"]:
            t["scols"] = None                       # trail colours to recompute
            if t.get("segs") is not None and info[t["aid"]]["lab"] is not None:
                t["scols"] = np.array([scol(l) for l in info[t["aid"]]["lab"][:-1]], dtype=object)
            _band(t)
            _draw(t)
        fig.canvas.draw_idle()

    @safe
    def on_click(event):
        t = next((t for t in st["tiles"] if t["ax"] is event.inaxes and t["aid"]), None)
        if t is None:
            return
        if "open_cuttlefish_viewer" not in G:
            _status("Detailed viewer missing: run the viewer cell first", ok=False)
            return
        st["playing"] = False
        b_play.label.set_text("▶ Play")
        st["detail"] = G["open_cuttlefish_viewer"](df, t["aid"], video_dir, cluster_col=st["col"])

    def on_close(event):
        timer.stop()
        _clear_tiles()

    radio_g.on_clicked(on_group)
    b_gprev.on_clicked(lambda e: step_group(-1))
    b_gnext.on_clicked(lambda e: step_group(+1))
    radio_n.on_clicked(on_n)
    radio_o.on_clicked(on_order)
    b_prev.on_clicked(lambda e: step_page(-1))
    b_next.on_clicked(lambda e: step_page(+1))
    b_play.on_clicked(on_play)
    b_again.on_clicked(on_again)
    s_speed.on_changed(on_speed)
    check.on_clicked(on_check)
    radio_c.on_clicked(on_col)
    fig.canvas.mpl_connect("button_press_event", on_click)
    fig.canvas.mpl_connect("close_event", on_close)

    st["widgets"] = dict(fig=fig, timer=timer, radio_g=radio_g, radio_n=radio_n, radio_o=radio_o,
                         check=check, radio_c=radio_c, b_gprev=b_gprev, b_gnext=b_gnext, b_prev=b_prev, b_next=b_next,
                         b_play=b_play, b_again=b_again, s_speed=s_speed)
    st["api"] = dict(refresh=_refresh, tick=_tick, build=_build_page, step_page=step_page, step_group=step_group)

    print(f"[BROWSER] groups : {src} | {len(names)} groups : "
          + ", ".join(f"{g} ({len(by_group[g])})" for g in names))
    _build_page()
    timer.start()
    plt.show()
    return st


print("[OK] Browser defined — usage :  sb = open_strategy_browser(n_show=4)")


In [ ]:
# Video browser by strategy (groups from mapping_optimal otherwise final_state)
%matplotlib qt

sb = open_strategy_browser(
    n_show = 4,           # videos shown at the same time: 1, 2, 4, 6 or 9
    speed  = 2,           # 1 = real time, 2 = twice as fast, ...
    order  = "typical",   # "typical" (closest to the group centre first) | "random" | "id"
    crop   = True,        # True = crop the arena (central square)
    cluster_col = "cluster",   # displayed states: "cluster" | "supercluster" (also a button in the window)
)


In [ ]:
# ══════════════════════════════════════════════════════════════
#  SESSION SAVE — into a NAMED folder
#  Everything needed to resume or compare: configuration, clusters,
#  UMAP, sequences, SeqVec distance, groups (explorer + regrouping),
#  group explanation, figures and a copy of the code. Folder: OUT_DIR/session_save/<SAVE_NAME>
# ══════════════════════════════════════════════════════════════
DO_SAVE   = False            # ⚠ SAFETY: set True then rerun THIS cell to save
SAVE_NAME = "reference"          # folder name, e.g. "before_duration_tests"
SAVE_NOTE = ""                   # free note (what this session contains)
OVERWRITE = False                # True = replaces an existing folder with the same name

if not DO_SAVE:
    print("[SAVE] disabled (DO_SAVE = False): nothing was written. "
          "To save: DO_SAVE = True, then rerun this cell.")
else:

    import io, json, pickle, shutil, contextlib, traceback
    from datetime import datetime
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    SAVE_DIR = RESULTS_DIR / SAVE_NAME
    if SAVE_DIR.exists() and any(SAVE_DIR.iterdir()) and not OVERWRITE:
        raise FileExistsError(f"This folder already exists (change SAVE_NAME or set OVERWRITE = True): {SAVE_DIR}")
    if SAVE_DIR.exists() and OVERWRITE:
        shutil.rmtree(SAVE_DIR)
    FIG_DIR = SAVE_DIR / "figures"
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    print(f"[SAVE] → {SAVE_DIR}\n")
    _report = []
    # figures created WITHOUT a window (no backend switch, no Qt popup)
    _was_interactive = plt.isinteractive()
    plt.ioff()


    def _g(name):
        return globals().get(name)


    def _save(label, fn):
        try:
            out = fn()
            if out is False:
                _report.append(("SKIP", label)); print(f"  [SKIP] {label}")
            else:
                _report.append(("OK", label)); print(f"  [OK]   {label}")
        except Exception as e:
            _report.append(("ERR", label)); print(f"  [ERR]  {label}  ({type(e).__name__}: {e})")


    def _jsonable(o):
        if isinstance(o, dict):
            return {str(k): _jsonable(v) for k, v in o.items()}
        if isinstance(o, (list, tuple, set)):
            return [_jsonable(v) for v in o]
        if isinstance(o, (np.integer,)):
            return int(o)
        if isinstance(o, (np.floating,)):
            return float(o)
        if isinstance(o, Path):
            return str(o)
        return o


    # ── 1. Configuration (cell 0 + explorer settings) ──
    def _cfg():
        fs = _g("final_state") or {}
        cfg = {
            "name": SAVE_NAME, "note": SAVE_NOTE, "date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "DATASET": _g("DATASET"), "SMOOTH_FILES": _g("SMOOTH_FILES"), "OUT_DIR": OUT_DIR,
            "SELECTED_WINDOW": _g("SELECTED_WINDOW"), "FEATURES": _g("FEATURES"),
            "FEATURE_WEIGHTS": _g("FEATURE_WEIGHTS"), "EXCLUDE_ANIMALS": _g("EXCLUDE_ANIMALS"),
            "ORBIT_MIN_DEG_S": _g("ORBIT_MIN_DEG_S"), "FRAME_STEP": _g("FRAME_STEP"),
            "LEIDEN_K": _g("LEIDEN_K"), "LEIDEN_RES": _g("LEIDEN_RES"),
            "LEIDEN_K2": _g("LEIDEN_K2"), "LEIDEN_RES2": _g("LEIDEN_RES2"),
            "explorer": {k: fs.get(k) for k in ("col", "n_s", "n_cl", "seqvec_weights", "knn_k",
                                                 "min_run_len", "desc")},
            "n_groups_explorer": len({g for g in fs.get("mapping_letters", {}).values() if g != "MONO"}),
            "n_groups_optimal": (len({g for g in _g("mapping_optimal").values() if g != "MONO"})
                                 if _g("mapping_optimal") else None),
        }
        json.dump(_jsonable(cfg), open(SAVE_DIR / "config.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
    _save("config.json (settings)", _cfg)

    # ── 2. Frame clusters + UMAP ──
    _save("results.pkl (clusters, superclusters, UMAP, X, per window)",
          lambda: pickle.dump(_g("results"), open(SAVE_DIR / "results.pkl", "wb")) if _g("results") else False)
    _save("df.pkl (current df)",
          lambda: _g("df").to_pickle(SAVE_DIR / "df.pkl") if _g("df") is not None else False)

    # ── 3. Sequences + SeqVec distance ──
    def _seqs():
        fs = _g("final_state")
        if not fs:
            return False
        pickle.dump(fs, open(SAVE_DIR / "final_state.pkl", "wb"))
        json.dump({a: {"seq": [int(c) for c in v["seq"]], "lengths": [int(x) for x in v["lengths"]],
                       "total": int(v.get("total", sum(v["lengths"])))} for a, v in fs["seqs"].items()},
                  open(SAVE_DIR / "sequences.json", "w"), indent=1)
        np.save(SAVE_DIR / "D_seqvec.npy", np.asarray(fs["D"]))
        json.dump([str(a) for a in fs["multi"]], open(SAVE_DIR / "D_seqvec_ids.json", "w"))
    _save("final_state.pkl + sequences.json + D_seqvec.npy (sequences, distances, SeqVec weights)", _seqs)

    # ── 4. Groups ──
    _save("mapping_letters.json (explorer groups)",
          lambda: json.dump(_jsonable(_g("final_state")["mapping_letters"]),
                            open(SAVE_DIR / "mapping_letters.json", "w"), indent=1) if _g("final_state") else False)
    def _opt():
        if not _g("mapping_optimal"):
            return False
        json.dump(_jsonable(_g("mapping_optimal")), open(SAVE_DIR / "mapping_optimal.json", "w"), indent=1)
        json.dump(_jsonable(_g("mapping_optimal")), open(OUT_DIR / "mapping_optimal_cuttle.json", "w"), indent=2)
    _save("mapping_optimal.json (regrouped groups)", _opt)
    for _nm in ("final_state_optimal", "history", "profiles"):
        _save(f"{_nm}.pkl", lambda _nm=_nm: pickle.dump(_g(_nm), open(SAVE_DIR / f"{_nm}.pkl", "wb"))
              if _g(_nm) is not None else False)
    for _nm, _fn in (("df_pairs", "df_pairs_power.csv"), ("df_merged", "df_merged_comparative.csv")):
        _save(_fn, lambda _nm=_nm, _fn=_fn: _g(_nm).to_csv(SAVE_DIR / _fn, index=False)
              if _g(_nm) is not None else False)

    # ── 5. Group explanation tables (+ figures) ──
    _expl_out = io.StringIO()
    def _figsaver(prefix):
        def _show(*a, **k):
            for num in plt.get_fignums():
                f = plt.figure(num)
                f.savefig(FIG_DIR / f"{prefix}_{num}.png", dpi=110, bbox_inches="tight")
                plt.close(f)
        return _show

    def _explain():
        if "explain_groups" not in globals() or not _g("final_state"):
            return False
        _show0 = plt.show
        try:
            plt.show = _figsaver("explication_groupes")
            with contextlib.redirect_stdout(_expl_out):
                ex = explain_groups(figure=True)
        finally:
            plt.show = _show0
        pickle.dump(ex, open(SAVE_DIR / "explication_groupes.pkl", "wb"))
        (SAVE_DIR / "explication_groupes.txt").write_text(_expl_out.getvalue(), encoding="utf-8")
        ex["medians"].to_csv(SAVE_DIR / "groupes_medianes.csv")
        ex["zscores"].to_csv(SAVE_DIR / "groupes_scores_z.csv")
        ex["families"].to_csv(SAVE_DIR / "groupes_familles.csv")
        tab = ex["individuals"].copy()
        tab["explorer group"] = [_g("final_state")["mapping_letters"].get(a, "") for a in tab.index]
        if _g("mapping_optimal"):
            tab["regrouped group"] = [_g("mapping_optimal").get(a, "") for a in tab.index]
        tab.to_csv(SAVE_DIR / "groupes_individus.csv")
    _save("group explanation (tables + figures)", _explain)

    # ── 6. Figures ──
    def _umap_fig():
        res, w = _g("results"), _g("SELECTED_WINDOW")
        if not res:
            return False
        w = w if w in res else list(res)[0]
        d, Zu = res[w]["df"], res[w]["Z"]
        fig, axes = plt.subplots(1, 2, figsize=(16, 7))
        for ax, col in zip(axes, ("cluster", "supercluster")):
            if col in d:
                labs = d[col].to_numpy()
                for k in np.unique(labs):
                    m = labs == k
                    ax.scatter(Zu[m, 0], Zu[m, 1], s=1, alpha=.4, color=plt.cm.tab20(int(k) % 20), label=str(k))
                ax.legend(markerscale=8, fontsize=6, ncol=2, title=col)
                ax.set_title(f"UMAP — {col} (window {w})")
        fig.savefig(FIG_DIR / "umap_clusters.png", dpi=110, bbox_inches="tight")
        plt.close(fig)
    _save("figures/umap_clusters.png", _umap_fig)

    def _mds_fig():
        if "visualize_strategy_groups" not in globals() or not _g("final_state"):
            return False
        _show0 = plt.show
        try:
            plt.show = _figsaver("groupes_mds")
            with contextlib.redirect_stdout(io.StringIO()):
                visualize_strategy_groups(_g("final_state"), method="mds")
        finally:
            plt.show = _show0
    _save("figures/groupes_mds_*.png", _mds_fig)

    # ── 7. Copy of the code (notebook without outputs + add_features_full.py) ──
    def _code():
        copied = []
        for nm in ("all_cuttlefish_english.ipynb", "add_features_full.py"):
            src = Path.cwd() / nm
            if not src.exists():
                continue
            if nm.endswith(".ipynb"):
                nb = json.load(open(src, encoding="utf-8"))
                for c in nb["cells"]:
                    if c["cell_type"] == "code":
                        c["outputs"], c["execution_count"] = [], None
                json.dump(nb, open(SAVE_DIR / f"code_{nm}", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
            else:
                shutil.copy(src, SAVE_DIR / f"code_{nm}")
            copied.append(nm)
        return copied or False
    _save("code (notebook without outputs + add_features)", _code)

    # ── Summary ──
    if _was_interactive:
        plt.ion()
    files = sorted(p for p in SAVE_DIR.rglob("*") if p.is_file())
    print(f"\n{'=' * 60}\n  {SAVE_NAME} : {len(files)} files ({sum(p.stat().st_size for p in files) / 1e6:.0f} Mo)")
    for p in files:
        print(f"    {str(p.relative_to(SAVE_DIR)):<45} {p.stat().st_size / 1024:>9.0f} Ko")
    bad = [l for s, l in _report if s == "ERR"]
    if bad:
        print(f"\n  ⚠️  items in error : {bad}")
    print("=" * 60)


In [ ]:
# ══════════════════════════════════════════════════════════════
#  RELOADING A SAVED SESSION
#  Run after cell 0 (and the cells defining the functions you need).
#  Do NOT rerun cell 2 afterwards: it would replace the reloaded clusters with the current cache.
# ══════════════════════════════════════════════════════════════
DO_LOAD        = False         # ⚠ SAFETY: set True then rerun THIS cell to reload
LOAD_NAME      = None            # folder name to reload; None = the most recent session
RESTORE_CONFIG = True            # True = also restores FEATURES, weights, Leiden, exclusions, orbit threshold of the session

if not DO_LOAD:
    print("[LOAD] disabled (DO_LOAD = False): memory was not changed. "
          "To reload: DO_LOAD = True, then rerun this cell.")
else:

    import json, pickle
    from pathlib import Path
    import numpy as np
    import pandas as pd

    SESSION_ROOT = RESULTS_DIR
    sessions = sorted([d for d in SESSION_ROOT.iterdir() if d.is_dir() and (d / "config.json").exists()], key=lambda d: d.stat().st_mtime)
    if not sessions:
        raise FileNotFoundError(f"No session in {SESSION_ROOT}")
    print("Available sessions :\n")
    for s in sessions:
        c = json.load(open(s / "config.json", encoding="utf-8")) if (s / "config.json").exists() else {}
        print(f"  • {s.name:<28} {c.get('date', '')}  groups explorer={c.get('n_groups_explorer', '?')}"
              f" / optimal={c.get('n_groups_optimal', '?')}  {c.get('note', '')}")
    LOAD_DIR = SESSION_ROOT / LOAD_NAME if LOAD_NAME else sessions[-1]
    if not LOAD_DIR.exists():
        raise FileNotFoundError(f"Session not found : {LOAD_DIR}")
    print(f"\n[LOAD] → {LOAD_DIR.name}\n")
    loaded, skipped = [], []


    def _load(label, fn):
        try:
            out = fn()
            if out is False:
                skipped.append(label); print(f"  [SKIP] {label}")
            else:
                loaded.append(label); print(f"  [OK]   {label}")
        except FileNotFoundError:
            skipped.append(label); print(f"  [SKIP] {label}  (file missing)")
        except Exception as e:
            skipped.append(label); print(f"  [ERR]  {label}  ({type(e).__name__}: {e})")


    # ── Configuration ──
    cfg = json.load(open(LOAD_DIR / "config.json", encoding="utf-8")) if (LOAD_DIR / "config.json").exists() else {}
    if cfg.get("DATASET") and cfg["DATASET"] != globals().get("DATASET"):
        print(f"  ⚠️  different DATASET : session={cfg['DATASET']} | current={globals().get('DATASET')}")
    def _load_cfg():
        if not RESTORE_CONFIG or not cfg:
            return False
        for k in ("SELECTED_WINDOW", "FEATURES", "FEATURE_WEIGHTS", "EXCLUDE_ANIMALS", "ORBIT_MIN_DEG_S",
                  "LEIDEN_K", "LEIDEN_RES", "LEIDEN_K2", "LEIDEN_RES2", "FRAME_STEP"):
            if cfg.get(k) is not None:
                v = cfg[k]
                if k == "SELECTED_WINDOW":
                    v = int(v)
                globals()[k] = v
    _load("config.json (features, weights, Leiden, exclusions, orbit threshold)", _load_cfg)

    # ── Frame clusters + UMAP ──
    def _load_results():
        global results, X, Z, n_cl, n_su, df
        if (LOAD_DIR / "results.pkl").exists():
            results = pickle.load(open(LOAD_DIR / "results.pkl", "rb"))
        else:                                            # old save format
            meta = json.load(open(LOAD_DIR / "results_meta.json"))
            results = {int(w): {"df": pd.read_csv(LOAD_DIR / f"df_window_{w}.csv"),
                                "X": np.load(LOAD_DIR / f"X_window_{w}.npy"),
                                "Z": np.load(LOAD_DIR / f"Z_umap_{w}.npy"),
                                "n_clusters": m["n_cl"], "n_superclusters": m["n_su"]} for w, m in meta.items()}
        w = globals().get("SELECTED_WINDOW", list(results)[0])
        w = w if w in results else list(results)[0]
        X, Z = results[w]["X"], results[w]["Z"]
        n_cl, n_su = results[w]["n_clusters"], results[w]["n_superclusters"]
        df = results[w]["df"]
    _load("results (clusters, superclusters, UMAP)", _load_results)
    def _load_df():
        global df
        if (LOAD_DIR / "df.pkl").exists():
            df = pd.read_pickle(LOAD_DIR / "df.pkl")
        elif (LOAD_DIR / "df_main.csv").exists():
            df = pd.read_csv(LOAD_DIR / "df_main.csv")
        else:
            return False
    _load("df (current df)", _load_df)

    # ── Sequences, distances and groups ──
    def _load_pickle(name, required=False):
        def _f():
            p = LOAD_DIR / f"{name}.pkl"
            if not p.exists():
                globals().pop(name, None)            # missing from the session -> removed from memory (no mixing of sessions)
                return False
            globals()[name] = pickle.load(open(p, "rb"))
        return _f
    _load("final_state (sequences, SeqVec distance, explorer groups)", _load_pickle("final_state"))
    def _load_mapping():
        p = LOAD_DIR / "mapping_optimal.json"
        if not p.exists():
            globals().pop("mapping_optimal", None)   # missing from the session -> removed from memory (no mixing of sessions)
            return False
        globals()["mapping_optimal"] = json.load(open(p))
    _load("mapping_optimal (regrouped groups)", _load_mapping)
    for _nm in ("final_state_optimal", "history", "profiles", "explication_groupes"):
        _load(_nm, _load_pickle(_nm))
    if "explication_groupes" in globals():
        explication = explication_groupes
    def _load_d():
        global D_seqvec, D_seqvec_ids
        D_seqvec = np.load(LOAD_DIR / "D_seqvec.npy")
        D_seqvec_ids = json.load(open(LOAD_DIR / "D_seqvec_ids.json")) if (LOAD_DIR / "D_seqvec_ids.json").exists() else None
    _load("D_seqvec", _load_d)
    for _nm, _fn in (("df_pairs", "df_pairs_power.csv"), ("df_merged", "df_merged_comparative.csv")):
        _load(_nm, lambda _nm=_nm, _fn=_fn: globals().__setitem__(_nm, pd.read_csv(LOAD_DIR / _fn)))
    if "final_state" in globals():
        mapping_letters = final_state.get("mapping_letters", {})

    # ── Summary ──
    print(f"\n{'=' * 60}\nSESSION RELOADED — {LOAD_DIR.name}  ({len(loaded)} loaded | {len(skipped)} missing)")
    checks = {
        "results / df":     lambda: f"{len(results)} window(s) | df {df.shape}",
        "final_state":      lambda: f"{final_state['n_cl']} states | {final_state['n_s']} groups | {final_state.get('desc', '')}",
        "mapping_optimal":  lambda: f"groups = {sorted(set(mapping_optimal.values()))}",
        "SeqVec":           lambda: f"{final_state.get('seqvec_weights')} | knn_k={final_state.get('knn_k')}",
        "FEATURE_WEIGHTS":  lambda: f"{FEATURE_WEIGHTS}",
        "LEIDEN":           lambda: f"K={LEIDEN_K} res={LEIDEN_RES} | orbit_min={globals().get('ORBIT_MIN_DEG_S')}",
    }
    for nm, fn in checks.items():
        try:
            print(f"    ✅  {nm:<18} {fn()}")
        except Exception:
            print(f"    —   {nm:<18} not defined")
    print(f"\n  Then: run the analysis / video / explain_groups() cells directly.\n  To recompute from these settings: rerun cell 2 then the explorer.\n{'=' * 60}")
